# SERF Functional Evaluation on HAI 21.03

**Companion to:** *A Segmentation-based Episodic Representation Framework for Cross-Design Comparison in Multi-Mode Industrial Time Series*

**Authors:** Ján Skalka, Małgorzata Przybyła-Kasperek, Ľubomír Benko, Martin Drlík, Dominik Halvoník, Kacper Książek

This notebook accompanies the article and reproduces the HAI 21.03 functional evaluation reported in Section 5 and the corresponding Supplementary Material. Its organization follows the manuscript workflow and keeps the normal-development and held-out evaluation roles explicit throughout.

**Input data**

Place the following CSV files directly in the notebook working directory before execution:

- `train1.csv` — the only normal-development run. All data-dependent preprocessing parameters, segmentation models, SERF mappings, and normal response references are estimated from this file.
- `test1.csv` to `test5.csv` — held-out evaluation runs. They are processed independently using the fixed train-derived pipeline.

Attack annotations contained in the test files are first accessed in **Section 5.7 — External functional validation**. All preceding construction and scoring steps are label-blind.

**Output convention**

All generated artifacts are written to relative paths under `outputs/`, with a separate subdirectory for each analysis step. The notebook contains no machine-specific output paths.

**Main dependencies:** `numpy`, `pandas`, `scikit-learn`, `scipy`, `hdbscan`, `hmmlearn`, `joblib`, and `IPython`.

Run the notebook from top to bottom in a clean kernel. The screening stages are computationally intensive; the embedded outputs correspond to the analysis used for the manuscript.


## Preparation — Train-only context audit

The preparation steps characterize the four candidate operating-context variables and their temporal structure using `train1.csv` only. They establish the empirical basis for the context representation and time scales used later, without fitting a segmentation model or using attack annotations.


### P.1 Context-variable audit


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# ============================================================
# HAI 21.03 — TRAIN1 CONTEXT VARIABLE AUDIT
#
# Purpose:
#   Descriptive train-only audit of the four set-point variables
#   used to construct the operating-context representation.
#
# This cell does not define or fit scaling parameters. The final
# train-derived level and delta scaling is constructed later.
#
# Input:
#   train1.csv
#
# Output:
#   outputs/preparation/context_variable_audit/
#       variable_summary.csv
#       frequent_values.csv
#       raw_scale_diagnostic.csv
#       spearman_correlation.csv
#       temporal_change_frequency.csv
# ============================================================

INPUT_FILE = Path("train1.csv")
OUTPUT_DIR = Path("outputs/preparation/context_variable_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTEXT_FEATURES = [
    "P1_B2004",
    "P1_B3004",
    "P1_B3005",
    "P1_B4002",
]

TOP_N_VALUES = 10


# ============================================================
# 1. LOAD AND VALIDATE
# ============================================================

train1 = pd.read_csv(INPUT_FILE)

missing = [c for c in CONTEXT_FEATURES if c not in train1.columns]
if missing:
    raise ValueError(f"Missing context variables: {missing}")

X = (
    train1[CONTEXT_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

if X.isna().any().any():
    bad = X.columns[X.isna().any()].tolist()
    counts = X[bad].isna().sum().to_dict()
    raise ValueError(
        "Missing, non-numeric, or infinite values found in context variables: "
        f"{counts}"
    )

print(f"train1 rows: {len(train1):,}")
print(f"context variables: {len(CONTEXT_FEATURES)}")


# ============================================================
# 2. VARIABLE SUMMARY
# ============================================================

summary_rows = []

for c in CONTEXT_FEATURES:
    s = X[c]
    vc = s.value_counts(dropna=False)

    summary_rows.append({
        "variable": c,
        "n_valid": int(s.notna().sum()),
        "n_missing": int(s.isna().sum()),
        "n_unique": int(s.nunique(dropna=True)),
        "min": float(s.min()),
        "q25": float(s.quantile(0.25)),
        "median": float(s.median()),
        "q75": float(s.quantile(0.75)),
        "max": float(s.max()),
        "range": float(s.max() - s.min()),
        "std": float(s.std(ddof=0)),
        "dominant_value": float(vc.index[0]),
        "dominant_share": float(vc.iloc[0] / len(s)),
    })

variable_summary = pd.DataFrame(summary_rows)
variable_summary.to_csv(OUTPUT_DIR / "variable_summary.csv", index=False)

print("\nVARIABLE SUMMARY")
display(variable_summary.round(6))


# ============================================================
# 3. MOST FREQUENT VALUES
# ============================================================

frequent_rows = []

for c in CONTEXT_FEATURES:
    counts = X[c].value_counts().head(TOP_N_VALUES)

    for rank, (value, count) in enumerate(counts.items(), start=1):
        frequent_rows.append({
            "variable": c,
            "rank": rank,
            "value": float(value),
            "count": int(count),
            "share": float(count / len(X)),
        })

frequent_values = pd.DataFrame(frequent_rows)
frequent_values.to_csv(OUTPUT_DIR / "frequent_values.csv", index=False)

print(f"\nTOP {TOP_N_VALUES} VALUES PER CONTEXT VARIABLE")
for c in CONTEXT_FEATURES:
    display(
        frequent_values.loc[
            frequent_values["variable"].eq(c),
            ["rank", "value", "count", "share"],
        ].round(6)
    )


# ============================================================
# 4. RAW-SCALE DIAGNOSTIC
#
# Descriptive only: quantifies how strongly the raw numerical
# ranges/variances differ before the train-derived level scaling.
# ============================================================

variance = X.var(ddof=0)
variance_total = float(variance.sum())

raw_scale_diagnostic = pd.DataFrame({
    "variable": CONTEXT_FEATURES,
    "range": (X.max() - X.min()).reindex(CONTEXT_FEATURES).to_numpy(),
    "std": X.std(ddof=0).reindex(CONTEXT_FEATURES).to_numpy(),
    "variance": variance.reindex(CONTEXT_FEATURES).to_numpy(),
    "variance_share": (
        variance.reindex(CONTEXT_FEATURES).to_numpy() / variance_total
        if variance_total > 0
        else np.nan
    ),
})

raw_scale_diagnostic = raw_scale_diagnostic.sort_values(
    "variance_share", ascending=False
).reset_index(drop=True)

raw_scale_diagnostic.to_csv(
    OUTPUT_DIR / "raw_scale_diagnostic.csv", index=False
)

print("\nRAW-SCALE DIAGNOSTIC")
display(raw_scale_diagnostic.round(6))


# ============================================================
# 5. SPEARMAN CORRELATION
# ============================================================

spearman_corr = X.corr(method="spearman")
spearman_corr.to_csv(OUTPUT_DIR / "spearman_correlation.csv")

print("\nSPEARMAN CORRELATION")
display(spearman_corr.round(3))


# ============================================================
# 6. TEMPORAL CHANGE FREQUENCY
#
# Counts signed level changes between consecutive valid 1-Hz
# observations. Detailed transition/plateau structure is audited
# separately in the subsequent temporal-context audit.
# ============================================================

change_rows = []

for c in CONTEXT_FEATURES:
    s = X[c]
    valid_pair = s.notna() & s.shift(1).notna()
    changed = s.ne(s.shift(1)) & valid_pair

    n_pairs = int(valid_pair.iloc[1:].sum())
    n_changes = int(changed.iloc[1:].sum())

    change_rows.append({
        "variable": c,
        "n_valid_consecutive_pairs": n_pairs,
        "n_changes": n_changes,
        "change_share": float(n_changes / n_pairs) if n_pairs else np.nan,
    })

temporal_change_frequency = pd.DataFrame(change_rows)
temporal_change_frequency.to_csv(
    OUTPUT_DIR / "temporal_change_frequency.csv", index=False
)

print("\nTEMPORAL CHANGE FREQUENCY")
display(temporal_change_frequency.round(6))


print(f"\nSaved audit outputs to: {OUTPUT_DIR}")

train1 rows: 216,001
context variables: 4

VARIABLE SUMMARY


,variable,n_valid,n_missing,n_unique,min,q25,median,q75,max,range,std,dominant_value,dominant_share
0,P1_B2004,216001,0,311,0.02989,0.09823,0.09913,0.10062,0.10135,0.07146,0.020949,0.10099,0.123699
1,P1_B3004,216001,0,364,369.75601,394.01361,400.94696,405.79410,443.27078,73.51477,15.014220,404.36404,0.123699
2,P1_B3005,216001,0,372,894.71869,1017.60492,1093.57519,1114.74206,1121.94116,227.22247,58.925749,1093.57519,0.123699
3,P1_B4002,216001,0,224,31.64864,32.00000,32.00000,32.56251,33.65550,2.00686,0.535567,32.00000,0.441919



TOP 10 VALUES PER CONTEXT VARIABLE


,rank,value,count,share
0,1,0.10099,26719,0.123699
1,2,0.09815,18362,0.085009
2,3,0.10121,16212,0.075055
3,4,0.09904,14114,0.065342
4,5,0.09938,9423,0.043625
5,6,0.10034,9421,0.043616
6,7,0.05943,9273,0.042930
7,8,0.09841,9085,0.042060
8,9,0.09933,9053,0.041912
9,10,0.06037,9052,0.041907


,rank,value,count,share
10,1,404.36404,26719,0.123699
11,2,405.79410,18360,0.085000
12,3,397.63785,16211,0.075051
13,4,399.73972,14113,0.065338
14,5,393.04804,9422,0.043620
15,6,429.62439,9420,0.043611
16,7,417.51267,9273,0.042930
17,8,406.22626,9084,0.042055
18,9,394.01361,9053,0.041912
19,10,416.51535,9052,0.041907


...


SPEARMAN CORRELATION


,P1_B2004,P1_B3004,P1_B3005,P1_B4002
P1_B2004,1.000,-0.039,-0.240,0.388
P1_B3004,-0.039,1.000,-0.064,0.034
P1_B3005,-0.240,-0.064,1.000,-0.053
P1_B4002,0.388,0.034,-0.053,1.000



TEMPORAL CHANGE FREQUENCY


,variable,n_valid_consecutive_pairs,n_changes,change_share
0,P1_B2004,216000,354,0.001639
1,P1_B3004,216000,363,0.001681
2,P1_B3005,216000,371,0.001718
3,P1_B4002,216000,230,0.001065



Saved audit outputs to: outputs/preparation/context_variable_audit


### P.2 Temporal operating-context audit


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# HAI 21.03 — TRAIN-ONLY TEMPORAL OPERATING-CONTEXT AUDIT
#
# Purpose:
#   Characterize the temporal structure of the four raw set-point
#   context variables in train1.csv before segmentation.
#
# This cell intentionally performs:
#   - no scaling for segmentation,
#   - no delta construction,
#   - no windowing,
#   - no clustering,
#   - no attack-label analysis.
#
# It supports the Section 5.3 statements about:
#   - the number and support of exact set-point configurations,
#   - dominant stable plateaus,
#   - low-support bridge intervals between plateaus,
#   - the empirical duration scale used to motivate later windows.
# ============================================================

INPUT_FILE = Path("train1.csv")
OUTPUT_DIR = Path("outputs/preparation/temporal_context_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTEXT_FEATURES = [
    "P1_B2004",
    "P1_B3004",
    "P1_B3005",
    "P1_B4002",
]

# Working coverage threshold for the dominant-configuration core.
# Sensitivity around this value is reported below.
CORE_COVERAGE_TARGET = 0.998
COVERAGE_LEVELS = [0.990, 0.995, 0.998, 0.999]
EXPECTED_SAMPLING_SEC = 1.0


# ============================================================
# 1. LOAD AND VALIDATE
# ============================================================

df = pd.read_csv(INPUT_FILE)

missing = [c for c in CONTEXT_FEATURES if c not in df.columns]
if missing:
    raise ValueError(f"Missing context variables: {missing}")

X = (
    df[CONTEXT_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

if X.isna().any().any():
    bad = X.columns[X.isna().any()].tolist()
    raise ValueError(f"Missing/non-numeric context values found in: {bad}")

n_rows = len(X)
if n_rows < 2:
    raise ValueError("train1.csv must contain at least two observations.")


# ============================================================
# 2. STRICT TIME-AXIS AUDIT
# ============================================================

time_candidates = [
    c for c in df.columns
    if c.lower() in {"time", "timestamp", "datetime", "date_time"}
]

if not time_candidates:
    raise ValueError(
        "No supported timestamp column found. Expected one of: "
        "time, timestamp, datetime, date_time."
    )

TIME_COL = time_candidates[0]
time_values = pd.to_datetime(df[TIME_COL], errors="coerce")

if time_values.isna().any():
    raise ValueError(f"Timestamp parsing failed for column '{TIME_COL}'.")

if time_values.duplicated().any():
    raise ValueError("Duplicate timestamps found in train1.csv.")

time_diff_sec = time_values.diff().dt.total_seconds().iloc[1:]

if (time_diff_sec <= 0).any():
    raise ValueError("Timestamps are not strictly increasing.")

if not np.allclose(
    time_diff_sec.to_numpy(dtype=float),
    EXPECTED_SAMPLING_SEC,
    rtol=0.0,
    atol=1e-9,
):
    bad = time_diff_sec[~np.isclose(
        time_diff_sec,
        EXPECTED_SAMPLING_SEC,
        rtol=0.0,
        atol=1e-9,
    )]
    raise ValueError(
        "train1.csv is not a continuous 1-Hz sequence. "
        f"Found {len(bad)} non-1-s intervals; examples: "
        f"{bad.head(10).tolist()}"
    )

sampling_sec = EXPECTED_SAMPLING_SEC


# ============================================================
# 3. EXACT SET-POINT CONFIGURATIONS
# ============================================================

configs = (
    X.groupby(CONTEXT_FEATURES, sort=False, dropna=False)
    .size()
    .reset_index(name="n_ticks")
    .sort_values("n_ticks", ascending=False, kind="stable")
    .reset_index(drop=True)
)

configs["rank"] = np.arange(1, len(configs) + 1)
configs["config_id"] = [f"C{i:03d}" for i in configs["rank"]]
configs["share"] = configs["n_ticks"] / n_rows
configs["cumulative_share"] = configs["share"].cumsum()


# ============================================================
# 4. COVERAGE SENSITIVITY
# ============================================================

def n_configs_for_coverage(threshold: float) -> tuple[int, float]:
    cumulative = configs["cumulative_share"].to_numpy(dtype=float)
    n_core = int(np.searchsorted(cumulative, threshold, side="left") + 1)
    achieved = float(cumulative[n_core - 1])
    return n_core, achieved


coverage_rows = []
for threshold in COVERAGE_LEVELS:
    n_core, achieved = n_configs_for_coverage(threshold)
    coverage_rows.append({
        "target_coverage": threshold,
        "n_configurations": n_core,
        "achieved_coverage": achieved,
        "remaining_share": 1.0 - achieved,
    })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(OUTPUT_DIR / "coverage_sensitivity.csv", index=False)

working_core_n, working_core_coverage = n_configs_for_coverage(
    CORE_COVERAGE_TARGET
)
working_core_ids = set(configs.iloc[:working_core_n]["config_id"])
configs["is_core_998"] = configs["config_id"].isin(working_core_ids)
configs.to_csv(OUTPUT_DIR / "configurations.csv", index=False)


# ============================================================
# 5. MAP EVERY TRAIN TICK TO ITS EXACT CONFIGURATION
# ============================================================

row_meta = (
    X.reset_index(drop=False)
    .rename(columns={"index": "row"})
    .merge(
        configs[
            CONTEXT_FEATURES
            + ["config_id", "rank", "share", "is_core_998"]
        ],
        on=CONTEXT_FEATURES,
        how="left",
        validate="many_to_one",
    )
    .sort_values("row")
    .reset_index(drop=True)
)

if row_meta["config_id"].isna().any():
    raise RuntimeError("Exact-configuration mapping failed.")

if not np.array_equal(row_meta["row"].to_numpy(), np.arange(n_rows)):
    raise RuntimeError("Row order changed during configuration mapping.")


# ============================================================
# 6. CONTIGUOUS EXACT-CONFIGURATION PLATEAUS
# ============================================================

new_episode = row_meta["config_id"].ne(row_meta["config_id"].shift(1))
episode_id = new_episode.cumsum().astype(int)

episode_source = pd.DataFrame({
    "episode_id": episode_id,
    "row": row_meta["row"],
    "config_id": row_meta["config_id"],
    "rank": row_meta["rank"],
    "is_core_998": row_meta["is_core_998"],
    "time": time_values.to_numpy(),
})

episode_rows = []
for _, g in episode_source.groupby("episode_id", sort=False):
    n_ticks = int(len(g))
    start_row = int(g["row"].iloc[0])
    end_row = int(g["row"].iloc[-1])

    # Under the strict 1-Hz continuity check, duration equals n_ticks seconds.
    duration_sec = float(n_ticks * sampling_sec)

    episode_rows.append({
        "episode_id": int(g["episode_id"].iloc[0]),
        "config_id": g["config_id"].iloc[0],
        "rank": int(g["rank"].iloc[0]),
        "is_core_998": bool(g["is_core_998"].iloc[0]),
        "start_row": start_row,
        "end_row": end_row,
        "start_time": g["time"].iloc[0],
        "end_time": g["time"].iloc[-1],
        "n_ticks": n_ticks,
        "duration_sec": duration_sec,
    })

episodes_df = pd.DataFrame(episode_rows)
episodes_df.to_csv(OUTPUT_DIR / "plateau_episodes.csv", index=False)


# ============================================================
# 7. CONFIGURATION-LEVEL TEMPORAL SUMMARY
# ============================================================

config_episode_summary = (
    episodes_df.groupby(
        ["config_id", "rank", "is_core_998"],
        as_index=False,
    )
    .agg(
        n_episodes=("episode_id", "count"),
        total_ticks=("n_ticks", "sum"),
        min_duration_sec=("duration_sec", "min"),
        median_duration_sec=("duration_sec", "median"),
        max_duration_sec=("duration_sec", "max"),
    )
    .merge(
        configs[["config_id", "share"] + CONTEXT_FEATURES],
        on="config_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("rank")
    .reset_index(drop=True)
)

config_episode_summary.to_csv(
    OUTPUT_DIR / "configuration_episode_summary.csv",
    index=False,
)


# ============================================================
# 8. LOW-SUPPORT RUNS BETWEEN DOMINANT PLATEAUS
# ============================================================

rare_flag = ~row_meta["is_core_998"]
rare_block_id = rare_flag.ne(rare_flag.shift(1)).cumsum()

rare_run_rows = []
if rare_flag.any():
    rare_groups = row_meta.loc[rare_flag].groupby(
        rare_block_id.loc[rare_flag],
        sort=False,
    )

    for block_id, g in rare_groups:
        start_idx = int(g.index.min())
        end_idx = int(g.index.max())
        previous_idx = start_idx - 1 if start_idx > 0 else None
        next_idx = end_idx + 1 if end_idx + 1 < n_rows else None

        previous_is_core = (
            bool(row_meta.loc[previous_idx, "is_core_998"])
            if previous_idx is not None else False
        )
        next_is_core = (
            bool(row_meta.loc[next_idx, "is_core_998"])
            if next_idx is not None else False
        )

        previous_core = (
            row_meta.loc[previous_idx, "config_id"]
            if previous_idx is not None else None
        )
        next_core = (
            row_meta.loc[next_idx, "config_id"]
            if next_idx is not None else None
        )

        if previous_idx is None or next_idx is None:
            run_type = "edge"
        elif previous_is_core and next_is_core:
            run_type = (
                "blink_same_core"
                if previous_core == next_core
                else "bridge_between_core"
            )
        else:
            run_type = "other"

        n_ticks = int(end_idx - start_idx + 1)

        rare_run_rows.append({
            "rare_run_id": int(block_id),
            "start_row": start_idx,
            "end_row": end_idx,
            "start_time": time_values.iloc[start_idx],
            "end_time": time_values.iloc[end_idx],
            "n_ticks": n_ticks,
            "duration_sec": float(n_ticks * sampling_sec),
            "n_exact_rare_configurations": int(
                row_meta.loc[start_idx:end_idx, "config_id"].nunique()
            ),
            "previous_core_config": previous_core,
            "next_core_config": next_core,
            "run_type": run_type,
        })

rare_runs_df = pd.DataFrame(rare_run_rows)
rare_runs_df.to_csv(OUTPUT_DIR / "low_support_runs.csv", index=False)


# ============================================================
# 9. LOW-SUPPORT RUN DURATION SUMMARY
# ============================================================

if len(rare_runs_df):
    transition_duration_summary = pd.DataFrame([{
        "n_runs": int(len(rare_runs_df)),
        "min_duration_sec": float(rare_runs_df["duration_sec"].min()),
        "p25_duration_sec": float(rare_runs_df["duration_sec"].quantile(0.25)),
        "median_duration_sec": float(rare_runs_df["duration_sec"].median()),
        "p75_duration_sec": float(rare_runs_df["duration_sec"].quantile(0.75)),
        "max_duration_sec": float(rare_runs_df["duration_sec"].max()),
        "n_bridge_between_core": int(
            (rare_runs_df["run_type"] == "bridge_between_core").sum()
        ),
        "n_blink_same_core": int(
            (rare_runs_df["run_type"] == "blink_same_core").sum()
        ),
        "n_edge": int((rare_runs_df["run_type"] == "edge").sum()),
        "n_other": int((rare_runs_df["run_type"] == "other").sum()),
    }])
else:
    transition_duration_summary = pd.DataFrame([{
        "n_runs": 0,
        "min_duration_sec": np.nan,
        "p25_duration_sec": np.nan,
        "median_duration_sec": np.nan,
        "p75_duration_sec": np.nan,
        "max_duration_sec": np.nan,
        "n_bridge_between_core": 0,
        "n_blink_same_core": 0,
        "n_edge": 0,
        "n_other": 0,
    }])

transition_duration_summary.to_csv(
    OUTPUT_DIR / "low_support_run_summary.csv",
    index=False,
)


# ============================================================
# 10. INTERNAL / MANUSCRIPT-CONSISTENCY CHECKS
# ============================================================

core_config_summary = config_episode_summary[
    config_episode_summary["is_core_998"]
].copy()

checks = pd.DataFrame([
    {
        "check": "continuous_1hz_time_axis",
        "value": True,
        "interpretation": "Required before treating contiguous rows as temporal intervals.",
    },
    {
        "check": "core_target_reached",
        "value": bool(working_core_coverage >= CORE_COVERAGE_TARGET),
        "interpretation": "Dominant exact configurations reach the requested coverage target.",
    },
    {
        "check": "each_core_configuration_is_one_plateau",
        "value": bool((core_config_summary["n_episodes"] == 1).all()),
        "interpretation": "Supports describing dominant configurations as long contiguous plateaus.",
    },
    {
        "check": "all_low_support_runs_are_core_to_core_bridges",
        "value": bool(
            len(rare_runs_df) > 0
            and (rare_runs_df["run_type"] == "bridge_between_core").all()
        ),
        "interpretation": "Supports describing low-support combinations as bridge intervals between dominant plateaus.",
    },
])

checks.to_csv(OUTPUT_DIR / "temporal_structure_checks.csv", index=False)


# ============================================================
# 11. SUMMARY
# ============================================================

core_episodes = episodes_df[episodes_df["is_core_998"]]
rare_ticks = int(rare_flag.sum())

summary_rows = [
    ("train1_rows", n_rows),
    ("time_column", TIME_COL),
    ("sampling_interval_sec", sampling_sec),
    ("exact_configurations", int(len(configs))),
    ("core_coverage_target", CORE_COVERAGE_TARGET),
    ("core_configurations", int(working_core_n)),
    ("core_achieved_coverage", float(working_core_coverage)),
    ("exact_configuration_episodes", int(len(episodes_df))),
    ("core_plateau_episodes", int(len(core_episodes))),
    (
        "core_configurations_with_multiple_episodes",
        int((core_config_summary["n_episodes"] > 1).sum()),
    ),
    ("low_support_ticks", rare_ticks),
    ("low_support_share", float(rare_ticks / n_rows)),
    ("low_support_runs", int(len(rare_runs_df))),
    (
        "bridge_between_core_runs",
        int((rare_runs_df["run_type"] == "bridge_between_core").sum())
        if len(rare_runs_df) else 0,
    ),
    (
        "low_support_run_min_sec",
        float(rare_runs_df["duration_sec"].min())
        if len(rare_runs_df) else np.nan,
    ),
    (
        "low_support_run_median_sec",
        float(rare_runs_df["duration_sec"].median())
        if len(rare_runs_df) else np.nan,
    ),
    (
        "low_support_run_max_sec",
        float(rare_runs_df["duration_sec"].max())
        if len(rare_runs_df) else np.nan,
    ),
]

summary_df = pd.DataFrame(summary_rows, columns=["metric", "value"])
summary_df.to_csv(OUTPUT_DIR / "summary.csv", index=False)


# ============================================================
# 12. FINISH
# ============================================================

print("=" * 76)
print("TRAIN-ONLY TEMPORAL OPERATING-CONTEXT AUDIT COMPLETE")
print("=" * 76)
print(f"Output directory: {OUTPUT_DIR}")
print()
print(summary_df.to_string(index=False))
print()
print("Temporal structure checks:")
print(checks[["check", "value"]].to_string(index=False))
print()
print(
    "This cell performs only the pre-segmentation temporal audit of exact "
    "set-point configurations; it does not define operating states."
)

TRAIN-ONLY TEMPORAL OPERATING-CONTEXT AUDIT COMPLETE
Output directory: outputs/preparation/temporal_context_audit

                                    metric     value
                               train1_rows    216001
                               time_column      time
                     sampling_interval_sec       1.0
                      exact_configurations       384
                      core_coverage_target     0.998
                       core_configurations        23
                    core_achieved_coverage  0.998329
              exact_configuration_episodes       384
                     core_plateau_episodes        23
core_configurations_with_multiple_episodes         0
                         low_support_ticks       361
                         low_support_share  0.001671
                          low_support_runs        22
                  bridge_between_core_runs        22
                   low_support_run_min_sec      15.0
                low_support_run_media

## Section 5.2 — Data preprocessing

The four context variables are converted into the fixed eight-dimensional tick representation used by every segmentation family: four train-range-scaled levels and four signed one-step deltas normalized by train-derived q99 absolute non-zero change. Trailing 4 s, 8 s, and 16 s windows are then constructed with a 1 s stride for the window-based branches.


### 5.2.1 Construct the train-derived 8D context representation


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# ============================================================
# HAI 21.03 — TRAIN1 8D OPERATING-CONTEXT CONSTRUCTION
#
# Purpose:
#   Construct the final train-derived tick-level operating-context
#   representation used by all segmentation designs.
#
# Representation:
#   4 range-scaled set-point levels
#   4 q99-scaled signed one-step deltas of those scaled levels
#
# All scaling parameters are estimated from train1.csv only and
# saved for unchanged out-of-sample application to test1-test5.
#
# Input:
#   train1.csv
#
# Output directory:
#   outputs/section_5_2/train_context_8d/
#       train1_context_8d.csv
#       scaling_parameters.csv
#       delta_scale_audit.csv
#       representation_audit.csv
#       feature_order.csv
#       time_continuity_audit.csv
# ============================================================

INPUT_FILE = Path("train1.csv")
OUTPUT_DIR = Path("outputs/section_5_2/train_context_8d")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTEXT_FEATURES = [
    "P1_B2004",
    "P1_B3004",
    "P1_B3005",
    "P1_B4002",
]

DELTA_QUANTILE = 0.99
EXPECTED_SAMPLING_SEC = 1.0
EPS = 1e-12


# ============================================================
# 1. LOAD AND VALIDATE
# ============================================================

df = pd.read_csv(INPUT_FILE)

missing = [c for c in CONTEXT_FEATURES if c not in df.columns]
if missing:
    raise ValueError(f"Missing context variables: {missing}")

X = (
    df[CONTEXT_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

if X.isna().any().any():
    bad = X.columns[X.isna().any()].tolist()
    counts = X[bad].isna().sum().to_dict()
    raise ValueError(
        "Missing, non-numeric, or infinite values found in context variables: "
        f"{counts}"
    )

if len(X) < 2:
    raise ValueError("train1.csv must contain at least two observations.")


# ============================================================
# 2. TIME AXIS AND DELTA-RESET LOCATIONS
#
# Consecutive 1-Hz observations receive ordinary one-step deltas.
# The first observation and the first observation after any temporal
# discontinuity receive delta = 0 for all four context variables.
# ============================================================

time_candidates = [
    c for c in df.columns
    if c.lower() in {"time", "timestamp", "datetime", "date_time"}
]

if not time_candidates:
    raise ValueError(
        "No supported timestamp column found. Expected one of: "
        "time, timestamp, datetime, date_time."
    )

TIME_COL = time_candidates[0]
time = pd.to_datetime(df[TIME_COL], errors="coerce")

if time.isna().any():
    raise ValueError(f"Timestamp parsing failed for column '{TIME_COL}'.")

if time.duplicated().any():
    raise ValueError("Duplicate timestamps found in train1.csv.")

# Strict ordering is required; positive gaps are permitted and treated
# as sequence breaks for delta construction.
time_diff_sec = time.diff().dt.total_seconds()

if (time_diff_sec.iloc[1:] <= 0).any():
    raise ValueError("Timestamps are not strictly increasing.")

reset_mask = time_diff_sec.isna() | ~np.isclose(
    time_diff_sec,
    EXPECTED_SAMPLING_SEC,
    rtol=0.0,
    atol=1e-9,
)

n_non_1s_steps = int(reset_mask.iloc[1:].sum())


# ============================================================
# 3. TRAIN-ONLY RANGE SCALING OF OPERATING LEVELS
#
# x'_j,t = (x_j,t - min_j^train) / (max_j^train - min_j^train)
# ============================================================

train_min = X.min()
train_max = X.max()
train_range = train_max - train_min

if (train_range <= 0).any():
    bad = train_range[train_range <= 0].index.tolist()
    raise ValueError(f"Zero-range context variables: {bad}")

X_level = (X - train_min) / train_range


# ============================================================
# 4. SIGNED ONE-STEP DELTAS OF SCALED LEVELS
# ============================================================

X_delta = X_level.diff()
X_delta.loc[reset_mask, :] = 0.0

if X_delta.isna().any().any():
    raise RuntimeError("Unexpected missing values after delta construction.")


# ============================================================
# 5. TRAIN-ONLY Q99 DELTA SCALE
#
# For each context variable j:
#   s_j = Q_0.99( |delta_j| | |delta_j| > 0 )
#
# The signed delta is then divided by s_j.
# ============================================================

delta_scale = {}
delta_audit_rows = []

for c in CONTEXT_FEATURES:
    d = X_delta[c].astype(float)
    active_mask = ~np.isclose(d.to_numpy(), 0.0, atol=EPS, rtol=0.0)
    active = d.loc[active_mask]

    if active.empty:
        raise ValueError(f"No non-zero delta values found for {c}.")

    abs_active = active.abs()
    q95 = float(abs_active.quantile(0.95))
    q99 = float(abs_active.quantile(DELTA_QUANTILE))
    abs_max = float(abs_active.max())

    if not np.isfinite(q99) or q99 <= 0:
        raise ValueError(f"Invalid q99 delta scale for {c}: {q99}")

    delta_scale[c] = q99

    delta_audit_rows.append({
        "variable": c,
        "n_nonzero_deltas": int(len(active)),
        "nonzero_share": float(len(active) / len(d)),
        "signed_min": float(active.min()),
        "signed_max": float(active.max()),
        "abs_min": float(abs_active.min()),
        "abs_q25": float(abs_active.quantile(0.25)),
        "abs_median": float(abs_active.median()),
        "abs_q75": float(abs_active.quantile(0.75)),
        "abs_q90": float(abs_active.quantile(0.90)),
        "abs_q95": q95,
        "abs_q99": q99,
        "abs_max": abs_max,
        "max_to_q95_ratio": float(abs_max / q95) if q95 > 0 else np.nan,
        "max_to_q99_ratio": float(abs_max / q99),
    })

X_delta_scaled = X_delta.copy()
for c in CONTEXT_FEATURES:
    X_delta_scaled[c] = X_delta[c] / delta_scale[c]


# ============================================================
# 6. BUILD FINAL 8D TICK-LEVEL CONTEXT REPRESENTATION
# ============================================================

LEVEL_COLUMNS = [f"{c}_level" for c in CONTEXT_FEATURES]
DELTA_COLUMNS = [f"{c}_delta" for c in CONTEXT_FEATURES]
MODEL_COLUMNS = LEVEL_COLUMNS + DELTA_COLUMNS

out = pd.DataFrame({
    "row_id": np.arange(len(df), dtype=np.int64),
    "time": time,
})

for source, target in zip(CONTEXT_FEATURES, LEVEL_COLUMNS):
    out[target] = X_level[source].to_numpy(dtype=float)

for source, target in zip(CONTEXT_FEATURES, DELTA_COLUMNS):
    out[target] = X_delta_scaled[source].to_numpy(dtype=float)

if len(MODEL_COLUMNS) != 8:
    raise RuntimeError("The operating-context representation must contain 8 model dimensions.")

if out[MODEL_COLUMNS].isna().any().any():
    raise RuntimeError("Missing values found in the final 8D representation.")

if not np.isfinite(out[MODEL_COLUMNS].to_numpy(dtype=float)).all():
    raise RuntimeError("Non-finite values found in the final 8D representation.")


# ============================================================
# 7. TRAIN-DERIVED SCALING PARAMETERS
# ============================================================

scaling_rows = []
for c in CONTEXT_FEATURES:
    scaling_rows.append({
        "variable": c,
        "train_min": float(train_min[c]),
        "train_max": float(train_max[c]),
        "train_range": float(train_range[c]),
        "delta_q99_abs_nonzero": float(delta_scale[c]),
    })

scaling_parameters = pd.DataFrame(scaling_rows)


# ============================================================
# 8. FEATURE-ORDER CONTRACT
#
# This file defines the exact column order to be reused by every
# segmentation model and during later out-of-sample application.
# ============================================================

feature_order_rows = []
for position, model_col in enumerate(MODEL_COLUMNS, start=1):
    if model_col.endswith("_level"):
        role = "level"
        source_variable = model_col.removesuffix("_level")
    else:
        role = "delta"
        source_variable = model_col.removesuffix("_delta")

    feature_order_rows.append({
        "position": position,
        "model_column": model_col,
        "source_variable": source_variable,
        "role": role,
    })

feature_order = pd.DataFrame(feature_order_rows)


# ============================================================
# 9. FINAL REPRESENTATION AUDIT
# ============================================================

representation_rows = []
for c in CONTEXT_FEATURES:
    level = out[f"{c}_level"].astype(float)
    delta = out[f"{c}_delta"].astype(float)

    delta_active_mask = ~np.isclose(
        delta.to_numpy(), 0.0, atol=EPS, rtol=0.0
    )
    active_abs_scaled = delta.loc[delta_active_mask].abs()

    representation_rows.append({
        "variable": c,
        "level_min": float(level.min()),
        "level_max": float(level.max()),
        "level_mean": float(level.mean()),
        "level_std": float(level.std(ddof=0)),
        "delta_min": float(delta.min()),
        "delta_max": float(delta.max()),
        "delta_mean": float(delta.mean()),
        "delta_std": float(delta.std(ddof=0)),
        "delta_zero_share": float((~delta_active_mask).mean()),
        "delta_nonzero_count": int(delta_active_mask.sum()),
        "active_delta_abs_q95": float(active_abs_scaled.quantile(0.95)),
        "active_delta_abs_q99": float(active_abs_scaled.quantile(0.99)),
        "active_delta_abs_max": float(active_abs_scaled.max()),
    })

representation_audit = pd.DataFrame(representation_rows)


# ============================================================
# 10. TIME-CONTINUITY AUDIT
# ============================================================

time_continuity_audit = pd.DataFrame([{
    "n_rows": int(len(out)),
    "start_time": time.iloc[0],
    "end_time": time.iloc[-1],
    "expected_sampling_sec": EXPECTED_SAMPLING_SEC,
    "n_non_1s_steps": n_non_1s_steps,
    "n_delta_reset_rows": int(reset_mask.sum()),
}])


# ============================================================
# 11. SAVE OUTPUTS
# ============================================================

out.to_csv(
    OUTPUT_DIR / "train1_context_8d.csv",
    index=False,
    float_format="%.17g",
)

scaling_parameters.to_csv(
    OUTPUT_DIR / "scaling_parameters.csv",
    index=False,
    float_format="%.17g",
)

pd.DataFrame(delta_audit_rows).to_csv(
    OUTPUT_DIR / "delta_scale_audit.csv",
    index=False,
    float_format="%.17g",
)

representation_audit.to_csv(
    OUTPUT_DIR / "representation_audit.csv",
    index=False,
    float_format="%.17g",
)

feature_order.to_csv(
    OUTPUT_DIR / "feature_order.csv",
    index=False,
)

time_continuity_audit.to_csv(
    OUTPUT_DIR / "time_continuity_audit.csv",
    index=False,
)


# ============================================================
# 12. DISPLAY KEY CHECKS
# ============================================================

print("=" * 72)
print("TRAIN1 8D OPERATING-CONTEXT REPRESENTATION CREATED")
print("=" * 72)
print(f"Rows: {len(out):,}")
print(f"Time column: {TIME_COL}")
print(f"Non-1 s steps: {n_non_1s_steps}")
print(f"Output directory: {OUTPUT_DIR}")

print("\nTRAIN-DERIVED SCALING PARAMETERS")
display(scaling_parameters)

print("\nDELTA SCALE AUDIT")
display(pd.DataFrame(delta_audit_rows).round(8))

print("\nFINAL 8D REPRESENTATION AUDIT")
display(representation_audit.round(8))

print("\nMODEL FEATURE ORDER")
display(feature_order)

TRAIN1 8D OPERATING-CONTEXT REPRESENTATION CREATED
Rows: 216,001
Time column: time
Non-1 s steps: 0
Output directory: outputs/section_5_2/train_context_8d

TRAIN-DERIVED SCALING PARAMETERS


,variable,train_min,train_max,train_range,delta_q99_abs_nonzero
0,P1_B2004,0.02989,0.10135,0.07146,0.062478
1,P1_B3004,369.75601,443.27078,73.51477,0.040601
2,P1_B3005,894.71869,1121.94116,227.22247,0.060520
3,P1_B4002,31.64864,33.65550,2.00686,0.051558



DELTA SCALE AUDIT


,variable,n_nonzero_deltas,nonzero_share,signed_min,signed_max,abs_min,abs_q25,abs_median,abs_q75,abs_q90,abs_q95,abs_q99,abs_max,max_to_q95_ratio,max_to_q99_ratio
0,P1_B2004,354,0.001639,-0.062552,0.060873,1.399400e-04,0.000840,0.002659,0.060174,0.061573,0.061853,0.062478,0.062552,1.011312,1.001187
1,P1_B3004,363,0.001681,-0.040601,0.048658,4.100000e-07,0.006776,0.015100,0.031006,0.032439,0.033078,0.040601,0.048658,1.471033,1.198441
2,P1_B3005,371,0.001718,-0.060520,0.084053,5.300000e-07,0.006260,0.015081,0.032553,0.056035,0.056448,0.060520,0.084053,1.489043,1.388850
3,P1_B4002,230,0.001065,-0.051558,0.047373,4.980000e-06,0.005098,0.010942,0.018178,0.047368,0.051558,0.051558,0.051558,1.000000,1.000000



FINAL 8D REPRESENTATION AUDIT


,variable,level_min,level_max,level_mean,level_std,delta_min,delta_max,delta_mean,delta_std,delta_zero_share,delta_nonzero_count,active_delta_abs_q95,active_delta_abs_q99,active_delta_abs_max
0,P1_B2004,0.0,1.0,0.852916,0.293161,-1.001187,0.974310,-0.000003,0.022710,0.998361,354,0.989988,1.0,1.001187
1,P1_B3004,0.0,1.0,0.435420,0.204234,-1.000002,1.198441,-0.000024,0.021960,0.998319,363,0.814693,1.0,1.198441
2,P1_B3005,0.0,1.0,0.770496,0.259331,-1.000000,1.388850,0.000040,0.019660,0.998282,371,0.932713,1.0,1.388850
3,P1_B4002,0.0,1.0,0.340458,0.266868,-1.000000,0.918817,-0.000074,0.014078,0.998935,230,1.000000,1.0,1.000000



MODEL FEATURE ORDER


,position,model_column,source_variable,role
0,1,P1_B2004_level,P1_B2004,level
1,2,P1_B3004_level,P1_B3004,level
2,3,P1_B3005_level,P1_B3005,level
3,4,P1_B4002_level,P1_B4002,level
4,5,P1_B2004_delta,P1_B2004,delta
5,6,P1_B3004_delta,P1_B3004,delta
6,7,P1_B3005_delta,P1_B3005,delta
7,8,P1_B4002_delta,P1_B4002,delta


### 5.2.2 Construct multiscale trailing-window representations


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# ============================================================
# HAI 21.03 — TRAIN1 MULTISCALE WINDOW CONSTRUCTION
#
# Purpose:
#   Convert the final 8D tick-level operating-context sequence
#   into the trailing/end-aligned window representations used by
#   the clustering-based segmentation branches.
#
# Window contract:
#   - lengths: 4, 8, 16 s
#   - stride: 1 s
#   - representation: arithmetic mean of all 8 context dimensions
#   - alignment: trailing / window end
#   - a window ending at row t covers rows t-L+1, ..., t
#
# Input:
#   outputs/section_5_2/train_context_8d/
#       train1_context_8d.csv
#       feature_order.csv
#
# Output directory:
#   outputs/section_5_2/multiscale_windows/
#       train1_window_4s.csv
#       train1_window_8s.csv
#       train1_window_16s.csv
#       windowing_summary.csv
#       window_feature_audit.csv
#       window_feature_order.csv
#       window_integrity_checks.csv
# ============================================================

INPUT_DIR = Path("outputs/section_5_2/train_context_8d")
INPUT_FILE = INPUT_DIR / "train1_context_8d.csv"
FEATURE_ORDER_FILE = INPUT_DIR / "feature_order.csv"

OUTPUT_DIR = Path("outputs/section_5_2/multiscale_windows")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_LENGTHS = [4, 8, 16]
EXPECTED_SAMPLING_SEC = 1.0
ATOL = 1e-12


# ============================================================
# 1. LOAD FEATURE-ORDER CONTRACT
# ============================================================

feature_order = pd.read_csv(FEATURE_ORDER_FILE)

required_feature_order_cols = {
    "position",
    "model_column",
    "source_variable",
    "role",
}

missing_feature_order_cols = (
    required_feature_order_cols - set(feature_order.columns)
)

if missing_feature_order_cols:
    raise ValueError(
        "feature_order.csv is missing required columns: "
        f"{sorted(missing_feature_order_cols)}"
    )

feature_order = feature_order.sort_values("position").reset_index(drop=True)

expected_positions = list(range(1, len(feature_order) + 1))
if feature_order["position"].tolist() != expected_positions:
    raise ValueError("feature_order.csv has a non-contiguous position sequence.")

MODEL_COLUMNS = feature_order["model_column"].tolist()

if len(MODEL_COLUMNS) != 8:
    raise ValueError(
        f"Expected exactly 8 operating-context dimensions, found {len(MODEL_COLUMNS)}."
    )

if feature_order["model_column"].duplicated().any():
    raise ValueError("Duplicate model columns found in feature_order.csv.")

LEVEL_COLUMNS = feature_order.loc[
    feature_order["role"].eq("level"), "model_column"
].tolist()

DELTA_COLUMNS = feature_order.loc[
    feature_order["role"].eq("delta"), "model_column"
].tolist()

if len(LEVEL_COLUMNS) != 4 or len(DELTA_COLUMNS) != 4:
    raise ValueError(
        "Expected four level dimensions and four delta dimensions in feature_order.csv."
    )


# ============================================================
# 2. LOAD AND VALIDATE FINAL 8D TRAIN REPRESENTATION
# ============================================================

df = pd.read_csv(INPUT_FILE)

required_cols = ["row_id", "time"] + MODEL_COLUMNS
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing columns in train1_context_8d.csv: {missing_cols}")

if len(df) < max(WINDOW_LENGTHS):
    raise ValueError(
        f"Input contains only {len(df)} rows; at least {max(WINDOW_LENGTHS)} are required."
    )

row_id = pd.to_numeric(df["row_id"], errors="coerce")
if row_id.isna().any():
    raise ValueError("row_id contains missing or non-numeric values.")

expected_row_id = np.arange(len(df), dtype=np.int64)
if not np.array_equal(row_id.to_numpy(dtype=np.int64), expected_row_id):
    raise ValueError("row_id must be exactly 0, 1, ..., n-1 in train1_context_8d.csv.")

time = pd.to_datetime(df["time"], errors="coerce")
if time.isna().any():
    raise ValueError("Invalid timestamps found in train1_context_8d.csv.")

if time.duplicated().any():
    raise ValueError("Duplicate timestamps found in train1_context_8d.csv.")

time_diff_sec = time.diff().dt.total_seconds()
non_1s_mask = time_diff_sec.notna() & ~np.isclose(
    time_diff_sec,
    EXPECTED_SAMPLING_SEC,
    rtol=0.0,
    atol=1e-9,
)

# train1 is the single continuous development run used in the manuscript.
if non_1s_mask.any():
    raise ValueError(
        f"Found {int(non_1s_mask.sum())} non-1-second steps in train1_context_8d.csv. "
        "Window construction must not cross temporal discontinuities."
    )

X = (
    df[MODEL_COLUMNS]
    .apply(pd.to_numeric, errors="coerce")
    .to_numpy(dtype=np.float64)
)

if not np.isfinite(X).all():
    raise ValueError("Missing or non-finite values found in the 8D context representation.")

n_rows = len(df)


# ============================================================
# 3. WINDOW FEATURE-ORDER CONTRACT
# ============================================================

WINDOW_MODEL_COLUMNS = [f"{c}_mean" for c in MODEL_COLUMNS]

window_feature_order = feature_order.copy()
window_feature_order["tick_model_column"] = window_feature_order["model_column"]
window_feature_order["model_column"] = WINDOW_MODEL_COLUMNS
window_feature_order["aggregation"] = "mean"

window_feature_order.to_csv(
    OUTPUT_DIR / "window_feature_order.csv",
    index=False,
)


# ============================================================
# 4. TRAILING / END-ALIGNED WINDOW CONSTRUCTION
# ============================================================

summary_rows = []
audit_rows = []
integrity_rows = []

for L in WINDOW_LENGTHS:
    # pandas rolling at index t uses rows t-L+1, ..., t.
    W = (
        df[MODEL_COLUMNS]
        .rolling(window=L, min_periods=L)
        .mean()
    )

    valid_mask = W.notna().all(axis=1)
    end_rows = np.flatnonzero(valid_mask.to_numpy())
    start_rows = end_rows - L + 1

    W_valid = W.loc[valid_mask, MODEL_COLUMNS].reset_index(drop=True)
    W_valid.columns = WINDOW_MODEL_COLUMNS

    out = pd.DataFrame({
        "window_index": np.arange(len(end_rows), dtype=np.int64),
        "window_start_row": start_rows.astype(np.int64),
        "window_end_row": end_rows.astype(np.int64),
        "window_start_time": time.iloc[start_rows].to_numpy(),
        "window_end_time": time.iloc[end_rows].to_numpy(),
        "window_samples": L,
        "window_sec": L,
        "stride_sec": 1,
        "alignment": "trailing_end",
    })

    out = pd.concat([out, W_valid], axis=1)

    # --------------------------------------------------------
    # Structural checks
    # --------------------------------------------------------
    expected_windows = n_rows - L + 1

    if len(out) != expected_windows:
        raise RuntimeError(
            f"{L}s: expected {expected_windows} windows, got {len(out)}."
        )

    if out[WINDOW_MODEL_COLUMNS].isna().any().any():
        raise RuntimeError(f"{L}s: missing values found in window features.")

    if not np.isfinite(out[WINDOW_MODEL_COLUMNS].to_numpy(dtype=float)).all():
        raise RuntimeError(f"{L}s: non-finite values found in window features.")

    if int(out["window_start_row"].iloc[0]) != 0:
        raise RuntimeError(f"{L}s: first window must start at row 0.")

    if int(out["window_end_row"].iloc[0]) != L - 1:
        raise RuntimeError(f"{L}s: first window must end at row {L - 1}.")

    if int(out["window_end_row"].iloc[-1]) != n_rows - 1:
        raise RuntimeError(f"{L}s: last window must end at the final train row.")

    if not np.all(
        out["window_start_row"].to_numpy(dtype=np.int64)
        == out["window_end_row"].to_numpy(dtype=np.int64) - L + 1
    ):
        raise RuntimeError(f"{L}s: start/end row alignment is inconsistent.")

    window_span_sec = (
        pd.to_datetime(out["window_end_time"])
        - pd.to_datetime(out["window_start_time"])
    ).dt.total_seconds()

    if not np.allclose(
        window_span_sec.to_numpy(dtype=float),
        float(L - 1),
        rtol=0.0,
        atol=1e-9,
    ):
        raise RuntimeError(f"{L}s: timestamp span does not match L samples at 1 Hz.")

    # --------------------------------------------------------
    # Direct mean-reconstruction checks
    #
    # Verify first, middle, and last windows against explicit
    # slices of the original 8D tick-level representation.
    # --------------------------------------------------------
    check_positions = sorted(set([0, len(out) // 2, len(out) - 1]))

    max_abs_reconstruction_error = 0.0

    for pos in check_positions:
        s = int(out.loc[pos, "window_start_row"])
        e = int(out.loc[pos, "window_end_row"])

        expected_mean = X[s : e + 1].mean(axis=0)
        observed_mean = out.loc[pos, WINDOW_MODEL_COLUMNS].to_numpy(dtype=float)

        err = float(np.max(np.abs(expected_mean - observed_mean)))
        max_abs_reconstruction_error = max(max_abs_reconstruction_error, err)

        if not np.allclose(
            expected_mean,
            observed_mean,
            rtol=1e-12,
            atol=1e-12,
        ):
            raise RuntimeError(
                f"{L}s: window mean reconstruction failed at window_index={pos}; "
                f"max_abs_error={err:.3e}."
            )

    # --------------------------------------------------------
    # Save window table
    # --------------------------------------------------------
    output_file = OUTPUT_DIR / f"train1_window_{L}s.csv"
    out.to_csv(output_file, index=False, float_format="%.17g")

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------
    summary_rows.append({
        "window_sec": L,
        "stride_sec": 1,
        "alignment": "trailing_end",
        "aggregation": "mean_8d_context",
        "n_input_rows": n_rows,
        "n_windows": len(out),
        "n_model_features": len(WINDOW_MODEL_COLUMNS),
        "first_window_start_row": int(out["window_start_row"].iloc[0]),
        "first_window_end_row": int(out["window_end_row"].iloc[0]),
        "first_window_start_time": out["window_start_time"].iloc[0],
        "first_window_end_time": out["window_end_time"].iloc[0],
        "last_window_start_row": int(out["window_start_row"].iloc[-1]),
        "last_window_end_row": int(out["window_end_row"].iloc[-1]),
        "last_window_start_time": out["window_start_time"].iloc[-1],
        "last_window_end_time": out["window_end_time"].iloc[-1],
        "output_file": output_file.name,
    })

    integrity_rows.append({
        "window_sec": L,
        "expected_windows": expected_windows,
        "actual_windows": len(out),
        "first_valid_end_row": int(out["window_end_row"].iloc[0]),
        "last_valid_end_row": int(out["window_end_row"].iloc[-1]),
        "checked_windows": len(check_positions),
        "max_abs_mean_reconstruction_error": max_abs_reconstruction_error,
        "status": "PASS",
    })

    # --------------------------------------------------------
    # Basic feature audit
    # --------------------------------------------------------
    for model_col, window_col, role in zip(
        MODEL_COLUMNS,
        WINDOW_MODEL_COLUMNS,
        feature_order["role"].tolist(),
    ):
        s = out[window_col].astype(float)

        audit_rows.append({
            "window_sec": L,
            "feature": window_col,
            "tick_feature": model_col,
            "feature_type": role,
            "min": float(s.min()),
            "q01": float(s.quantile(0.01)),
            "q05": float(s.quantile(0.05)),
            "median": float(s.median()),
            "q95": float(s.quantile(0.95)),
            "q99": float(s.quantile(0.99)),
            "max": float(s.max()),
            "mean": float(s.mean()),
            "std": float(s.std(ddof=0)),
            "zero_share": float(np.isclose(s, 0.0, atol=ATOL, rtol=0.0).mean()),
        })


# ============================================================
# 5. SAVE AUDITS
# ============================================================

windowing_summary = pd.DataFrame(summary_rows)
window_feature_audit = pd.DataFrame(audit_rows)
window_integrity_checks = pd.DataFrame(integrity_rows)

windowing_summary.to_csv(
    OUTPUT_DIR / "windowing_summary.csv",
    index=False,
)

window_feature_audit.to_csv(
    OUTPUT_DIR / "window_feature_audit.csv",
    index=False,
    float_format="%.17g",
)

window_integrity_checks.to_csv(
    OUTPUT_DIR / "window_integrity_checks.csv",
    index=False,
    float_format="%.17g",
)


# ============================================================
# 6. DISPLAY KEY CHECKS
# ============================================================

print("=" * 72)
print("TRAIN1 MULTISCALE WINDOW CONSTRUCTION COMPLETE")
print("=" * 72)
print(f"Input rows: {n_rows:,}")
print(f"Input model dimensions: {len(MODEL_COLUMNS)}")
print(f"Output directory: {OUTPUT_DIR}")

print("\nWINDOWING SUMMARY")
display(
    windowing_summary[
        [
            "window_sec",
            "stride_sec",
            "alignment",
            "n_windows",
            "n_model_features",
            "first_window_end_row",
            "last_window_end_row",
        ]
    ]
)

print("\nWINDOW INTEGRITY CHECKS")
display(window_integrity_checks)

print("\nWINDOW FEATURE ORDER")
display(window_feature_order)

TRAIN1 MULTISCALE WINDOW CONSTRUCTION COMPLETE
Input rows: 216,001
Input model dimensions: 8
Output directory: outputs/section_5_2/multiscale_windows

WINDOWING SUMMARY


,window_sec,stride_sec,alignment,n_windows,n_model_features,first_window_end_row,last_window_end_row
0,4,1,trailing_end,215998,8,3,216000
1,8,1,trailing_end,215994,8,7,216000
2,16,1,trailing_end,215986,8,15,216000



WINDOW INTEGRITY CHECKS


,window_sec,expected_windows,actual_windows,first_valid_end_row,last_valid_end_row,checked_windows,max_abs_mean_reconstruction_error,status
0,4,215998,215998,3,216000,3,0.0,PASS
1,8,215994,215994,7,216000,3,0.0,PASS
2,16,215986,215986,15,216000,3,0.0,PASS



WINDOW FEATURE ORDER


,position,model_column,source_variable,role,tick_model_column,aggregation
0,1,P1_B2004_level_mean,P1_B2004,level,P1_B2004_level,mean
1,2,P1_B3004_level_mean,P1_B3004,level,P1_B3004_level,mean
2,3,P1_B3005_level_mean,P1_B3005,level,P1_B3005_level,mean
3,4,P1_B4002_level_mean,P1_B4002,level,P1_B4002_level,mean
4,5,P1_B2004_delta_mean,P1_B2004,delta,P1_B2004_delta,mean
5,6,P1_B3004_delta_mean,P1_B3004,delta,P1_B3004_delta,mean
6,7,P1_B3005_delta_mean,P1_B3005,delta,P1_B3005_delta,mean
7,8,P1_B4002_delta_mean,P1_B4002,delta,P1_B4002_delta,mean


### 5.2.3 Audit window geometry

This diagnostic verifies the numerical geometry of the window representations used by the clustering branches. It is retained as a reproducibility check; it does not select a downstream configuration.


In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# ============================================================
# HAI 21.03 — TRAIN1 WINDOW-GEOMETRY AUDIT
#
# Purpose:
#   Audit the geometry of the final 8D trailing-window
#   representations before clustering-based segmentation.
#
# This cell is descriptive only:
#   - no model fitting
#   - no candidate selection
#   - no attack labels
#   - no additional scaling
#
# It checks whether the level and delta blocks contribute
# sensibly to Euclidean geometry at L = 4, 8, and 16 s.
#
# Input:
#   outputs/section_5_2/multiscale_windows/
#       train1_window_4s.csv
#       train1_window_8s.csv
#       train1_window_16s.csv
#       window_feature_order.csv
#
# Output directory:
#   outputs/section_5_2/window_geometry_audit/
#       feature_variance.csv
#       block_variance_summary.csv
#       delta_activity_summary.csv
#       random_pair_geometry.csv
#       adjacent_step_geometry.csv
#       integrity_checks.csv
# ============================================================

INPUT_DIR = Path("outputs/section_5_2/multiscale_windows")
FEATURE_ORDER_FILE = INPUT_DIR / "window_feature_order.csv"

OUTPUT_DIR = Path("outputs/section_5_2/window_geometry_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_LENGTHS = [4, 8, 16]
RANDOM_STATE = 42
N_RANDOM_PAIRS = 100_000
ACTIVE_TOL = 1e-12
ATOL = 1e-12


# ============================================================
# 1. HELPERS
# ============================================================

def finite_quantiles(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return {
            "p05": np.nan,
            "p25": np.nan,
            "median": np.nan,
            "p75": np.nan,
            "p95": np.nan,
            "p99": np.nan,
            "max": np.nan,
        }

    return {
        "p05": float(np.quantile(values, 0.05)),
        "p25": float(np.quantile(values, 0.25)),
        "median": float(np.quantile(values, 0.50)),
        "p75": float(np.quantile(values, 0.75)),
        "p95": float(np.quantile(values, 0.95)),
        "p99": float(np.quantile(values, 0.99)),
        "max": float(np.max(values)),
    }


def block_distance_summary(level_sq, delta_sq):
    level_sq = np.asarray(level_sq, dtype=float)
    delta_sq = np.asarray(delta_sq, dtype=float)

    total_sq = level_sq + delta_sq
    valid = np.isfinite(total_sq) & (total_sq > 0)

    if not valid.any():
        return None

    total_distance = np.sqrt(total_sq[valid])
    delta_share = delta_sq[valid] / total_sq[valid]
    level_share = level_sq[valid] / total_sq[valid]

    q_dist = finite_quantiles(total_distance)
    q_delta = finite_quantiles(delta_share)
    q_level = finite_quantiles(level_share)

    return {
        "n_valid": int(valid.sum()),
        "distance_p05": q_dist["p05"],
        "distance_median": q_dist["median"],
        "distance_p95": q_dist["p95"],
        "distance_p99": q_dist["p99"],
        "distance_max": q_dist["max"],
        "delta_share_p05": q_delta["p05"],
        "delta_share_p25": q_delta["p25"],
        "delta_share_median": q_delta["median"],
        "delta_share_p75": q_delta["p75"],
        "delta_share_p95": q_delta["p95"],
        "delta_share_p99": q_delta["p99"],
        "level_share_median": q_level["median"],
    }


# ============================================================
# 2. LOAD FEATURE-ORDER CONTRACT
# ============================================================

feature_order = pd.read_csv(FEATURE_ORDER_FILE)

required_order_cols = {
    "position",
    "model_column",
    "role",
}

missing_order_cols = required_order_cols - set(feature_order.columns)
if missing_order_cols:
    raise ValueError(
        "window_feature_order.csv is missing required columns: "
        f"{sorted(missing_order_cols)}"
    )

feature_order = feature_order.sort_values("position").reset_index(drop=True)

if feature_order["position"].tolist() != list(range(1, len(feature_order) + 1)):
    raise ValueError("Feature positions are not contiguous.")

MODEL_COLS = feature_order["model_column"].tolist()
LEVEL_COLS = feature_order.loc[
    feature_order["role"].eq("level"), "model_column"
].tolist()
DELTA_COLS = feature_order.loc[
    feature_order["role"].eq("delta"), "model_column"
].tolist()

if len(MODEL_COLS) != 8:
    raise ValueError(f"Expected 8 window-model features, found {len(MODEL_COLS)}.")

if len(LEVEL_COLS) != 4 or len(DELTA_COLS) != 4:
    raise ValueError(
        "Expected four level and four delta window features."
    )

if len(set(MODEL_COLS)) != len(MODEL_COLS):
    raise ValueError("Duplicate model columns found in window feature order.")


# ============================================================
# 3. STORAGE
# ============================================================

feature_variance_rows = []
block_variance_rows = []
delta_activity_rows = []
random_pair_rows = []
adjacent_rows = []
integrity_rows = []


# ============================================================
# 4. AUDIT EACH WINDOW LENGTH
# ============================================================

for L in WINDOW_LENGTHS:
    input_file = INPUT_DIR / f"train1_window_{L}s.csv"
    df = pd.read_csv(input_file)

    required_cols = [
        "window_index",
        "window_start_row",
        "window_end_row",
        "window_sec",
        "stride_sec",
        "alignment",
    ] + MODEL_COLS

    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{L}s: missing columns: {missing_cols}")

    if len(df) < 2:
        raise ValueError(f"{L}s: fewer than two windows available.")

    # --------------------------------------------------------
    # Structural contract checks
    # --------------------------------------------------------
    window_index = pd.to_numeric(df["window_index"], errors="coerce")
    if window_index.isna().any():
        raise ValueError(f"{L}s: non-numeric window_index values.")

    expected_index = np.arange(len(df), dtype=np.int64)
    if not np.array_equal(window_index.to_numpy(dtype=np.int64), expected_index):
        raise ValueError(f"{L}s: window_index is not 0, 1, ..., n-1.")

    if not (pd.to_numeric(df["window_sec"], errors="coerce") == L).all():
        raise ValueError(f"{L}s: inconsistent window_sec metadata.")

    if not (pd.to_numeric(df["stride_sec"], errors="coerce") == 1).all():
        raise ValueError(f"{L}s: stride_sec is not uniformly 1.")

    if not df["alignment"].astype(str).eq("trailing_end").all():
        raise ValueError(f"{L}s: alignment is not uniformly trailing_end.")

    start_rows = pd.to_numeric(df["window_start_row"], errors="coerce")
    end_rows = pd.to_numeric(df["window_end_row"], errors="coerce")

    if start_rows.isna().any() or end_rows.isna().any():
        raise ValueError(f"{L}s: invalid window row indices.")

    if not np.array_equal(
        start_rows.to_numpy(dtype=np.int64),
        end_rows.to_numpy(dtype=np.int64) - L + 1,
    ):
        raise ValueError(f"{L}s: start/end row alignment is inconsistent.")

    X = (
        df[MODEL_COLS]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=np.float64)
    )

    if not np.isfinite(X).all():
        raise ValueError(f"{L}s: NaN or infinite values in model features.")

    level = df[LEVEL_COLS].to_numpy(dtype=np.float64)
    delta = df[DELTA_COLS].to_numpy(dtype=np.float64)
    n = len(df)

    # --------------------------------------------------------
    # 4.1 Per-feature variance
    # Population variance is used because the purpose is to
    # describe the complete train1 window geometry.
    # --------------------------------------------------------
    variances = pd.Series(
        np.var(X, axis=0, ddof=0),
        index=MODEL_COLS,
    )

    means = pd.Series(
        np.mean(X, axis=0),
        index=MODEL_COLS,
    )

    stds = pd.Series(
        np.std(X, axis=0, ddof=0),
        index=MODEL_COLS,
    )

    for col in MODEL_COLS:
        role = "level" if col in LEVEL_COLS else "delta"
        feature_variance_rows.append({
            "window_sec": L,
            "feature": col,
            "role": role,
            "mean": float(means[col]),
            "std": float(stds[col]),
            "variance": float(variances[col]),
        })

    level_variance = float(variances[LEVEL_COLS].sum())
    delta_variance = float(variances[DELTA_COLS].sum())
    total_variance = level_variance + delta_variance

    if total_variance <= 0:
        raise RuntimeError(f"{L}s: total feature variance is not positive.")

    block_variance_rows.append({
        "window_sec": L,
        "n_windows": n,
        "level_total_variance": level_variance,
        "delta_total_variance": delta_variance,
        "total_variance": total_variance,
        "level_variance_share": level_variance / total_variance,
        "delta_variance_share": delta_variance / total_variance,
        "delta_to_level_variance_ratio": (
            delta_variance / level_variance if level_variance > 0 else np.nan
        ),
    })

    # --------------------------------------------------------
    # 4.2 Dynamic activity in the delta block
    # --------------------------------------------------------
    delta_norm = np.linalg.norm(delta, axis=1)
    active = delta_norm > ACTIVE_TOL

    q_all = finite_quantiles(delta_norm)
    q_active = finite_quantiles(delta_norm[active])

    delta_activity_rows.append({
        "window_sec": L,
        "n_windows": n,
        "n_active_windows": int(active.sum()),
        "active_window_share": float(active.mean()),
        "zero_delta_window_share": float((~active).mean()),
        "delta_norm_median_all": q_all["median"],
        "delta_norm_p95_all": q_all["p95"],
        "delta_norm_p99_all": q_all["p99"],
        "delta_norm_max_all": q_all["max"],
        "delta_norm_p05_active": q_active["p05"],
        "delta_norm_median_active": q_active["median"],
        "delta_norm_p95_active": q_active["p95"],
        "delta_norm_p99_active": q_active["p99"],
        "delta_norm_max_active": q_active["max"],
    })

    # --------------------------------------------------------
    # 4.3 Random-pair Euclidean geometry
    # Fixed seed per L -> reproducible audit.
    # --------------------------------------------------------
    rng = np.random.default_rng(RANDOM_STATE + L)
    n_pairs = min(N_RANDOM_PAIRS, max(1, n * 2))

    i = rng.integers(0, n, size=n_pairs)
    j = rng.integers(0, n, size=n_pairs)

    same = i == j
    while same.any():
        j[same] = rng.integers(0, n, size=int(same.sum()))
        same = i == j

    level_diff = level[i] - level[j]
    delta_diff = delta[i] - delta[j]

    level_sq = np.sum(level_diff ** 2, axis=1)
    delta_sq = np.sum(delta_diff ** 2, axis=1)

    pair_summary = block_distance_summary(level_sq, delta_sq)
    if pair_summary is None:
        raise RuntimeError(f"{L}s: no non-identical random pairs with positive distance.")

    random_pair_rows.append({
        "window_sec": L,
        "n_sampled_pairs": n_pairs,
        **pair_summary,
    })

    # --------------------------------------------------------
    # 4.4 Adjacent-window step geometry
    # This is descriptive of local movement in the overlapping
    # stride-1 representation; it is not a clustering criterion.
    # --------------------------------------------------------
    level_step = np.diff(level, axis=0)
    delta_step = np.diff(delta, axis=0)

    level_step_sq = np.sum(level_step ** 2, axis=1)
    delta_step_sq = np.sum(delta_step ** 2, axis=1)

    adjacent_summary = block_distance_summary(level_step_sq, delta_step_sq)

    if adjacent_summary is None:
        adjacent_rows.append({
            "window_sec": L,
            "n_adjacent_pairs": n - 1,
            "n_valid": 0,
            "distance_p05": np.nan,
            "distance_median": np.nan,
            "distance_p95": np.nan,
            "distance_p99": np.nan,
            "distance_max": np.nan,
            "delta_share_p05": np.nan,
            "delta_share_p25": np.nan,
            "delta_share_median": np.nan,
            "delta_share_p75": np.nan,
            "delta_share_p95": np.nan,
            "delta_share_p99": np.nan,
            "level_share_median": np.nan,
        })
    else:
        adjacent_rows.append({
            "window_sec": L,
            "n_adjacent_pairs": n - 1,
            **adjacent_summary,
        })

    # --------------------------------------------------------
    # Integrity summary
    # --------------------------------------------------------
    integrity_rows.append({
        "window_sec": L,
        "n_windows": n,
        "n_features": X.shape[1],
        "n_level_features": len(LEVEL_COLS),
        "n_delta_features": len(DELTA_COLS),
        "all_finite": True,
        "positive_total_variance": bool(total_variance > 0),
        "status": "PASS",
    })


# ============================================================
# 5. SAVE OUTPUTS
# ============================================================

feature_variance_df = pd.DataFrame(feature_variance_rows)
block_variance_df = pd.DataFrame(block_variance_rows)
delta_activity_df = pd.DataFrame(delta_activity_rows)
random_pair_df = pd.DataFrame(random_pair_rows)
adjacent_df = pd.DataFrame(adjacent_rows)
integrity_df = pd.DataFrame(integrity_rows)

feature_variance_df.to_csv(
    OUTPUT_DIR / "feature_variance.csv",
    index=False,
)
block_variance_df.to_csv(
    OUTPUT_DIR / "block_variance_summary.csv",
    index=False,
)
delta_activity_df.to_csv(
    OUTPUT_DIR / "delta_activity_summary.csv",
    index=False,
)
random_pair_df.to_csv(
    OUTPUT_DIR / "random_pair_geometry.csv",
    index=False,
)
adjacent_df.to_csv(
    OUTPUT_DIR / "adjacent_step_geometry.csv",
    index=False,
)
integrity_df.to_csv(
    OUTPUT_DIR / "integrity_checks.csv",
    index=False,
)


# ============================================================
# 6. DISPLAY COMPACT RESULTS
# ============================================================

print("TRAIN1 WINDOW-GEOMETRY AUDIT")
print("=" * 72)
print(f"Input directory : {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Feature order   : {MODEL_COLS}")
print()

print("Block variance summary")
display(block_variance_df)

print("Delta activity summary")
display(delta_activity_df)

print("Random-pair Euclidean geometry")
display(random_pair_df)

print("Adjacent-window step geometry")
display(adjacent_df)

print("Integrity checks")
display(integrity_df)

if not integrity_df["status"].eq("PASS").all():
    raise RuntimeError("At least one geometry-audit integrity check failed.")

print()
print("WINDOW-GEOMETRY AUDIT STATUS: PASS")

TRAIN1 WINDOW-GEOMETRY AUDIT
Input directory : outputs/section_5_2/multiscale_windows
Output directory: outputs/section_5_2/window_geometry_audit
Feature order   : ['P1_B2004_level_mean', 'P1_B3004_level_mean', 'P1_B3005_level_mean', 'P1_B4002_level_mean', 'P1_B2004_delta_mean', 'P1_B3004_delta_mean', 'P1_B3005_delta_mean', 'P1_B4002_delta_mean']

Block variance summary


,window_sec,n_windows,level_total_variance,delta_total_variance,total_variance,level_variance_share,delta_variance_share,delta_to_level_variance_ratio
0,4,215998,0.266119,0.001455,0.267574,0.994562,0.005438,0.005468
1,8,215994,0.266101,0.001318,0.267419,0.995071,0.004929,0.004954
2,16,215986,0.266039,0.001055,0.267094,0.996052,0.003948,0.003964


Delta activity summary


,window_sec,n_windows,n_active_windows,active_window_share,zero_delta_window_share,delta_norm_median_all,delta_norm_p95_all,delta_norm_p99_all,delta_norm_max_all,delta_norm_p05_active,delta_norm_median_active,delta_norm_p95_active,delta_norm_p99_active,delta_norm_max_active
0,4,215998,449,0.002079,0.997921,0.0,0.0,0.0,1.501065,0.069760,0.819382,1.310511,1.501065,1.501065
1,8,215994,537,0.002486,0.997514,0.0,0.0,0.0,1.501065,0.043553,0.619824,1.280954,1.500874,1.501065
2,16,215986,713,0.003301,0.996699,0.0,0.0,0.0,1.418759,0.035585,0.423219,1.011175,1.232455,1.418759


Random-pair Euclidean geometry


...

Adjacent-window step geometry


,window_sec,n_adjacent_pairs,n_valid,distance_p05,distance_median,distance_p95,distance_p99,distance_max,delta_share_p05,delta_share_p25,delta_share_median,delta_share_p75,delta_share_p95,delta_share_p99,level_share_median
0,4,215997,471,0.007511,0.066203,0.311522,0.342168,0.380778,1.025120e-25,4.035196e-07,0.142504,0.987913,1.000000,1.0,0.857496
1,8,215993,559,0.015206,0.098992,0.167038,0.193963,0.201909,3.955913e-11,1.765676e-04,0.917751,0.984232,1.000000,1.0,0.082249
2,16,215985,735,0.011985,0.065080,0.098469,0.114281,0.124595,5.759673e-01,7.187482e-01,0.862773,0.968222,0.999477,1.0,0.137227


Integrity checks


,window_sec,n_windows,n_features,n_level_features,n_delta_features,all_finite,positive_total_variance,status
0,4,215998,8,4,4,True,True,PASS
1,8,215994,8,4,4,True,True,PASS
2,16,215986,8,4,4,True,True,PASS



WINDOW-GEOMETRY AUDIT STATUS: PASS


## Section 5.3 — Operating-context construction

All candidate segmentation models are fitted and screened on `train1.csv` only. Retained models are then reconstructed and checked against their screening outputs. Boundary stabilization and shared meta-mode alignment are also derived exclusively from the normal-development run.


### 5.3.1 K-means screening


In [5]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)

# ============================================================
# HAI 21.03 — TRAIN1 K-MEANS SEGMENTATION SCREENING
#
# Purpose:
#   Audit a fixed, label-blind k-means design grid on the final
#   train1 multiscale operating-context windows.
#
# Design grid:
#   - window lengths: 4, 8, 16 s
#   - K: 3, ..., 16
#
# Methodological contract:
#   - train1 only
#   - all models fitted on all train1 windows for their scale
#   - fixed random state and n_init
#   - no attack labels
#   - no automatic winner selection
#   - exact native-state labels and centroids from every audited
#     fit are saved for later review/reproduction
#
# Input:
#   outputs/section_5_2/multiscale_windows/
#       train1_window_4s.csv
#       train1_window_8s.csv
#       train1_window_16s.csv
#       window_feature_order.csv
#
# Output:
#   outputs/section_5_3/kmeans_screening/
#       settings.csv
#       model_summary.csv
#       cluster_summary.csv
#       cluster_centers.csv
#       native_runs.csv
#       transition_counts.csv
#       fit_diagnostics.csv
#       integrity_checks.csv
#       diagnostic_ranking.csv
#       silhouette_sample_rows.csv
#       labels_4s.csv
#       labels_8s.csv
#       labels_16s.csv
# ============================================================

INPUT_DIR = Path("outputs/section_5_2/multiscale_windows")
FEATURE_ORDER_FILE = INPUT_DIR / "window_feature_order.csv"

OUTPUT_DIR = Path("outputs/section_5_3/kmeans_screening")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_LENGTHS = [4, 8, 16]
K_VALUES = list(range(3, 17))

RANDOM_STATE = 42
N_INIT = 30
MAX_ITER = 500
TOL = 1e-4
ALGORITHM = "lloyd"

# Silhouette is evaluated on the same deterministic subsample
# for every K within a given window length. All other metrics
# use the complete train1 window sequence.
SILHOUETTE_SAMPLE_SIZE = 10_000


# ============================================================
# HELPERS
# ============================================================

def quantile_or_nan(values, p):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return np.nan
    return float(np.quantile(values, p))


def normalized_entropy(counts):
    counts = np.asarray(counts, dtype=float)
    if counts.sum() <= 0:
        return np.nan
    p = counts / counts.sum()
    p = p[p > 0]
    if len(p) <= 1:
        return 0.0
    h = -np.sum(p * np.log(p))
    return float(h / np.log(len(counts)))


def build_native_runs(labels):
    """Contiguous runs of identical native k-means labels."""
    labels = np.asarray(labels, dtype=int)
    if labels.size == 0:
        return pd.DataFrame(
            columns=[
                "native_run_index",
                "cluster",
                "start_window_index",
                "end_window_index",
                "duration_sec",
            ]
        )

    starts = np.r_[0, np.flatnonzero(labels[1:] != labels[:-1]) + 1]
    ends = np.r_[starts[1:] - 1, len(labels) - 1]

    return pd.DataFrame(
        {
            "native_run_index": np.arange(1, len(starts) + 1),
            "cluster": labels[starts],
            "start_window_index": starts,
            "end_window_index": ends,
            # stride = 1 s, so the number of consecutive window
            # endpoints equals run duration in seconds.
            "duration_sec": ends - starts + 1,
        }
    )


# ============================================================
# 1. LOAD AND VALIDATE WINDOW FEATURE CONTRACT
# ============================================================

feature_order = pd.read_csv(FEATURE_ORDER_FILE)

required_feature_cols = {
    "position",
    "model_column",
    "source_variable",
    "role",
    "aggregation",
}

missing_feature_cols = required_feature_cols - set(feature_order.columns)
if missing_feature_cols:
    raise ValueError(
        "window_feature_order.csv is missing required columns: "
        f"{sorted(missing_feature_cols)}"
    )

feature_order = feature_order.sort_values("position").reset_index(drop=True)
expected_positions = list(range(1, len(feature_order) + 1))
if feature_order["position"].tolist() != expected_positions:
    raise ValueError("window_feature_order.csv has a non-contiguous position sequence.")

MODEL_COLS = feature_order["model_column"].tolist()
LEVEL_COLS = feature_order.loc[
    feature_order["role"].eq("level"), "model_column"
].tolist()
DELTA_COLS = feature_order.loc[
    feature_order["role"].eq("delta"), "model_column"
].tolist()

if len(MODEL_COLS) != 8 or len(LEVEL_COLS) != 4 or len(DELTA_COLS) != 4:
    raise ValueError(
        "Expected exactly 8 window features: four level means and four delta means."
    )

if feature_order["model_column"].duplicated().any():
    raise ValueError("Duplicate model columns found in window_feature_order.csv.")

if not feature_order["aggregation"].eq("mean").all():
    raise ValueError("All window model features must use mean aggregation.")


# ============================================================
# 2. SAVE REPRODUCIBILITY SETTINGS
# ============================================================

settings = pd.DataFrame(
    [
        {"parameter": "window_lengths_sec", "value": "4,8,16"},
        {"parameter": "K_values", "value": "3-16"},
        {"parameter": "random_state", "value": RANDOM_STATE},
        {"parameter": "n_init", "value": N_INIT},
        {"parameter": "max_iter", "value": MAX_ITER},
        {"parameter": "tol", "value": TOL},
        {"parameter": "algorithm", "value": ALGORITHM},
        {
            "parameter": "silhouette_sample_size",
            "value": SILHOUETTE_SAMPLE_SIZE,
        },
        {"parameter": "feature_order", "value": ",".join(MODEL_COLS)},
        {
            "parameter": "selection_rule",
            "value": "diagnostic screening only; no automatic winner selection",
        },
    ]
)
settings.to_csv(OUTPUT_DIR / "settings.csv", index=False)


# ============================================================
# 3. STORAGE
# ============================================================

summary_rows = []
cluster_rows = []
center_rows = []
native_run_frames = []
transition_frames = []
silhouette_sample_frames = []
fit_rows = []
integrity_rows = []


# ============================================================
# 4. SCREENING LOOP
# ============================================================

for L in WINDOW_LENGTHS:
    print("\n" + "=" * 76)
    print(f"K-MEANS SCREENING — WINDOW LENGTH {L} s")
    print("=" * 76)

    input_file = INPUT_DIR / f"train1_window_{L}s.csv"
    df = pd.read_csv(input_file)

    required_cols = [
        "window_index",
        "window_start_row",
        "window_end_row",
        "window_start_time",
        "window_end_time",
        "window_samples",
        "window_sec",
        "stride_sec",
        "alignment",
    ] + MODEL_COLS

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{L}s: missing columns: {missing}")

    if len(df) <= max(K_VALUES):
        raise ValueError(f"{L}s: too few windows ({len(df)}) for K up to {max(K_VALUES)}.")

    # Structural contract inherited from multiscale-window construction.
    window_index = pd.to_numeric(df["window_index"], errors="coerce")
    expected_window_index = np.arange(len(df), dtype=np.int64)
    if window_index.isna().any() or not np.array_equal(
        window_index.to_numpy(dtype=np.int64), expected_window_index
    ):
        raise ValueError(f"{L}s: window_index is not exactly 0,1,...,n-1.")

    if not pd.to_numeric(df["window_samples"], errors="coerce").eq(L).all():
        raise ValueError(f"{L}s: inconsistent window_samples values.")
    if not pd.to_numeric(df["window_sec"], errors="coerce").eq(L).all():
        raise ValueError(f"{L}s: inconsistent window_sec values.")
    if not pd.to_numeric(df["stride_sec"], errors="coerce").eq(1).all():
        raise ValueError(f"{L}s: stride_sec must be 1 for all windows.")
    if not df["alignment"].astype(str).eq("trailing_end").all():
        raise ValueError(f"{L}s: alignment must be trailing_end for all windows.")

    start_row = pd.to_numeric(df["window_start_row"], errors="coerce")
    end_row = pd.to_numeric(df["window_end_row"], errors="coerce")
    if start_row.isna().any() or end_row.isna().any():
        raise ValueError(f"{L}s: invalid window row indices.")
    if not np.array_equal(
        (end_row - start_row + 1).to_numpy(dtype=np.int64),
        np.full(len(df), L, dtype=np.int64),
    ):
        raise ValueError(f"{L}s: window row spans do not equal the declared length.")

    X = (
        df[MODEL_COLS]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=np.float64)
    )
    if not np.isfinite(X).all():
        raise ValueError(f"{L}s: missing or non-finite model features.")

    n = len(X)

    # --------------------------------------------------------
    # Deterministic silhouette sample shared across K for L.
    # --------------------------------------------------------
    sample_n = min(SILHOUETTE_SAMPLE_SIZE, n)
    rng = np.random.default_rng(RANDOM_STATE + L)
    sample_idx = np.sort(rng.choice(n, size=sample_n, replace=False))
    X_sil = X[sample_idx]

    silhouette_sample_frames.append(
        pd.DataFrame(
            {
                "window_sec": L,
                "window_index": sample_idx,
                "window_end_row": df.loc[sample_idx, "window_end_row"].to_numpy(),
                "window_end_time": df.loc[sample_idx, "window_end_time"].to_numpy(),
            }
        )
    )

    # Exact labels from every audited fit are stored side-by-side.
    labels_out = pd.DataFrame(
        {
            "window_index": np.arange(n, dtype=np.int64),
            "window_start_row": df["window_start_row"].to_numpy(),
            "window_end_row": df["window_end_row"].to_numpy(),
            "window_start_time": df["window_start_time"].to_numpy(),
            "window_end_time": df["window_end_time"].to_numpy(),
        }
    )

    for K in K_VALUES:
        design_id = f"KM{K}_{L}"
        print(f"  {design_id}: fitting ...", end=" ", flush=True)

        model = KMeans(
            n_clusters=K,
            init="k-means++",
            n_init=N_INIT,
            max_iter=MAX_ITER,
            tol=TOL,
            random_state=RANDOM_STATE,
            algorithm=ALGORITHM,
        )

        warning_text = ""
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            labels = model.fit_predict(X)
            if caught:
                warning_text = " | ".join(str(w.message) for w in caught)

        labels = np.asarray(labels, dtype=np.int64)
        centers = np.asarray(model.cluster_centers_, dtype=np.float64)
        labels_out[design_id] = labels

        # ----------------------------------------------------
        # Integrity checks for the exact audited fit.
        # ----------------------------------------------------
        predicted = model.predict(X).astype(np.int64)
        predict_matches_fit = bool(np.array_equal(predicted, labels))

        unique_labels = np.unique(labels)
        all_clusters_present = bool(
            np.array_equal(unique_labels, np.arange(K, dtype=np.int64))
        )

        counts = np.bincount(labels, minlength=K)
        shares = counts / n
        no_empty_clusters = bool((counts > 0).all())

        assigned_center_dist_sq = np.sum((X - centers[labels]) ** 2, axis=1)
        reconstructed_inertia = float(assigned_center_dist_sq.sum())
        inertia_matches = bool(
            np.isclose(
                reconstructed_inertia,
                float(model.inertia_),
                rtol=1e-10,
                atol=1e-8,
            )
        )

        integrity_status = (
            "PASS"
            if predict_matches_fit and all_clusters_present and no_empty_clusters and inertia_matches
            else "FAIL"
        )

        integrity_rows.append(
            {
                "design_id": design_id,
                "window_sec": L,
                "K": K,
                "predict_matches_fit_labels": predict_matches_fit,
                "all_cluster_ids_present": all_clusters_present,
                "no_empty_clusters": no_empty_clusters,
                "inertia_reconstruction_matches": inertia_matches,
                "status": integrity_status,
            }
        )

        if integrity_status != "PASS":
            raise RuntimeError(f"{design_id}: k-means integrity check failed.")

        # ----------------------------------------------------
        # Internal geometric metrics.
        # ----------------------------------------------------
        inertia = float(model.inertia_)
        db = float(davies_bouldin_score(X, labels))
        ch = float(calinski_harabasz_score(X, labels))

        sample_labels = labels[sample_idx]
        n_sample_clusters = int(len(np.unique(sample_labels)))
        if n_sample_clusters >= 2:
            sil = float(silhouette_score(X_sil, sample_labels, metric="euclidean"))
        else:
            sil = np.nan

        min_cluster_n = int(counts.min())
        max_cluster_n = int(counts.max())
        min_cluster_share = float(shares.min())
        max_cluster_share = float(shares.max())
        cluster_entropy = normalized_entropy(counts)

        # ----------------------------------------------------
        # Native-state temporal structure.
        # ----------------------------------------------------
        native_runs = build_native_runs(labels)
        durations = native_runs["duration_sec"].to_numpy(dtype=float)
        n_native_runs = len(native_runs)
        n_switches = max(n_native_runs - 1, 0)
        switching_density = n_switches / (n - 1) if n > 1 else np.nan

        share_runs_le_1 = float((durations <= 1).mean())
        share_runs_le_2 = float((durations <= 2).mean())
        share_runs_le_4 = float((durations <= 4).mean())
        share_runs_le_8 = float((durations <= 8).mean())
        share_runs_le_16 = float((durations <= 16).mean())

        share_ticks_runs_le_4 = float(durations[durations <= 4].sum() / n)
        share_ticks_runs_le_8 = float(durations[durations <= 8].sum() / n)
        share_ticks_runs_le_16 = float(durations[durations <= 16].sum() / n)

        # ----------------------------------------------------
        # Centroid geometry.
        # ----------------------------------------------------
        diff = centers[:, None, :] - centers[None, :, :]
        center_dist = np.sqrt(np.sum(diff**2, axis=2))
        offdiag = ~np.eye(K, dtype=bool)
        pair_center_dist = center_dist[offdiag]

        nearest_matrix = center_dist.copy()
        np.fill_diagonal(nearest_matrix, np.inf)
        nearest_distance = nearest_matrix.min(axis=1)
        nearest_cluster = nearest_matrix.argmin(axis=1)

        # ----------------------------------------------------
        # Per-cluster diagnostics and centroids.
        # ----------------------------------------------------
        for cluster_id in range(K):
            member_mask = labels == cluster_id
            member_dist_sq = assigned_center_dist_sq[member_mask]
            cluster_runs = native_runs[native_runs["cluster"].eq(cluster_id)]
            cluster_durations = cluster_runs["duration_sec"].to_numpy(dtype=float)

            center_level = centers[cluster_id, : len(LEVEL_COLS)]
            center_delta = centers[cluster_id, len(LEVEL_COLS) :]

            cluster_rows.append(
                {
                    "design_id": design_id,
                    "window_sec": L,
                    "K": K,
                    "cluster": cluster_id,
                    "n_windows": int(counts[cluster_id]),
                    "share": float(shares[cluster_id]),
                    "n_native_runs": int(len(cluster_runs)),
                    "run_duration_min": (
                        float(cluster_durations.min()) if len(cluster_durations) else np.nan
                    ),
                    "run_duration_p10": quantile_or_nan(cluster_durations, 0.10),
                    "run_duration_median": quantile_or_nan(cluster_durations, 0.50),
                    "run_duration_p90": quantile_or_nan(cluster_durations, 0.90),
                    "run_duration_max": (
                        float(cluster_durations.max()) if len(cluster_durations) else np.nan
                    ),
                    "within_cluster_mean_sq_distance": (
                        float(member_dist_sq.mean()) if len(member_dist_sq) else np.nan
                    ),
                    "within_cluster_rmse": (
                        float(np.sqrt(member_dist_sq.mean())) if len(member_dist_sq) else np.nan
                    ),
                    "center_level_norm": float(np.linalg.norm(center_level)),
                    "center_delta_norm": float(np.linalg.norm(center_delta)),
                    "nearest_cluster": int(nearest_cluster[cluster_id]),
                    "nearest_center_distance": float(nearest_distance[cluster_id]),
                }
            )

            center_row = {
                "design_id": design_id,
                "window_sec": L,
                "K": K,
                "cluster": cluster_id,
                "n_windows": int(counts[cluster_id]),
                "share": float(shares[cluster_id]),
                "nearest_cluster": int(nearest_cluster[cluster_id]),
                "nearest_center_distance": float(nearest_distance[cluster_id]),
            }
            for idx, feature in enumerate(MODEL_COLS):
                center_row[feature] = float(centers[cluster_id, idx])
            center_row["center_level_norm"] = float(np.linalg.norm(center_level))
            center_row["center_delta_norm"] = float(np.linalg.norm(center_delta))
            center_rows.append(center_row)

        # ----------------------------------------------------
        # Save contiguous native runs from this exact fit.
        # ----------------------------------------------------
        native_runs = native_runs.copy()
        native_runs.insert(0, "design_id", design_id)
        native_runs.insert(1, "window_sec", L)
        native_runs.insert(2, "K", K)

        start_idx = native_runs["start_window_index"].to_numpy(dtype=int)
        end_idx = native_runs["end_window_index"].to_numpy(dtype=int)
        native_runs["start_window_end_row"] = df.loc[
            start_idx, "window_end_row"
        ].to_numpy()
        native_runs["end_window_end_row"] = df.loc[
            end_idx, "window_end_row"
        ].to_numpy()
        native_runs["start_time"] = df.loc[
            start_idx, "window_end_time"
        ].to_numpy()
        native_runs["end_time"] = df.loc[
            end_idx, "window_end_time"
        ].to_numpy()
        native_run_frames.append(native_runs)

        # ----------------------------------------------------
        # Consecutive native-label transition counts.
        # ----------------------------------------------------
        previous = labels[:-1]
        current = labels[1:]
        changed = previous != current
        if changed.any():
            transitions = pd.DataFrame(
                {
                    "from_cluster": previous[changed],
                    "to_cluster": current[changed],
                }
            )
            transitions = (
                transitions.value_counts()
                .rename("count")
                .reset_index()
            )
            transitions.insert(0, "design_id", design_id)
            transitions.insert(1, "window_sec", L)
            transitions.insert(2, "K", K)
            transition_frames.append(transitions)

        # ----------------------------------------------------
        # Model-level summary.
        # ----------------------------------------------------
        summary_rows.append(
            {
                "design_id": design_id,
                "window_sec": L,
                "K": K,
                "n_windows": n,
                "n_features": X.shape[1],
                "inertia": inertia,
                "inertia_per_window": inertia / n,
                "silhouette_sample": sil,
                "silhouette_sample_size": sample_n,
                "silhouette_sample_clusters": n_sample_clusters,
                "davies_bouldin": db,
                "calinski_harabasz": ch,
                "min_cluster_n": min_cluster_n,
                "min_cluster_share": min_cluster_share,
                "max_cluster_n": max_cluster_n,
                "max_cluster_share": max_cluster_share,
                "normalized_cluster_entropy": cluster_entropy,
                "n_native_runs": n_native_runs,
                "n_switches": n_switches,
                "switching_density": switching_density,
                "run_duration_min": float(durations.min()),
                "run_duration_p10": quantile_or_nan(durations, 0.10),
                "run_duration_p25": quantile_or_nan(durations, 0.25),
                "run_duration_median": quantile_or_nan(durations, 0.50),
                "run_duration_p75": quantile_or_nan(durations, 0.75),
                "run_duration_p90": quantile_or_nan(durations, 0.90),
                "run_duration_p95": quantile_or_nan(durations, 0.95),
                "run_duration_max": float(durations.max()),
                "share_native_runs_le_1s": share_runs_le_1,
                "share_native_runs_le_2s": share_runs_le_2,
                "share_native_runs_le_4s": share_runs_le_4,
                "share_native_runs_le_8s": share_runs_le_8,
                "share_native_runs_le_16s": share_runs_le_16,
                "share_ticks_in_native_runs_le_4s": share_ticks_runs_le_4,
                "share_ticks_in_native_runs_le_8s": share_ticks_runs_le_8,
                "share_ticks_in_native_runs_le_16s": share_ticks_runs_le_16,
                "center_distance_min": float(pair_center_dist.min()),
                "center_distance_p10": quantile_or_nan(pair_center_dist, 0.10),
                "center_distance_median": quantile_or_nan(pair_center_dist, 0.50),
                "center_distance_max": float(pair_center_dist.max()),
                "n_iter": int(model.n_iter_),
                "warning": warning_text,
            }
        )

        fit_rows.append(
            {
                "design_id": design_id,
                "window_sec": L,
                "K": K,
                "random_state": RANDOM_STATE,
                "n_init": N_INIT,
                "max_iter": MAX_ITER,
                "tol": TOL,
                "algorithm": ALGORITHM,
                "n_iter": int(model.n_iter_),
                "converged_before_max_iter": bool(model.n_iter_ < MAX_ITER),
                "warning": warning_text,
            }
        )

        print(
            f"sil={sil:.4f}, DB={db:.4f}, "
            f"runs={n_native_runs}, min_share={min_cluster_share:.5f}"
        )

    labels_out.to_csv(OUTPUT_DIR / f"labels_{L}s.csv", index=False)


# ============================================================
# 5. COMBINE AND SAVE OUTPUT TABLES
# ============================================================

summary_df = pd.DataFrame(summary_rows).sort_values(["window_sec", "K"])
cluster_df = pd.DataFrame(cluster_rows).sort_values(
    ["window_sec", "K", "cluster"]
)
center_df = pd.DataFrame(center_rows).sort_values(
    ["window_sec", "K", "cluster"]
)
native_runs_df = pd.concat(native_run_frames, ignore_index=True)
transition_df = (
    pd.concat(transition_frames, ignore_index=True)
    if transition_frames
    else pd.DataFrame(
        columns=[
            "design_id",
            "window_sec",
            "K",
            "from_cluster",
            "to_cluster",
            "count",
        ]
    )
)
silhouette_sample_df = pd.concat(silhouette_sample_frames, ignore_index=True)
fit_df = pd.DataFrame(fit_rows).sort_values(["window_sec", "K"])
integrity_df = pd.DataFrame(integrity_rows).sort_values(["window_sec", "K"])

summary_df.to_csv(OUTPUT_DIR / "model_summary.csv", index=False)
cluster_df.to_csv(OUTPUT_DIR / "cluster_summary.csv", index=False)
center_df.to_csv(OUTPUT_DIR / "cluster_centers.csv", index=False)
native_runs_df.to_csv(OUTPUT_DIR / "native_runs.csv", index=False)
transition_df.to_csv(OUTPUT_DIR / "transition_counts.csv", index=False)
silhouette_sample_df.to_csv(OUTPUT_DIR / "silhouette_sample_rows.csv", index=False)
fit_df.to_csv(OUTPUT_DIR / "fit_diagnostics.csv", index=False)
integrity_df.to_csv(OUTPUT_DIR / "integrity_checks.csv", index=False)


# ============================================================
# 6. DIAGNOSTIC RANKING — NO AUTOMATIC SELECTION
# ============================================================

ranking = summary_df.copy()
ranking["rank_silhouette_within_L"] = ranking.groupby("window_sec")[
    "silhouette_sample"
].rank(ascending=False, method="min")
ranking["rank_DB_within_L"] = ranking.groupby("window_sec")[
    "davies_bouldin"
].rank(ascending=True, method="min")
ranking["rank_CH_within_L"] = ranking.groupby("window_sec")[
    "calinski_harabasz"
].rank(ascending=False, method="min")
ranking["rank_center_separation_within_L"] = ranking.groupby("window_sec")[
    "center_distance_min"
].rank(ascending=False, method="min")
ranking.to_csv(OUTPUT_DIR / "diagnostic_ranking.csv", index=False)


# ============================================================
# 7. FINAL CHECKS AND COMPACT DISPLAY
# ============================================================

expected_models = len(WINDOW_LENGTHS) * len(K_VALUES)
if len(summary_df) != expected_models:
    raise RuntimeError(
        f"Expected {expected_models} audited models, found {len(summary_df)}."
    )

if not integrity_df["status"].eq("PASS").all():
    raise RuntimeError("At least one k-means model failed integrity checks.")

if not fit_df["converged_before_max_iter"].all():
    warnings.warn(
        "At least one k-means fit reached MAX_ITER. Review fit_diagnostics.csv."
    )

print("\n" + "=" * 76)
print("K-MEANS SCREENING COMPLETE")
print("=" * 76)
print(f"Audited models: {len(summary_df)}")
print(f"All integrity checks: {'PASS' if integrity_df['status'].eq('PASS').all() else 'FAIL'}")
print(f"Output directory: {OUTPUT_DIR}")
print("No model was selected automatically.")

compact_cols = [
    "design_id",
    "window_sec",
    "K",
    "silhouette_sample",
    "davies_bouldin",
    "calinski_harabasz",
    "min_cluster_share",
    "n_native_runs",
    "switching_density",
    "run_duration_median",
    "center_distance_min",
]

display(summary_df[compact_cols])


K-MEANS SCREENING — WINDOW LENGTH 4 s
  KM3_4: fitting ... sil=0.5767, DB=0.8210, runs=18, min_share=0.16417
  KM4_4: fitting ... sil=0.6305, DB=0.5779, runs=18, min_share=0.04948
  KM5_4: fitting ... sil=0.5613, DB=0.6958, runs=20, min_share=0.04947
  KM6_4: fitting ... sil=0.6389, DB=0.5600, runs=22, min_share=0.04947
  KM7_4: fitting ... sil=0.6894, DB=0.4295, runs=27, min_share=0.04947
  KM8_4: fitting ... sil=0.7119, DB=0.4948, runs=31, min_share=0.04947
  KM9_4: fitting ... sil=0.7513, DB=0.4842, runs=35, min_share=0.04946
  KM10_4: fitting ... sil=0.7817, DB=0.3424, runs=38, min_share=0.04222
  KM11_4: fitting ... sil=0.7986, DB=0.3042, runs=40, min_share=0.04209
  KM12_4: fitting ... sil=0.8340, DB=0.2803, runs=40, min_share=0.04209
  KM13_4: fitting ... sil=0.8521, DB=0.2757, runs=41, min_share=0.04040
  KM14_4: fitting ... sil=0.8444, DB=0.2897, runs=43, min_share=0.02367
  KM15_4: fitting ... sil=0.8402, DB=0.3071, runs=45, min_share=0.02367
  KM16_4: fitting ... sil=0.8646

,design_id,window_sec,K,silhouette_sample,davies_bouldin,calinski_harabasz,min_cluster_share,n_native_runs,switching_density,run_duration_median,center_distance_min
0,KM3_4,4,3,0.576709,0.821013,1.683971e+05,0.164173,18,0.000079,8628.5,0.759604
1,KM4_4,4,4,0.630518,0.577881,1.784637e+05,0.049477,18,0.000079,8628.0,0.789421
2,KM5_4,4,5,0.561348,0.695811,2.252079e+05,0.049468,20,0.000088,8898.5,0.387969
3,KM6_4,4,6,0.638912,0.560044,2.946167e+05,0.049473,22,0.000097,7739.0,0.353831
4,KM7_4,4,7,0.689379,0.429525,5.048785e+05,0.049473,27,0.000120,5886.0,0.353834
5,KM8_4,4,8,0.711866,0.494830,6.062302e+05,0.049473,31,0.000139,5685.0,0.260958
6,KM9_4,4,9,0.751317,0.484232,7.401679e+05,0.049463,35,0.000157,5622.0,0.228671
7,KM10_4,4,10,0.781664,0.342431,8.600584e+05,0.042223,38,0.000171,5404.5,0.233745
8,KM11_4,4,11,0.798584,0.304227,9.614631e+05,0.042093,40,0.000181,5204.0,0.226581
9,KM12_4,4,12,0.833981,0.280283,1.036530e+06,0.042093,40,0.000181,5204.0,0.122372


### 5.3.2 HDBSCAN screening


In [8]:
import tempfile
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import hdbscan
from joblib import Memory
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# ============================================================
# HAI 21.03 — TRAIN1 HDBSCAN SCREENING
#
# Screening only: no prediction objects are saved here.
# Retained HDBSCAN designs will be refitted later with
# prediction_data=True and exported for test assignment.
# ============================================================

INPUT_DIR = Path("outputs/section_5_2/multiscale_windows")
OUTPUT_DIR = Path("outputs/section_5_3/hdbscan_screening")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_LENGTHS = [4, 8, 16]
MCS_SHARES = [0.0050, 0.0065, 0.0100, 0.0150, 0.0200, 0.0250, 0.0300, 0.0400, 0.0500, 0.0750, 0.1000]
MIN_SAMPLES_VALUES = [50, 100, 200]
METRIC = "euclidean"
CLUSTER_SELECTION_METHOD = "eom"
SILHOUETTE_SAMPLE_N = 5_000
RANDOM_STATE = 42
SMALL_CLUSTER_SHARE_THRESHOLD = 0.01

# ------------------------------------------------------------
# Feature contract from multiscale-window construction
# ------------------------------------------------------------
feature_order = pd.read_csv(INPUT_DIR / "window_feature_order.csv").sort_values("position")
MODEL_COLS = feature_order["model_column"].tolist()

if len(MODEL_COLS) != 8 or feature_order["model_column"].duplicated().any():
    raise ValueError("Expected exactly eight unique model columns in window_feature_order.csv.")


def q(x, p):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.quantile(x, p)) if x.size else np.nan


def build_runs(labels):
    labels = np.asarray(labels, dtype=int)
    starts = np.r_[0, np.flatnonzero(labels[1:] != labels[:-1]) + 1]
    ends = np.r_[starts[1:] - 1, len(labels) - 1]
    return pd.DataFrame({
        "state": labels[starts],
        "duration_sec": ends - starts + 1,
    })


def internal_scores(X, labels):
    mask = labels != -1
    Xn, yn = X[mask], labels[mask]
    if len(Xn) < 3 or np.unique(yn).size < 2:
        return np.nan, np.nan, np.nan, 0

    sample_n = min(SILHOUETTE_SAMPLE_N, len(Xn))
    try:
        sil = silhouette_score(
            Xn, yn,
            metric="euclidean",
            sample_size=sample_n if sample_n < len(Xn) else None,
            random_state=RANDOM_STATE,
        )
    except Exception:
        sil = np.nan

    try:
        db = davies_bouldin_score(Xn, yn)
    except Exception:
        db = np.nan

    try:
        ch = calinski_harabasz_score(Xn, yn)
    except Exception:
        ch = np.nan

    return sil, db, ch, sample_n


settings = pd.DataFrame({
    "parameter": [
        "window_lengths_sec", "mcs_shares", "min_samples_values",
        "metric", "cluster_selection_method", "silhouette_sample_n",
        "small_cluster_share_threshold", "feature_order",
    ],
    "value": [
        ",".join(map(str, WINDOW_LENGTHS)),
        ",".join(map(str, MCS_SHARES)),
        ",".join(map(str, MIN_SAMPLES_VALUES)),
        METRIC,
        CLUSTER_SELECTION_METHOD,
        SILHOUETTE_SAMPLE_N,
        SMALL_CLUSTER_SHARE_THRESHOLD,
        ",".join(MODEL_COLS),
    ],
})
settings.to_csv(OUTPUT_DIR / "settings.csv", index=False)

screening_rows = []
cluster_tables = []
fit_rows = []

for L in WINDOW_LENGTHS:
    df = pd.read_csv(INPUT_DIR / f"train1_window_{L}s.csv")

    required = ["window_end_row", "window_end_time"] + MODEL_COLS
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{L}s: missing columns: {missing}")

    X = df[MODEL_COLS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(X).all():
        raise ValueError(f"{L}s: non-finite feature values.")

    n = len(X)
    mcs_values = [max(2, round(share * n)) for share in MCS_SHARES]

    print(f"\nHDBSCAN {L}s: {n:,} windows")

    # Reuse expensive HDBSCAN tree calculations across MCS values.
    with tempfile.TemporaryDirectory() as cache_dir:
        memory = Memory(cache_dir, verbose=0)

        for min_samples in MIN_SAMPLES_VALUES:
            for mcs_share, min_cluster_size in zip(MCS_SHARES, mcs_values):
                config_id = f"L{L}_MCS{mcs_share:.4f}_MS{min_samples}"
                t0 = time.perf_counter()

                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")
                    model = hdbscan.HDBSCAN(
                        min_cluster_size=min_cluster_size,
                        min_samples=min_samples,
                        metric=METRIC,
                        cluster_selection_method=CLUSTER_SELECTION_METHOD,
                        prediction_data=False,
                        approx_min_span_tree=True,
                        core_dist_n_jobs=-1,
                        memory=memory,
                    )
                    labels = model.fit_predict(X).astype(int)

                probabilities = np.asarray(model.probabilities_, dtype=float)
                warning_text = " | ".join(str(w.message) for w in caught)

                if len(labels) != n or len(probabilities) != n:
                    raise RuntimeError(f"{config_id}: output length mismatch.")

                assigned = labels != -1
                n_noise = int((~assigned).sum())
                assigned_n = int(assigned.sum())
                cluster_ids, counts = np.unique(labels[assigned], return_counts=True)
                n_clusters = len(cluster_ids)
                shares = counts / n if n_clusters else np.array([])

                sil, db, ch, sil_n = internal_scores(X, labels)
                runs = build_runs(labels)
                durations = runs["duration_sec"].to_numpy(dtype=float)
                noise_runs = runs.loc[runs["state"] == -1, "duration_sec"]

                persistence = np.asarray(model.cluster_persistence_, dtype=float)
                persistence_map = dict(zip(cluster_ids, persistence)) if len(persistence) == n_clusters else {}

                # One compact cluster-level table per fit.
                if n_clusters:
                    cdf = pd.DataFrame({"cluster": labels[assigned], "probability": probabilities[assigned]})
                    cdf = cdf.groupby("cluster", as_index=False).agg(
                        n_windows=("cluster", "size"),
                        membership_probability_mean=("probability", "mean"),
                        membership_probability_median=("probability", "median"),
                    )

                    rdf = runs[runs["state"] != -1].groupby("state", as_index=False).agg(
                        n_runs=("duration_sec", "size"),
                        run_duration_min=("duration_sec", "min"),
                        run_duration_median=("duration_sec", "median"),
                        run_duration_max=("duration_sec", "max"),
                    ).rename(columns={"state": "cluster"})

                    cdf = cdf.merge(rdf, on="cluster", how="left")
                    cdf["share_all_windows"] = cdf["n_windows"] / n
                    cdf["share_assigned_windows"] = cdf["n_windows"] / assigned_n
                    cdf["persistence"] = cdf["cluster"].map(persistence_map)
                    cdf.insert(0, "min_samples", min_samples)
                    cdf.insert(0, "min_cluster_size", min_cluster_size)
                    cdf.insert(0, "mcs_share", mcs_share)
                    cdf.insert(0, "window_sec", L)
                    cluster_tables.append(cdf)

                if n_clusters:
                    p = counts / counts.sum()
                    entropy = -np.sum(p * np.log(p))
                    normalized_entropy = float(entropy / np.log(n_clusters)) if n_clusters > 1 else 0.0
                    effective_n_clusters = float(np.exp(entropy))
                else:
                    normalized_entropy = np.nan
                    effective_n_clusters = np.nan

                screening_rows.append({
                    "config_id": config_id,
                    "window_sec": L,
                    "mcs_share": mcs_share,
                    "min_cluster_size": min_cluster_size,
                    "min_samples": min_samples,
                    "n_windows": n,
                    "n_clusters": n_clusters,
                    "noise_share": n_noise / n,
                    "assigned_share": assigned_n / n,
                    "min_cluster_share": shares.min() if shares.size else np.nan,
                    "max_cluster_share": shares.max() if shares.size else np.nan,
                    "n_clusters_below_1pct": int((shares < SMALL_CLUSTER_SHARE_THRESHOLD).sum()),
                    "normalized_cluster_entropy": normalized_entropy,
                    "effective_n_clusters": effective_n_clusters,
                    "cluster_persistence_median": q(persistence, 0.50),
                    "membership_probability_median": q(probabilities[assigned], 0.50),
                    "silhouette_non_noise": sil,
                    "silhouette_sample_n": sil_n,
                    "davies_bouldin_non_noise": db,
                    "calinski_harabasz_non_noise": ch,
                    "n_runs_including_noise": len(runs),
                    "switching_density": (len(runs) - 1) / (n - 1) if n > 1 else np.nan,
                    "run_duration_median": q(durations, 0.50),
                    "run_duration_p10": q(durations, 0.10),
                    "run_duration_max": q(durations, 1.00),
                    "share_runs_le_4s": float((durations <= 4).mean()),
                    "share_runs_le_8s": float((durations <= 8).mean()),
                    "share_runs_le_16s": float((durations <= 16).mean()),
                    "share_ticks_in_runs_le_16s": float(durations[durations <= 16].sum() / n),
                    "n_noise_runs": len(noise_runs),
                    "noise_run_median_sec": q(noise_runs, 0.50),
                    "noise_run_max_sec": q(noise_runs, 1.00),
                })

                fit_rows.append({
                    "config_id": config_id,
                    "window_sec": L,
                    "mcs_share": mcs_share,
                    "min_cluster_size": min_cluster_size,
                    "min_samples": min_samples,
                    "fit_seconds": time.perf_counter() - t0,
                    "n_clusters": n_clusters,
                    "noise_share": n_noise / n,
                    "warning": warning_text,
                })

                print(
                    f"  {config_id}: clusters={n_clusters:2d}, "
                    f"noise={n_noise/n:6.2%}, sil={sil:.3f}, "
                    f"runs={len(runs)}"
                )

screening_df = pd.DataFrame(screening_rows)
cluster_df = pd.concat(cluster_tables, ignore_index=True) if cluster_tables else pd.DataFrame()
fit_df = pd.DataFrame(fit_rows)

screening_df.to_csv(OUTPUT_DIR / "screening_summary.csv", index=False)
cluster_df.to_csv(OUTPUT_DIR / "cluster_summary.csv", index=False)
fit_df.to_csv(OUTPUT_DIR / "fit_diagnostics.csv", index=False)

print(f"\nCompleted {len(screening_df)} HDBSCAN fits.")
print(f"Outputs: {OUTPUT_DIR}")


HDBSCAN 4s: 215,998 windows
  L4_MCS0.0050_MS50: clusters=23, noise= 0.19%, sil=1.000, runs=45
  L4_MCS0.0065_MS50: clusters=23, noise= 0.19%, sil=1.000, runs=45
  L4_MCS0.0100_MS50: clusters=23, noise= 0.19%, sil=1.000, runs=45
  L4_MCS0.0150_MS50: clusters=23, noise= 0.19%, sil=1.000, runs=45
  L4_MCS0.0200_MS50: clusters=23, noise= 0.19%, sil=1.000, runs=45
  L4_MCS0.0250_MS50: clusters=20, noise= 0.18%, sil=0.976, runs=45
  L4_MCS0.0300_MS50: clusters=18, noise= 0.18%, sil=0.964, runs=45
  L4_MCS0.0400_MS50: clusters=15, noise= 0.18%, sil=0.906, runs=45
  L4_MCS0.0500_MS50: clusters= 9, noise=17.43%, sil=0.890, runs=35
  L4_MCS0.0750_MS50: clusters= 8, noise=23.97%, sil=0.874, runs=33
  L4_MCS0.1000_MS50: clusters= 3, noise= 5.06%, sil=0.619, runs=39
  L4_MCS0.0050_MS100: clusters=23, noise= 0.19%, sil=1.000, runs=45
  L4_MCS0.0065_MS100: clusters=23, noise= 0.19%, sil=1.000, runs=45
  L4_MCS0.0100_MS100: clusters=23, noise= 0.19%, sil=1.000, runs=45
  L4_MCS0.0150_MS100: clusters

### 5.3.3 HMM screening


In [9]:
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from hmmlearn.hmm import GaussianHMM

# Keep the public notebook output focused on explicit fit diagnostics.
logging.getLogger("hmmlearn.base").setLevel(logging.ERROR)

# ============================================================
# HAI 21.03 — HMM SCREENING: TRAIN-ONLY HMM SCREENING
#
# Direct 1-Hz sequence model on the final 8D train1 context.
# No windowing, no attack labels, no smoothing.
#
# Candidate state counts: K = 3 ... 16
# Five train-only random initialisations per K.
# Representative model for each K = valid fit with highest
# train log-likelihood.
# ============================================================

INPUT_DIR = Path("outputs/section_5_2/train_context_8d")
INPUT_FILE = INPUT_DIR / "train1_context_8d.csv"
FEATURE_ORDER_FILE = INPUT_DIR / "feature_order.csv"

OUTPUT_DIR = Path("outputs/section_5_3/hmm_screening")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

K_VALUES = list(range(3, 17))
SEEDS = [11, 23, 37, 53, 71]

COVARIANCE_TYPE = "diag"
N_ITER = 200
TOL = 1e-2
MIN_COVAR = 1e-3
TRANSMAT_PRIOR = 1.01
HMM_PARAMS = "tmc"       # start probabilities remain fixed
HMM_INIT_PARAMS = "tmc"  # start probabilities are not reinitialised
IMPLEMENTATION = "log"

SILHOUETTE_SAMPLE_N = 5_000
SILHOUETTE_RANDOM_STATE = 42


# ============================================================
# HELPERS
# ============================================================

def q(x, p):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.quantile(x, p)) if len(x) else np.nan


def normalized_entropy(counts):
    counts = np.asarray(counts, dtype=float)
    if counts.sum() <= 0:
        return np.nan
    p = counts[counts > 0] / counts.sum()
    return float(-np.sum(p * np.log(p)) / np.log(len(p))) if len(p) > 1 else 0.0


def effective_state_count(counts):
    counts = np.asarray(counts, dtype=float)
    if counts.sum() <= 0:
        return np.nan
    p = counts[counts > 0] / counts.sum()
    return float(np.exp(-np.sum(p * np.log(p))))


def build_native_runs(labels):
    labels = np.asarray(labels, dtype=int)
    starts = np.r_[0, np.flatnonzero(labels[1:] != labels[:-1]) + 1]
    ends = np.r_[starts[1:] - 1, len(labels) - 1]
    return pd.DataFrame({
        "run_index": np.arange(1, len(starts) + 1),
        "state": labels[starts],
        "start_row": starts,
        "end_row": ends,
        "duration_sec": ends - starts + 1,
    })


def get_diag_covars(model):
    covars = np.asarray(model.covars_, dtype=float)
    if covars.ndim == 2:
        return covars
    if covars.ndim == 3:
        return np.diagonal(covars, axis1=1, axis2=2)
    raise ValueError(f"Unexpected covariance shape: {covars.shape}")


def validate_model(model):
    startprob = np.asarray(model.startprob_, dtype=float)
    transmat = np.asarray(model.transmat_, dtype=float)
    means = np.asarray(model.means_, dtype=float)
    variances = get_diag_covars(model)

    if not np.isfinite(startprob).all() or not np.isclose(startprob.sum(), 1.0, atol=1e-8):
        raise FloatingPointError("Invalid start probabilities.")
    if not np.isfinite(transmat).all() or not np.allclose(transmat.sum(axis=1), 1.0, atol=1e-8):
        raise FloatingPointError("Invalid transition matrix.")
    if not np.isfinite(means).all():
        raise FloatingPointError("Non-finite emission means.")
    if not np.isfinite(variances).all() or np.any(variances <= 0):
        raise FloatingPointError("Invalid emission variances.")


def n_free_parameters(K, D):
    # startprob_ is fixed; transition rows have K-1 free values each.
    return K * (K - 1) + 2 * K * D


# ============================================================
# 1. LOAD FINAL TRAIN1 8D CONTEXT
# ============================================================

df = pd.read_csv(INPUT_FILE)
feature_order = pd.read_csv(FEATURE_ORDER_FILE)

if "model_column" not in feature_order.columns:
    raise ValueError("feature_order.csv must contain a 'model_column' column.")

MODEL_COLS = feature_order.sort_values("position")["model_column"].tolist()
if len(MODEL_COLS) != 8:
    raise ValueError(f"Expected 8 modelling features, found {len(MODEL_COLS)}.")

missing = [c for c in ["time"] + MODEL_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in train1_context_8d.csv: {missing}")

df["time"] = pd.to_datetime(df["time"], errors="coerce")
if df["time"].isna().any():
    raise ValueError("Invalid timestamps found.")

step = df["time"].diff().dt.total_seconds()
if (step.iloc[1:] != 1.0).any():
    raise ValueError("HMM screening expects one continuous 1-Hz train1 sequence.")

X = df[MODEL_COLS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
if not np.isfinite(X).all():
    raise ValueError("Non-finite values found in HMM input.")

N, D = X.shape
print(f"Rows     : {N:,}")
print(f"Features : {D}")
print(f"K grid   : {K_VALUES}")
print(f"Seeds    : {SEEDS}")


# ============================================================
# 2. SCREENING
# ============================================================

fit_rows = []
screening_rows = []
state_rows = []
run_tables = []
mean_rows = []
variance_rows = []
startprob_rows = []
transition_rows = []

labels_out = pd.DataFrame({
    "row_id": df["row_id"] if "row_id" in df.columns else np.arange(N),
    "time": df["time"],
})

for K in K_VALUES:
    print(f"\nHMM K={K}")

    best_model = None
    best_seed = None
    best_loglik = -np.inf
    successful_logliks = []

    for seed in SEEDS:
        model = GaussianHMM(
            n_components=K,
            covariance_type=COVARIANCE_TYPE,
            n_iter=N_ITER,
            tol=TOL,
            min_covar=MIN_COVAR,
            transmat_prior=TRANSMAT_PRIOR,
            random_state=seed,
            algorithm="viterbi",
            params=HMM_PARAMS,
            init_params=HMM_INIT_PARAMS,
            implementation=IMPLEMENTATION,
            verbose=False,
        )

        # Fixed uniformly and excluded from EM updates.
        model.startprob_ = np.full(K, 1.0 / K, dtype=float)

        warning_text = ""
        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                model.fit(X)
                warning_text = " | ".join(str(w.message) for w in caught)

            validate_model(model)
            loglik = float(model.score(X))
            if not np.isfinite(loglik):
                raise FloatingPointError("Non-finite train log-likelihood.")

            labels = np.asarray(model.predict(X), dtype=int)
            counts = np.bincount(labels, minlength=K)
            runs = build_native_runs(labels)

            fit_rows.append({
                "K": K,
                "seed": seed,
                "status": "success",
                "log_likelihood": loglik,
                "log_likelihood_per_tick": loglik / N,
                "converged": bool(model.monitor_.converged),
                "n_iterations": int(model.monitor_.iter),
                "n_states_used": int((counts > 0).sum()),
                "min_state_share": float(counts.min() / N),
                "max_state_share": float(counts.max() / N),
                "n_native_runs": len(runs),
                "median_run_sec": q(runs["duration_sec"], 0.50),
                "warning": warning_text,
                "error": "",
            })

            successful_logliks.append(loglik)
            if loglik > best_loglik:
                best_loglik = loglik
                best_seed = seed
                best_model = model

            print(
                f"  seed={seed}: logL/tick={loglik / N:.6f}, "
                f"runs={len(runs)}, min_share={counts.min() / N:.4%}"
            )

        except Exception as exc:
            fit_rows.append({
                "K": K,
                "seed": seed,
                "status": "failed",
                "log_likelihood": np.nan,
                "log_likelihood_per_tick": np.nan,
                "converged": False,
                "n_iterations": int(getattr(getattr(model, "monitor_", None), "iter", 0)),
                "n_states_used": np.nan,
                "min_state_share": np.nan,
                "max_state_share": np.nan,
                "n_native_runs": np.nan,
                "median_run_sec": np.nan,
                "warning": warning_text,
                "error": f"{type(exc).__name__}: {exc}",
            })
            print(f"  seed={seed}: FAILED — {type(exc).__name__}: {exc}")

    if best_model is None:
        screening_rows.append({
            "K": K,
            "status": "all_seeds_failed",
            "best_seed": np.nan,
            "n_successful_seeds": 0,
            "n_failed_seeds": len(SEEDS),
        })
        continue

    model = best_model
    labels = np.asarray(model.predict(X), dtype=int)
    posterior = model.predict_proba(X)
    posterior_max = posterior.max(axis=1)
    posterior_argmax = posterior.argmax(axis=1)

    counts = np.bincount(labels, minlength=K)
    shares = counts / N
    runs = build_native_runs(labels)
    durations = runs["duration_sec"].to_numpy(dtype=float)

    loglik = float(model.score(X))
    p = n_free_parameters(K, D)
    aic = 2 * p - 2 * loglik
    bic = np.log(N) * p - 2 * loglik

    # Internal geometry is diagnostic only.
    if (counts > 0).sum() >= 2:
        try:
            sil = float(silhouette_score(
                X, labels,
                sample_size=min(SILHOUETTE_SAMPLE_N, N),
                random_state=SILHOUETTE_RANDOM_STATE,
            ))
        except Exception:
            sil = np.nan
        try:
            db = float(davies_bouldin_score(X, labels))
        except Exception:
            db = np.nan
        try:
            ch = float(calinski_harabasz_score(X, labels))
        except Exception:
            ch = np.nan
    else:
        sil = db = ch = np.nan

    logliks = np.asarray(successful_logliks, dtype=float)
    screening_rows.append({
        "K": K,
        "status": "success",
        "best_seed": best_seed,
        "n_successful_seeds": len(logliks),
        "n_failed_seeds": len(SEEDS) - len(logliks),
        "seed_loglik_range": float(logliks.max() - logliks.min()),
        "seed_loglik_std": float(logliks.std(ddof=0)),
        "log_likelihood": loglik,
        "log_likelihood_per_tick": loglik / N,
        "AIC": aic,
        "BIC": bic,
        "n_states_used": int((counts > 0).sum()),
        "min_state_share": float(shares.min()),
        "max_state_share": float(shares.max()),
        "normalized_state_entropy": normalized_entropy(counts),
        "effective_n_states": effective_state_count(counts),
        "n_native_runs": len(runs),
        "switching_density": float((len(runs) - 1) / (N - 1)),
        "run_duration_min": float(durations.min()),
        "run_duration_p10": q(durations, 0.10),
        "run_duration_median": q(durations, 0.50),
        "run_duration_p90": q(durations, 0.90),
        "run_duration_max": float(durations.max()),
        "share_runs_le_4s": float((durations <= 4).mean()),
        "share_runs_le_8s": float((durations <= 8).mean()),
        "share_runs_le_16s": float((durations <= 16).mean()),
        "share_ticks_in_runs_le_4s": float(durations[durations <= 4].sum() / N),
        "share_ticks_in_runs_le_8s": float(durations[durations <= 8].sum() / N),
        "share_ticks_in_runs_le_16s": float(durations[durations <= 16].sum() / N),
        "posterior_max_mean": float(posterior_max.mean()),
        "posterior_max_median": q(posterior_max, 0.50),
        "posterior_max_p10": q(posterior_max, 0.10),
        "viterbi_posterior_agreement": float(np.mean(posterior_argmax == labels)),
        "silhouette_sample": sil,
        "davies_bouldin": db,
        "calinski_harabasz": ch,
        "converged": bool(model.monitor_.converged),
        "n_iterations": int(model.monitor_.iter),
    })

    # Save exact best-model native labels for later reproducibility checks.
    labels_out[f"HMM{K}"] = labels

    # Native runs from the representative model.
    runs.insert(0, "best_seed", best_seed)
    runs.insert(0, "K", K)
    runs["start_time"] = df.loc[runs["start_row"].to_numpy(), "time"].to_numpy()
    runs["end_time"] = df.loc[runs["end_row"].to_numpy(), "time"].to_numpy()
    run_tables.append(runs)

    means = np.asarray(model.means_, dtype=float)
    variances = get_diag_covars(model)
    transmat = np.asarray(model.transmat_, dtype=float)

    # State-level audit and compact parameter exports.
    for state in range(K):
        state_runs = runs[runs["state"] == state]
        state_durations = state_runs["duration_sec"].to_numpy(dtype=float)
        p_self = float(transmat[state, state])

        state_rows.append({
            "K": K,
            "best_seed": best_seed,
            "state": state,
            "n_ticks": int(counts[state]),
            "share": float(shares[state]),
            "n_native_runs": len(state_runs),
            "run_duration_median": q(state_durations, 0.50),
            "run_duration_max": float(state_durations.max()) if len(state_durations) else np.nan,
            "self_transition_probability": p_self,
            "expected_dwell_sec": float(1.0 / (1.0 - p_self)) if p_self < 1.0 else np.inf,
            "own_posterior_mean": float(posterior[labels == state, state].mean()) if counts[state] else np.nan,
        })

        mean_row = {"K": K, "best_seed": best_seed, "state": state}
        variance_row = {"K": K, "best_seed": best_seed, "state": state}
        for j, feature in enumerate(MODEL_COLS):
            mean_row[feature] = float(means[state, j])
            variance_row[feature] = float(variances[state, j])
        mean_rows.append(mean_row)
        variance_rows.append(variance_row)

        startprob_rows.append({
            "K": K,
            "best_seed": best_seed,
            "state": state,
            "start_probability": float(model.startprob_[state]),
        })

        for to_state in range(K):
            transition_rows.append({
                "K": K,
                "best_seed": best_seed,
                "from_state": state,
                "to_state": to_state,
                "probability": float(transmat[state, to_state]),
            })

    print(f"  -> representative seed: {best_seed}, native runs: {len(runs)}")


# ============================================================
# 3. SAVE RESULTS
# ============================================================

pd.DataFrame(fit_rows).to_csv(OUTPUT_DIR / "fit_diagnostics_all_seeds.csv", index=False)
pd.DataFrame(screening_rows).to_csv(OUTPUT_DIR / "hmm_screening.csv", index=False)
pd.DataFrame(state_rows).to_csv(OUTPUT_DIR / "state_summary.csv", index=False)
pd.concat(run_tables, ignore_index=True).to_csv(OUTPUT_DIR / "native_runs.csv", index=False)
pd.DataFrame(mean_rows).to_csv(OUTPUT_DIR / "emission_means.csv", index=False)
pd.DataFrame(variance_rows).to_csv(OUTPUT_DIR / "emission_variances.csv", index=False)
pd.DataFrame(startprob_rows).to_csv(OUTPUT_DIR / "start_probabilities.csv", index=False)
pd.DataFrame(transition_rows).to_csv(OUTPUT_DIR / "transition_probabilities.csv", index=False)
labels_out.to_csv(OUTPUT_DIR / "labels_best_models.csv", index=False)

pd.DataFrame([
    {"parameter": "K_values", "value": "3-16"},
    {"parameter": "seeds", "value": ",".join(map(str, SEEDS))},
    {"parameter": "covariance_type", "value": COVARIANCE_TYPE},
    {"parameter": "n_iter", "value": N_ITER},
    {"parameter": "tol", "value": TOL},
    {"parameter": "min_covar", "value": MIN_COVAR},
    {"parameter": "transmat_prior", "value": TRANSMAT_PRIOR},
    {"parameter": "startprob", "value": "fixed_uniform"},
    {"parameter": "params", "value": HMM_PARAMS},
    {"parameter": "init_params", "value": HMM_INIT_PARAMS},
    {"parameter": "implementation", "value": IMPLEMENTATION},
    {"parameter": "feature_order", "value": ",".join(MODEL_COLS)},
]).to_csv(OUTPUT_DIR / "settings.csv", index=False)

print("\nHMM screening complete.")
print(f"Requested fits: {len(K_VALUES) * len(SEEDS)}")
print(f"Output directory: {OUTPUT_DIR}")

Rows     : 216,001
Features : 8
K grid   : [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
Seeds    : [11, 23, 37, 53, 71]

HMM K=3


  seed=11: logL/tick=30.958069, runs=43, min_share=11.8824%
  seed=23: logL/tick=29.192989, runs=35, min_share=16.5485%
  seed=37: logL/tick=29.192989, runs=35, min_share=16.5485%


  seed=53: logL/tick=29.192989, runs=35, min_share=16.5485%
  seed=71: logL/tick=29.192989, runs=35, min_share=16.5485%
  -> representative seed: 11, native runs: 43

HMM K=4
  seed=11: logL/tick=29.811834, runs=35, min_share=4.9324%
  seed=23: logL/tick=31.576069, runs=50, min_share=0.0657%


  seed=37: logL/tick=31.180108, runs=43, min_share=4.9324%


  seed=53: logL/tick=29.191837, runs=38, min_share=5.0264%
  seed=71: logL/tick=33.125626, runs=45, min_share=0.1731%
  -> representative seed: 71, native runs: 45

HMM K=5
  seed=11: logL/tick=32.572569, runs=54, min_share=0.0028%
  seed=23: logL/tick=33.744471, runs=45, min_share=0.1731%


...

  seed=37: logL/tick=36.551433, runs=29024, min_share=0.0074%


  seed=53: logL/tick=36.371506, runs=25345, min_share=0.0000%
  seed=71: logL/tick=34.480766, runs=57, min_share=0.0000%
  -> representative seed: 37, native runs: 29024

HMM K=16


  seed=11: logL/tick=34.415196, runs=124555, min_share=0.0000%
  seed=23: logL/tick=34.249846, runs=149854, min_share=0.0014%
  seed=37: logL/tick=33.943031, runs=35419, min_share=0.0000%


  seed=53: logL/tick=35.019182, runs=45129, min_share=0.0009%


  seed=71: logL/tick=35.681318, runs=8769, min_share=0.0000%
  -> representative seed: 71, native runs: 8769

HMM screening complete.
Requested fits: 70
Output directory: outputs/section_5_3/hmm_screening


### 5.3.4 Fit and verify the retained k-means designs


In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# HAI 21.03 — RETAINED K-MEANS ARTIFACTS
#
# Purpose:
#   Convert the retained train-only k-means screening solutions
#   into fixed artifacts for later out-of-sample assignment.
#
# Methodological contract:
#   - no refitting
#   - centroids come directly from k-means screening screening
#   - train windows are reassigned by nearest Euclidean centroid
#   - reassigned labels must reproduce the audited train labels
#     exactly before an artifact is accepted
#
# Retained designs:
#   KM4_8, KM7_8, KM13_8, KM13_16
#
# Inputs:
#   outputs/section_5_2/multiscale_windows/
#       train1_window_8s.csv
#       train1_window_16s.csv
#       window_feature_order.csv
#
#   outputs/section_5_3/kmeans_screening/
#       model_summary.csv
#       cluster_centers.csv
#       integrity_checks.csv
#       labels_8s.csv
#       labels_16s.csv
#
# Output:
#   outputs/section_5_3/retained_kmeans/
#       retained_designs.csv
#       feature_order.csv
#       reproduction_checks.csv
#       KM4_8/
#           centroids.csv
#           metadata.csv
#           train_native_labels.csv
#       ...
# ============================================================

WINDOW_DIR = Path("outputs/section_5_2/multiscale_windows")
KM_DIR = Path("outputs/section_5_3/kmeans_screening")
OUTPUT_DIR = Path("outputs/section_5_3/retained_kmeans")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RETAINED = [
    {"design_id": "KM4_8", "window_sec": 8, "K": 4},
    {"design_id": "KM7_8", "window_sec": 8, "K": 7},
    {"design_id": "KM13_8", "window_sec": 8, "K": 13},
    {"design_id": "KM13_16", "window_sec": 16, "K": 13},
]

# ============================================================
# 1. FEATURE CONTRACT
# ============================================================

feature_order = pd.read_csv(WINDOW_DIR / "window_feature_order.csv")

required_feature_cols = {
    "position",
    "model_column",
    "source_variable",
    "role",
    "aggregation",
}
missing = required_feature_cols - set(feature_order.columns)
if missing:
    raise ValueError(
        "window_feature_order.csv is missing required columns: "
        f"{sorted(missing)}"
    )

feature_order = feature_order.sort_values("position").reset_index(drop=True)
MODEL_COLS = feature_order["model_column"].tolist()

if feature_order["position"].tolist() != list(range(1, len(feature_order) + 1)):
    raise ValueError("Feature positions are not contiguous from 1 to n.")
if len(MODEL_COLS) != 8 or feature_order["model_column"].duplicated().any():
    raise ValueError("Expected exactly eight unique model features.")
if not feature_order["aggregation"].eq("mean").all():
    raise ValueError("All retained k-means features must use mean aggregation.")

feature_order.to_csv(OUTPUT_DIR / "feature_order.csv", index=False)

# ============================================================
# 2. LOAD SCREENING ARTIFACTS
# ============================================================

model_summary = pd.read_csv(KM_DIR / "model_summary.csv")
centers_all = pd.read_csv(KM_DIR / "cluster_centers.csv")
integrity = pd.read_csv(KM_DIR / "integrity_checks.csv")

required_summary = {"design_id", "window_sec", "K", "n_windows", "inertia"}
required_centers = {"design_id", "window_sec", "K", "cluster", *MODEL_COLS}
required_integrity = {"design_id", "status"}

if required_summary - set(model_summary.columns):
    raise ValueError("model_summary.csv does not match the expected k-means screening schema.")
if required_centers - set(centers_all.columns):
    raise ValueError("cluster_centers.csv does not match the expected k-means screening schema.")
if required_integrity - set(integrity.columns):
    raise ValueError("integrity_checks.csv does not match the expected k-means screening schema.")

# ============================================================
# 3. EXPORT RETAINED ARTIFACTS AND VERIFY TRAIN REPRODUCTION
# ============================================================

manifest_rows = []
check_rows = []

for spec in RETAINED:
    design_id = spec["design_id"]
    L = spec["window_sec"]
    K = spec["K"]

    print("\n" + "=" * 72)
    print(f"RETAINED K-MEANS — {design_id}")
    print("=" * 72)

    # Screening metadata must identify exactly one audited model.
    summary_row = model_summary.loc[model_summary["design_id"].eq(design_id)]
    if len(summary_row) != 1:
        raise ValueError(f"{design_id}: expected exactly one row in model_summary.csv.")
    summary_row = summary_row.iloc[0]

    if int(summary_row["window_sec"]) != L or int(summary_row["K"]) != K:
        raise ValueError(f"{design_id}: screening metadata does not match retained specification.")

    integrity_row = integrity.loc[integrity["design_id"].eq(design_id)]
    if len(integrity_row) != 1 or str(integrity_row.iloc[0]["status"]).upper() != "PASS":
        raise ValueError(f"{design_id}: k-means screening integrity status is not PASS.")

    # Load the same train windows used in screening.
    windows = pd.read_csv(WINDOW_DIR / f"train1_window_{L}s.csv")
    required_window_cols = {
        "window_index",
        "window_start_row",
        "window_end_row",
        "window_start_time",
        "window_end_time",
        *MODEL_COLS,
    }
    missing_window = required_window_cols - set(windows.columns)
    if missing_window:
        raise ValueError(f"{design_id}: window file is missing {sorted(missing_window)}")

    X = windows[MODEL_COLS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(X).all():
        raise ValueError(f"{design_id}: non-finite values found in train window features.")

    if len(windows) != int(summary_row["n_windows"]):
        raise ValueError(f"{design_id}: window count differs from k-means screening screening summary.")

    # Load exact audited labels saved by k-means screening.
    labels_file = pd.read_csv(KM_DIR / f"labels_{L}s.csv")
    if design_id not in labels_file.columns:
        raise ValueError(f"{design_id}: audited labels are missing from labels_{L}s.csv.")
    if len(labels_file) != len(windows):
        raise ValueError(f"{design_id}: label count differs from train window count.")

    saved_labels = pd.to_numeric(labels_file[design_id], errors="coerce")
    if saved_labels.isna().any():
        raise ValueError(f"{design_id}: audited labels contain missing/non-numeric values.")
    saved_labels = saved_labels.to_numpy(dtype=np.int64)

    # Load retained centroids in exact cluster-id order.
    center_df = centers_all.loc[centers_all["design_id"].eq(design_id)].copy()
    center_df["cluster"] = pd.to_numeric(center_df["cluster"], errors="raise").astype(int)
    center_df = center_df.sort_values("cluster").reset_index(drop=True)

    if len(center_df) != K:
        raise ValueError(f"{design_id}: expected {K} centroids, found {len(center_df)}.")
    if center_df["cluster"].tolist() != list(range(K)):
        raise ValueError(f"{design_id}: centroid cluster IDs are not 0..K-1.")

    centers = center_df[MODEL_COLS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(centers).all():
        raise ValueError(f"{design_id}: centroid table contains non-finite values.")

    # Fixed out-of-sample rule: nearest Euclidean centroid.
    dist_sq = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    reproduced_labels = np.argmin(dist_sq, axis=1).astype(np.int64)

    mismatch_mask = reproduced_labels != saved_labels
    n_mismatch = int(mismatch_mask.sum())
    exact_agreement = float((~mismatch_mask).mean())

    # Independent reconstruction checks.
    reconstructed_inertia = float(dist_sq[np.arange(len(X)), reproduced_labels].sum())
    screening_inertia = float(summary_row["inertia"])
    inertia_matches = bool(
        np.isclose(reconstructed_inertia, screening_inertia, rtol=1e-10, atol=1e-8)
    )

    # KMeans.cluster_centers_ are the authoritative fitted centroids.
    # Do NOT require them to equal the arithmetic means of the final
    # predict-consistent labels: scikit-learn may reassign labels after
    # the last optimization step when stopping by tolerance.
    empirical_centers = np.vstack(
        [X[reproduced_labels == cluster].mean(axis=0) for cluster in range(K)]
    )
    max_abs_centroid_mean_difference = float(
        np.max(np.abs(empirical_centers - centers))
    )

    all_states_occupied = bool(
        np.array_equal(np.unique(reproduced_labels), np.arange(K, dtype=np.int64))
    )

    status = (
        "PASS"
        if n_mismatch == 0 and inertia_matches and all_states_occupied
        else "FAIL"
    )

    if status != "PASS":
        raise RuntimeError(
            f"{design_id}: retained artifact failed reproduction check "
            f"(mismatches={n_mismatch}, inertia_matches={inertia_matches}, "
            f"all_states_occupied={all_states_occupied})."
        )

    # --------------------------------------------------------
    # Save only the artifacts required for later assignment.
    # --------------------------------------------------------
    design_dir = OUTPUT_DIR / design_id
    design_dir.mkdir(parents=True, exist_ok=True)

    centroid_export = center_df[["cluster"] + MODEL_COLS].copy()
    centroid_export.to_csv(design_dir / "centroids.csv", index=False)

    train_labels = windows[
        [
            "window_index",
            "window_start_row",
            "window_end_row",
            "window_start_time",
            "window_end_time",
        ]
    ].copy()
    train_labels["native_state"] = reproduced_labels
    train_labels.to_csv(design_dir / "train_native_labels.csv", index=False)

    metadata = pd.DataFrame(
        [
            {"parameter": "design_id", "value": design_id},
            {"parameter": "family", "value": "kmeans"},
            {"parameter": "window_sec", "value": L},
            {"parameter": "K", "value": K},
            {"parameter": "distance", "value": "euclidean"},
            {"parameter": "assignment_rule", "value": "nearest fixed train centroid"},
            {"parameter": "alignment", "value": "trailing_end"},
            {"parameter": "stride_sec", "value": 1},
            {"parameter": "feature_order", "value": ",".join(MODEL_COLS)},
            {"parameter": "source", "value": "k-means screening train1 screening artifacts"},
            {"parameter": "refit_in_cell09", "value": False},
        ]
    )
    metadata.to_csv(design_dir / "metadata.csv", index=False)

    check_rows.append(
        {
            "design_id": design_id,
            "window_sec": L,
            "K": K,
            "n_train_windows": len(X),
            "n_label_mismatches": n_mismatch,
            "exact_label_agreement": exact_agreement,
            "screening_inertia": screening_inertia,
            "reconstructed_inertia": reconstructed_inertia,
            "inertia_matches": inertia_matches,
            "max_abs_centroid_mean_difference": max_abs_centroid_mean_difference,
            "all_states_occupied": all_states_occupied,
            "status": status,
        }
    )

    manifest_rows.append(
        {
            "design_id": design_id,
            "family": "kmeans",
            "window_sec": L,
            "K": K,
            "assignment_rule": "nearest fixed train centroid",
            "artifact_dir": str(design_dir),
            "train_reproduction_status": status,
        }
    )

    print(
        f"PASS: {len(X):,} train windows, "
        f"label agreement={exact_agreement:.6f}, "
        f"max centroid-vs-final-mean difference={max_abs_centroid_mean_difference:.3e}"
    )

# ============================================================
# 4. CONSOLIDATED MANIFEST AND CHECKS
# ============================================================

retained_designs = pd.DataFrame(manifest_rows)
reproduction_checks = pd.DataFrame(check_rows)

retained_designs.to_csv(OUTPUT_DIR / "retained_designs.csv", index=False)
reproduction_checks.to_csv(OUTPUT_DIR / "reproduction_checks.csv", index=False)

print("\nRetained k-means artifacts:")
display(retained_designs)

print("\nTrain reproduction checks:")
display(reproduction_checks)

if not reproduction_checks["status"].eq("PASS").all():
    raise RuntimeError("At least one retained k-means artifact failed validation.")

print("\nRETAINED K-MEANS FITTING COMPLETE — all retained k-means artifacts reproduced train labels exactly.")


RETAINED K-MEANS — KM4_8
PASS: 215,994 train windows, label agreement=1.000000, max centroid-vs-final-mean difference=6.649e-13

RETAINED K-MEANS — KM7_8
PASS: 215,994 train windows, label agreement=1.000000, max centroid-vs-final-mean difference=1.078e-12

RETAINED K-MEANS — KM13_8
PASS: 215,994 train windows, label agreement=1.000000, max centroid-vs-final-mean difference=8.761e-05

RETAINED K-MEANS — KM13_16
PASS: 215,986 train windows, label agreement=1.000000, max centroid-vs-final-mean difference=4.609e-13

Retained k-means artifacts:


,design_id,family,window_sec,K,assignment_rule,artifact_dir,train_reproduction_status
0,KM4_8,kmeans,8,4,nearest fixed train centroid,outputs/section_5_3/retained_kmeans\KM4_8,PASS
1,KM7_8,kmeans,8,7,nearest fixed train centroid,outputs/section_5_3/retained_kmeans\KM7_8,PASS
2,KM13_8,kmeans,8,13,nearest fixed train centroid,outputs/section_5_3/retained_kmeans\KM13_8,PASS
3,KM13_16,kmeans,16,13,nearest fixed train centroid,outputs/section_5_3/retained_kmeans\KM13_16,PASS



Train reproduction checks:


,design_id,window_sec,K,n_train_windows,n_label_mismatches,exact_label_agreement,screening_inertia,reconstructed_inertia,inertia_matches,max_abs_centroid_mean_difference,all_states_occupied,status
0,KM4_8,8,4,215994,0,1.0,16583.533027,16583.533027,True,6.649126e-13,True,PASS
1,KM7_8,8,7,215994,0,1.0,3817.169358,3817.169358,True,1.077693e-12,True,PASS
2,KM13_8,8,13,215994,0,1.0,860.328803,860.328803,True,8.760820e-05,True,PASS
3,KM13_16,16,13,215986,0,1.0,806.495638,806.495638,True,4.608536e-13,True,PASS



RETAINED K-MEANS FITTING COMPLETE — all retained k-means artifacts reproduced train labels exactly.


### 5.3.5 Fit and verify the retained HDBSCAN designs


In [12]:
from pathlib import Path
import importlib.metadata
import warnings

import hdbscan
import joblib
import numpy as np
import pandas as pd
from hdbscan.prediction import approximate_predict
from sklearn.metrics import adjusted_rand_score

# ============================================================
# HAI 21.03 — RETAINED HDBSCAN FITTING
# RETAINED HDBSCAN MODELS + TRAIN REPRODUCTION CHECK
# ============================================================

WINDOW_DIR = Path("outputs/section_5_2/multiscale_windows")
SCREEN_DIR = Path("outputs/section_5_3/hdbscan_screening")
OUTPUT_DIR = Path("outputs/section_5_3/retained_hdbscan")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RETAINED = [
    ("HDB23_8",  8, 0.02),
    ("HDB18_8",  8, 0.03),
    ("HDB15_8",  8, 0.04),
    ("HDB18_16", 16, 0.03),
]
MIN_SAMPLES = 100

feature_order = pd.read_csv(WINDOW_DIR / "window_feature_order.csv").sort_values("position")
MODEL_COLS = feature_order["model_column"].tolist()
if len(MODEL_COLS) != 8 or len(set(MODEL_COLS)) != 8:
    raise ValueError("Expected exactly eight unique window model columns.")

screening = pd.read_csv(SCREEN_DIR / "screening_summary.csv")
cluster_screening = pd.read_csv(SCREEN_DIR / "cluster_summary.csv")

summary_rows = []
check_rows = []
warnings.filterwarnings("ignore", message=".*force_all_finite.*")

for design_id, L, mcs_share in RETAINED:
    print(f"\n{design_id}")

    df = pd.read_csv(WINDOW_DIR / f"train1_window_{L}s.csv")
    required = ["window_index", "window_start_row", "window_end_row",
                "window_start_time", "window_end_time"] + MODEL_COLS
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{design_id}: missing columns {missing}")

    X = df[MODEL_COLS].to_numpy(dtype=np.float64)
    if not np.isfinite(X).all():
        raise ValueError(f"{design_id}: non-finite model features.")

    n = len(X)
    min_cluster_size = max(2, round(mcs_share * n))

    s = screening[
        (screening["window_sec"] == L)
        & np.isclose(screening["mcs_share"], mcs_share)
        & (screening["min_samples"] == MIN_SAMPLES)
    ]
    if len(s) != 1:
        raise RuntimeError(f"{design_id}: matching screening row not unique.")
    s = s.iloc[0]

    cs = cluster_screening[
        (cluster_screening["window_sec"] == L)
        & np.isclose(cluster_screening["mcs_share"], mcs_share)
        & (cluster_screening["min_samples"] == MIN_SAMPLES)
    ]

    if int(s["min_cluster_size"]) != min_cluster_size:
        raise RuntimeError(f"{design_id}: min_cluster_size differs from HDBSCAN screening.")

    common = dict(
        min_cluster_size=min_cluster_size,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        cluster_selection_method="eom",
        approx_min_span_tree=True,
        core_dist_n_jobs=-1,
    )

    # Reference fit reproduces the exact HDBSCAN screening configuration.
    ref_model = hdbscan.HDBSCAN(**common, prediction_data=False)
    ref_labels = ref_model.fit_predict(X).astype(np.int64)

    # Deployment fit adds only the prediction structure needed for test data.
    model = hdbscan.HDBSCAN(**common, prediction_data=True)
    labels = model.fit_predict(X).astype(np.int64)

    states = np.unique(labels[labels >= 0])
    n_clusters = len(states)
    noise_share = float(np.mean(labels == -1))

    ari = float(adjusted_rand_score(ref_labels, labels))
    noise_mask_match = bool(np.array_equal(ref_labels == -1, labels == -1))
    n_clusters_match = n_clusters == int(s["n_clusters"])
    noise_share_match = bool(np.isclose(noise_share, s["noise_share"], atol=1e-12, rtol=0))

    fitted_counts = np.sort([(labels == k).sum() for k in states])
    screening_counts = np.sort(cs["n_windows"].to_numpy(dtype=np.int64))
    cluster_counts_match = bool(np.array_equal(fitted_counts, screening_counts))

    # Prediction-readiness smoke test; not used as a clustering criterion.
    idx = np.linspace(0, n - 1, min(1000, n), dtype=np.int64)
    pred_labels, pred_strengths = approximate_predict(model, X[idx])
    valid_labels = set(np.unique(pred_labels)).issubset(set(states.tolist()) | {-1})
    valid_strengths = bool(
        np.isfinite(pred_strengths).all()
        and np.all((pred_strengths >= 0) & (pred_strengths <= 1))
    )

    passed = (
        np.isclose(ari, 1.0, atol=1e-12, rtol=0)
        and noise_mask_match
        and n_clusters_match
        and noise_share_match
        and cluster_counts_match
        and valid_labels
        and valid_strengths
    )

    check_rows.append({
        "design_id": design_id,
        "reference_vs_prediction_fit_ari": ari,
        "noise_mask_match": noise_mask_match,
        "screening_n_clusters_match": n_clusters_match,
        "screening_noise_share_match": noise_share_match,
        "screening_cluster_counts_match": cluster_counts_match,
        "prediction_smoke_labels_valid": valid_labels,
        "prediction_smoke_strengths_valid": valid_strengths,
        "status": "PASS" if passed else "FAIL",
    })

    if not passed:
        raise RuntimeError(f"{design_id}: reproduction/deployment check failed.")

    design_dir = OUTPUT_DIR / design_id
    design_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, design_dir / "model.joblib", compress=3)

    out = df[["window_index", "window_start_row", "window_end_row",
              "window_start_time", "window_end_time"]].copy()
    out["native_state"] = labels
    out["membership_probability"] = model.probabilities_
    out.to_csv(design_dir / "train_native_labels.csv", index=False)

    pd.DataFrame([{
        "design_id": design_id,
        "family": "HDBSCAN",
        "window_sec": L,
        "mcs_share": mcs_share,
        "min_cluster_size": min_cluster_size,
        "min_samples": MIN_SAMPLES,
        "metric": "euclidean",
        "cluster_selection_method": "eom",
        "prediction_data": True,
        "n_train_windows": n,
        "n_native_clusters": n_clusters,
        "train_noise_share": noise_share,
        "feature_order": ",".join(MODEL_COLS),
        "hdbscan_version": importlib.metadata.version("hdbscan"),
    }]).to_csv(design_dir / "metadata.csv", index=False)

    summary_rows.append({
        "design_id": design_id,
        "window_sec": L,
        "mcs_share": mcs_share,
        "min_cluster_size": min_cluster_size,
        "min_samples": MIN_SAMPLES,
        "n_native_clusters": n_clusters,
        "train_noise_share": noise_share,
        "model_file": str(design_dir / "model.joblib"),
    })

    print(f"  PASS | clusters={n_clusters} | noise={100*noise_share:.3f}% | ARI={ari:.6f}")

pd.DataFrame(summary_rows).to_csv(OUTPUT_DIR / "retained_designs.csv", index=False)
pd.DataFrame(check_rows).to_csv(OUTPUT_DIR / "reproduction_checks.csv", index=False)
feature_order.to_csv(OUTPUT_DIR / "feature_order.csv", index=False)

print("\nretained HDBSCAN fitting complete: all retained HDBSCAN models are prediction-ready.")


HDB23_8
  PASS | clusters=23 | noise=0.219% | ARI=1.000000

HDB18_8
  PASS | clusters=18 | noise=0.195% | ARI=1.000000

HDB15_8
  PASS | clusters=15 | noise=0.192% | ARI=1.000000

HDB18_16
  PASS | clusters=18 | noise=0.231% | ARI=1.000000

retained HDBSCAN fitting complete: all retained HDBSCAN models are prediction-ready.


### 5.3.6 Fit and verify the retained HMM designs


In [13]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM

# ============================================================
# HAI 21.03 — RETAINED HMM FITTING: RETAINED HMM ARTIFACTS
#
# Reconstruct retained HMM4/HMM5/HMM8 directly from the
# representative train1 parameters saved by HMM screening.
# No refitting is performed here.
#
# Required checks:
#   1) exact Viterbi-label reproduction on train1,
#   2) reconstructed train log-likelihood matches HMM screening,
#   3) all probability/covariance parameters are valid.
# ============================================================

CONTEXT_DIR = Path("outputs/section_5_2/train_context_8d")
HMM_DIR = Path("outputs/section_5_3/hmm_screening")
OUTPUT_DIR = Path("outputs/section_5_3/retained_hmm")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RETAINED = {
    "HMM4": 4,
    "HMM5": 5,
    "HMM8": 8,
}

# These settings reproduce the inference configuration used in HMM screening.
COVARIANCE_TYPE = "diag"
MIN_COVAR = 1e-3
TRANSMAT_PRIOR = 1.01
IMPLEMENTATION = "log"
ALGORITHM = "viterbi"

# CSV round-trip can introduce tiny floating-point differences in score.
LOGLIK_RTOL = 1e-10
LOGLIK_ATOL = 1e-3


# ============================================================
# 1. LOAD TRAIN1 CONTEXT AND HMM SCREENING OUTPUTS
# ============================================================

context = pd.read_csv(CONTEXT_DIR / "train1_context_8d.csv")
feature_order = pd.read_csv(CONTEXT_DIR / "feature_order.csv")

if "model_column" not in feature_order.columns:
    raise ValueError("feature_order.csv must contain a 'model_column' column.")

MODEL_COLS = feature_order.sort_values("position")["model_column"].tolist()
if len(MODEL_COLS) != 8:
    raise ValueError(f"Expected 8 modelling features, found {len(MODEL_COLS)}.")

missing = [c for c in MODEL_COLS if c not in context.columns]
if missing:
    raise ValueError(f"Missing modelling columns in train1 context: {missing}")

X = context[MODEL_COLS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
if not np.isfinite(X).all():
    raise ValueError("Non-finite values found in train1 HMM input.")

screening = pd.read_csv(HMM_DIR / "hmm_screening.csv")
means_all = pd.read_csv(HMM_DIR / "emission_means.csv")
vars_all = pd.read_csv(HMM_DIR / "emission_variances.csv")
start_all = pd.read_csv(HMM_DIR / "start_probabilities.csv")
trans_all = pd.read_csv(HMM_DIR / "transition_probabilities.csv")
labels_all = pd.read_csv(HMM_DIR / "labels_best_models.csv")

required_label_cols = list(RETAINED)
missing_labels = [c for c in required_label_cols if c not in labels_all.columns]
if missing_labels:
    raise ValueError(f"labels_best_models.csv is missing columns: {missing_labels}")

if len(labels_all) != len(X):
    raise ValueError(
        f"Label/context length mismatch: labels={len(labels_all):,}, context={len(X):,}."
    )

# Preserve the same train timeline identifiers in exported native labels.
label_base = pd.DataFrame()
if "row_id" in labels_all.columns:
    label_base["row_id"] = labels_all["row_id"]
elif "row_id" in context.columns:
    label_base["row_id"] = context["row_id"]
else:
    label_base["row_id"] = np.arange(len(X))

if "time" in labels_all.columns:
    label_base["time"] = labels_all["time"]
elif "time" in context.columns:
    label_base["time"] = context["time"]


# ============================================================
# 2. RECONSTRUCT RETAINED MODELS AND VERIFY TRAIN REPRODUCTION
# ============================================================

check_rows = []
design_rows = []

for design, K in RETAINED.items():
    print(f"\n{design}")

    screen_row = screening.loc[(screening["K"] == K) & (screening["status"] == "success")]
    if len(screen_row) != 1:
        raise ValueError(f"{design}: expected one successful screening row for K={K}.")
    screen_row = screen_row.iloc[0]
    best_seed = int(screen_row["best_seed"])

    means_df = means_all.loc[means_all["K"] == K].sort_values("state")
    vars_df = vars_all.loc[vars_all["K"] == K].sort_values("state")
    start_df = start_all.loc[start_all["K"] == K].sort_values("state")
    trans_df = trans_all.loc[trans_all["K"] == K].sort_values(["from_state", "to_state"])

    if not (len(means_df) == len(vars_df) == len(start_df) == K):
        raise ValueError(f"{design}: incomplete state-parameter tables.")
    if len(trans_df) != K * K:
        raise ValueError(f"{design}: incomplete transition matrix.")

    # Ensure every retained parameter table refers to the representative seed.
    for name, table in {
        "means": means_df,
        "variances": vars_df,
        "start probabilities": start_df,
        "transition probabilities": trans_df,
    }.items():
        seeds = set(pd.to_numeric(table["best_seed"], errors="raise").astype(int))
        if seeds != {best_seed}:
            raise ValueError(
                f"{design}: {name} use seed(s) {sorted(seeds)}, expected {best_seed}."
            )

    means = means_df[MODEL_COLS].to_numpy(dtype=np.float64)
    variances = vars_df[MODEL_COLS].to_numpy(dtype=np.float64)
    startprob = start_df["start_probability"].to_numpy(dtype=np.float64)
    transmat = trans_df.pivot(
        index="from_state", columns="to_state", values="probability"
    ).sort_index().sort_index(axis=1).to_numpy(dtype=np.float64)

    # Parameter-integrity checks before model reconstruction.
    if means.shape != (K, len(MODEL_COLS)):
        raise ValueError(f"{design}: invalid means shape {means.shape}.")
    if variances.shape != means.shape:
        raise ValueError(f"{design}: invalid variance shape {variances.shape}.")
    if not np.isfinite(means).all():
        raise ValueError(f"{design}: non-finite emission means.")
    if not np.isfinite(variances).all() or np.any(variances <= 0):
        raise ValueError(f"{design}: emission variances must be finite and > 0.")
    if not np.isfinite(startprob).all() or not np.isclose(startprob.sum(), 1.0, atol=1e-10):
        raise ValueError(f"{design}: invalid start probabilities.")
    if not np.isfinite(transmat).all() or not np.allclose(
        transmat.sum(axis=1), 1.0, atol=1e-10
    ):
        raise ValueError(f"{design}: invalid transition matrix.")

    # Reconstruct directly from fitted parameters. No fit() call is made.
    model = GaussianHMM(
        n_components=K,
        covariance_type=COVARIANCE_TYPE,
        min_covar=MIN_COVAR,
        transmat_prior=TRANSMAT_PRIOR,
        algorithm=ALGORITHM,
        params="",
        init_params="",
        implementation=IMPLEMENTATION,
        verbose=False,
    )
    model.startprob_ = startprob.copy()
    model.transmat_ = transmat.copy()
    model.means_ = means.copy()
    model.covars_ = variances.copy()

    # hmmlearn derives n_features from observations during normal fitting;
    # set it explicitly because this model is reconstructed without fit().
    model.n_features = len(MODEL_COLS)

    expected_labels = pd.to_numeric(labels_all[design], errors="raise").to_numpy(dtype=int)
    reconstructed_labels = np.asarray(model.predict(X), dtype=int)

    mismatch_count = int(np.sum(reconstructed_labels != expected_labels))
    exact_agreement = float(np.mean(reconstructed_labels == expected_labels))

    reconstructed_loglik = float(model.score(X))
    expected_loglik = float(screen_row["log_likelihood"])
    loglik_abs_diff = abs(reconstructed_loglik - expected_loglik)
    loglik_matches = bool(
        np.isclose(
            reconstructed_loglik,
            expected_loglik,
            rtol=LOGLIK_RTOL,
            atol=LOGLIK_ATOL,
        )
    )

    n_states_expected = int(screen_row["n_states_used"])
    n_states_reconstructed = int(len(np.unique(reconstructed_labels)))
    state_usage_matches = n_states_reconstructed == n_states_expected

    passed = (
        mismatch_count == 0
        and loglik_matches
        and state_usage_matches
    )

    check_rows.append({
        "design": design,
        "K": K,
        "best_seed": best_seed,
        "n_ticks": len(X),
        "label_mismatches": mismatch_count,
        "exact_label_agreement": exact_agreement,
        "expected_log_likelihood": expected_loglik,
        "reconstructed_log_likelihood": reconstructed_loglik,
        "log_likelihood_abs_diff": loglik_abs_diff,
        "log_likelihood_matches": loglik_matches,
        "expected_states_used": n_states_expected,
        "reconstructed_states_used": n_states_reconstructed,
        "state_usage_matches": state_usage_matches,
        "status": "PASS" if passed else "FAIL",
    })

    if not passed:
        raise RuntimeError(
            f"{design}: retained HMM failed reproduction check "
            f"(mismatches={mismatch_count}, "
            f"loglik_matches={loglik_matches}, "
            f"state_usage_matches={state_usage_matches})."
        )

    # --------------------------------------------------------
    # Export deployable retained artifact.
    # --------------------------------------------------------
    design_dir = OUTPUT_DIR / design
    design_dir.mkdir(parents=True, exist_ok=True)

    joblib.dump(model, design_dir / "model.joblib")

    means_df.to_csv(design_dir / "emission_means.csv", index=False)
    vars_df.to_csv(design_dir / "emission_variances.csv", index=False)
    start_df.to_csv(design_dir / "start_probabilities.csv", index=False)
    trans_df.to_csv(design_dir / "transition_probabilities.csv", index=False)

    train_labels = label_base.copy()
    train_labels["native_state"] = reconstructed_labels
    train_labels.to_csv(design_dir / "train_native_labels.csv", index=False)

    metadata = pd.DataFrame([
        {"parameter": "design", "value": design},
        {"parameter": "family", "value": "HMM"},
        {"parameter": "K", "value": K},
        {"parameter": "best_seed_from_screening", "value": best_seed},
        {"parameter": "input_resolution", "value": "1 Hz"},
        {"parameter": "input_representation", "value": "train-derived 8D tick-level context"},
        {"parameter": "covariance_type", "value": COVARIANCE_TYPE},
        {"parameter": "algorithm", "value": ALGORITHM},
        {"parameter": "implementation", "value": IMPLEMENTATION},
        {"parameter": "min_covar", "value": MIN_COVAR},
        {"parameter": "transmat_prior_used_during_training", "value": TRANSMAT_PRIOR},
        {"parameter": "deployment_refit", "value": "none"},
        {"parameter": "feature_order", "value": ",".join(MODEL_COLS)},
    ])
    metadata.to_csv(design_dir / "metadata.csv", index=False)

    design_rows.append({
        "design": design,
        "family": "HMM",
        "K": K,
        "best_seed": best_seed,
        "input_resolution": "1 Hz",
        "n_features": len(MODEL_COLS),
        "artifact": f"{design}/model.joblib",
        "train_reproduction": "PASS",
    })

    print(
        f"  seed={best_seed}, labels={exact_agreement:.6f}, "
        f"logL diff={loglik_abs_diff:.6g} -> PASS"
    )


# ============================================================
# 3. SAVE COMMON RETAINED-HMM CONTRACT
# ============================================================

checks = pd.DataFrame(check_rows)
designs = pd.DataFrame(design_rows)

checks.to_csv(OUTPUT_DIR / "reproduction_checks.csv", index=False)
designs.to_csv(OUTPUT_DIR / "retained_designs.csv", index=False)
feature_order.sort_values("position").to_csv(OUTPUT_DIR / "feature_order.csv", index=False)

print("\nRetained HMM export complete.")
print(checks[[
    "design",
    "label_mismatches",
    "exact_label_agreement",
    "log_likelihood_abs_diff",
    "status",
]].to_string(index=False))
print(f"Output directory: {OUTPUT_DIR}")


HMM4
  seed=71, labels=1.000000, logL diff=0 -> PASS

HMM5
  seed=37, labels=1.000000, logL diff=0 -> PASS

HMM8
  seed=53, labels=1.000000, logL diff=0 -> PASS

Retained HMM export complete.
design  label_mismatches  exact_label_agreement  log_likelihood_abs_diff status
  HMM4                 0                    1.0                      0.0   PASS
  HMM5                 0                    1.0                      0.0   PASS
  HMM8                 0                    1.0                      0.0   PASS
Output directory: outputs/section_5_3/retained_hmm


### 5.3.7 Train-only boundary stabilization

The retained native train sequences are converted to episodes and processed with the two predefined minimum-duration settings, `dmin = 5 s` and `dmin = 10 s`. The procedure is deterministic and remains independent of attack labels.


In [16]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# HAI 21.03 — TRAIN-ONLY BOUNDARY STABILIZATION
# TRAIN-ONLY EPISODE CONSTRUCTION + BOUNDARY STABILIZATION
#
# Purpose:
#   - convert retained train1 native-state sequences to episodes
#   - apply the fixed SERF boundary-stabilization rule
#   - retain HDBSCAN -1 as protected density-uncertainty/unassigned
#   - prepare stabilized train representations for later prototypes
#
# No test data.
# No attack labels.
# No response variables.
# No model refitting.
# ============================================================

CONTEXT_FILE = Path("outputs/section_5_2/train_context_8d/train1_context_8d.csv")
KM_DIR = Path("outputs/section_5_3/retained_kmeans")
HDB_DIR = Path("outputs/section_5_3/retained_hdbscan")
HMM_DIR = Path("outputs/section_5_3/retained_hmm")
OUTPUT_DIR = Path("outputs/section_5_3/train_boundary_stabilization")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DMIN_VALUES = [5, 10]
PROTECTED_NOISE = -1

DESIGNS = {
    "KM4_8":   ("kmeans",  KM_DIR / "KM4_8" / "train_native_labels.csv"),
    "KM7_8":   ("kmeans",  KM_DIR / "KM7_8" / "train_native_labels.csv"),
    "KM13_8":  ("kmeans",  KM_DIR / "KM13_8" / "train_native_labels.csv"),
    "KM13_16": ("kmeans",  KM_DIR / "KM13_16" / "train_native_labels.csv"),
    "HDB23_8": ("hdbscan", HDB_DIR / "HDB23_8" / "train_native_labels.csv"),
    "HDB18_8": ("hdbscan", HDB_DIR / "HDB18_8" / "train_native_labels.csv"),
    "HDB15_8": ("hdbscan", HDB_DIR / "HDB15_8" / "train_native_labels.csv"),
    "HDB18_16":("hdbscan", HDB_DIR / "HDB18_16" / "train_native_labels.csv"),
    "HMM4":    ("hmm",     HMM_DIR / "HMM4" / "train_native_labels.csv"),
    "HMM5":    ("hmm",     HMM_DIR / "HMM5" / "train_native_labels.csv"),
    "HMM8":    ("hmm",     HMM_DIR / "HMM8" / "train_native_labels.csv"),
}

# ------------------------------------------------------------
# Small helpers
# ------------------------------------------------------------

def build_episodes(labels):
    """Build contiguous episodes over non-missing labels."""
    episodes = []
    start = None
    prev = None

    for i, value in enumerate(labels):
        if pd.isna(value):
            if start is not None:
                episodes.append([start, i - 1, int(prev)])
                start = None
                prev = None
            continue

        value = int(value)
        if start is None:
            start = i
            prev = value
        elif value != prev:
            episodes.append([start, i - 1, int(prev)])
            start = i
            prev = value

    if start is not None:
        episodes.append([start, len(labels) - 1, int(prev)])

    return episodes


def merge_adjacent_same(episodes):
    if not episodes:
        return []
    merged = [episodes[0].copy()]
    for start, end, label in episodes[1:]:
        last = merged[-1]
        if label == last[2] and start == last[1] + 1:
            last[1] = end
        else:
            merged.append([start, end, label])
    return merged


def episode_mean(X, start, end):
    return X[start:end + 1].mean(axis=0)


def stabilize(episodes, X, dmin, protect_noise=False):
    """
    Iterative local stabilization.

    Rule for a short non-protected episode:
      1) same-label non-noise neighbours -> bridge merge;
      2) otherwise merge to the non-noise adjacent episode whose
         8D mean context profile is nearest in Euclidean distance;
      3) if no valid neighbour exists, retain it.

    HDBSCAN -1 is never reassigned and is never used as a merge target.
    """
    eps = [e.copy() for e in episodes]
    actions = []

    while True:
        changed = False

        for i, (start, end, label) in enumerate(eps):
            duration = end - start + 1
            if duration >= dmin:
                continue
            if protect_noise and label == PROTECTED_NOISE:
                continue

            left = eps[i - 1] if i > 0 else None
            right = eps[i + 1] if i + 1 < len(eps) else None

            # Never cross a discontinuity in labelled coverage.
            if left is not None and left[1] + 1 != start:
                left = None
            if right is not None and end + 1 != right[0]:
                right = None

            valid_left = left is not None and not (protect_noise and left[2] == PROTECTED_NOISE)
            valid_right = right is not None and not (protect_noise and right[2] == PROTECTED_NOISE)

            # Bridge-like interruption between equal labels.
            if valid_left and valid_right and left[2] == right[2]:
                old_label = label
                new_label = left[2]
                eps[i][2] = new_label
                eps = merge_adjacent_same(eps)
                actions.append({
                    "rule": "bridge_same_neighbours",
                    "start_row": start,
                    "end_row": end,
                    "duration": duration,
                    "from_label": old_label,
                    "to_label": new_label,
                    "distance": 0.0,
                })
                changed = True
                break

            candidates = []
            current_profile = episode_mean(X, start, end)

            if valid_left:
                dist = float(np.linalg.norm(
                    current_profile - episode_mean(X, left[0], left[1])
                ))
                candidates.append((dist, -(left[1] - left[0] + 1), 0, left[2], "left"))

            if valid_right:
                dist = float(np.linalg.norm(
                    current_profile - episode_mean(X, right[0], right[1])
                ))
                candidates.append((dist, -(right[1] - right[0] + 1), 1, right[2], "right"))

            if not candidates:
                continue

            # Deterministic tie-breaking: distance, longer neighbour, left.
            candidates.sort()
            dist, _, _, new_label, side = candidates[0]
            old_label = label
            eps[i][2] = new_label
            eps = merge_adjacent_same(eps)
            actions.append({
                "rule": f"nearest_profile_{side}",
                "start_row": start,
                "end_row": end,
                "duration": duration,
                "from_label": old_label,
                "to_label": new_label,
                "distance": dist,
            })
            changed = True
            break

        if not changed:
            break

    return eps, actions


def episodes_to_frame(episodes, time_values, X, feature_cols):
    rows = []
    for episode_id, (start, end, label) in enumerate(episodes):
        profile = episode_mean(X, start, end)
        row = {
            "episode_id": episode_id,
            "start_row": start,
            "end_row": end,
            "start_time": time_values[start],
            "end_time": time_values[end],
            "duration_sec": end - start + 1,
            "native_state": label,
        }
        for col, value in zip(feature_cols, profile):
            row[f"mean_{col}"] = float(value)
        rows.append(row)
    return pd.DataFrame(rows)


# ============================================================
# 1. LOAD TRAIN CONTEXT
# ============================================================

context = pd.read_csv(CONTEXT_FILE)
context["time"] = pd.to_datetime(context["time"], errors="coerce")

if context["time"].isna().any():
    raise ValueError("Invalid timestamps in train1 context file.")

FEATURE_COLS = [c for c in context.columns if c.endswith("_level") or c.endswith("_delta")]
if len(FEATURE_COLS) != 8:
    raise ValueError(f"Expected 8 context features, found {len(FEATURE_COLS)}.")

X = context[FEATURE_COLS].to_numpy(dtype=float)
if not np.isfinite(X).all():
    raise ValueError("Non-finite values in train1 8D context representation.")

n_rows = len(context)
time_values = context["time"].to_numpy()

# train1 is expected to be one contiguous 1-Hz sequence.
dt = context["time"].diff().dt.total_seconds()
if int((dt.notna() & (dt != 1.0)).sum()) != 0:
    raise ValueError("train-only boundary stabilization expects contiguous 1-Hz train1 context data.")


# ============================================================
# 2. LOAD EACH RETAINED NATIVE SEQUENCE AND STABILIZE
# ============================================================

summary_rows = []
action_rows = []
check_rows = []

for design, (family, label_file) in DESIGNS.items():
    labels_df = pd.read_csv(label_file)
    full_labels = pd.Series(pd.array([pd.NA] * n_rows, dtype="Int64"))

    if "window_end_row" in labels_df.columns:
        end_rows = pd.to_numeric(labels_df["window_end_row"], errors="raise").astype(int)
        if end_rows.duplicated().any():
            raise ValueError(f"{design}: duplicated window_end_row values.")
        if end_rows.min() < 0 or end_rows.max() >= n_rows:
            raise ValueError(f"{design}: window_end_row outside train1 range.")
        full_labels.iloc[end_rows.to_numpy()] = pd.to_numeric(
            labels_df["native_state"], errors="raise"
        ).astype(int).to_numpy()

    elif "row_id" in labels_df.columns:
        rows = pd.to_numeric(labels_df["row_id"], errors="raise").astype(int)
        if rows.duplicated().any():
            raise ValueError(f"{design}: duplicated row_id values.")
        if rows.min() < 0 or rows.max() >= n_rows:
            raise ValueError(f"{design}: row_id outside train1 range.")
        full_labels.iloc[rows.to_numpy()] = pd.to_numeric(
            labels_df["native_state"], errors="raise"
        ).astype(int).to_numpy()

    else:
        raise ValueError(f"{design}: cannot locate row alignment column.")

    labelled_mask = full_labels.notna().to_numpy()
    preliminary = build_episodes(full_labels.to_numpy(dtype=object))

    design_dir = OUTPUT_DIR / design
    design_dir.mkdir(parents=True, exist_ok=True)

    prelim_df = episodes_to_frame(preliminary, time_values, X, FEATURE_COLS)
    prelim_df.to_csv(design_dir / "preliminary_episodes.csv", index=False)

    # Explicit timestamp-level contract for the unstabilized episodic representation.
    # This avoids reconstructing episode membership later from start/end rows.
    preliminary_episode_id = pd.Series(pd.array([pd.NA] * n_rows, dtype="Int64"))
    for episode_id, (start, end, _) in enumerate(preliminary):
        preliminary_episode_id.iloc[start:end + 1] = episode_id

    timeline = pd.DataFrame({
        "row_id": context["row_id"] if "row_id" in context.columns else np.arange(n_rows),
        "time": context["time"],
        "native_state": full_labels,
        "preliminary_episode_id": preliminary_episode_id,
    })
    timeline.to_csv(design_dir / "preliminary_episode_labels.csv", index=False)

    prelim_noise_mask = None
    if family == "hdbscan":
        prelim_noise_mask = (full_labels == PROTECTED_NOISE).fillna(False).to_numpy()

    for dmin in DMIN_VALUES:
        stabilized, actions = stabilize(
            preliminary,
            X,
            dmin=dmin,
            protect_noise=(family == "hdbscan"),
        )

        stable_state = pd.Series(pd.array([pd.NA] * n_rows, dtype="Int64"))
        for start, end, label in stabilized:
            stable_state.iloc[start:end + 1] = int(label)

        stable_df = episodes_to_frame(stabilized, time_values, X, FEATURE_COLS)
        stable_df.to_csv(design_dir / f"stabilized_episodes_dmin{dmin}.csv", index=False)

        out_timeline = timeline.copy()
        out_timeline[f"stabilized_state_dmin{dmin}"] = stable_state
        out_timeline.to_csv(design_dir / f"stabilized_labels_dmin{dmin}.csv", index=False)

        for action_id, action in enumerate(actions):
            action_rows.append({
                "design": design,
                "family": family,
                "dmin": dmin,
                "action_id": action_id,
                **action,
            })

        stable_labelled_mask = stable_state.notna().to_numpy()
        coverage_same = bool(np.array_equal(labelled_mask, stable_labelled_mask))

        noise_mask_same = True
        if family == "hdbscan":
            stable_noise_mask = (stable_state == PROTECTED_NOISE).fillna(False).to_numpy()
            noise_mask_same = bool(np.array_equal(prelim_noise_mask, stable_noise_mask))

        labels_subset = set(stable_state.dropna().astype(int).unique()).issubset(
            set(full_labels.dropna().astype(int).unique())
        )

        remaining_short_nonnoise = int(sum(
            (end - start + 1) < dmin and not (family == "hdbscan" and label == PROTECTED_NOISE)
            for start, end, label in stabilized
        ))

        status = "PASS" if coverage_same and noise_mask_same and labels_subset else "FAIL"
        if status != "PASS":
            raise RuntimeError(
                f"{design}, dmin={dmin}: stabilization integrity check failed."
            )

        prelim_durations = np.array([e[1] - e[0] + 1 for e in preliminary], dtype=int)
        stable_durations = np.array([e[1] - e[0] + 1 for e in stabilized], dtype=int)

        summary_rows.append({
            "design": design,
            "family": family,
            "dmin": dmin,
            "n_labelled_ticks": int(labelled_mask.sum()),
            "preliminary_episodes": len(preliminary),
            "stabilized_episodes": len(stabilized),
            "episode_reduction_share": 1.0 - len(stabilized) / len(preliminary),
            "preliminary_median_duration_sec": float(np.median(prelim_durations)),
            "stabilized_median_duration_sec": float(np.median(stable_durations)),
            "preliminary_short_share": float(np.mean(prelim_durations < dmin)),
            "n_reassignments": len(actions),
            "remaining_short_nonnoise_episodes": remaining_short_nonnoise,
        })

        check_rows.append({
            "design": design,
            "family": family,
            "dmin": dmin,
            "label_coverage_preserved": coverage_same,
            "hdbscan_noise_mask_preserved": noise_mask_same,
            "no_new_native_labels": labels_subset,
            "remaining_short_nonnoise_episodes": remaining_short_nonnoise,
            "status": status,
        })


# ============================================================
# 3. SAVE COMBINED AUDIT TABLES
# ============================================================

summary = pd.DataFrame(summary_rows)
actions = pd.DataFrame(action_rows)
checks = pd.DataFrame(check_rows)

summary.to_csv(OUTPUT_DIR / "stabilization_summary.csv", index=False)
actions.to_csv(OUTPUT_DIR / "stabilization_actions.csv", index=False)
checks.to_csv(OUTPUT_DIR / "integrity_checks.csv", index=False)

pd.DataFrame({
    "parameter": [
        "dmin_values",
        "local_profile",
        "distance",
        "same_neighbour_rule",
        "hdbscan_noise_policy",
        "test_data_used",
        "attack_labels_used",
        "response_variables_used",
    ],
    "value": [
        ",".join(map(str, DMIN_VALUES)),
        "mean of train-derived 8D context over episode interval",
        "euclidean",
        "short bridge between equal non-noise labels -> merge",
        "-1 protected; never reassigned and never used as merge target",
        False,
        False,
        False,
    ],
}).to_csv(OUTPUT_DIR / "settings.csv", index=False)

print("=" * 72)
print("TRAIN-ONLY BOUNDARY STABILIZATION COMPLETE")
print("=" * 72)
print(summary.to_string(index=False))
print()
print(checks.groupby(["dmin", "status"]).size().rename("n").reset_index().to_string(index=False))
print(f"\nOutput directory: {OUTPUT_DIR}")

TRAIN-ONLY BOUNDARY STABILIZATION COMPLETE — TRAIN-ONLY BOUNDARY STABILIZATION
  design  family  dmin  n_labelled_ticks  preliminary_episodes  stabilized_episodes  episode_reduction_share  preliminary_median_duration_sec  stabilized_median_duration_sec  preliminary_short_share  n_reassignments  remaining_short_nonnoise_episodes
   KM4_8  kmeans     5            215994                    18                   17                 0.055556                           8628.0                          8737.0                 0.055556                1                                  0
   KM4_8  kmeans    10            215994                    18                   17                 0.055556                           8628.0                          8737.0                 0.055556                1                                  0
   KM7_8  kmeans     5            215994                    27                   21                 0.222222                           5886.0                          8

### 5.3.8 Train-only prototype construction and meta-mode alignment

Native-state prototypes are built independently for both stabilization branches and pooled across the 11 retained designs. Ward alignment is screened over the predefined range and the retained `q = 7` mapping is frozen for held-out use.


In [18]:
import numpy as np
import pandas as pd
from pathlib import Path

from scipy.optimize import linear_sum_assignment
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)

# ============================================================
# HAI 21.03 — TRAIN-ONLY META-MODE ALIGNMENT
# TRAIN-ONLY RAW-LABEL PROTOTYPES + META-MODE ALIGNMENT SCREENING
#
# Purpose:
#   1) aggregate stabilized train1 episodes into one robust prototype
#      per design-specific non-noise native state;
#   2) align prototypes directly in the 16D prototype space derived
#      from the already train-scaled 8D operating-context representation;
#   3) screen q = 3..23 non-noise meta-modes for dmin = 5 and 10;
#   4) keep HDBSCAN -1 outside Ward and map it deterministically to M0.
#
# Important:
#   - train1 only
#   - no test data
#   - no attack labels
#   - no response variables
#   - no model refitting
#   - no q is selected here from downstream anomaly performance
# ============================================================

INPUT_DIR = Path("outputs/section_5_3/train_boundary_stabilization")
OUTPUT_DIR = Path("outputs/section_5_3/train_meta_alignment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DMIN_VALUES = [5, 10]
Q_VALUES = list(range(3, 24))
PROTECTED_NOISE = -1

DESIGNS = {
    "KM4_8": "kmeans",
    "KM7_8": "kmeans",
    "KM13_8": "kmeans",
    "KM13_16": "kmeans",
    "HDB23_8": "hdbscan",
    "HDB18_8": "hdbscan",
    "HDB15_8": "hdbscan",
    "HDB18_16": "hdbscan",
    "HMM4": "hmm",
    "HMM5": "hmm",
    "HMM8": "hmm",
}

# train-only boundary stabilization stores the episode descriptor as mean_<context feature>.
EPISODE_FEATURE_PREFIX = "mean_"

# ------------------------------------------------------------
# Small helpers
# ------------------------------------------------------------

def canonicalize_meta_labels(raw_labels, supports, X):
    """
    Convert arbitrary Ward cluster IDs to deterministic M1..Mq IDs.

    Ordering rule:
      1) descending total train tick support represented by prototypes;
      2) descending number of prototypes in the meta-mode;
      3) lexicographic prototype-space centroid as deterministic tie-break.

    Support is used only to name the resulting clusters, not to fit Ward.
    """
    raw_labels = np.asarray(raw_labels, dtype=int)
    supports = np.asarray(supports, dtype=float)
    X = np.asarray(X, dtype=float)

    cluster_rows = []
    for raw_id in sorted(np.unique(raw_labels)):
        mask = raw_labels == raw_id
        centroid = X[mask].mean(axis=0)
        cluster_rows.append({
            "raw_id": int(raw_id),
            "support": float(supports[mask].sum()),
            "n_prototypes": int(mask.sum()),
            "centroid_key": tuple(np.round(centroid, 12).tolist()),
        })

    cluster_rows.sort(
        key=lambda r: (
            -r["support"],
            -r["n_prototypes"],
            r["centroid_key"],
            r["raw_id"],
        )
    )

    mapping = {
        row["raw_id"]: i + 1
        for i, row in enumerate(cluster_rows)
    }
    canonical = np.array([mapping[x] for x in raw_labels], dtype=int)
    return canonical, mapping


def best_matched_agreement(labels_a, labels_b):
    """Label-invariant exact agreement after optimal one-to-one matching."""
    a = np.asarray(labels_a, dtype=int)
    b = np.asarray(labels_b, dtype=int)
    if len(a) != len(b):
        raise ValueError("Label sequences must have equal length.")
    if len(a) == 0:
        return np.nan

    ua = np.unique(a)
    ub = np.unique(b)
    contingency = np.zeros((len(ua), len(ub)), dtype=int)

    pos_a = {v: i for i, v in enumerate(ua)}
    pos_b = {v: i for i, v in enumerate(ub)}

    for x, y in zip(a, b):
        contingency[pos_a[x], pos_b[y]] += 1

    row_ind, col_ind = linear_sum_assignment(-contingency)
    matched = int(contingency[row_ind, col_ind].sum())
    return matched / len(a)


def fit_ward(X, q):
    model = AgglomerativeClustering(
        n_clusters=q,
        linkage="ward",
    )
    return model.fit_predict(X)


# ============================================================
# 1. BUILD ROBUST RAW-LABEL PROTOTYPES FOR EACH DMIN
# ============================================================

all_prototypes = {}
all_ward_input = {}
prototype_feature_cols = None
base_descriptor_cols = None

prototype_integrity_rows = []
prototype_inventory_rows = []

for dmin in DMIN_VALUES:
    rows = []
    expected_keys = set()
    state_inventory_preserved_all = True

    for design, family in DESIGNS.items():
        episode_file = INPUT_DIR / design / f"stabilized_episodes_dmin{dmin}.csv"
        if not episode_file.exists():
            raise FileNotFoundError(f"Missing train-only boundary stabilization output: {episode_file}")

        eps = pd.read_csv(episode_file)

        required = {"native_state", "duration_sec"}
        missing = sorted(required - set(eps.columns))
        if missing:
            raise ValueError(f"{design}, dmin={dmin}: missing columns {missing}")

        descriptor_cols = [
            c for c in eps.columns
            if c.startswith(EPISODE_FEATURE_PREFIX)
        ]
        if len(descriptor_cols) != 8:
            raise ValueError(
                f"{design}, dmin={dmin}: expected 8 episode descriptor columns, "
                f"found {len(descriptor_cols)}."
            )

        if base_descriptor_cols is None:
            base_descriptor_cols = descriptor_cols.copy()
        elif descriptor_cols != base_descriptor_cols:
            raise ValueError(
                f"{design}, dmin={dmin}: episode descriptor order differs across designs."
            )

        eps["native_state"] = pd.to_numeric(eps["native_state"], errors="raise").astype(int)
        eps["duration_sec"] = pd.to_numeric(eps["duration_sec"], errors="raise").astype(int)

        if (eps["duration_sec"] <= 0).any():
            raise ValueError(f"{design}, dmin={dmin}: non-positive episode duration.")

        # Deployment-critical check: stabilization must not eliminate an
        # entire non-noise native state, because the fixed native->meta mapping
        # must remain defined for every state that the retained model can emit.
        prelim_file = INPUT_DIR / design / "preliminary_episodes.csv"
        if not prelim_file.exists():
            raise FileNotFoundError(f"Missing train-only boundary stabilization output: {prelim_file}")
        prelim = pd.read_csv(prelim_file)
        prelim_states = set(
            pd.to_numeric(prelim["native_state"], errors="raise").astype(int).unique()
        )
        stable_states = set(eps["native_state"].unique())
        if family == "hdbscan":
            prelim_states.discard(PROTECTED_NOISE)
            stable_states.discard(PROTECTED_NOISE)

        state_inventory_preserved = prelim_states == stable_states
        state_inventory_preserved_all &= state_inventory_preserved
        if not state_inventory_preserved:
            lost = sorted(prelim_states - stable_states)
            gained = sorted(stable_states - prelim_states)
            raise RuntimeError(
                f"{design}, dmin={dmin}: stabilization changed the non-noise "
                f"native-state inventory. lost={lost}, gained={gained}"
            )

        expected_keys.update((design, int(s)) for s in prelim_states)

        Xeps = eps[descriptor_cols].apply(pd.to_numeric, errors="coerce")
        if Xeps.isna().any().any() or not np.isfinite(Xeps.to_numpy(dtype=float)).all():
            raise ValueError(f"{design}, dmin={dmin}: invalid episode descriptor values.")
        eps[descriptor_cols] = Xeps

        # HDBSCAN -1 remains a protected non-operating-state category (M0)
        # and is intentionally excluded from Ward prototype alignment.
        ward_eps = eps.loc[eps["native_state"] != PROTECTED_NOISE].copy()

        if ward_eps.empty:
            raise RuntimeError(f"{design}, dmin={dmin}: no non-noise episodes available.")

        for native_state, grp in ward_eps.groupby("native_state", sort=True):
            values = grp[descriptor_cols]

            row = {
                "dmin": dmin,
                "design": design,
                "family": family,
                "native_state": int(native_state),
                "n_episodes": int(len(grp)),
                "total_duration_sec": int(grp["duration_sec"].sum()),
                "median_episode_duration_sec": float(grp["duration_sec"].median()),
            }

            for col in descriptor_cols:
                s = values[col]
                row[f"median__{col}"] = float(s.median())
                row[f"iqr__{col}"] = float(s.quantile(0.75) - s.quantile(0.25))

            rows.append(row)

        # Inventory audit includes protected HDBSCAN noise separately.
        inventory = eps.groupby("native_state", sort=True).agg(
            n_episodes=("native_state", "size"),
            total_duration_sec=("duration_sec", "sum"),
        ).reset_index()

        for r in inventory.itertuples(index=False):
            prototype_inventory_rows.append({
                "dmin": dmin,
                "design": design,
                "family": family,
                "native_state": int(r.native_state),
                "is_protected_noise": bool(
                    family == "hdbscan" and int(r.native_state) == PROTECTED_NOISE
                ),
                "n_episodes": int(r.n_episodes),
                "total_duration_sec": int(r.total_duration_sec),
            })

    prototypes = pd.DataFrame(rows)
    prototypes = prototypes.sort_values(
        ["design", "native_state"],
        kind="mergesort",
    ).reset_index(drop=True)

    prototype_feature_cols = [
        c for c in prototypes.columns
        if c.startswith("median__") or c.startswith("iqr__")
    ]

    expected_features = 16
    if len(prototype_feature_cols) != expected_features:
        raise RuntimeError(
            f"dmin={dmin}: expected {expected_features} prototype features, "
            f"found {len(prototype_feature_cols)}."
        )

    if prototypes[["design", "native_state"]].duplicated().any():
        raise RuntimeError(f"dmin={dmin}: duplicate design/native_state prototypes.")

    Xp = prototypes[prototype_feature_cols].to_numpy(dtype=float)
    if not np.isfinite(Xp).all():
        raise RuntimeError(f"dmin={dmin}: non-finite prototype values.")

    # No additional scaling is applied here. The prototype descriptors are
    # derived from the common train-scaled 8D operating-context space created
    # in train-derived context representation. Ward therefore operates directly on the resulting 16D
    # median+IQR prototype vectors.
    Xward = Xp.copy()

    if not np.isfinite(Xward).all():
        raise RuntimeError(f"dmin={dmin}: non-finite Ward input prototype values.")

    # State coverage integrity: every non-noise native state available before
    # stabilization must contribute exactly one stabilized prototype.
    actual_keys = set(
        zip(prototypes["design"], prototypes["native_state"].astype(int))
    )

    coverage_ok = expected_keys == actual_keys
    if not coverage_ok:
        missing_keys = sorted(expected_keys - actual_keys)
        extra_keys = sorted(actual_keys - expected_keys)
        raise RuntimeError(
            f"dmin={dmin}: prototype key coverage failed. "
            f"missing={missing_keys[:10]}, extra={extra_keys[:10]}"
        )

    all_prototypes[dmin] = prototypes
    all_ward_input[dmin] = Xward

    prototypes.to_csv(
        OUTPUT_DIR / f"raw_label_prototypes_dmin{dmin}.csv",
        index=False,
    )

    ward_df = prototypes[[
        "dmin", "design", "family", "native_state",
        "n_episodes", "total_duration_sec",
    ]].copy()
    for j, col in enumerate(prototype_feature_cols):
        ward_df[f"ward__{col}"] = Xward[:, j]
    ward_df.to_csv(
        OUTPUT_DIR / f"ward_input_prototypes_dmin{dmin}.csv",
        index=False,
    )

    prototype_integrity_rows.append({
        "dmin": dmin,
        "n_nonnoise_prototypes": len(prototypes),
        "n_prototype_features": len(prototype_feature_cols),
        "state_inventory_preserved": bool(state_inventory_preserved_all),
        "prototype_key_coverage_exact": coverage_ok,
        "all_prototypes_finite": bool(np.isfinite(Xp).all()),
        "all_ward_inputs_finite": bool(np.isfinite(Xward).all()),
        "status": "PASS",
    })


# ============================================================
# 2. ALIGN NON-NOISE PROTOTYPES WITH WARD FOR q = 3..23
# ============================================================

alignment_rows = []
meta_summary_rows = []
mapping_rows = []
order_sensitivity_rows = []

for dmin in DMIN_VALUES:
    prototypes = all_prototypes[dmin]
    Xward = all_ward_input[dmin]
    supports = prototypes["total_duration_sec"].to_numpy(dtype=float)

    if max(Q_VALUES) >= len(prototypes):
        raise RuntimeError(
            f"dmin={dmin}: q grid is too large for {len(prototypes)} prototypes."
        )

    for q in Q_VALUES:
        raw = fit_ward(Xward, q)
        canonical, _ = canonicalize_meta_labels(raw, supports, Xward)

        # Independent order-sensitivity diagnostic: fit the same geometry
        # after reversing object order and compare partitions by ARI.
        raw_rev = fit_ward(Xward[::-1], q)[::-1]
        order_ari = float(adjusted_rand_score(raw, raw_rev))

        unique_modes, mode_counts = np.unique(canonical, return_counts=True)
        if len(unique_modes) != q:
            raise RuntimeError(f"dmin={dmin}, q={q}: Ward produced {len(unique_modes)} modes.")

        sil = float(silhouette_score(Xward, canonical)) if q > 1 else np.nan
        db = float(davies_bouldin_score(Xward, canonical)) if q > 1 else np.nan
        ch = float(calinski_harabasz_score(Xward, canonical)) if q > 1 else np.nan

        singleton_modes = int(np.sum(mode_counts == 1))

        # Mapping rows for ordinary non-noise native states.
        for i, r in prototypes.iterrows():
            mapping_rows.append({
                "dmin": dmin,
                "q": q,
                "design": r["design"],
                "family": r["family"],
                "native_state": int(r["native_state"]),
                "meta_mode": int(canonical[i]),
                "mapping_source": "ward_nonnoise_prototype",
                "n_episodes": int(r["n_episodes"]),
                "total_duration_sec": int(r["total_duration_sec"]),
            })

        # HDBSCAN -1 is appended deterministically as M0 and never enters Ward.
        for design, family in DESIGNS.items():
            if family == "hdbscan":
                mapping_rows.append({
                    "dmin": dmin,
                    "q": q,
                    "design": design,
                    "family": family,
                    "native_state": PROTECTED_NOISE,
                    "meta_mode": 0,
                    "mapping_source": "protected_hdbscan_noise",
                    "n_episodes": np.nan,
                    "total_duration_sec": np.nan,
                })

        # Per-meta-mode structural summary.
        tmp = prototypes[[
            "design", "family", "native_state",
            "n_episodes", "total_duration_sec",
        ]].copy()
        tmp["meta_mode"] = canonical

        for meta_mode, grp in tmp.groupby("meta_mode", sort=True):
            meta_summary_rows.append({
                "dmin": dmin,
                "q": q,
                "meta_mode": int(meta_mode),
                "n_prototypes": int(len(grp)),
                "n_designs": int(grp["design"].nunique()),
                "n_families": int(grp["family"].nunique()),
                "families": ",".join(sorted(grp["family"].unique())),
                "designs": ",".join(sorted(grp["design"].unique())),
                "total_train_tick_support": int(grp["total_duration_sec"].sum()),
                "prototype_share": float(len(grp) / len(tmp)),
            })

        alignment_rows.append({
            "dmin": dmin,
            "q_nonnoise_meta_modes": q,
            "n_nonnoise_prototypes": len(prototypes),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch,
            "singleton_meta_modes": singleton_modes,
            "min_prototypes_per_meta_mode": int(mode_counts.min()),
            "max_prototypes_per_meta_mode": int(mode_counts.max()),
            "order_sensitivity_ari": order_ari,
        })

        order_sensitivity_rows.append({
            "dmin": dmin,
            "q": q,
            "ari_original_vs_reversed_order": order_ari,
            "status": "PASS" if np.isclose(order_ari, 1.0, atol=1e-12) else "REVIEW",
        })


# ============================================================
# 3. CROSS-DMIN PROTOTYPE AND ALIGNMENT STABILITY
# ============================================================

proto5 = all_prototypes[5].copy()
proto10 = all_prototypes[10].copy()

keys = ["design", "native_state"]
common = proto5.merge(
    proto10,
    on=keys,
    suffixes=("_d5", "_d10"),
    how="inner",
)

prototype_stability_rows = []
for r in common.itertuples(index=False):
    v5 = np.array([
        getattr(r, f"{c}_d5")
        for c in prototype_feature_cols
    ], dtype=float)
    v10 = np.array([
        getattr(r, f"{c}_d10")
        for c in prototype_feature_cols
    ], dtype=float)

    prototype_stability_rows.append({
        "design": r.design,
        "native_state": int(r.native_state),
        "euclidean_raw_prototype_shift": float(np.linalg.norm(v5 - v10)),
        "max_abs_feature_shift": float(np.max(np.abs(v5 - v10))),
    })

cross_dmin_rows = []
all_mapping = pd.DataFrame(mapping_rows)

for q in Q_VALUES:
    m5 = all_mapping[
        (all_mapping["dmin"] == 5)
        & (all_mapping["q"] == q)
        & (all_mapping["meta_mode"] != 0)
    ][["design", "native_state", "meta_mode"]].rename(
        columns={"meta_mode": "meta5"}
    )

    m10 = all_mapping[
        (all_mapping["dmin"] == 10)
        & (all_mapping["q"] == q)
        & (all_mapping["meta_mode"] != 0)
    ][["design", "native_state", "meta_mode"]].rename(
        columns={"meta_mode": "meta10"}
    )

    pair = m5.merge(m10, on=["design", "native_state"], how="inner")

    if len(pair) == 0:
        raise RuntimeError(f"q={q}: no common non-noise prototypes across dmin branches.")

    ari = float(adjusted_rand_score(pair["meta5"], pair["meta10"]))
    matched = float(best_matched_agreement(pair["meta5"], pair["meta10"]))

    cross_dmin_rows.append({
        "q": q,
        "n_common_prototypes": len(pair),
        "ari_dmin5_vs_dmin10": ari,
        "optimal_label_matched_agreement": matched,
    })


# ============================================================
# 4. SAVE OUTPUTS
# ============================================================

alignment_screening = pd.DataFrame(alignment_rows)
meta_mode_summary = pd.DataFrame(meta_summary_rows)
native_to_meta_mapping = pd.DataFrame(mapping_rows)
order_sensitivity = pd.DataFrame(order_sensitivity_rows)
prototype_integrity = pd.DataFrame(prototype_integrity_rows)
prototype_inventory = pd.DataFrame(prototype_inventory_rows)
prototype_stability = pd.DataFrame(prototype_stability_rows)
cross_dmin_stability = pd.DataFrame(cross_dmin_rows)

alignment_screening.to_csv(OUTPUT_DIR / "alignment_screening.csv", index=False)
meta_mode_summary.to_csv(OUTPUT_DIR / "meta_mode_summary.csv", index=False)
native_to_meta_mapping.to_csv(OUTPUT_DIR / "native_to_meta_mapping.csv", index=False)
order_sensitivity.to_csv(OUTPUT_DIR / "order_sensitivity.csv", index=False)
prototype_integrity.to_csv(OUTPUT_DIR / "prototype_integrity_checks.csv", index=False)
prototype_inventory.to_csv(OUTPUT_DIR / "prototype_inventory.csv", index=False)
prototype_stability.to_csv(OUTPUT_DIR / "prototype_stability_across_dmin.csv", index=False)
cross_dmin_stability.to_csv(OUTPUT_DIR / "cross_dmin_alignment_stability.csv", index=False)

pd.DataFrame({
    "prototype_feature": prototype_feature_cols,
    "role": [
        "median of stabilized episode-level mean context descriptor"
        if c.startswith("median__")
        else "IQR of stabilized episode-level mean context descriptor"
        for c in prototype_feature_cols
    ],
}).to_csv(OUTPUT_DIR / "prototype_feature_order.csv", index=False)

pd.DataFrame({
    "parameter": [
        "dmin_values",
        "q_values",
        "q_upper_bound_rationale",
        "episode_descriptor",
        "prototype_location",
        "prototype_dispersion",
        "prototype_weighting_in_ward",
        "prototype_scaling",
        "alignment_method",
        "alignment_metric",
        "hdbscan_noise_policy",
        "meta_mode_numbering",
        "test_data_used",
        "attack_labels_used",
        "response_variables_used",
    ],
    "value": [
        ",".join(map(str, DMIN_VALUES)),
        ",".join(map(str, Q_VALUES)),
        "screen through 23 non-noise meta-modes, matching the train-only temporal audit upper reference of 23 dominant stable configurations",
        "8D mean train-derived operating-context vector per stabilized episode",
        "component-wise median across episodes of the same design/native_state",
        "component-wise IQR across episodes of the same design/native_state",
        "unweighted prototypes; prevalence used only for diagnostics/canonical naming",
        "none beyond the train-derived 8D context scaling already applied before episode/prototype construction",
        "agglomerative hierarchical clustering, Ward linkage",
        "euclidean in the unrescaled 16D median+IQR prototype space derived from the common train-scaled 8D context",
        "native_state -1 excluded from Ward and mapped deterministically to M0",
        "M1..Mq ordered by descending represented train tick support; M0 reserved for HDBSCAN -1",
        False,
        False,
        False,
    ],
}).to_csv(OUTPUT_DIR / "settings.csv", index=False)

# Hard integrity checks that should always hold.
if not (prototype_integrity["status"] == "PASS").all():
    raise RuntimeError("Prototype integrity check failed.")

mapping_counts = (
    native_to_meta_mapping
    .groupby(["dmin", "q", "design", "native_state"])
    .size()
)
if not (mapping_counts == 1).all():
    raise RuntimeError("Native-state to meta-mode mapping is not one-to-one per dmin/q branch.")

print("=" * 72)
print("TRAIN-ONLY META-MODE ALIGNMENT COMPLETE")
print("=" * 72)
print()
print("Prototype integrity:")
print(prototype_integrity.to_string(index=False))
print()
print("Alignment screening:")
print(alignment_screening.to_string(index=False))
print()
print("Cross-dmin alignment stability:")
print(cross_dmin_stability.to_string(index=False))
print()
print(f"Output directory: {OUTPUT_DIR}")

TRAIN-ONLY META-MODE ALIGNMENT COMPLETE — TRAIN-ONLY PROTOTYPES + META-MODE ALIGNMENT

Prototype integrity:
 dmin  n_nonnoise_prototypes  n_prototype_features  state_inventory_preserved  prototype_key_coverage_exact  all_prototypes_finite  all_ward_inputs_finite status
    5                    128                    16                       True                          True                   True                    True   PASS
   10                    128                    16                       True                          True                   True                    True   PASS

Alignment screening:
 dmin  q_nonnoise_meta_modes  n_nonnoise_prototypes  silhouette  davies_bouldin  calinski_harabasz  singleton_meta_modes  min_prototypes_per_meta_mode  max_prototypes_per_meta_mode  order_sensitivity_ari
    5                      3                    128    0.457966        1.047774          58.510689                     0                            25                            75

## Section 5.4 — Operating-context assignment on held-out test runs

The five test runs are processed independently using only train-derived transformations and retained model artifacts. No segmentation model is refitted or adapted on test data. K-means uses nearest-centroid assignment, HDBSCAN uses frozen approximate prediction while preserving the `-1` state, and HMMs are decoded with fixed fitted parameters within each contiguous test segment.


### 5.4.1 Native-state assignment


In [14]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
from hdbscan.prediction import approximate_predict

# ============================================================
# HAI 21.03 — HELD-OUT NATIVE ASSIGNMENT: OUT-OF-SAMPLE NATIVE-STATE ASSIGNMENT
#
# Purpose:
#   Apply the fixed train1-derived context transformation and the
#   retained segmentation artifacts to unseen test1-test5 runs.
#
# Important:
#   - no fitting or adaptation is performed on test data
#   - attack-label columns are not used
#   - each test file is processed independently
#   - temporal gaps create new contiguous segments
#   - windows never cross segment boundaries
#   - HMMs are decoded separately within each contiguous segment
#   - this cell stops at native states; no SERF stabilization or
#     meta-mode mapping is performed here
# ============================================================

TEST_FILES = [Path(f"test{i}.csv") for i in range(1, 6)]

CONTEXT_DIR = Path("outputs/section_5_2/train_context_8d")
KM_DIR = Path("outputs/section_5_3/retained_kmeans")
HDB_DIR = Path("outputs/section_5_3/retained_hdbscan")
HMM_DIR = Path("outputs/section_5_3/retained_hmm")
OUTPUT_DIR = Path("outputs/section_5_4/test_native_assignment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SAMPLING_SEC = 1.0
ATOL = 1e-12

KM_SPECS = {
    "KM4_8": (8, 4),
    "KM7_8": (8, 7),
    "KM13_8": (8, 13),
    "KM13_16": (16, 13),
}
HDB_SPECS = {
    "HDB23_8": 8,
    "HDB18_8": 8,
    "HDB15_8": 8,
    "HDB18_16": 16,
}
HMM_SPECS = {
    "HMM4": 4,
    "HMM5": 5,
    "HMM8": 8,
}
ALL_DESIGNS = list(KM_SPECS) + list(HDB_SPECS) + list(HMM_SPECS)
WINDOW_LENGTHS = sorted(set(L for L, _ in KM_SPECS.values()) | set(HDB_SPECS.values()))

# ============================================================
# 1. LOAD AND VALIDATE TRAIN-DERIVED CONTRACTS
# ============================================================

scaling = pd.read_csv(CONTEXT_DIR / "scaling_parameters.csv")
tick_features = pd.read_csv(CONTEXT_DIR / "feature_order.csv").sort_values("position")

required_scaling = {"variable", "train_min", "train_max", "train_range", "delta_q99_abs_nonzero"}
if required_scaling - set(scaling.columns):
    raise ValueError("scaling_parameters.csv does not match the train-derived context representation schema.")
if {"position", "model_column", "source_variable", "role"} - set(tick_features.columns):
    raise ValueError("train-derived context representation feature_order.csv does not match the expected schema.")

TICK_MODEL_COLS = tick_features["model_column"].tolist()
if len(TICK_MODEL_COLS) != 8 or len(set(TICK_MODEL_COLS)) != 8:
    raise ValueError("Expected exactly eight unique tick-level context features.")

level_contract = tick_features.loc[tick_features["role"].eq("level")]
CONTEXT_FEATURES = level_contract["source_variable"].tolist()
if len(CONTEXT_FEATURES) != 4 or len(set(CONTEXT_FEATURES)) != 4:
    raise ValueError("Expected exactly four unique context source variables.")

scaling = scaling.set_index("variable").loc[CONTEXT_FEATURES]
for c in ["train_min", "train_max", "train_range", "delta_q99_abs_nonzero"]:
    scaling[c] = pd.to_numeric(scaling[c], errors="raise")
if not np.isfinite(scaling.to_numpy(dtype=float)).all():
    raise ValueError("Non-finite train-derived scaling parameters.")
if (scaling["train_range"] <= 0).any() or (scaling["delta_q99_abs_nonzero"] <= 0).any():
    raise ValueError("Train-derived level ranges and delta scales must be positive.")

# Window feature contract must match for k-means and HDBSCAN.
km_window_features = pd.read_csv(KM_DIR / "feature_order.csv").sort_values("position")
hdb_window_features = pd.read_csv(HDB_DIR / "feature_order.csv").sort_values("position")
WINDOW_MODEL_COLS = km_window_features["model_column"].tolist()

if WINDOW_MODEL_COLS != hdb_window_features["model_column"].tolist():
    raise ValueError("K-means and HDBSCAN window feature orders differ.")
if WINDOW_MODEL_COLS != [f"{c}_mean" for c in TICK_MODEL_COLS]:
    raise ValueError("Window feature order is inconsistent with the train-derived context representation tick-level contract.")

# Do not use retained artifacts unless their train reproduction checks passed.
for family, directory in [("k-means", KM_DIR), ("HDBSCAN", HDB_DIR), ("HMM", HMM_DIR)]:
    checks = pd.read_csv(directory / "reproduction_checks.csv")
    if "status" not in checks.columns or not checks["status"].astype(str).str.upper().eq("PASS").all():
        raise RuntimeError(f"{family} retained artifacts are not fully validated.")

# Load all retained model artifacts once. No fit() call occurs in this cell.
km_centroids = {}
for design, (L, K) in KM_SPECS.items():
    centers = pd.read_csv(KM_DIR / design / "centroids.csv").sort_values("cluster")
    if centers["cluster"].astype(int).tolist() != list(range(K)):
        raise ValueError(f"{design}: centroid IDs are not 0..K-1.")
    C = centers[WINDOW_MODEL_COLS].to_numpy(dtype=np.float64)
    if C.shape != (K, 8) or not np.isfinite(C).all():
        raise ValueError(f"{design}: invalid centroid artifact.")
    km_centroids[design] = C

hdb_models = {}
for design in HDB_SPECS:
    model = joblib.load(HDB_DIR / design / "model.joblib")
    train_states = np.unique(np.asarray(model.labels_, dtype=int))
    hdb_models[design] = (model, set(train_states[train_states >= 0].tolist()))

hmm_models = {}
for design, K in HMM_SPECS.items():
    model = joblib.load(HMM_DIR / design / "model.joblib")
    if int(model.n_components) != K:
        raise ValueError(f"{design}: unexpected number of HMM states.")
    if int(getattr(model, "n_features", 8)) != 8:
        raise ValueError(f"{design}: HMM does not use eight context features.")
    hmm_models[design] = model

# ============================================================
# 2. SMALL REUSABLE OPERATIONS
# ============================================================

def build_test_context(raw, test_id):
    """Apply train-derived context train-derived scaling to one unseen test file."""
    missing = [c for c in CONTEXT_FEATURES if c not in raw.columns]
    if missing:
        raise ValueError(f"{test_id}: missing context variables {missing}")

    time_candidates = [
        c for c in raw.columns
        if c.lower() in {"time", "timestamp", "datetime", "date_time"}
    ]
    if not time_candidates:
        raise ValueError(f"{test_id}: no supported timestamp column found.")
    time_col = time_candidates[0]
    time = pd.to_datetime(raw[time_col], errors="coerce")
    if time.isna().any():
        raise ValueError(f"{test_id}: timestamp parsing failed for '{time_col}'.")
    if time.duplicated().any():
        raise ValueError(f"{test_id}: duplicate timestamps found.")

    dt = time.diff().dt.total_seconds()
    if (dt.iloc[1:] <= 0).any():
        raise ValueError(f"{test_id}: timestamps are not strictly increasing.")

    reset = dt.isna() | ~np.isclose(dt, EXPECTED_SAMPLING_SEC, atol=1e-9, rtol=0.0)
    segment_id = reset.cumsum().astype(np.int64) - 1

    Xraw = raw[CONTEXT_FEATURES].apply(pd.to_numeric, errors="coerce")
    if Xraw.isna().any().any() or not np.isfinite(Xraw.to_numpy(dtype=float)).all():
        raise ValueError(f"{test_id}: invalid values in context variables.")

    level = pd.DataFrame(index=raw.index)
    for c in CONTEXT_FEATURES:
        level[c] = (Xraw[c] - scaling.loc[c, "train_min"]) / scaling.loc[c, "train_range"]

    delta = level.diff()
    delta.loc[reset, :] = 0.0
    for c in CONTEXT_FEATURES:
        delta[c] = delta[c] / scaling.loc[c, "delta_q99_abs_nonzero"]

    context = pd.DataFrame({
        "row_id": np.arange(len(raw), dtype=np.int64),
        "time": time,
        "segment_id": segment_id.to_numpy(dtype=np.int64),
    })

    for c in CONTEXT_FEATURES:
        context[f"{c}_level"] = level[c].to_numpy(dtype=float)
    for c in CONTEXT_FEATURES:
        context[f"{c}_delta"] = delta[c].to_numpy(dtype=float)

    if context[TICK_MODEL_COLS].isna().any().any():
        raise RuntimeError(f"{test_id}: missing values after context construction.")
    if not np.isfinite(context[TICK_MODEL_COLS].to_numpy(dtype=float)).all():
        raise RuntimeError(f"{test_id}: non-finite values after context construction.")

    range_rows = []
    for c in CONTEXT_FEATURES:
        z = level[c].to_numpy(dtype=float)
        range_rows.append({
            "test_id": test_id,
            "variable": c,
            "n_below_train_range": int(np.sum(z < -ATOL)),
            "n_above_train_range": int(np.sum(z > 1.0 + ATOL)),
            "scaled_min": float(np.min(z)),
            "scaled_max": float(np.max(z)),
        })

    return context, pd.DataFrame(range_rows), int(reset.iloc[1:].sum())


def build_windows(context, L):
    """Construct trailing mean windows without crossing contiguous-segment boundaries."""
    parts = []
    for seg_id, seg in context.groupby("segment_id", sort=True):
        seg = seg.sort_values("row_id")
        if len(seg) < L:
            continue

        W = seg[TICK_MODEL_COLS].rolling(window=L, min_periods=L).mean()
        valid = W.notna().all(axis=1)
        end_pos = np.flatnonzero(valid.to_numpy())
        start_pos = end_pos - L + 1

        seg_rows = seg["row_id"].to_numpy(dtype=np.int64)
        seg_times = pd.to_datetime(seg["time"]).to_numpy()
        W_valid = W.loc[valid, TICK_MODEL_COLS].reset_index(drop=True)
        W_valid.columns = WINDOW_MODEL_COLS

        part = pd.DataFrame({
            "window_start_row": seg_rows[start_pos],
            "window_end_row": seg_rows[end_pos],
            "window_start_time": seg_times[start_pos],
            "window_end_time": seg_times[end_pos],
            "segment_id": int(seg_id),
            "window_samples": L,
            "window_sec": L,
            "stride_sec": 1,
            "alignment": "trailing_end",
        })
        parts.append(pd.concat([part, W_valid], axis=1))

    if not parts:
        return pd.DataFrame(columns=[
            "window_index", "window_start_row", "window_end_row",
            "window_start_time", "window_end_time", "segment_id",
            "window_samples", "window_sec", "stride_sec", "alignment",
            *WINDOW_MODEL_COLS,
        ])

    out = pd.concat(parts, ignore_index=True)
    out.insert(0, "window_index", np.arange(len(out), dtype=np.int64))
    return out

# ============================================================
# 3. PROCESS EACH TEST RUN INDEPENDENTLY
# ============================================================

integrity_rows = []
test_summary_rows = []
all_range_audits = []
warnings.filterwarnings("ignore", message=".*force_all_finite.*")

for test_file in TEST_FILES:
    test_id = test_file.stem
    print("\n" + "=" * 72)
    print(f"OUT-OF-SAMPLE ASSIGNMENT — {test_id}")
    print("=" * 72)

    if not test_file.exists():
        raise FileNotFoundError(f"Missing test file: {test_file}")

    # Attack labels, if present in the raw CSV, are deliberately ignored.
    raw = pd.read_csv(test_file)
    context, range_audit, n_gaps = build_test_context(raw, test_id)
    all_range_audits.append(range_audit)

    test_dir = OUTPUT_DIR / test_id
    test_dir.mkdir(parents=True, exist_ok=True)
    context.to_csv(test_dir / "context_8d.csv", index=False, float_format="%.17g")
    range_audit.to_csv(test_dir / "context_range_audit.csv", index=False)

    # Only 8 s and 16 s windows are required by the retained portfolio.
    windows = {}
    segment_sizes = context.groupby("segment_id").size().to_numpy(dtype=int)
    for L in WINDOW_LENGTHS:
        w = build_windows(context, L)
        expected = int(np.maximum(segment_sizes - L + 1, 0).sum())
        if len(w) != expected:
            raise RuntimeError(f"{test_id}, {L}s: expected {expected} windows, got {len(w)}.")
        if len(w) and not np.isfinite(w[WINDOW_MODEL_COLS].to_numpy(dtype=float)).all():
            raise RuntimeError(f"{test_id}, {L}s: non-finite window features.")
        windows[L] = w
        w.to_csv(test_dir / f"window_{L}s.csv", index=False, float_format="%.17g")

    native = context[["row_id", "time", "segment_id"]].copy()
    hdb_strengths = context[["row_id", "time", "segment_id"]].copy()

    # --------------------------------------------------------
    # K-MEANS: nearest fixed train centroid
    # --------------------------------------------------------
    for design, (L, K) in KM_SPECS.items():
        w = windows[L]
        Xw = w[WINDOW_MODEL_COLS].to_numpy(dtype=np.float64)
        C = km_centroids[design]

        if len(Xw):
            dist_sq = ((Xw[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
            labels = np.argmin(dist_sq, axis=1).astype(np.int64)
        else:
            labels = np.empty(0, dtype=np.int64)

        if not set(np.unique(labels)).issubset(set(range(K))):
            raise RuntimeError(f"{test_id}, {design}: invalid k-means state IDs.")

        s = pd.Series(pd.array([pd.NA] * len(context), dtype="Int64"))
        if len(labels):
            s.iloc[w["window_end_row"].to_numpy(dtype=int)] = labels
        native[design] = s

        expected = len(w)
        observed = int(native[design].notna().sum())
        passed = observed == expected
        integrity_rows.append({
            "test_id": test_id, "design": design, "family": "kmeans",
            "n_expected_assignments": expected, "n_observed_assignments": observed,
            "status": "PASS" if passed else "FAIL",
        })
        if not passed:
            raise RuntimeError(f"{test_id}, {design}: assignment-count mismatch.")

    # --------------------------------------------------------
    # HDBSCAN: approximate prediction against fixed train density model
    # --------------------------------------------------------
    for design, L in HDB_SPECS.items():
        w = windows[L]
        Xw = w[WINDOW_MODEL_COLS].to_numpy(dtype=np.float64)
        model, train_states = hdb_models[design]

        if len(Xw):
            labels, strengths = approximate_predict(model, Xw)
            labels = np.asarray(labels, dtype=np.int64)
            strengths = np.asarray(strengths, dtype=float)
        else:
            labels = np.empty(0, dtype=np.int64)
            strengths = np.empty(0, dtype=float)

        if not set(np.unique(labels)).issubset(train_states | {-1}):
            raise RuntimeError(f"{test_id}, {design}: invalid HDBSCAN state IDs.")
        if not np.isfinite(strengths).all() or np.any((strengths < 0) | (strengths > 1)):
            raise RuntimeError(f"{test_id}, {design}: invalid HDBSCAN prediction strengths.")

        s = pd.Series(pd.array([pd.NA] * len(context), dtype="Int64"))
        q = np.full(len(context), np.nan, dtype=float)
        if len(labels):
            end_rows = w["window_end_row"].to_numpy(dtype=int)
            s.iloc[end_rows] = labels
            q[end_rows] = strengths
        native[design] = s
        hdb_strengths[design] = q

        expected = len(w)
        observed = int(native[design].notna().sum())
        passed = observed == expected
        integrity_rows.append({
            "test_id": test_id, "design": design, "family": "HDBSCAN",
            "n_expected_assignments": expected, "n_observed_assignments": observed,
            "status": "PASS" if passed else "FAIL",
        })
        if not passed:
            raise RuntimeError(f"{test_id}, {design}: assignment-count mismatch.")

    # --------------------------------------------------------
    # HMM: Viterbi decode each contiguous test segment separately
    # --------------------------------------------------------
    Xtick = context[TICK_MODEL_COLS].to_numpy(dtype=np.float64)
    seg_ids = context["segment_id"].to_numpy(dtype=np.int64)

    for design, K in HMM_SPECS.items():
        model = hmm_models[design]
        labels = np.full(len(context), -1, dtype=np.int64)

        for seg_id in np.unique(seg_ids):
            idx = np.flatnonzero(seg_ids == seg_id)
            labels[idx] = np.asarray(model.predict(Xtick[idx]), dtype=np.int64)

        if np.any(labels < 0) or not set(np.unique(labels)).issubset(set(range(K))):
            raise RuntimeError(f"{test_id}, {design}: invalid HMM state IDs.")

        native[design] = pd.Series(labels, dtype="Int64")
        observed = int(native[design].notna().sum())
        passed = observed == len(context)
        integrity_rows.append({
            "test_id": test_id, "design": design, "family": "HMM",
            "n_expected_assignments": len(context), "n_observed_assignments": observed,
            "status": "PASS" if passed else "FAIL",
        })
        if not passed:
            raise RuntimeError(f"{test_id}, {design}: assignment-count mismatch.")

    # Keep the native label table narrow and explicit.
    native = native[["row_id", "time", "segment_id", *ALL_DESIGNS]]
    native.to_csv(test_dir / "native_labels.csv", index=False)
    hdb_strengths.to_csv(test_dir / "hdbscan_strengths.csv", index=False, float_format="%.17g")

    assignment_rows = []
    for design in ALL_DESIGNS:
        assigned = native[design].dropna().astype(int)
        row = {
            "test_id": test_id,
            "design": design,
            "n_assigned": int(len(assigned)),
            "assigned_share": float(len(assigned) / len(native)),
            "n_distinct_native_states": int(assigned.nunique()),
        }
        if design.startswith("HDB"):
            row["noise_share_among_assigned"] = float((assigned == -1).mean()) if len(assigned) else np.nan
        else:
            row["noise_share_among_assigned"] = np.nan
        assignment_rows.append(row)
    pd.DataFrame(assignment_rows).to_csv(test_dir / "assignment_summary.csv", index=False)

    test_summary_rows.append({
        "test_id": test_id,
        "n_rows": len(context),
        "n_contiguous_segments": int(context["segment_id"].nunique()),
        "n_non_1s_steps": n_gaps,
        "n_windows_8s": len(windows[8]),
        "n_windows_16s": len(windows[16]),
    })

    print(
        f"PASS | ticks={len(context):,} | segments={context['segment_id'].nunique()} | "
        f"8s windows={len(windows[8]):,} | 16s windows={len(windows[16]):,}"
    )

# ============================================================
# 4. CONSOLIDATED CHECKS
# ============================================================

integrity = pd.DataFrame(integrity_rows)
test_summary = pd.DataFrame(test_summary_rows)
range_audit = pd.concat(all_range_audits, ignore_index=True)

integrity.to_csv(OUTPUT_DIR / "integrity_checks.csv", index=False)
test_summary.to_csv(OUTPUT_DIR / "test_summary.csv", index=False)
range_audit.to_csv(OUTPUT_DIR / "context_range_audit_all_tests.csv", index=False)

if not integrity["status"].eq("PASS").all():
    raise RuntimeError("At least one test native-state assignment failed validation.")

print("\nHELD-OUT NATIVE ASSIGNMENT COMPLETE — all five tests assigned using fixed train-derived artifacts only.")
print(test_summary.to_string(index=False))


OUT-OF-SAMPLE ASSIGNMENT — test1
PASS | ticks=43,201 | segments=1 | 8s windows=43,194 | 16s windows=43,186

OUT-OF-SAMPLE ASSIGNMENT — test2
PASS | ticks=118,801 | segments=1 | 8s windows=118,794 | 16s windows=118,786

OUT-OF-SAMPLE ASSIGNMENT — test3
PASS | ticks=108,001 | segments=1 | 8s windows=107,994 | 16s windows=107,986

OUT-OF-SAMPLE ASSIGNMENT — test4
PASS | ticks=39,601 | segments=1 | 8s windows=39,594 | 16s windows=39,586

OUT-OF-SAMPLE ASSIGNMENT — test5
PASS | ticks=92,401 | segments=1 | 8s windows=92,394 | 16s windows=92,386

HELD-OUT NATIVE ASSIGNMENT COMPLETE — all five tests assigned using fixed train-derived artifacts only.
test_id  n_rows  n_contiguous_segments  n_non_1s_steps  n_windows_8s  n_windows_16s
  test1   43201                      1               0         43194          43186
  test2  118801                      1               0        118794         118786
  test3  108001                      1               0        107994         107986
  test4   396

### 5.4.2 HDBSCAN held-out density-support audit


In [15]:
from pathlib import Path

import numpy as np
import pandas as pd

# ============================================================
# HAI 21.03 — HDBSCAN HELD-OUT ASSIGNMENT AUDIT: HDBSCAN OUT-OF-SAMPLE COVERAGE AUDIT
#
# Purpose:
#   Diagnose the high HDBSCAN -1 rates observed after fixed,
#   train-derived out-of-sample assignment in held-out native assignment.
#
# This cell is label-blind:
#   - no attack labels are used
#   - no response variables are used
#   - no model is refit or adapted
#   - native assignments from held-out native assignment are not changed
#
# It asks three questions:
#   1) Is test noise concentrated in short or long contiguous runs?
#   2) Is test noise mainly associated with windows containing
#      set-point changes, or with otherwise stable context windows?
#   3) How often do noise windows contain context levels outside
#      the marginal train1 min-max range?
#
# Note:
#   Marginal train-range exceedance is only a diagnostic. A window
#   can be outside the learned HDBSCAN density support even when all
#   individual level variables remain inside their train min-max ranges.
# ============================================================

NATIVE_ASSIGNMENT_DIR = Path("outputs/section_5_4/test_native_assignment")
HDB_DIR = Path("outputs/section_5_3/retained_hdbscan")
TEMPORAL_AUDIT_DIR = Path("outputs/preparation/temporal_context_audit")
OUTPUT_DIR = Path("hai21_cell13_hdbscan_coverage_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_IDS = [f"test{i}" for i in range(1, 6)]
HDB_SPECS = {
    "HDB23_8": 8,
    "HDB18_8": 8,
    "HDB15_8": 8,
    "HDB18_16": 16,
}
LEVEL_ATOL = 1e-12
DELTA_ATOL = 1e-12

# ============================================================
# 1. LOAD TRAIN-DERIVED REFERENCE INFORMATION
# ============================================================

retained = pd.read_csv(HDB_DIR / "retained_designs.csv")
required_retained = {"design_id", "window_sec", "train_noise_share"}
if required_retained - set(retained.columns):
    raise ValueError("retained HDBSCAN fitting retained_designs.csv does not match the expected schema.")
retained = retained.set_index("design_id")

if set(HDB_SPECS) - set(retained.index):
    raise ValueError("retained HDBSCAN fitting retained HDBSCAN design list is incomplete.")

for design, L in HDB_SPECS.items():
    if int(retained.loc[design, "window_sec"]) != L:
        raise ValueError(f"{design}: window length differs from the retained HDBSCAN fitting contract.")

transition_summary = pd.read_csv(TEMPORAL_AUDIT_DIR / "low_support_run_summary.csv")
if len(transition_summary) != 1 or "max_duration_sec" not in transition_summary.columns:
    raise ValueError("Temporal audit summary does not contain a unique max_duration_sec.")
TRAIN_TRANSITION_MAX_SEC = float(transition_summary.loc[0, "max_duration_sec"])
if not np.isfinite(TRAIN_TRANSITION_MAX_SEC) or TRAIN_TRANSITION_MAX_SEC <= 0:
    raise ValueError("Invalid train transition-duration reference.")

# ============================================================
# 2. AUDIT EACH TEST / HDBSCAN DESIGN
# ============================================================

summary_rows = []
group_rows = []
noise_run_rows = []
integrity_rows = []

for test_id in TEST_IDS:
    test_dir = NATIVE_ASSIGNMENT_DIR / test_id

    context = pd.read_csv(test_dir / "context_8d.csv")
    native = pd.read_csv(test_dir / "native_labels.csv")
    strengths = pd.read_csv(test_dir / "hdbscan_strengths.csv")
    assignment_summary = pd.read_csv(test_dir / "assignment_summary.csv")

    required_base = {"row_id", "time", "segment_id"}
    for name, df in [("context", context), ("native", native), ("strengths", strengths)]:
        if required_base - set(df.columns):
            raise ValueError(f"{test_id}: {name} file is missing row/time/segment columns.")

    if not np.array_equal(context["row_id"].to_numpy(), np.arange(len(context))):
        raise ValueError(f"{test_id}: context row_id must be exactly 0..n-1.")
    if not np.array_equal(native["row_id"].to_numpy(), context["row_id"].to_numpy()):
        raise ValueError(f"{test_id}: native label row_id does not match context row_id.")
    if not np.array_equal(strengths["row_id"].to_numpy(), context["row_id"].to_numpy()):
        raise ValueError(f"{test_id}: HDBSCAN strength row_id does not match context row_id.")

    level_cols = [c for c in context.columns if c.endswith("_level")]
    delta_cols = [c for c in context.columns if c.endswith("_delta")]
    if len(level_cols) != 4 or len(delta_cols) != 4:
        raise ValueError(f"{test_id}: expected four level and four delta context columns.")

    levels = context[level_cols].to_numpy(dtype=float)
    deltas = context[delta_cols].to_numpy(dtype=float)
    if not np.isfinite(levels).all() or not np.isfinite(deltas).all():
        raise ValueError(f"{test_id}: non-finite context values.")

    # Per-tick diagnostics. These are derived only from the fixed held-out native assignment context representation.
    tick_marginal_oor = np.any((levels < -LEVEL_ATOL) | (levels > 1.0 + LEVEL_ATOL), axis=1).astype(np.int64)
    tick_context_change = np.any(np.abs(deltas) > DELTA_ATOL, axis=1).astype(np.int64)
    tick_abs_delta_l1 = np.abs(deltas).sum(axis=1)

    # Prefix sums make overlapping-window diagnostics O(n), not O(n*L).
    oor_prefix = np.r_[0, np.cumsum(tick_marginal_oor)]
    change_prefix = np.r_[0, np.cumsum(tick_context_change)]
    delta_prefix = np.r_[0.0, np.cumsum(tick_abs_delta_l1)]

    native_idx = native.set_index("row_id")
    strengths_idx = strengths.set_index("row_id")

    for design, L in HDB_SPECS.items():
        windows = pd.read_csv(test_dir / f"window_{L}s.csv")
        required_window = {
            "window_start_row", "window_end_row", "window_end_time",
            "segment_id", "window_samples",
        }
        if required_window - set(windows.columns):
            raise ValueError(f"{test_id}, {design}: window file schema is incomplete.")

        starts = windows["window_start_row"].to_numpy(dtype=np.int64)
        ends = windows["window_end_row"].to_numpy(dtype=np.int64)
        if np.any(ends - starts + 1 != L):
            raise RuntimeError(f"{test_id}, {design}: window support length mismatch.")
        if np.any(windows["window_samples"].to_numpy(dtype=int) != L):
            raise RuntimeError(f"{test_id}, {design}: window_samples mismatch.")

        labels = pd.to_numeric(native_idx.loc[ends, design], errors="raise").to_numpy(dtype=np.int64)
        pred_strength = pd.to_numeric(strengths_idx.loc[ends, design], errors="raise").to_numpy(dtype=float)
        if len(labels) != len(windows) or not np.isfinite(pred_strength).all():
            raise RuntimeError(f"{test_id}, {design}: label/strength alignment failure.")

        n_oor = oor_prefix[ends + 1] - oor_prefix[starts]
        n_change = change_prefix[ends + 1] - change_prefix[starts]
        delta_sum = delta_prefix[ends + 1] - delta_prefix[starts]

        wf = pd.DataFrame({
            "row_id": ends,
            "time": pd.to_datetime(windows["window_end_time"], errors="raise"),
            "segment_id": windows["segment_id"].to_numpy(dtype=np.int64),
            "label": labels,
            "strength": pred_strength,
            "marginal_oor_tick_share": n_oor / L,
            "has_marginal_oor": n_oor > 0,
            "change_tick_share": n_change / L,
            "has_context_change": n_change > 0,
            "is_stable_window": n_change == 0,
            "mean_abs_delta_l1": delta_sum / L,
        })
        wf["is_noise"] = wf["label"].eq(-1)

        # Cross-check against the held-out native assignment assignment summary.
        expected_row = assignment_summary.loc[assignment_summary["design"].eq(design)]
        if len(expected_row) != 1:
            raise RuntimeError(f"{test_id}, {design}: missing unique held-out native assignment assignment summary row.")
        observed_noise_share = float(wf["is_noise"].mean())
        expected_noise_share = float(expected_row.iloc[0]["noise_share_among_assigned"])
        noise_share_match = np.isclose(observed_noise_share, expected_noise_share, atol=1e-12, rtol=0.0)

        # Noise vs non-noise context profiles.
        for is_noise, group_name in [(True, "noise"), (False, "non_noise")]:
            g = wf.loc[wf["is_noise"].eq(is_noise)]
            group_rows.append({
                "test_id": test_id,
                "design": design,
                "window_sec": L,
                "group": group_name,
                "n_windows": int(len(g)),
                "window_share": float(len(g) / len(wf)) if len(wf) else np.nan,
                "stable_window_share": float(g["is_stable_window"].mean()) if len(g) else np.nan,
                "context_change_window_share": float(g["has_context_change"].mean()) if len(g) else np.nan,
                "mean_change_tick_share": float(g["change_tick_share"].mean()) if len(g) else np.nan,
                "marginal_oor_window_share": float(g["has_marginal_oor"].mean()) if len(g) else np.nan,
                "mean_marginal_oor_tick_share": float(g["marginal_oor_tick_share"].mean()) if len(g) else np.nan,
                "mean_abs_delta_l1": float(g["mean_abs_delta_l1"].mean()) if len(g) else np.nan,
                "median_abs_delta_l1": float(g["mean_abs_delta_l1"].median()) if len(g) else np.nan,
                "mean_prediction_strength": float(g["strength"].mean()) if len(g) else np.nan,
                "median_prediction_strength": float(g["strength"].median()) if len(g) else np.nan,
            })

        # Contiguous -1 runs on the 1-Hz window-end timeline.
        # A train transition lasting T seconds can affect up to T + L - 1
        # trailing-window assignments, so use that conservative duration
        # as the comparison threshold rather than T itself.
        transition_affected_window_max_sec = TRAIN_TRANSITION_MAX_SEC + L - 1
        noise = wf.loc[wf["is_noise"]].copy()
        if len(noise):
            new_run = (
                noise["segment_id"].ne(noise["segment_id"].shift())
                | noise["row_id"].diff().ne(1)
            )
            noise["noise_run_id"] = new_run.cumsum().astype(np.int64) - 1

            run_durations = []
            for run_id, r in noise.groupby("noise_run_id", sort=True):
                duration_sec = int(len(r))
                run_durations.append(duration_sec)
                noise_run_rows.append({
                    "test_id": test_id,
                    "design": design,
                    "window_sec": L,
                    "noise_run_id": int(run_id),
                    "segment_id": int(r["segment_id"].iloc[0]),
                    "start_row": int(r["row_id"].iloc[0]),
                    "end_row": int(r["row_id"].iloc[-1]),
                    "start_time": r["time"].iloc[0],
                    "end_time": r["time"].iloc[-1],
                    "duration_sec": duration_sec,
                    "longer_than_transition_affected_window_max": bool(duration_sec > transition_affected_window_max_sec),
                    "stable_window_share": float(r["is_stable_window"].mean()),
                    "context_change_window_share": float(r["has_context_change"].mean()),
                    "marginal_oor_window_share": float(r["has_marginal_oor"].mean()),
                    "mean_marginal_oor_tick_share": float(r["marginal_oor_tick_share"].mean()),
                    "mean_abs_delta_l1": float(r["mean_abs_delta_l1"].mean()),
                    "mean_prediction_strength": float(r["strength"].mean()),
                })

            run_durations = np.asarray(run_durations, dtype=float)
            run_size = noise.groupby("noise_run_id")["row_id"].transform("size")
            long_noise_share = float((run_size > transition_affected_window_max_sec).mean())

            n_noise_runs = int(len(run_durations))
            median_noise_run = float(np.median(run_durations))
            p90_noise_run = float(np.quantile(run_durations, 0.90))
            max_noise_run = float(np.max(run_durations))
            n_long_noise_runs = int(np.sum(run_durations > transition_affected_window_max_sec))
        else:
            long_noise_share = 0.0
            n_noise_runs = 0
            median_noise_run = np.nan
            p90_noise_run = np.nan
            max_noise_run = np.nan
            n_long_noise_runs = 0

        train_noise_share = float(retained.loc[design, "train_noise_share"])
        summary_rows.append({
            "test_id": test_id,
            "design": design,
            "window_sec": L,
            "train_noise_share": train_noise_share,
            "test_noise_share": observed_noise_share,
            "test_to_train_noise_ratio": float(observed_noise_share / train_noise_share) if train_noise_share > 0 else np.nan,
            "n_noise_runs": n_noise_runs,
            "median_noise_run_sec": median_noise_run,
            "p90_noise_run_sec": p90_noise_run,
            "max_noise_run_sec": max_noise_run,
            "train_transition_max_sec": TRAIN_TRANSITION_MAX_SEC,
            "transition_affected_window_max_sec": transition_affected_window_max_sec,
            "n_noise_runs_longer_than_transition_affected_window_max": n_long_noise_runs,
            "share_noise_windows_in_long_runs": long_noise_share,
            "noise_stable_window_share": float(wf.loc[wf["is_noise"], "is_stable_window"].mean()) if wf["is_noise"].any() else np.nan,
            "noise_marginal_oor_window_share": float(wf.loc[wf["is_noise"], "has_marginal_oor"].mean()) if wf["is_noise"].any() else np.nan,
            "nonnoise_stable_window_share": float(wf.loc[~wf["is_noise"], "is_stable_window"].mean()) if (~wf["is_noise"]).any() else np.nan,
            "nonnoise_marginal_oor_window_share": float(wf.loc[~wf["is_noise"], "has_marginal_oor"].mean()) if (~wf["is_noise"]).any() else np.nan,
        })

        integrity_rows.append({
            "test_id": test_id,
            "design": design,
            "n_windows": int(len(wf)),
            "n_native_assignments": int(pd.Series(labels).notna().sum()),
            "noise_share_matches_cell12": bool(noise_share_match),
            "window_end_rows_unique": bool(pd.Series(ends).is_unique),
            "strengths_finite": bool(np.isfinite(pred_strength).all()),
            "status": "PASS" if (noise_share_match and pd.Series(ends).is_unique and np.isfinite(pred_strength).all()) else "FAIL",
        })

# ============================================================
# 3. SAVE CONSOLIDATED AUDIT OUTPUTS
# ============================================================

summary = pd.DataFrame(summary_rows)
groups = pd.DataFrame(group_rows)
noise_runs = pd.DataFrame(noise_run_rows)
integrity = pd.DataFrame(integrity_rows)

summary.to_csv(OUTPUT_DIR / "hdb_noise_summary.csv", index=False)
groups.to_csv(OUTPUT_DIR / "noise_vs_nonnoise_context.csv", index=False)
noise_runs.to_csv(OUTPUT_DIR / "noise_runs.csv", index=False)
integrity.to_csv(OUTPUT_DIR / "integrity_checks.csv", index=False)

pd.DataFrame([{
    "train_transition_max_sec": TRAIN_TRANSITION_MAX_SEC,
    "transition_window_rule": "a transition lasting T seconds can affect at most T + L - 1 trailing-window assignments",
    "level_out_of_range_rule": "any of four scaled level variables < 0 or > 1 within the window",
    "context_change_rule": "any non-zero scaled signed delta within the window",
    "noise_definition": "HDBSCAN approximate_predict label == -1",
    "models_modified": False,
    "attack_labels_used": False,
    "response_variables_used": False,
}]).to_csv(OUTPUT_DIR / "settings.csv", index=False)

if not integrity["status"].eq("PASS").all():
    raise RuntimeError("At least one HDBSCAN held-out assignment audit HDBSCAN audit integrity check failed.")

print("HDBSCAN HELD-OUT ASSIGNMENT AUDIT COMPLETE — HDBSCAN out-of-sample coverage audit passed integrity checks.")
print(f"Train transition-duration reference: max = {TRAIN_TRANSITION_MAX_SEC:.0f} s")
print(summary[[
    "test_id", "design", "test_noise_share", "n_noise_runs",
    "median_noise_run_sec", "max_noise_run_sec",
    "share_noise_windows_in_long_runs", "noise_stable_window_share",
    "noise_marginal_oor_window_share",
]].to_string(index=False))

HDBSCAN HELD-OUT ASSIGNMENT AUDIT COMPLETE — HDBSCAN out-of-sample coverage audit passed integrity checks.
Train transition-duration reference: max = 18 s
test_id   design  test_noise_share  n_noise_runs  median_noise_run_sec  max_noise_run_sec  share_noise_windows_in_long_runs  noise_stable_window_share  noise_marginal_oor_window_share
  test1  HDB23_8          0.797588             2               17225.5            17876.0                          1.000000                   0.998026                         0.217381
  test1  HDB18_8          0.797588             2               17225.5            17876.0                          1.000000                   0.998026                         0.217381
  test1  HDB15_8          0.384197             2                8297.5            16575.0                          0.998795                   0.996023                         0.451281
  test1 HDB18_16          0.797689             2               17224.5            17875.0                    

### 5.4.3 Apply frozen stabilization and meta-mode mappings


In [20]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# HAI 21.03 — HELD-OUT SERF POST-PROCESSING
# OUT-OF-SAMPLE SERF POST-PROCESSING
#
# Purpose:
#   - take the fixed native-state assignments from held-out native assignment;
#   - construct preliminary (unstabilized) test episodes;
#   - apply the same deterministic boundary-stabilization rule used
#     on train1, for dmin = 5 s (primary) and 10 s (sensitivity);
#   - apply the FIXED train-derived native_state -> meta_mode mapping
#     from train-only meta-mode alignment for q = 7;
#   - retain HDBSCAN -1 as protected M0;
#   - never refit, recluster, or remap using test data.
#
# Test context values are used only by the deterministic local
# stabilization rule (episode-profile distance), exactly as specified
# by the fixed SERF post-processing rule. No parameter is estimated
# from test data.
# ============================================================

NATIVE_ASSIGNMENT_DIR = Path("outputs/section_5_4/test_native_assignment")

# train-only meta-mode alignment v2 writes to outputs/section_5_3/train_meta_alignment.
# Keep the older name only as a compatibility fallback.
META_ALIGNMENT_DIR_CANDIDATES = [
    Path("outputs/section_5_3/train_meta_alignment"),
    Path("hai21_cell15_train_prototypes_meta_alignment"),
]
META_ALIGNMENT_DIR = next(
    (d for d in META_ALIGNMENT_DIR_CANDIDATES if (d / "native_to_meta_mapping.csv").exists()),
    None,
)
if META_ALIGNMENT_DIR is None:
    searched = ", ".join(str(d / "native_to_meta_mapping.csv") for d in META_ALIGNMENT_DIR_CANDIDATES)
    raise FileNotFoundError(
        "train-only meta-mode alignment mapping not found. Searched: " + searched
    )

OUTPUT_DIR = Path("outputs/section_5_4/test_serf_postprocessing")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_IDS = [f"test{i}" for i in range(1, 6)]
DMIN_VALUES = [5, 10]
Q_FIXED = 7
PROTECTED_NOISE = -1

DESIGNS = {
    "KM4_8": "kmeans",
    "KM7_8": "kmeans",
    "KM13_8": "kmeans",
    "KM13_16": "kmeans",
    "HDB23_8": "hdbscan",
    "HDB18_8": "hdbscan",
    "HDB15_8": "hdbscan",
    "HDB18_16": "hdbscan",
    "HMM4": "hmm",
    "HMM5": "hmm",
    "HMM8": "hmm",
}

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def build_episodes(labels, row_ids, segment_id):
    """Build contiguous episodes within one already isolated segment."""
    episodes = []
    start_pos = None
    prev = None

    for pos, value in enumerate(labels):
        if pd.isna(value):
            if start_pos is not None:
                episodes.append([
                    int(row_ids[start_pos]),
                    int(row_ids[pos - 1]),
                    int(prev),
                    int(segment_id),
                ])
                start_pos = None
                prev = None
            continue

        value = int(value)
        if start_pos is None:
            start_pos = pos
            prev = value
        elif value != prev:
            episodes.append([
                int(row_ids[start_pos]),
                int(row_ids[pos - 1]),
                int(prev),
                int(segment_id),
            ])
            start_pos = pos
            prev = value

    if start_pos is not None:
        episodes.append([
            int(row_ids[start_pos]),
            int(row_ids[len(labels) - 1]),
            int(prev),
            int(segment_id),
        ])

    return episodes


def merge_adjacent_same(episodes):
    if not episodes:
        return []
    merged = [episodes[0].copy()]
    for start, end, label, seg_id in episodes[1:]:
        last = merged[-1]
        if seg_id == last[3] and label == last[2] and start == last[1] + 1:
            last[1] = end
        else:
            merged.append([start, end, label, seg_id])
    return merged


def episode_mean(X, start, end):
    return X[start:end + 1].mean(axis=0)


def stabilize_segment(episodes, X, dmin, protect_noise=False):
    """
    Same deterministic local rule as train-only boundary stabilization, applied within one
    contiguous test segment only.
    """
    eps = [e.copy() for e in episodes]
    actions = []

    while True:
        changed = False

        for i, (start, end, label, seg_id) in enumerate(eps):
            duration = end - start + 1
            if duration >= dmin:
                continue
            if protect_noise and label == PROTECTED_NOISE:
                continue

            left = eps[i - 1] if i > 0 else None
            right = eps[i + 1] if i + 1 < len(eps) else None

            # Never cross missing coverage or segment boundaries.
            if left is not None and (left[3] != seg_id or left[1] + 1 != start):
                left = None
            if right is not None and (right[3] != seg_id or end + 1 != right[0]):
                right = None

            valid_left = left is not None and not (
                protect_noise and left[2] == PROTECTED_NOISE
            )
            valid_right = right is not None and not (
                protect_noise and right[2] == PROTECTED_NOISE
            )

            # Bridge-like interruption between equal non-noise labels.
            if valid_left and valid_right and left[2] == right[2]:
                old_label = label
                new_label = left[2]
                eps[i][2] = new_label
                eps = merge_adjacent_same(eps)
                actions.append({
                    "segment_id": seg_id,
                    "rule": "bridge_same_neighbours",
                    "start_row": start,
                    "end_row": end,
                    "duration_sec": duration,
                    "from_label": old_label,
                    "to_label": new_label,
                    "distance": 0.0,
                })
                changed = True
                break

            current_profile = episode_mean(X, start, end)
            candidates = []

            if valid_left:
                dist = float(np.linalg.norm(
                    current_profile - episode_mean(X, left[0], left[1])
                ))
                candidates.append((
                    dist,
                    -(left[1] - left[0] + 1),
                    0,  # deterministic left tie-break
                    left[2],
                    "left",
                ))

            if valid_right:
                dist = float(np.linalg.norm(
                    current_profile - episode_mean(X, right[0], right[1])
                ))
                candidates.append((
                    dist,
                    -(right[1] - right[0] + 1),
                    1,
                    right[2],
                    "right",
                ))

            if not candidates:
                continue

            candidates.sort()
            dist, _, _, new_label, side = candidates[0]
            old_label = label
            eps[i][2] = new_label
            eps = merge_adjacent_same(eps)
            actions.append({
                "segment_id": seg_id,
                "rule": f"nearest_profile_{side}",
                "start_row": start,
                "end_row": end,
                "duration_sec": duration,
                "from_label": old_label,
                "to_label": new_label,
                "distance": dist,
            })
            changed = True
            break

        if not changed:
            break

    return eps, actions


def episodes_to_frame(episodes, time_values, X, feature_cols):
    rows = []
    for episode_id, (start, end, label, seg_id) in enumerate(episodes):
        profile = episode_mean(X, start, end)
        row = {
            "episode_id": episode_id,
            "segment_id": seg_id,
            "start_row": start,
            "end_row": end,
            "start_time": time_values[start],
            "end_time": time_values[end],
            "duration_sec": end - start + 1,
            "native_state": label,
        }
        for col, value in zip(feature_cols, profile):
            row[f"mean_{col}"] = float(value)
        rows.append(row)
    return pd.DataFrame(rows)


def labels_from_episodes(n_rows, episodes):
    state = pd.Series(pd.array([pd.NA] * n_rows, dtype="Int64"))
    episode_id = pd.Series(pd.array([pd.NA] * n_rows, dtype="Int64"))
    for eid, (start, end, label, _) in enumerate(episodes):
        state.iloc[start:end + 1] = int(label)
        episode_id.iloc[start:end + 1] = int(eid)
    return state, episode_id


def episode_ids_from_labels(labels, segment_ids):
    """Contiguous episode IDs for an arbitrary timestamp-level label series."""
    out = pd.Series(pd.array([pd.NA] * len(labels), dtype="Int64"))
    next_id = 0

    for seg_id in pd.unique(segment_ids):
        idx = np.flatnonzero(np.asarray(segment_ids) == seg_id)
        if len(idx) == 0:
            continue
        values = labels.iloc[idx].to_numpy(dtype=object)
        rows = idx.astype(int)
        eps = build_episodes(values, rows, int(seg_id))
        for start, end, _, _ in eps:
            out.iloc[start:end + 1] = next_id
            next_id += 1

    return out


# ============================================================
# 1. LOAD FIXED TRAIN-DERIVED q=7 MAPPINGS
# ============================================================

mapping_all = pd.read_csv(META_ALIGNMENT_DIR / "native_to_meta_mapping.csv")
required_mapping_cols = {
    "dmin", "q", "design", "family", "native_state", "meta_mode"
}
if required_mapping_cols - set(mapping_all.columns):
    raise ValueError("train-only meta-mode alignment native_to_meta_mapping.csv has an unexpected schema.")

mapping_all["dmin"] = pd.to_numeric(mapping_all["dmin"], errors="raise").astype(int)
mapping_all["q"] = pd.to_numeric(mapping_all["q"], errors="raise").astype(int)
mapping_all["native_state"] = pd.to_numeric(
    mapping_all["native_state"], errors="raise"
).astype(int)
mapping_all["meta_mode"] = pd.to_numeric(
    mapping_all["meta_mode"], errors="raise"
).astype(int)

mapping_q7 = mapping_all.loc[
    mapping_all["q"].eq(Q_FIXED) & mapping_all["dmin"].isin(DMIN_VALUES)
].copy()

if set(mapping_q7["dmin"].unique()) != set(DMIN_VALUES):
    raise RuntimeError("train-only meta-mode alignment does not contain q=7 mappings for both dmin=5 and dmin=10.")
if set(mapping_q7["design"].unique()) != set(DESIGNS):
    raise RuntimeError("train-only meta-mode alignment q=7 mapping does not cover all retained designs.")
if mapping_q7.duplicated(["dmin", "design", "native_state"]).any():
    raise RuntimeError("Duplicate native-state mappings detected in train-only meta-mode alignment q=7 contract.")

# HDBSCAN -1 must be M0 in both stabilization branches.
hdb_noise_map = mapping_q7.loc[
    mapping_q7["design"].str.startswith("HDB") & mapping_q7["native_state"].eq(-1)
]
if len(hdb_noise_map) != 2 * 4 or not hdb_noise_map["meta_mode"].eq(0).all():
    raise RuntimeError("HDBSCAN -1 -> M0 mapping is incomplete or inconsistent.")

mapping_q7.to_csv(OUTPUT_DIR / "fixed_q7_mapping_contract.csv", index=False)

# ============================================================
# 2. PROCESS TEST RUNS INDEPENDENTLY
# ============================================================

summary_rows = []
action_rows = []
check_rows = []

for test_id in TEST_IDS:
    test_in = NATIVE_ASSIGNMENT_DIR / test_id
    native_file = test_in / "native_labels.csv"
    context_file = test_in / "context_8d.csv"

    if not native_file.exists() or not context_file.exists():
        raise FileNotFoundError(f"{test_id}: missing held-out native assignment native/context outputs.")

    native = pd.read_csv(native_file)
    context = pd.read_csv(context_file)

    for df, name in [(native, "native_labels"), (context, "context_8d")]:
        if not {"row_id", "time", "segment_id"}.issubset(df.columns):
            raise ValueError(f"{test_id}: {name} missing row_id/time/segment_id.")

    native["time"] = pd.to_datetime(native["time"], errors="coerce")
    context["time"] = pd.to_datetime(context["time"], errors="coerce")
    if native["time"].isna().any() or context["time"].isna().any():
        raise ValueError(f"{test_id}: invalid timestamps.")

    if len(native) != len(context):
        raise RuntimeError(f"{test_id}: native/context row-count mismatch.")
    if not np.array_equal(native["row_id"].to_numpy(), context["row_id"].to_numpy()):
        raise RuntimeError(f"{test_id}: native/context row_id mismatch.")
    if not np.array_equal(native["segment_id"].to_numpy(), context["segment_id"].to_numpy()):
        raise RuntimeError(f"{test_id}: native/context segment_id mismatch.")
    if not native["time"].equals(context["time"]):
        raise RuntimeError(f"{test_id}: native/context timestamp mismatch.")

    feature_cols = [
        c for c in context.columns if c.endswith("_level") or c.endswith("_delta")
    ]
    if len(feature_cols) != 8:
        raise ValueError(f"{test_id}: expected 8 context features, found {len(feature_cols)}.")

    X = context[feature_cols].to_numpy(dtype=float)
    if not np.isfinite(X).all():
        raise ValueError(f"{test_id}: non-finite context values.")

    n_rows = len(context)
    row_ids = context["row_id"].to_numpy(dtype=int)
    if not np.array_equal(row_ids, np.arange(n_rows, dtype=int)):
        raise RuntimeError(
            f"{test_id}: held-out SERF post-processing expects row_id to be the zero-based row position."
        )

    segment_ids = context["segment_id"].to_numpy(dtype=int)
    time_values = context["time"].to_numpy()

    test_out = OUTPUT_DIR / test_id
    test_out.mkdir(parents=True, exist_ok=True)

    for design, family in DESIGNS.items():
        if design not in native.columns:
            raise ValueError(f"{test_id}: native label column missing for {design}.")

        design_out = test_out / design
        design_out.mkdir(parents=True, exist_ok=True)

        native_state = pd.to_numeric(native[design], errors="coerce").astype("Int64")
        labelled_mask = native_state.notna().to_numpy()

        # ----------------------------------------------------
        # Preliminary / unstabilized episodes, per contiguous segment
        # ----------------------------------------------------
        preliminary = []
        for seg_id in pd.unique(segment_ids):
            idx = np.flatnonzero(segment_ids == seg_id)
            seg_labels = native_state.iloc[idx].to_numpy(dtype=object)
            preliminary.extend(
                build_episodes(seg_labels, row_ids[idx], int(seg_id))
            )

        prelim_df = episodes_to_frame(
            preliminary, time_values, X, feature_cols
        )
        prelim_df.to_csv(design_out / "preliminary_episodes.csv", index=False)

        prelim_state, prelim_episode_id = labels_from_episodes(n_rows, preliminary)
        if not prelim_state.equals(native_state):
            raise RuntimeError(
                f"{test_id}, {design}: preliminary episode reconstruction does not reproduce native labels."
            )

        prelim_timeline = pd.DataFrame({
            "row_id": row_ids,
            "time": context["time"],
            "segment_id": segment_ids,
            "native_state": native_state,
            "preliminary_episode_id": prelim_episode_id,
        })
        prelim_timeline.to_csv(
            design_out / "preliminary_episode_labels.csv", index=False
        )

        original_noise_mask = None
        if family == "hdbscan":
            original_noise_mask = native_state.eq(PROTECTED_NOISE).fillna(False).to_numpy()

        # ----------------------------------------------------
        # Fixed stabilization branches + fixed q=7 mapping
        # ----------------------------------------------------
        for dmin in DMIN_VALUES:
            stabilized = []
            local_actions = []

            # Apply the same deterministic stabilization rule separately
            # within each contiguous segment; never cross a time gap.
            for seg_id in pd.unique(segment_ids):
                seg_prelim = [e for e in preliminary if e[3] == int(seg_id)]
                seg_stable, seg_actions = stabilize_segment(
                    seg_prelim,
                    X,
                    dmin=dmin,
                    protect_noise=(family == "hdbscan"),
                )
                stabilized.extend(seg_stable)
                local_actions.extend(seg_actions)

            stabilized = sorted(stabilized, key=lambda e: (e[3], e[0]))
            stable_state, stable_episode_id = labels_from_episodes(n_rows, stabilized)

            mapping_part = mapping_q7.loc[
                mapping_q7["dmin"].eq(dmin) & mapping_q7["design"].eq(design),
                ["native_state", "meta_mode"],
            ]
            state_to_meta = dict(zip(
                mapping_part["native_state"].astype(int),
                mapping_part["meta_mode"].astype(int),
            ))

            encountered_states = set(stable_state.dropna().astype(int).unique())
            missing_map = encountered_states - set(state_to_meta)
            if missing_map:
                raise RuntimeError(
                    f"{test_id}, {design}, dmin={dmin}: q=7 mapping missing states {sorted(missing_map)}."
                )

            meta_mode = pd.Series(pd.array([pd.NA] * n_rows, dtype="Int64"))
            mask = stable_state.notna()
            meta_mode.loc[mask] = stable_state.loc[mask].astype(int).map(state_to_meta).astype("Int64")
            meta_episode_id = episode_ids_from_labels(meta_mode, segment_ids)

            # ------------------------------------------------
            # Integrity checks
            # ------------------------------------------------
            coverage_same = bool(np.array_equal(labelled_mask, stable_state.notna().to_numpy()))
            no_new_states = encountered_states.issubset(
                set(native_state.dropna().astype(int).unique())
            )
            mapping_complete = bool(meta_mode.notna().to_numpy().tolist() == stable_state.notna().to_numpy().tolist())

            noise_mask_same = True
            m0_matches_noise = True
            if family == "hdbscan":
                stable_noise_mask = stable_state.eq(PROTECTED_NOISE).fillna(False).to_numpy()
                noise_mask_same = bool(np.array_equal(original_noise_mask, stable_noise_mask))
                m0_mask = meta_mode.eq(0).fillna(False).to_numpy()
                m0_matches_noise = bool(np.array_equal(stable_noise_mask, m0_mask))
            else:
                # M0 is reserved for HDBSCAN out-of-support/noise only.
                m0_matches_noise = int(meta_mode.eq(0).fillna(False).sum()) == 0

            valid_meta_ids = set(meta_mode.dropna().astype(int).unique()).issubset(
                set(range(0 if family == "hdbscan" else 1, Q_FIXED + 1))
            )

            # Ensure no episode spans more than one segment.
            no_cross_segment_episode = True
            for start, end, _, seg_id in stabilized:
                if not np.all(segment_ids[start:end + 1] == seg_id):
                    no_cross_segment_episode = False
                    break

            remaining_short_nonnoise = int(sum(
                (end - start + 1) < dmin
                and not (family == "hdbscan" and label == PROTECTED_NOISE)
                for start, end, label, _ in stabilized
            ))

            status = "PASS" if all([
                coverage_same,
                no_new_states,
                mapping_complete,
                noise_mask_same,
                m0_matches_noise,
                valid_meta_ids,
                no_cross_segment_episode,
            ]) else "FAIL"

            if status != "PASS":
                raise RuntimeError(
                    f"{test_id}, {design}, dmin={dmin}: SERF test post-processing integrity failure."
                )

            # ------------------------------------------------
            # Save stabilized and mapped outputs
            # ------------------------------------------------
            stable_df = episodes_to_frame(
                stabilized, time_values, X, feature_cols
            )
            stable_df.to_csv(
                design_out / f"stabilized_episodes_dmin{dmin}.csv", index=False
            )

            serf_labels = pd.DataFrame({
                "row_id": row_ids,
                "time": context["time"],
                "segment_id": segment_ids,
                "native_state": native_state,
                "preliminary_episode_id": prelim_episode_id,
                "stabilized_state": stable_state,
                "stabilized_episode_id": stable_episode_id,
                "meta_mode_q7": meta_mode,
                "meta_mode_episode_id_q7": meta_episode_id,
            })
            serf_labels.to_csv(
                design_out / f"serf_labels_dmin{dmin}_q7.csv", index=False
            )

            for action_id, action in enumerate(local_actions):
                action_rows.append({
                    "test_id": test_id,
                    "design": design,
                    "family": family,
                    "dmin": dmin,
                    "action_id": action_id,
                    **action,
                })

            n_changed_ticks = int((
                native_state.notna()
                & stable_state.notna()
                & native_state.ne(stable_state)
            ).sum())

            preliminary_durations = np.array(
                [end - start + 1 for start, end, _, _ in preliminary], dtype=int
            )
            stabilized_durations = np.array(
                [end - start + 1 for start, end, _, _ in stabilized], dtype=int
            )

            summary_rows.append({
                "test_id": test_id,
                "design": design,
                "family": family,
                "dmin": dmin,
                "q": Q_FIXED,
                "n_labelled_ticks": int(labelled_mask.sum()),
                "preliminary_episodes": int(len(preliminary)),
                "stabilized_episodes": int(len(stabilized)),
                "episode_reduction_share": (
                    1.0 - len(stabilized) / len(preliminary)
                    if len(preliminary) else 0.0
                ),
                "n_stabilization_actions": int(len(local_actions)),
                "n_reassigned_ticks": n_changed_ticks,
                "remaining_short_nonnoise_episodes": remaining_short_nonnoise,
                "n_meta_modes_used": int(meta_mode.dropna().nunique()),
                "m0_share_among_assigned": (
                    float(meta_mode.dropna().eq(0).mean())
                    if family == "hdbscan" and meta_mode.notna().any()
                    else np.nan
                ),
                "preliminary_median_duration_sec": (
                    float(np.median(preliminary_durations))
                    if len(preliminary_durations) else np.nan
                ),
                "stabilized_median_duration_sec": (
                    float(np.median(stabilized_durations))
                    if len(stabilized_durations) else np.nan
                ),
            })

            check_rows.append({
                "test_id": test_id,
                "design": design,
                "family": family,
                "dmin": dmin,
                "q": Q_FIXED,
                "label_coverage_preserved": coverage_same,
                "no_new_native_states": no_new_states,
                "q7_mapping_complete": mapping_complete,
                "hdbscan_noise_mask_preserved": noise_mask_same,
                "m0_exactly_matches_hdbscan_noise": m0_matches_noise,
                "valid_meta_mode_ids": valid_meta_ids,
                "no_cross_segment_episode": no_cross_segment_episode,
                "remaining_short_nonnoise_episodes": remaining_short_nonnoise,
                "status": status,
            })

# ============================================================
# 3. CONSOLIDATED AUDIT OUTPUTS
# ============================================================

summary = pd.DataFrame(summary_rows)
actions = pd.DataFrame(action_rows)
checks = pd.DataFrame(check_rows)

summary.to_csv(OUTPUT_DIR / "postprocessing_summary.csv", index=False)
actions.to_csv(OUTPUT_DIR / "stabilization_actions.csv", index=False)
checks.to_csv(OUTPUT_DIR / "integrity_checks.csv", index=False)

pd.DataFrame({
    "parameter": [
        "dmin_values",
        "q_fixed",
        "stabilization_rule",
        "local_profile",
        "distance",
        "hdbscan_noise_policy",
        "meta_mode_mapping",
        "test_refitting",
        "test_reclustering",
        "attack_labels_used",
        "response_variables_used",
    ],
    "value": [
        ",".join(map(str, DMIN_VALUES)),
        Q_FIXED,
        "same deterministic local boundary-stabilization rule as train-only boundary stabilization, applied separately within each contiguous test segment",
        "mean of fixed-train-scaled 8D context over the local test episode interval",
        "euclidean",
        "native_state -1 protected during stabilization and mapped to fixed M0",
        "fixed train-derived native_state -> meta_mode mapping from train-only meta-mode alignment",
        False,
        False,
        False,
        False,
    ],
}).to_csv(OUTPUT_DIR / "settings.csv", index=False)

if not checks["status"].eq("PASS").all():
    raise RuntimeError("At least one held-out SERF post-processing integrity check failed.")

print("=" * 72)
print("HELD-OUT SERF POST-PROCESSING COMPLETE")
print("=" * 72)
print(
    summary.groupby(["dmin", "family"], as_index=False)
    .agg(
        designs=("design", "count"),
        stabilization_actions=("n_stabilization_actions", "sum"),
        reassigned_ticks=("n_reassigned_ticks", "sum"),
    )
    .to_string(index=False)
)
print(f"\nOutput directory: {OUTPUT_DIR}")

HELD-OUT SERF POST-PROCESSING COMPLETE — OUT-OF-SAMPLE SERF POST-PROCESSING
 dmin  family  designs  stabilization_actions  reassigned_ticks
    5 hdbscan       20                      1                 1
    5     hmm       15                      4                 9
    5  kmeans       20                     37                99
   10 hdbscan       20                      1                 1
   10     hmm       15                      7                31
   10  kmeans       20                     55               207

Output directory: outputs/section_5_4/test_serf_postprocessing


## Section 5.5 — Response space and reference regions

The response vector is defined independently of the operating-context variables. Empirical reference intervals are estimated from `train1.csv` only. The 53 train-varying channels receive empirical intervals, while the 22 train-constant channels retain singleton intervals. Context-specific references follow the predefined tail-support rule and use GLOBAL fallback when the context lacks sufficient training support.


### 5.5.1 Define the fixed response-vector contract


In [5]:
# response-vector contract — Train-only response-vector contract
# HAI 21.03 / SERF
#
# Purpose
# -------
# Define the fixed response vector Y_t used by the downstream extremeness analysis.
#
# Fixed analytical contract
# -------------------------
# - train1.csv only
# - explicit exclusion of:
#     * time
#     * four attack/label columns
#     * four operating-context variables used to construct C_t
# - no test data
# - no attack-label-driven selection
# - no correlation/IQR/MAD/low-cardinality filtering
# - no response-space scaling or standardization
#
# The remaining 75 response channels are retained in the response vector:
#   - 53 train-varying channels -> empirical reference intervals in train-only reference-interval contract
#   - 22 train-constant channels -> singleton reference intervals [a_j, a_j]
#
# The response-vector order is frozen here and must be reused unchanged
# in all subsequent cells.
#
# Canonical outputs
# -----------------
# response_vector_contract.csv
# response_variables_train_varying.csv
# response_variables_train_constant.csv
# response_variable_audit.csv
# column_role_audit.csv
# summary.csv
# settings.csv

from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

TRAIN_CSV = Path("train1.csv")
if not TRAIN_CSV.exists():
    raise FileNotFoundError("train1.csv must be placed in the notebook working directory.")

OUT_DIR = Path("outputs/section_5_5/response_vector_contract")
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. FIXED ANALYTICAL ROLES
# ============================================================

TIME_COL = "time"

ATTACK_LABEL_COLS = [
    "attack",
    "attack_P1",
    "attack_P2",
    "attack_P3",
]

CONTEXT_COLS = [
    "P1_B2004",
    "P1_B3004",
    "P1_B3005",
    "P1_B4002",
]

HARD_EXCLUSIONS = [TIME_COL] + ATTACK_LABEL_COLS + CONTEXT_COLS

# Dataset-specific guards.
EXPECTED_RAW_COLUMNS = 84
EXPECTED_RESPONSE_CHANNELS = 75
EXPECTED_TRAIN_VARYING = 53
EXPECTED_TRAIN_CONSTANT = 22
EXPECTED_INVALID = 0


# ============================================================
# 3. LOAD TRAIN1 ONLY
# ============================================================

train = pd.read_csv(TRAIN_CSV)

missing_required = [c for c in HARD_EXCLUSIONS if c not in train.columns]
if missing_required:
    raise ValueError(
        "response-vector contract contract failed: required columns are missing from train1.csv: "
        + ", ".join(missing_required)
    )

if train.shape[1] != EXPECTED_RAW_COLUMNS:
    raise ValueError(
        f"response-vector contract contract failed: expected {EXPECTED_RAW_COLUMNS} raw columns, "
        f"found {train.shape[1]}."
    )

# The response candidate pool is defined purely by analytical role.
# Original column order is preserved and becomes the frozen response-vector order.
response_channels = [c for c in train.columns if c not in HARD_EXCLUSIONS]

if len(response_channels) != EXPECTED_RESPONSE_CHANNELS:
    raise ValueError(
        f"response-vector contract contract failed: expected {EXPECTED_RESPONSE_CHANNELS} response "
        f"channels after hard exclusions, found {len(response_channels)}."
    )


# ============================================================
# 4. TRAIN-ONLY RESPONSE-CHANNEL AUDIT
# ============================================================

audit_rows = []

for response_index, col in enumerate(response_channels):
    raw = train[col]

    # Existing NaN values remain NaN.
    # Newly created NaN values from non-numeric content count as conversion failures.
    numeric = pd.to_numeric(raw, errors="coerce")

    conversion_failures = int(
        (raw.notna() & numeric.isna()).sum()
    )

    values = numeric.to_numpy(dtype=float, na_value=np.nan)
    finite_mask = np.isfinite(values)
    finite = numeric.loc[finite_mask]

    n_rows = int(len(raw))
    n_valid = int(finite.shape[0])
    n_missing_or_nonfinite = int(n_rows - n_valid)
    n_unique = int(finite.nunique(dropna=True)) if n_valid > 0 else 0

    if conversion_failures > 0 or n_valid == 0:
        train_status = "invalid"
        reference_type = "invalid"
        singleton_value = np.nan

    elif n_unique == 1:
        train_status = "train_constant"
        reference_type = "singleton"
        singleton_value = float(finite.iloc[0])

    else:
        train_status = "train_varying"
        reference_type = "empirical"
        singleton_value = np.nan

    # For a truly singleton channel, the mathematical population SD is
    # exactly zero. Some floating-point reduction algorithms can return a
    # tiny non-zero value even when all stored values are identical, so we
    # set it explicitly from the already verified n_unique_train == 1 rule.
    if n_valid == 0:
        sd_train_population = np.nan
    elif n_unique == 1:
        sd_train_population = 0.0
    else:
        sd_train_population = float(finite.std(ddof=0))

    audit_rows.append(
        {
            "response_index": response_index,
            "variable": col,
            "train_status": train_status,
            "reference_type": reference_type,
            "singleton_reference_value": singleton_value,
            "n_rows": n_rows,
            "n_valid_train": n_valid,
            "n_missing_or_nonfinite_train": n_missing_or_nonfinite,
            "missing_or_nonfinite_share": (
                n_missing_or_nonfinite / n_rows if n_rows else np.nan
            ),
            "conversion_failures": conversion_failures,
            "n_unique_train": n_unique,
            "min_train": float(finite.min()) if n_valid else np.nan,
            "max_train": float(finite.max()) if n_valid else np.nan,
            "mean_train": float(finite.mean()) if n_valid else np.nan,
            "sd_train_population": sd_train_population,
        }
    )

response_audit = pd.DataFrame(audit_rows)

train_varying = response_audit.loc[
    response_audit["train_status"] == "train_varying"
].copy()

train_constant = response_audit.loc[
    response_audit["train_status"] == "train_constant"
].copy()

invalid = response_audit.loc[
    response_audit["train_status"] == "invalid"
].copy()


# ============================================================
# 5. FIXED-CONTRACT CHECKS
# ============================================================

observed_counts = {
    "raw_columns": int(train.shape[1]),
    "context_variables": len(CONTEXT_COLS),
    "attack_label_variables": len(ATTACK_LABEL_COLS),
    "time_variables": 1,
    "response_channels_total": int(len(response_channels)),
    "train_varying_response_channels": int(len(train_varying)),
    "train_constant_response_channels": int(len(train_constant)),
    "invalid_response_channels": int(len(invalid)),
}

expected_counts = {
    "raw_columns": EXPECTED_RAW_COLUMNS,
    "response_channels_total": EXPECTED_RESPONSE_CHANNELS,
    "train_varying_response_channels": EXPECTED_TRAIN_VARYING,
    "train_constant_response_channels": EXPECTED_TRAIN_CONSTANT,
    "invalid_response_channels": EXPECTED_INVALID,
}

for key, expected in expected_counts.items():
    observed = observed_counts[key]
    if observed != expected:
        raise ValueError(
            f"response-vector contract contract failed for '{key}': "
            f"expected {expected}, observed {observed}."
        )

# All 75 response channels must be classifiable as empirical or singleton.
if not set(response_audit["reference_type"]).issubset({"empirical", "singleton"}):
    bad = response_audit.loc[
        ~response_audit["reference_type"].isin({"empirical", "singleton"}),
        ["variable", "train_status", "reference_type"],
    ]
    raise RuntimeError(
        "response-vector contract found invalid response channels:\n"
        + bad.to_string(index=False)
    )

# Singleton channels must have exactly one finite train value and zero population SD.
singleton_check = train_constant[
    (train_constant["n_unique_train"] != 1)
    | (train_constant["sd_train_population"] != 0.0)
    | (train_constant["singleton_reference_value"].isna())
]

if not singleton_check.empty:
    raise RuntimeError(
        "response-vector contract singleton-reference contract failed:\n"
        + singleton_check.to_string(index=False)
    )

# Empirical channels must vary globally in train1.
empirical_check = train_varying[
    (train_varying["n_unique_train"] <= 1)
    | (train_varying["sd_train_population"] <= 0.0)
]

if not empirical_check.empty:
    raise RuntimeError(
        "response-vector contract empirical-reference contract failed:\n"
        + empirical_check.to_string(index=False)
    )

# Frozen response indices must cover exactly 0..74 without gaps or duplicates.
expected_indices = list(range(EXPECTED_RESPONSE_CHANNELS))
observed_indices = response_audit["response_index"].astype(int).tolist()

if observed_indices != expected_indices:
    raise RuntimeError(
        "response-vector contract response_index contract failed: expected contiguous indices 0..74 "
        "in original response-column order."
    )


# ============================================================
# 6. AUTHORITATIVE RESPONSE-VECTOR CONTRACT
# ============================================================

response_vector_contract = response_audit[
    [
        "response_index",
        "variable",
        "train_status",
        "reference_type",
        "singleton_reference_value",
        "n_unique_train",
        "n_valid_train",
        "n_missing_or_nonfinite_train",
        "min_train",
        "max_train",
        "mean_train",
        "sd_train_population",
    ]
].copy()

# This file is the canonical downstream contract.
# train-only reference-interval contract and all test-scoring cells should read this file and preserve its order.
response_vector_contract.to_csv(
    OUT_DIR / "response_vector_contract.csv",
    index=False,
)


# ============================================================
# 7. SUBSET OUTPUTS
# ============================================================

train_varying[
    [
        "response_index",
        "variable",
        "n_unique_train",
        "n_valid_train",
        "n_missing_or_nonfinite_train",
        "min_train",
        "max_train",
        "mean_train",
        "sd_train_population",
    ]
].to_csv(
    OUT_DIR / "response_variables_train_varying.csv",
    index=False,
)

train_constant[
    [
        "response_index",
        "variable",
        "singleton_reference_value",
        "n_valid_train",
        "n_missing_or_nonfinite_train",
        "min_train",
        "max_train",
        "mean_train",
        "sd_train_population",
    ]
].to_csv(
    OUT_DIR / "response_variables_train_constant.csv",
    index=False,
)

response_audit.to_csv(
    OUT_DIR / "response_variable_audit.csv",
    index=False,
)


# ============================================================
# 8. FULL RAW-COLUMN ROLE AUDIT
# ============================================================

role_rows = []

response_contract_by_var = (
    response_vector_contract
    .set_index("variable")
    [["response_index", "train_status", "reference_type"]]
    .to_dict(orient="index")
)

for raw_column_index, col in enumerate(train.columns):
    row = {
        "raw_column_index": raw_column_index,
        "variable": col,
        "response_index": np.nan,
        "role": None,
        "train_status": np.nan,
        "reference_type": np.nan,
    }

    if col == TIME_COL:
        row["role"] = "time"

    elif col in ATTACK_LABEL_COLS:
        row["role"] = "attack_label_excluded"

    elif col in CONTEXT_COLS:
        row["role"] = "operating_context_excluded_from_response"

    elif col in response_contract_by_var:
        info = response_contract_by_var[col]
        row["response_index"] = int(info["response_index"])
        row["role"] = "response"
        row["train_status"] = info["train_status"]
        row["reference_type"] = info["reference_type"]

    else:
        raise RuntimeError(
            f"Unclassified train1 column in response-vector contract: {col}"
        )

    role_rows.append(row)

column_roles = pd.DataFrame(role_rows)

column_roles.to_csv(
    OUT_DIR / "column_role_audit.csv",
    index=False,
)


# ============================================================
# 9. SUMMARY + SETTINGS
# ============================================================

summary_rows = [
    {"metric": key, "value": value}
    for key, value in observed_counts.items()
]

summary_rows.extend([
    {
        "metric": "response_vector_dimension",
        "value": EXPECTED_RESPONSE_CHANNELS,
    },
    {
        "metric": "empirical_reference_channels",
        "value": EXPECTED_TRAIN_VARYING,
    },
    {
        "metric": "singleton_reference_channels",
        "value": EXPECTED_TRAIN_CONSTANT,
    },
])

pd.DataFrame(summary_rows).to_csv(
    OUT_DIR / "summary.csv",
    index=False,
)

settings_rows = [
    ("input_file", str(TRAIN_CSV)),
    ("train_only", True),
    ("test_data_used", False),
    ("attack_labels_used_for_selection", False),
    ("time_column", TIME_COL),
    ("attack_label_columns", "|".join(ATTACK_LABEL_COLS)),
    (
        "context_columns_excluded_from_response",
        "|".join(CONTEXT_COLS),
    ),
    (
        "response_vector_rule",
        "all columns remaining after explicit hard exclusions",
    ),
    (
        "response_vector_dimension",
        EXPECTED_RESPONSE_CHANNELS,
    ),
    (
        "train_varying_rule",
        "numeric finite train values with n_unique_train > 1",
    ),
    (
        "train_constant_rule",
        "numeric finite train values with n_unique_train == 1",
    ),
    (
        "train_varying_reference_type",
        "empirical reference intervals constructed in train-only reference-interval contract",
    ),
    (
        "train_constant_reference_type",
        "singleton interval [a_j, a_j] equal to the observed train value",
    ),
    (
        "response_vector_order",
        "original train1 column order after hard exclusions",
    ),
    ("correlation_filtering", False),
    ("iqr_filtering", False),
    ("mad_filtering", False),
    ("low_cardinality_filtering", False),
    ("response_scaling_or_standardization", False),
]

pd.DataFrame(
    settings_rows,
    columns=["setting", "value"],
).to_csv(
    OUT_DIR / "settings.csv",
    index=False,
)


# ============================================================
# 10. COMPACT NOTEBOOK REPORT
# ============================================================

print("RESPONSE-VECTOR CONTRACT COMPLETE")
print(f"Input: {TRAIN_CSV}")
print(f"Rows: {len(train):,}")
print(f"Raw columns: {train.shape[1]}")
print(
    f"Hard exclusions: {len(HARD_EXCLUSIONS)} "
    f"(1 time + {len(ATTACK_LABEL_COLS)} labels "
    f"+ {len(CONTEXT_COLS)} context variables)"
)
print(f"Response-vector dimension: {len(response_vector_contract)}")
print(f"  Train-varying / empirical-reference channels: {len(train_varying)}")
print(f"  Train-constant / singleton-reference channels: {len(train_constant)}")
print(f"  Invalid response channels: {len(invalid)}")
print(f"Frozen response indices: 0..{len(response_vector_contract) - 1}")
print(f"Outputs: {OUT_DIR}")

print("\nSingleton-reference channels:")
print(
    train_constant[
        [
            "response_index",
            "variable",
            "singleton_reference_value",
        ]
    ].to_string(index=False)
)


RESPONSE-VECTOR CONTRACT COMPLETE — TRAIN-ONLY RESPONSE-VECTOR CONTRACT
Input: train1.csv
Rows: 216,001
Raw columns: 84
Hard exclusions: 9 (1 time + 4 labels + 4 context variables)
Response-vector dimension: 75
  Train-varying / empirical-reference channels: 53
  Train-constant / singleton-reference channels: 22
  Invalid response channels: 0
Frozen response indices: 0..74
Outputs: outputs/section_5_5/response_vector_contract

Singleton-reference channels:
 response_index    variable  singleton_reference_value
             21   P1_PCV02D                   12.00000
             25   P1_PP01AD               540833.00000
             26   P1_PP01AR               540833.00000
             27   P1_PP01BD                    0.00000
             28   P1_PP01BR                    0.00000
             29    P1_PP02D                    1.00000
             30    P1_PP02R                    1.00000
             31     P1_STSP                    1.00000
             35      P2_ASD                 

### 5.5.2 Estimate GLOBAL and context-conditioned train references


In [6]:
# train-only reference-interval contract — Train-only reference-interval contract
# HAI 21.03 / SERF
#
# Purpose
# -------
# Construct the complete train-derived reference contract for the fixed
# 75-channel response vector Y_t defined in response-vector contract.
#
# Response-channel classes
# ------------------------
# - 53 train-varying channels:
#     empirical global and context-conditioned reference intervals
# - 22 train-constant channels:
#     singleton reference intervals [a_j, a_j], retained unchanged for all
#     contexts and later test scoring
#
# Fixed analytical contract
# -------------------------
# - train1.csv only
# - q99 is the PRIMARY extremeness definition
# - q95 and q98 are SENSITIVITY levels
# - empirical quantiles use Hyndman-Fan Type 8 ("median_unbiased")
# - a context-conditioned EMPIRICAL interval is used only when the train
#   support implies at least 10 expected observations in EACH tail:
#       q95 -> >= 400 train ticks
#       q98 -> >= 1000 train ticks
#       q99 -> >= 2000 train ticks
# - if an empirical context is below the support threshold, the GLOBAL
#   empirical interval of the same coverage is used
# - HDBSCAN native/stabilized -1 and aligned meta-mode M0 do not receive
#   context-conditioned empirical intervals; the GLOBAL interval is used later
# - singleton channels are never re-estimated by context and never widened
# - no test files
# - no attack labels
# - no response scaling/standardization
# - no PCA/covariance/Mahalanobis scoring
# - no threshold selection from attack performance
#
# Canonical outputs
# -----------------
# global_reference_intervals.csv
# context_conditioned_reference_intervals.csv
# reference_support_audit.csv
# reference_coverage_summary.csv
# integrity_checks.csv
# settings.csv

from pathlib import Path
import math
import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS AND FIXED SETTINGS
# ============================================================

TRAIN_FILE = Path("train1.csv")

RESPONSE_CONTRACT_DIR = Path("outputs/section_5_5/response_vector_contract")
TRAIN_STABILIZATION_DIR = Path("outputs/section_5_3/train_boundary_stabilization")
META_ALIGNMENT_DIR = Path("outputs/section_5_3/train_meta_alignment")

OUTPUT_DIR = Path("outputs/section_5_5/train_reference_intervals")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESPONSE_CONTRACT_FILE = RESPONSE_CONTRACT_DIR / "response_vector_contract.csv"
META_MAPPING_FILE = META_ALIGNMENT_DIR / "native_to_meta_mapping.csv"

EXPECTED_RESPONSE_COUNT = 75
EXPECTED_EMPIRICAL_COUNT = 53
EXPECTED_SINGLETON_COUNT = 22

DMIN_VALUES = [5, 10]
Q_FIXED = 7

HDB_NOISE = -1
META_M0 = 0

PRIMARY_COVERAGE = 0.99
SENSITIVITY_COVERAGES = [0.95, 0.98]
COVERAGES = SENSITIVITY_COVERAGES + [PRIMARY_COVERAGE]

MIN_EXPECTED_OBS_PER_TAIL = 10.0
QUANTILE_METHOD = "median_unbiased"  # Hyndman-Fan Type 8

DESIGNS = {
    "KM4_8": "kmeans",
    "KM7_8": "kmeans",
    "KM13_8": "kmeans",
    "KM13_16": "kmeans",
    "HDB23_8": "hdbscan",
    "HDB18_8": "hdbscan",
    "HDB15_8": "hdbscan",
    "HDB18_16": "hdbscan",
    "HMM4": "hmm",
    "HMM5": "hmm",
    "HMM8": "hmm",
}


# ============================================================
# 2. HELPERS
# ============================================================

def coverage_tag(coverage: float) -> str:
    return str(int(round(coverage * 100)))


def tail_probability(coverage: float) -> float:
    # Rounded to avoid floating-point ceiling artefacts.
    return round((1.0 - coverage) / 2.0, 12)


def minimum_context_support(coverage: float) -> int:
    tail = tail_probability(coverage)
    return int(math.ceil(MIN_EXPECTED_OBS_PER_TAIL / tail - 1e-12))


MIN_SUPPORT = {
    coverage: minimum_context_support(coverage)
    for coverage in COVERAGES
}


def empirical_bounds(block: pd.DataFrame, coverage: float):
    """
    Column-wise empirical central interval using Hyndman-Fan Type 8.
    Returns q_low, q_high, lower_series, upper_series.
    """
    alpha = 1.0 - coverage
    q_low = alpha / 2.0
    q_high = 1.0 - alpha / 2.0

    arr = block.to_numpy(dtype=float)

    lower = np.quantile(
        arr,
        q_low,
        axis=0,
        method=QUANTILE_METHOD,
    )
    upper = np.quantile(
        arr,
        q_high,
        axis=0,
        method=QUANTILE_METHOD,
    )

    return (
        q_low,
        q_high,
        pd.Series(lower, index=block.columns, dtype=float),
        pd.Series(upper, index=block.columns, dtype=float),
    )


def outside_rates(
    block: pd.DataFrame,
    lower: pd.Series,
    upper: pd.Series,
):
    below = block.lt(lower, axis=1)
    above = block.gt(upper, axis=1)
    outside = below | above

    return (
        below.mean(axis=0),
        above.mean(axis=0),
        outside.mean(axis=0),
    )


def load_labels(path: Path, value_col: str, n_rows: int) -> pd.Series:
    """Load a row_id-aligned nullable integer label series."""
    frame = pd.read_csv(path)

    required = {"row_id", value_col}
    if not required.issubset(frame.columns):
        raise ValueError(
            f"{path}: expected columns {sorted(required)}, "
            f"found {list(frame.columns)}"
        )

    row_id = pd.to_numeric(
        frame["row_id"],
        errors="raise",
    ).astype(int)

    if row_id.duplicated().any():
        raise ValueError(f"{path}: duplicated row_id")

    if len(row_id) and (
        row_id.min() < 0
        or row_id.max() >= n_rows
    ):
        raise ValueError(
            f"{path}: row_id outside train1 range"
        )

    values = pd.to_numeric(
        frame[value_col],
        errors="coerce",
    )

    out = pd.Series(
        pd.array([pd.NA] * n_rows, dtype="Int64")
    )

    valid = values.notna().to_numpy()

    if valid.any():
        out.iloc[row_id.to_numpy()[valid]] = (
            values.to_numpy()[valid].astype(int)
        )

    return out


def intervals_nested(frame: pd.DataFrame) -> bool:
    """Check q99 contains q98 contains q95."""
    return bool(
        (frame["lower_q99"] <= frame["lower_q98"]).all()
        and (frame["lower_q98"] <= frame["lower_q95"]).all()
        and (frame["lower_q95"] <= frame["upper_q95"]).all()
        and (frame["upper_q95"] <= frame["upper_q98"]).all()
        and (frame["upper_q98"] <= frame["upper_q99"]).all()
    )


def context_source_hierarchy_valid(
    frame: pd.DataFrame,
) -> bool:
    """
    On empirical channels, support thresholds must be monotone:
    q99 context -> q98 context -> q95 context.
    Singleton channels are excluded because their source is always singleton.
    """
    empirical = frame[
        frame["reference_type"] == "empirical"
    ].copy()

    q95 = empirical["reference_source_q95"] == "context"
    q98 = empirical["reference_source_q98"] == "context"
    q99 = empirical["reference_source_q99"] == "context"

    return bool(
        (~q99 | q98).all()
        and (~q98 | q95).all()
    )


def same_source_intervals_nested(
    frame: pd.DataFrame,
) -> bool:
    """
    Nestedness is checked only where all three coverages use the same
    reference source. Mixed context/global-support rows are intentionally
    not compared across coverage levels.
    """
    same_source = (
        (
            frame["reference_source_q95"]
            == frame["reference_source_q98"]
        )
        & (
            frame["reference_source_q98"]
            == frame["reference_source_q99"]
        )
    )

    subset = frame.loc[same_source]

    if subset.empty:
        return True

    return intervals_nested(subset)


# ============================================================
# 3. FIXED RESPONSE-VECTOR CONTRACT FROM RESPONSE-VECTOR CONTRACT
# ============================================================

if not RESPONSE_CONTRACT_FILE.exists():
    raise FileNotFoundError(RESPONSE_CONTRACT_FILE)

contract = pd.read_csv(RESPONSE_CONTRACT_FILE)

required_contract_cols = {
    "response_index",
    "variable",
    "train_status",
    "reference_type",
    "singleton_reference_value",
}

if not required_contract_cols.issubset(contract.columns):
    missing_cols = sorted(
        required_contract_cols - set(contract.columns)
    )
    raise ValueError(
        f"{RESPONSE_CONTRACT_FILE}: missing columns {missing_cols}"
    )

contract["response_index"] = pd.to_numeric(
    contract["response_index"],
    errors="raise",
).astype(int)

contract = contract.sort_values(
    "response_index",
    kind="mergesort",
).reset_index(drop=True)

if len(contract) != EXPECTED_RESPONSE_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_RESPONSE_COUNT} response channels from response-vector contract, "
        f"found {len(contract)}"
    )

if contract["response_index"].tolist() != list(
    range(EXPECTED_RESPONSE_COUNT)
):
    raise RuntimeError(
        "response-vector contract response_index must be contiguous 0..74"
    )

if contract["variable"].duplicated().any():
    raise RuntimeError(
        "Duplicated response variables in response-vector contract contract"
    )

allowed_reference_types = {"empirical", "singleton"}

if not set(contract["reference_type"]).issubset(
    allowed_reference_types
):
    bad = sorted(
        set(contract["reference_type"])
        - allowed_reference_types
    )
    raise RuntimeError(
        f"Unexpected reference types in response-vector contract contract: {bad}"
    )

empirical_contract = contract[
    contract["reference_type"] == "empirical"
].copy()

singleton_contract = contract[
    contract["reference_type"] == "singleton"
].copy()

if len(empirical_contract) != EXPECTED_EMPIRICAL_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_EMPIRICAL_COUNT} empirical channels, "
        f"found {len(empirical_contract)}"
    )

if len(singleton_contract) != EXPECTED_SINGLETON_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_SINGLETON_COUNT} singleton channels, "
        f"found {len(singleton_contract)}"
    )

if singleton_contract[
    "singleton_reference_value"
].isna().any():
    raise RuntimeError(
        "Singleton channels must have a fixed train reference value"
    )

RESPONSE_VARS = contract["variable"].astype(str).tolist()
EMPIRICAL_VARS = (
    empirical_contract["variable"].astype(str).tolist()
)
SINGLETON_VARS = (
    singleton_contract["variable"].astype(str).tolist()
)


# ============================================================
# 4. TRAIN RESPONSE MATRIX
# ============================================================

train = pd.read_csv(TRAIN_FILE)

missing = [
    c for c in RESPONSE_VARS
    if c not in train.columns
]

if missing:
    raise ValueError(
        f"train1.csv is missing response channels: {missing}"
    )

Y = train[RESPONSE_VARS].apply(
    pd.to_numeric,
    errors="coerce",
)

Y = Y.replace(
    [np.inf, -np.inf],
    np.nan,
)

if Y.isna().any().any():
    bad = Y.columns[
        Y.isna().any()
    ].tolist()

    raise ValueError(
        f"Non-finite train response values in: {bad}"
    )

Y = Y.astype(float)
Y_empirical = Y[EMPIRICAL_VARS]

n_rows = len(Y)


# ============================================================
# 5. VERIFY SINGLETON CHANNELS AGAINST TRAIN1
# ============================================================

singleton_values = (
    singleton_contract
    .set_index("variable")["singleton_reference_value"]
    .astype(float)
)

singleton_check_rows = []

for variable in SINGLETON_VARS:
    expected = float(singleton_values.loc[variable])
    values = Y[variable].to_numpy(dtype=float)

    exact_match = bool(
        np.all(values == expected)
    )

    singleton_check_rows.append({
        "variable": variable,
        "expected_singleton_value": expected,
        "n_train_ticks": n_rows,
        "n_unique_train": int(
            pd.Series(values).nunique(dropna=True)
        ),
        "all_train_values_equal_singleton": exact_match,
    })

singleton_check = pd.DataFrame(
    singleton_check_rows
)

if not singleton_check[
    "all_train_values_equal_singleton"
].all():
    bad = singleton_check.loc[
        ~singleton_check[
            "all_train_values_equal_singleton"
        ]
    ]

    raise RuntimeError(
        "response-vector contract singleton contract does not match train1:\n"
        + bad.to_string(index=False)
    )


# ============================================================
# 6. GLOBAL REFERENCE INTERVALS FOR ALL 75 CHANNELS
# ============================================================

g_mu = Y.mean(axis=0)
g_sd = Y.std(axis=0, ddof=0)

# Singleton channels are already verified above by exact equality to the
# train singleton value. Their mathematical population SD is therefore
# exactly zero; force it to zero to avoid tiny floating-point artefacts.
g_sd.loc[SINGLETON_VARS] = 0.0

if (
    g_sd.loc[EMPIRICAL_VARS] <= 0
).any():
    bad = g_sd.loc[
        EMPIRICAL_VARS
    ].index[
        g_sd.loc[EMPIRICAL_VARS] <= 0
    ].tolist()

    raise RuntimeError(
        "Empirical channels contain globally zero-SD variables: "
        f"{bad}"
    )

if (
    g_sd.loc[SINGLETON_VARS] != 0
).any():
    bad = g_sd.loc[
        SINGLETON_VARS
    ].index[
        g_sd.loc[SINGLETON_VARS] != 0
    ].tolist()

    raise RuntimeError(
        "Singleton channels are not globally constant: "
        f"{bad}"
    )

global_reference = contract[
    [
        "response_index",
        "variable",
        "train_status",
        "reference_type",
        "singleton_reference_value",
    ]
].copy()

global_reference["n_train_ticks"] = n_rows
global_reference["mean"] = (
    global_reference["variable"]
    .map(g_mu)
    .astype(float)
)
global_reference["sd_population"] = (
    global_reference["variable"]
    .map(g_sd)
    .astype(float)
)

global_reference = global_reference.set_index(
    "variable"
)

for coverage in COVERAGES:
    tag = coverage_tag(coverage)

    q_low = tail_probability(coverage)
    q_high = 1.0 - q_low

    # Empirical channels: train-derived quantiles.
    _, _, lower_emp, upper_emp = empirical_bounds(
        Y_empirical,
        coverage,
    )

    # Start with singleton limits for all singleton channels.
    lower = pd.Series(
        index=RESPONSE_VARS,
        dtype=float,
    )
    upper = pd.Series(
        index=RESPONSE_VARS,
        dtype=float,
    )

    lower.loc[EMPIRICAL_VARS] = (
        lower_emp.loc[EMPIRICAL_VARS]
    )
    upper.loc[EMPIRICAL_VARS] = (
        upper_emp.loc[EMPIRICAL_VARS]
    )

    lower.loc[SINGLETON_VARS] = (
        singleton_values.loc[SINGLETON_VARS]
    )
    upper.loc[SINGLETON_VARS] = (
        singleton_values.loc[SINGLETON_VARS]
    )

    below_rate, above_rate, outside_rate = (
        outside_rates(
            Y,
            lower,
            upper,
        )
    )

    global_reference[
        f"lower_q{tag}"
    ] = lower

    global_reference[
        f"upper_q{tag}"
    ] = upper

    global_reference[
        f"reference_source_q{tag}"
    ] = np.where(
        global_reference["reference_type"] == "singleton",
        "singleton",
        "global",
    )

    global_reference[
        f"train_below_rate_q{tag}"
    ] = below_rate

    global_reference[
        f"train_above_rate_q{tag}"
    ] = above_rate

    global_reference[
        f"train_outside_rate_q{tag}"
    ] = outside_rate

    global_reference[
        f"nominal_lower_quantile_q{tag}"
    ] = q_low

    global_reference[
        f"nominal_upper_quantile_q{tag}"
    ] = q_high

global_reference = (
    global_reference
    .reset_index()
    .sort_values(
        "response_index",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

global_reference.to_csv(
    OUTPUT_DIR / "global_reference_intervals.csv",
    index=False,
)


# ============================================================
# 7. FIXED q=7 NATIVE -> META MAPPING FROM TRAIN-ONLY META-MODE ALIGNMENT
# ============================================================

if not META_MAPPING_FILE.exists():
    raise FileNotFoundError(META_MAPPING_FILE)

mapping = pd.read_csv(
    META_MAPPING_FILE
)

required_mapping_cols = {
    "design",
    "dmin",
    "q",
    "native_state",
    "meta_mode",
}

if not required_mapping_cols.issubset(
    mapping.columns
):
    missing_cols = sorted(
        required_mapping_cols - set(mapping.columns)
    )
    raise ValueError(
        f"{META_MAPPING_FILE}: missing columns {missing_cols}"
    )

for col in [
    "dmin",
    "q",
    "native_state",
    "meta_mode",
]:
    mapping[col] = pd.to_numeric(
        mapping[col],
        errors="raise",
    ).astype(int)

mapping = mapping[
    (mapping["q"] == Q_FIXED)
    & (mapping["dmin"].isin(DMIN_VALUES))
].copy()

if mapping.empty:
    raise RuntimeError(
        f"No q={Q_FIXED} native-to-meta mapping found"
    )


# ============================================================
# 8. CONTEXT-REFERENCE BUILDER
# ============================================================

global_by_var = (
    global_reference
    .set_index("variable")
)


def build_context_reference(
    labels: pd.Series,
    design: str,
    family: str,
    representation: str,
    dmin=None,
    q=None,
    excluded_context=None,
):
    """
    Build the complete 75-channel reference table for every eligible
    operating-context group.

    Empirical channels:
      - use a context-conditioned empirical interval when support is sufficient
      - otherwise use the global empirical interval of the same coverage

    Singleton channels:
      - always use their fixed [a_j, a_j] interval
      - never depend on context support
    """

    labelled = labels.notna()
    eligible_label = labelled.copy()

    if excluded_context is not None:
        eligible_label &= (
            labels != excluded_context
        )

    n_labelled = int(
        labelled.sum()
    )

    n_eligible_rows = int(
        eligible_label.sum()
    )

    n_predefined_global_rows = int(
        len(Y) - n_eligible_rows
    )

    if n_eligible_rows == 0:
        raise RuntimeError(
            f"{design}/{representation}: "
            "no train rows with an eligible operating context"
        )

    lab = (
        labels.loc[eligible_label]
        .astype(int)
    )

    context_supports = (
        lab.value_counts()
        .sort_index()
    )

    ref_rows = []
    support_rows = []
    coverage_rows = []

    for context_id, support in (
        context_supports.items()
    ):
        context_id = int(
            context_id
        )
        support = int(
            support
        )

        mask = (
            eligible_label
            & (labels == context_id)
        )

        block_all = Y.loc[
            mask
        ]

        block_empirical = (
            block_all[
                EMPIRICAL_VARS
            ]
        )

        context_mu = (
            block_all.mean(axis=0)
        )

        context_sd = (
            block_all.std(
                axis=0,
                ddof=0,
            )
        )

        # Coverage-independent support audit for the context.
        support_row = {
            "reference_scope": "context",
            "design": design,
            "family": family,
            "representation": representation,
            "dmin": dmin,
            "q": q,
            "context_id": context_id,
            "n_train_ticks": support,
        }

        per_coverage = {}

        for coverage in COVERAGES:
            tag = coverage_tag(
                coverage
            )

            tail = tail_probability(
                coverage
            )

            min_support = MIN_SUPPORT[
                coverage
            ]

            context_supported = (
                support >= min_support
            )

            support_row[
                f"tail_probability_per_side_q{tag}"
            ] = tail

            support_row[
                f"minimum_train_ticks_q{tag}"
            ] = min_support

            support_row[
                f"expected_train_ticks_per_tail_q{tag}"
            ] = (
                support * tail
            )

            support_row[
                f"context_supported_q{tag}"
            ] = bool(
                context_supported
            )

            if context_supported:
                (
                    _,
                    _,
                    lower_emp,
                    upper_emp,
                ) = empirical_bounds(
                    block_empirical,
                    coverage,
                )

                empirical_source = (
                    "context"
                )

            else:
                lower_emp = (
                    global_by_var.loc[
                        EMPIRICAL_VARS,
                        f"lower_q{tag}",
                    ]
                )

                upper_emp = (
                    global_by_var.loc[
                        EMPIRICAL_VARS,
                        f"upper_q{tag}",
                    ]
                )

                empirical_source = (
                    "global_support"
                )

            lower = pd.Series(
                index=RESPONSE_VARS,
                dtype=float,
            )

            upper = pd.Series(
                index=RESPONSE_VARS,
                dtype=float,
            )

            source = pd.Series(
                index=RESPONSE_VARS,
                dtype="object",
            )

            # 53 empirical channels.
            lower.loc[
                EMPIRICAL_VARS
            ] = lower_emp.loc[
                EMPIRICAL_VARS
            ]

            upper.loc[
                EMPIRICAL_VARS
            ] = upper_emp.loc[
                EMPIRICAL_VARS
            ]

            source.loc[
                EMPIRICAL_VARS
            ] = empirical_source

            # 22 singleton channels.
            lower.loc[
                SINGLETON_VARS
            ] = singleton_values.loc[
                SINGLETON_VARS
            ]

            upper.loc[
                SINGLETON_VARS
            ] = singleton_values.loc[
                SINGLETON_VARS
            ]

            source.loc[
                SINGLETON_VARS
            ] = "singleton"

            (
                below_rate,
                above_rate,
                outside_rate,
            ) = outside_rates(
                block_all,
                lower,
                upper,
            )

            per_coverage[
                coverage
            ] = {
                "lower": lower,
                "upper": upper,
                "source": source,
                "below_rate": below_rate,
                "above_rate": above_rate,
                "outside_rate": outside_rate,
            }

        # One complete row per response channel.
        for _, contract_row in (
            contract.iterrows()
        ):
            variable = str(
                contract_row["variable"]
            )

            row = {
                "response_index": int(
                    contract_row[
                        "response_index"
                    ]
                ),
                "design": design,
                "family": family,
                "representation": representation,
                "dmin": dmin,
                "q": q,
                "context_id": context_id,
                "variable": variable,
                "train_status": (
                    contract_row[
                        "train_status"
                    ]
                ),
                "reference_type": (
                    contract_row[
                        "reference_type"
                    ]
                ),
                "singleton_reference_value": (
                    contract_row[
                        "singleton_reference_value"
                    ]
                ),
                "n_train_ticks": support,
                "mean_context": float(
                    context_mu.loc[
                        variable
                    ]
                ),
                "sd_population_context": float(
                    context_sd.loc[
                        variable
                    ]
                ),
            }

            for coverage in COVERAGES:
                tag = coverage_tag(
                    coverage
                )

                info = per_coverage[
                    coverage
                ]

                row[
                    f"lower_q{tag}"
                ] = float(
                    info["lower"].loc[
                        variable
                    ]
                )

                row[
                    f"upper_q{tag}"
                ] = float(
                    info["upper"].loc[
                        variable
                    ]
                )

                row[
                    f"reference_source_q{tag}"
                ] = str(
                    info["source"].loc[
                        variable
                    ]
                )

                row[
                    f"train_below_rate_q{tag}"
                ] = float(
                    info["below_rate"].loc[
                        variable
                    ]
                )

                row[
                    f"train_above_rate_q{tag}"
                ] = float(
                    info["above_rate"].loc[
                        variable
                    ]
                )

                row[
                    f"train_outside_rate_q{tag}"
                ] = float(
                    info["outside_rate"].loc[
                        variable
                    ]
                )

            ref_rows.append(
                row
            )

        support_rows.append(
            support_row
        )

    # Timestamp-level empirical-reference coverage summary.
    # Singleton channels are always available and therefore do not affect
    # this support calculation.
    for coverage in COVERAGES:
        min_support = MIN_SUPPORT[
            coverage
        ]

        supported_context_ids = set(
            context_supports.index[
                context_supports >= min_support
            ].astype(int)
        )

        context_conditioned_mask = (
            eligible_label
            & labels.isin(
                supported_context_ids
            )
        )

        n_context_conditioned_rows = int(
            context_conditioned_mask.sum()
        )

        n_global_empirical_rows = int(
            len(Y)
            - n_context_conditioned_rows
        )

        coverage_rows.append({
            "design": design,
            "family": family,
            "representation": representation,
            "dmin": dmin,
            "q": q,
            "coverage": coverage,
            "minimum_train_ticks": min_support,
            "n_train_rows": len(Y),
            "n_labelled_rows": n_labelled,
            "n_eligible_context_rows": n_eligible_rows,
            "n_predefined_global_rows": n_predefined_global_rows,
            "n_contexts_total": int(
                len(context_supports)
            ),
            "n_contexts_supported": int(
                (
                    context_supports
                    >= min_support
                ).sum()
            ),
            "n_contexts_global_by_support": int(
                (
                    context_supports
                    < min_support
                ).sum()
            ),
            "n_context_conditioned_rows": (
                n_context_conditioned_rows
            ),
            "n_global_empirical_rows": (
                n_global_empirical_rows
            ),
            "context_conditioned_share": (
                n_context_conditioned_rows
                / len(Y)
            ),
            "global_empirical_share": (
                n_global_empirical_rows
                / len(Y)
            ),
            "n_empirical_response_channels": (
                EXPECTED_EMPIRICAL_COUNT
            ),
            "n_singleton_response_channels": (
                EXPECTED_SINGLETON_COUNT
            ),
        })

    return (
        pd.DataFrame(ref_rows),
        pd.DataFrame(support_rows),
        pd.DataFrame(coverage_rows),
    )


# ============================================================
# 9. BUILD CONTEXT-CONDITIONED REFERENCES
# ============================================================

reference_frames = []
support_frames = []
coverage_frames = []
check_rows = []

for design, family in DESIGNS.items():
    design_dir = (
        TRAIN_STABILIZATION_DIR / design
    )

    # --------------------------------------------------------
    # Native representation
    # --------------------------------------------------------
    native = load_labels(
        design_dir
        / "preliminary_episode_labels.csv",
        "native_state",
        n_rows,
    )

    ref, supp, cov = (
        build_context_reference(
            labels=native,
            design=design,
            family=family,
            representation="native",
            excluded_context=(
                HDB_NOISE
                if family == "hdbscan"
                else None
            ),
        )
    )

    reference_frames.append(
        ref
    )
    support_frames.append(
        supp
    )
    coverage_frames.append(
        cov
    )

    check_rows.append({
        "design": design,
        "representation": "native",
        "dmin": np.nan,
        "q": np.nan,
        "check": "75_response_channels_per_context",
        "status": (
            "PASS"
            if (
                ref.groupby(
                    "context_id"
                ).size()
                == EXPECTED_RESPONSE_COUNT
            ).all()
            else "FAIL"
        ),
    })

    # --------------------------------------------------------
    # Stabilized + fixed q=7 meta-mode representations
    # --------------------------------------------------------
    for dmin in DMIN_VALUES:
        stable = load_labels(
            design_dir
            / f"stabilized_labels_dmin{dmin}.csv",
            f"stabilized_state_dmin{dmin}",
            n_rows,
        )

        ref, supp, cov = (
            build_context_reference(
                labels=stable,
                design=design,
                family=family,
                representation="stabilized",
                dmin=dmin,
                excluded_context=(
                    HDB_NOISE
                    if family == "hdbscan"
                    else None
                ),
            )
        )

        reference_frames.append(
            ref
        )
        support_frames.append(
            supp
        )
        coverage_frames.append(
            cov
        )

        check_rows.append({
            "design": design,
            "representation": "stabilized",
            "dmin": dmin,
            "q": np.nan,
            "check": "75_response_channels_per_context",
            "status": (
                "PASS"
                if (
                    ref.groupby(
                        "context_id"
                    ).size()
                    == EXPECTED_RESPONSE_COUNT
                ).all()
                else "FAIL"
            ),
        })

        # Fixed q=7 native -> meta mapping.
        map_part = mapping[
            (
                mapping["design"]
                == design
            )
            & (
                mapping["dmin"]
                == dmin
            )
        ][
            [
                "native_state",
                "meta_mode",
            ]
        ].copy()

        if map_part[
            "native_state"
        ].duplicated().any():
            raise RuntimeError(
                f"{design}, dmin={dmin}: "
                "duplicated q7 mapping"
            )

        map_dict = dict(
            zip(
                map_part[
                    "native_state"
                ],
                map_part[
                    "meta_mode"
                ],
            )
        )

        present_states = set(
            stable.dropna()
            .astype(int)
            .unique()
        )

        missing_map = sorted(
            present_states
            - set(map_dict)
        )

        if missing_map:
            raise RuntimeError(
                f"{design}, dmin={dmin}: "
                "stabilized states missing q7 mapping: "
                f"{missing_map}"
            )

        meta = (
            stable.map(
                map_dict
            ).astype(
                "Int64"
            )
        )

        if family == "hdbscan":
            hdb_noise = (
                stable == HDB_NOISE
            ).fillna(
                False
            ).to_numpy()

            meta_zero = (
                meta == META_M0
            ).fillna(
                False
            ).to_numpy()

            if not np.array_equal(
                hdb_noise,
                meta_zero,
            ):
                raise RuntimeError(
                    f"{design}, dmin={dmin}: "
                    "HDBSCAN -1 does not match M0"
                )

        elif (
            meta == META_M0
        ).fillna(
            False
        ).any():
            raise RuntimeError(
                f"{design}, dmin={dmin}: "
                "non-HDBSCAN design produced M0"
            )

        ref, supp, cov = (
            build_context_reference(
                labels=meta,
                design=design,
                family=family,
                representation="meta_q7",
                dmin=dmin,
                q=Q_FIXED,
                excluded_context=META_M0,
            )
        )

        reference_frames.append(
            ref
        )
        support_frames.append(
            supp
        )
        coverage_frames.append(
            cov
        )

        check_rows.append({
            "design": design,
            "representation": "meta_q7",
            "dmin": dmin,
            "q": Q_FIXED,
            "check": "75_response_channels_per_context",
            "status": (
                "PASS"
                if (
                    ref.groupby(
                        "context_id"
                    ).size()
                    == EXPECTED_RESPONSE_COUNT
                ).all()
                else "FAIL"
            ),
        })


# ============================================================
# 10. COMBINE OUTPUTS
# ============================================================

context_reference = pd.concat(
    reference_frames,
    ignore_index=True,
)

support_audit = pd.concat(
    support_frames,
    ignore_index=True,
)

coverage_summary = pd.concat(
    coverage_frames,
    ignore_index=True,
)

checks = pd.DataFrame(
    check_rows
)

if (
    context_reference.empty
    or support_audit.empty
    or coverage_summary.empty
):
    raise RuntimeError(
        "train-only reference-interval contract produced an empty output"
    )

context_reference = (
    context_reference
    .sort_values(
        [
            "design",
            "representation",
            "dmin",
            "q",
            "context_id",
            "response_index",
        ],
        kind="mergesort",
        na_position="first",
    )
    .reset_index(drop=True)
)

key = [
    "design",
    "representation",
    "dmin",
    "q",
    "context_id",
    "response_index",
]

if context_reference.duplicated(
    key
).any():
    raise RuntimeError(
        "Duplicated context reference rows"
    )


# ============================================================
# 11. INTEGRITY CHECKS
# ============================================================

# Global singleton intervals must be [a, a] for every coverage.
global_singleton = global_reference[
    global_reference[
        "reference_type"
    ] == "singleton"
].copy()

global_singleton_ok = True

for coverage in COVERAGES:
    tag = coverage_tag(
        coverage
    )

    expected = (
        global_singleton[
            "singleton_reference_value"
        ].astype(float)
    )

    global_singleton_ok &= bool(
        np.array_equal(
            global_singleton[
                f"lower_q{tag}"
            ].to_numpy(float),
            expected.to_numpy(float),
        )
        and np.array_equal(
            global_singleton[
                f"upper_q{tag}"
            ].to_numpy(float),
            expected.to_numpy(float),
        )
        and (
            global_singleton[
                f"train_outside_rate_q{tag}"
            ] == 0.0
        ).all()
    )

# Context singleton intervals must be identical to the fixed singleton value.
context_singleton = context_reference[
    context_reference[
        "reference_type"
    ] == "singleton"
].copy()

context_singleton_ok = True

for coverage in COVERAGES:
    tag = coverage_tag(
        coverage
    )

    expected = (
        context_singleton[
            "singleton_reference_value"
        ].astype(float)
    )

    context_singleton_ok &= bool(
        np.array_equal(
            context_singleton[
                f"lower_q{tag}"
            ].to_numpy(float),
            expected.to_numpy(float),
        )
        and np.array_equal(
            context_singleton[
                f"upper_q{tag}"
            ].to_numpy(float),
            expected.to_numpy(float),
        )
        and (
            context_singleton[
                f"reference_source_q{tag}"
            ] == "singleton"
        ).all()
    )

# Reference-source vocabulary.
source_ok = True

for coverage in COVERAGES:
    tag = coverage_tag(
        coverage
    )

    allowed_context_sources = {
        "context",
        "global_support",
        "singleton",
    }

    if not set(
        context_reference[
            f"reference_source_q{tag}"
        ].unique()
    ).issubset(
        allowed_context_sources
    ):
        source_ok = False
        break

# Empirical source must agree with support audit.
empirical_source_support_ok = True

for coverage in COVERAGES:
    tag = coverage_tag(
        coverage
    )

    support_lookup = (
        support_audit[
            [
                "design",
                "representation",
                "dmin",
                "q",
                "context_id",
                f"context_supported_q{tag}",
            ]
        ]
        .drop_duplicates()
    )

    empirical_rows = context_reference[
        context_reference[
            "reference_type"
        ] == "empirical"
    ][
        [
            "design",
            "representation",
            "dmin",
            "q",
            "context_id",
            f"reference_source_q{tag}",
        ]
    ].drop_duplicates()

    merged = empirical_rows.merge(
        support_lookup,
        on=[
            "design",
            "representation",
            "dmin",
            "q",
            "context_id",
        ],
        how="left",
        validate="one_to_one",
    )

    expected_source = np.where(
        merged[
            f"context_supported_q{tag}"
        ],
        "context",
        "global_support",
    )

    if not np.array_equal(
        merged[
            f"reference_source_q{tag}"
        ].to_numpy(str),
        expected_source.astype(str),
    ):
        empirical_source_support_ok = False
        break

# Train rates must be valid probabilities.
rate_cols_global = []
rate_cols_context = []

for coverage in COVERAGES:
    tag = coverage_tag(
        coverage
    )

    rate_cols_global.extend([
        f"train_below_rate_q{tag}",
        f"train_above_rate_q{tag}",
        f"train_outside_rate_q{tag}",
    ])

    rate_cols_context.extend([
        f"train_below_rate_q{tag}",
        f"train_above_rate_q{tag}",
        f"train_outside_rate_q{tag}",
    ])

global_rate_ok = (
    global_reference[
        rate_cols_global
    ]
    .stack()
    .between(0, 1)
    .all()
)

context_rate_ok = (
    context_reference[
        rate_cols_context
    ]
    .stack()
    .between(0, 1)
    .all()
)

checks = pd.concat(
    [
        checks,
        pd.DataFrame([
            {
                "design": "GLOBAL",
                "representation": "global",
                "dmin": np.nan,
                "q": np.nan,
                "check": "response_count_is_75",
                "status": (
                    "PASS"
                    if len(
                        global_reference
                    ) == EXPECTED_RESPONSE_COUNT
                    else "FAIL"
                ),
            },
            {
                "design": "GLOBAL",
                "representation": "global",
                "dmin": np.nan,
                "q": np.nan,
                "check": "53_empirical_channels",
                "status": (
                    "PASS"
                    if (
                        global_reference[
                            "reference_type"
                        ] == "empirical"
                    ).sum()
                    == EXPECTED_EMPIRICAL_COUNT
                    else "FAIL"
                ),
            },
            {
                "design": "GLOBAL",
                "representation": "global",
                "dmin": np.nan,
                "q": np.nan,
                "check": "22_singleton_channels",
                "status": (
                    "PASS"
                    if (
                        global_reference[
                            "reference_type"
                        ] == "singleton"
                    ).sum()
                    == EXPECTED_SINGLETON_COUNT
                    else "FAIL"
                ),
            },
            {
                "design": "GLOBAL",
                "representation": "global",
                "dmin": np.nan,
                "q": np.nan,
                "check": "empirical_global_sd_positive",
                "status": (
                    "PASS"
                    if (
                        global_reference.loc[
                            global_reference[
                                "reference_type"
                            ] == "empirical",
                            "sd_population",
                        ] > 0
                    ).all()
                    else "FAIL"
                ),
            },
            {
                "design": "GLOBAL",
                "representation": "global",
                "dmin": np.nan,
                "q": np.nan,
                "check": "singleton_global_sd_zero",
                "status": (
                    "PASS"
                    if (
                        global_reference.loc[
                            global_reference[
                                "reference_type"
                            ] == "singleton",
                            "sd_population",
                        ] == 0
                    ).all()
                    else "FAIL"
                ),
            },
            {
                "design": "GLOBAL",
                "representation": "global",
                "dmin": np.nan,
                "q": np.nan,
                "check": "global_singleton_intervals_valid",
                "status": (
                    "PASS"
                    if global_singleton_ok
                    else "FAIL"
                ),
            },
            {
                "design": "GLOBAL",
                "representation": "global",
                "dmin": np.nan,
                "q": np.nan,
                "check": "global_intervals_nested",
                "status": (
                    "PASS"
                    if intervals_nested(
                        global_reference
                    )
                    else "FAIL"
                ),
            },
            {
                "design": "ALL",
                "representation": "all",
                "dmin": np.nan,
                "q": np.nan,
                "check": "context_singleton_intervals_valid",
                "status": (
                    "PASS"
                    if context_singleton_ok
                    else "FAIL"
                ),
            },
            {
                "design": "ALL",
                "representation": "all",
                "dmin": np.nan,
                "q": np.nan,
                "check": "context_reference_source_hierarchy_valid",
                "status": (
                    "PASS"
                    if context_source_hierarchy_valid(
                        context_reference
                    )
                    else "FAIL"
                ),
            },
            {
                "design": "ALL",
                "representation": "all",
                "dmin": np.nan,
                "q": np.nan,
                "check": "same_source_context_intervals_nested",
                "status": (
                    "PASS"
                    if same_source_intervals_nested(
                        context_reference
                    )
                    else "FAIL"
                ),
            },
            {
                "design": "ALL",
                "representation": "all",
                "dmin": np.nan,
                "q": np.nan,
                "check": "reference_source_values_valid",
                "status": (
                    "PASS"
                    if source_ok
                    else "FAIL"
                ),
            },
            {
                "design": "ALL",
                "representation": "all",
                "dmin": np.nan,
                "q": np.nan,
                "check": "empirical_reference_source_matches_support",
                "status": (
                    "PASS"
                    if empirical_source_support_ok
                    else "FAIL"
                ),
            },
            {
                "design": "ALL",
                "representation": "all",
                "dmin": np.nan,
                "q": np.nan,
                "check": "all_train_rates_in_unit_interval",
                "status": (
                    "PASS"
                    if (
                        global_rate_ok
                        and context_rate_ok
                    )
                    else "FAIL"
                ),
            },
            {
                "design": "ALL",
                "representation": "all",
                "dmin": np.nan,
                "q": np.nan,
                "check": "all_context_support_counts_positive",
                "status": (
                    "PASS"
                    if (
                        support_audit[
                            "n_train_ticks"
                        ] > 0
                    ).all()
                    else "FAIL"
                ),
            },
        ]),
    ],
    ignore_index=True,
)

if not (
    checks["status"] == "PASS"
).all():
    bad = checks.loc[
        checks["status"] != "PASS"
    ]

    raise RuntimeError(
        "train-only reference-interval contract integrity checks failed:\n"
        + bad.to_string(index=False)
    )


# ============================================================
# 12. SAVE
# ============================================================

context_reference.to_csv(
    OUTPUT_DIR
    / "context_conditioned_reference_intervals.csv",
    index=False,
)

support_audit.to_csv(
    OUTPUT_DIR
    / "reference_support_audit.csv",
    index=False,
)

coverage_summary.to_csv(
    OUTPUT_DIR
    / "reference_coverage_summary.csv",
    index=False,
)

singleton_check.to_csv(
    OUTPUT_DIR
    / "singleton_reference_audit.csv",
    index=False,
)

checks.to_csv(
    OUTPUT_DIR
    / "integrity_checks.csv",
    index=False,
)

settings_rows = [
    (
        "train_file",
        str(TRAIN_FILE),
    ),
    (
        "train_only",
        True,
    ),
    (
        "test_data_used",
        False,
    ),
    (
        "attack_labels_used",
        False,
    ),
    (
        "response_vector_dimension",
        EXPECTED_RESPONSE_COUNT,
    ),
    (
        "empirical_reference_channels",
        EXPECTED_EMPIRICAL_COUNT,
    ),
    (
        "singleton_reference_channels",
        EXPECTED_SINGLETON_COUNT,
    ),
    (
        "primary_coverage",
        PRIMARY_COVERAGE,
    ),
    (
        "sensitivity_coverages",
        ",".join(
            str(c)
            for c in SENSITIVITY_COVERAGES
        ),
    ),
    (
        "quantile_method",
        (
            "Hyndman-Fan Type 8 / "
            f"numpy method='{QUANTILE_METHOD}'"
        ),
    ),
    (
        "minimum_expected_train_observations_per_tail",
        MIN_EXPECTED_OBS_PER_TAIL,
    ),
]

for coverage in COVERAGES:
    tag = coverage_tag(
        coverage
    )

    settings_rows.extend([
        (
            f"tail_probability_per_side_q{tag}",
            tail_probability(
                coverage
            ),
        ),
        (
            f"minimum_context_train_ticks_q{tag}",
            MIN_SUPPORT[
                coverage
            ],
        ),
    ])

settings_rows.extend([
    (
        "empirical_context_support_policy",
        (
            "use context-conditioned interval only when "
            "minimum train support is satisfied; otherwise "
            "use global empirical interval of the same coverage"
        ),
    ),
    (
        "singleton_policy",
        (
            "retain fixed train singleton interval [a_j, a_j] "
            "for all contexts and all coverage levels"
        ),
    ),
    (
        "hdbscan_minus1_policy",
        (
            "no context-conditioned empirical interval; "
            "use global empirical interval later"
        ),
    ),
    (
        "meta_m0_policy",
        (
            "no context-conditioned empirical interval; "
            "use global empirical interval later"
        ),
    ),
    (
        "dmin_values",
        ",".join(
            map(
                str,
                DMIN_VALUES,
            )
        ),
    ),
    (
        "meta_q",
        Q_FIXED,
    ),
    (
        "response_scaling_or_standardization",
        False,
    ),
    (
        "coverage_selected_from_test_performance",
        False,
    ),
])

pd.DataFrame(
    settings_rows,
    columns=[
        "setting",
        "value",
    ],
).to_csv(
    OUTPUT_DIR
    / "settings.csv",
    index=False,
)


# ============================================================
# 13. COMPACT NOTEBOOK REPORT
# ============================================================

print(
    "TRAIN-ONLY REFERENCE-INTERVAL CONTRACT COMPLETE"
)
print(
    f"Response-vector dimension: {EXPECTED_RESPONSE_COUNT}"
)
print(
    f"  Empirical-reference channels: {EXPECTED_EMPIRICAL_COUNT}"
)
print(
    f"  Singleton-reference channels: {EXPECTED_SINGLETON_COUNT}"
)
print(
    f"Primary coverage: {int(PRIMARY_COVERAGE * 100)}%"
)
print(
    "Sensitivity coverages: "
    + ", ".join(
        f"{int(c * 100)}%"
        for c in SENSITIVITY_COVERAGES
    )
)
print(
    f"Quantile method: Hyndman-Fan Type 8 ({QUANTILE_METHOD})"
)
print(
    "Minimum expected train observations per tail: "
    f"{MIN_EXPECTED_OBS_PER_TAIL:g}"
)

print(
    "\nMinimum context support for empirical channels:"
)
for coverage in COVERAGES:
    print(
        f"  q{coverage_tag(coverage)}: "
        f"{MIN_SUPPORT[coverage]:,} train ticks"
    )

print(
    f"\nGlobal reference rows: {len(global_reference):,}"
)
print(
    "Context-reference rows: "
    f"{len(context_reference):,}"
)
print(
    "Support-audit context groups: "
    f"{len(support_audit):,}"
)
print(
    "Integrity checks: "
    f"{(checks['status'] == 'PASS').sum()}/{len(checks)} PASS"
)
print(
    f"Output directory: {OUTPUT_DIR}"
)

print(
    "\nTRAIN-ONLY EMPIRICAL-CONTEXT COVERAGE — PRIMARY q99"
)

q99_summary = coverage_summary[
    coverage_summary[
        "coverage"
    ] == PRIMARY_COVERAGE
].copy()

display_cols = [
    "design",
    "representation",
    "dmin",
    "n_contexts_total",
    "n_contexts_supported",
    "context_conditioned_share",
    "global_empirical_share",
]

print(
    q99_summary[
        display_cols
    ]
    .sort_values(
        [
            "design",
            "representation",
            "dmin",
        ],
        kind="mergesort",
        na_position="first",
    )
    .to_string(index=False)
)

print(
    "\nSINGLETON REFERENCE CHANNELS"
)

print(
    singleton_contract[
        [
            "response_index",
            "variable",
            "singleton_reference_value",
        ]
    ].to_string(
        index=False
    )
)


TRAIN-ONLY REFERENCE-INTERVAL CONTRACT COMPLETE — TRAIN-ONLY REFERENCE-INTERVAL CONTRACT
Response-vector dimension: 75
  Empirical-reference channels: 53
  Singleton-reference channels: 22
Primary coverage: 99%
Sensitivity coverages: 95%, 98%
Quantile method: Hyndman-Fan Type 8 (median_unbiased)
Minimum expected train observations per tail: 10

Minimum context support for empirical channels:
  q95: 400 train ticks
  q98: 1,000 train ticks
  q99: 2,000 train ticks

Global reference rows: 75
Context-reference rows: 38,250
Support-audit context groups: 510
Integrity checks: 69/69 PASS
Output directory: outputs/section_5_5/train_reference_intervals

TRAIN-ONLY EMPIRICAL-CONTEXT COVERAGE — PRIMARY q99
  design representation dmin  n_contexts_total  n_contexts_supported  context_conditioned_share  global_empirical_share
 HDB15_8        meta_q7    5                 6                     6                   0.998051                0.001949
 HDB15_8        meta_q7   10                 6        

## Section 5.6 — Extremeness scoring and functional comparison

The held-out response observations are linked to the already fixed operating-context assignments. Multichannel extremeness is then computed without attack labels under one GLOBAL reference and 55 SERF-conditioned configurations. The 99% reference coverage is primary; 98% and 95% are predefined sensitivity settings.


### 5.6.1 Link held-out response vectors to frozen operating contexts


In [8]:
# held-out response/context contract — Test response-vector construction and exact held-out SERF post-processing context linkage
# HAI 21.03 / SERF
#
# Purpose
# -------
# For each unseen test run, create one tick-level table containing:
#   row_id + time
#   + the fixed 75-channel response vector Y_t from response-vector contract
#   + 55 already-computed operating-context assignments:
#       11 retained designs x
#       [native, stabilized dmin=5, stabilized dmin=10,
#        meta q=7/dmin=5, meta q=7/dmin=10]
#
# IMPORTANT
# ---------
# This cell DOES NOT recompute stabilization or meta alignment.
# It reads the exact per-tick outputs already written by Cells 12 and 16:
#
# held-out native assignment:
#   outputs/section_5_4/test_native_assignment/testX/native_labels.csv
#
# held-out SERF post-processing:
#   outputs/section_5_4/test_serf_postprocessing/
#       testX/DESIGN/serf_labels_dmin5_q7.csv
#       testX/DESIGN/serf_labels_dmin10_q7.csv
#
# held-out SERF post-processing per-tick schema used here:
#   row_id, time, segment_id, native_state,
#   preliminary_episode_id, stabilized_state,
#   stabilized_episode_id, meta_mode_q7, meta_mode_episode_id_q7
#
# This cell DOES NOT:
#   - apply reference intervals
#   - score extremeness
#   - read attack labels
#   - refit segmentation
#   - recompute stabilization
#   - remap meta-modes
#
# Output per test:
#   outputs/section_5_6/test_response_context_contract/
#       testX/test_response_context.csv
#
# Width:
#   2 + 75 + 55 = 132 columns

from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS AND FIXED CONTRACT
# ============================================================

RESPONSE_CONTRACT_DIR = Path("outputs/section_5_5/response_vector_contract")
NATIVE_ASSIGNMENT_DIR = Path("outputs/section_5_4/test_native_assignment")
SERF_TEST_DIR = Path("outputs/section_5_4/test_serf_postprocessing")

RESPONSE_CONTRACT_FILE = RESPONSE_CONTRACT_DIR / "response_vector_contract.csv"

OUTPUT_DIR = Path("outputs/section_5_6/test_response_context_contract")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_RUNS = ["test1", "test2", "test3", "test4", "test5"]

EXPECTED_TEST_ROWS = {
    "test1": 43_201,
    "test2": 118_801,
    "test3": 108_001,
    "test4": 39_601,
    "test5": 92_401,
}

EXPECTED_TOTAL_TEST_ROWS = 402_005
EXPECTED_RESPONSE_COUNT = 75
EXPECTED_CONTEXT_COLUMNS = 55

TIME_COL = "time"
DMIN_VALUES = [5, 10]
Q_FIXED = 7

DESIGNS = [
    "KM4_8",
    "KM7_8",
    "KM13_8",
    "KM13_16",
    "HDB23_8",
    "HDB18_8",
    "HDB15_8",
    "HDB18_16",
    "HMM4",
    "HMM5",
    "HMM8",
]

FAMILIES = {
    "KM4_8": "kmeans",
    "KM7_8": "kmeans",
    "KM13_8": "kmeans",
    "KM13_16": "kmeans",
    "HDB23_8": "hdbscan",
    "HDB18_8": "hdbscan",
    "HDB15_8": "hdbscan",
    "HDB18_16": "hdbscan",
    "HMM4": "hmm",
    "HMM5": "hmm",
    "HMM8": "hmm",
}


# ============================================================
# 2. HELPERS
# ============================================================

def resolve_test_file(test_run: str) -> Path:
    """Return testX.csv from the notebook working directory."""
    path = Path(f"{test_run}.csv")
    if not path.exists():
        raise FileNotFoundError(f"{path.name} must be placed in the notebook working directory.")
    return path


def as_int64_nullable(series: pd.Series, name: str) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")

    valid = numeric.notna()
    if valid.any():
        values = numeric.loc[valid].to_numpy(float)
        if not np.array_equal(values, np.round(values)):
            raise ValueError(f"{name}: non-integer labels found.")

    return numeric.round().astype("Int64")


def series_equal_with_na(a: pd.Series, b: pd.Series) -> bool:
    a = as_int64_nullable(a, "left")
    b = as_int64_nullable(b, "right")

    if len(a) != len(b):
        return False

    a_na = a.isna().to_numpy()
    b_na = b.isna().to_numpy()

    if not np.array_equal(a_na, b_na):
        return False

    valid = ~a_na
    if not valid.any():
        return True

    return np.array_equal(
        a.loc[valid].astype(int).to_numpy(),
        b.loc[valid].astype(int).to_numpy(),
    )


def time_equal(a: pd.Series, b: pd.Series) -> bool:
    """
    Compare timestamps robustly. If both parse as datetimes, compare parsed
    datetimes; otherwise compare string representations.
    """
    adt = pd.to_datetime(a, errors="coerce")
    bdt = pd.to_datetime(b, errors="coerce")

    if adt.notna().all() and bdt.notna().all():
        return adt.equals(bdt)

    return a.astype(str).reset_index(drop=True).equals(
        b.astype(str).reset_index(drop=True)
    )


# ============================================================
# 3. LOAD FIXED 75D RESPONSE CONTRACT
# ============================================================

if not RESPONSE_CONTRACT_FILE.exists():
    raise FileNotFoundError(RESPONSE_CONTRACT_FILE)

response_contract = pd.read_csv(RESPONSE_CONTRACT_FILE)

required_contract_cols = {
    "response_index",
    "variable",
    "train_status",
    "reference_type",
}

if not required_contract_cols.issubset(response_contract.columns):
    missing = sorted(required_contract_cols - set(response_contract.columns))
    raise ValueError(
        f"{RESPONSE_CONTRACT_FILE}: missing columns {missing}"
    )

response_contract["response_index"] = pd.to_numeric(
    response_contract["response_index"],
    errors="raise",
).astype(int)

response_contract = (
    response_contract
    .sort_values("response_index", kind="mergesort")
    .reset_index(drop=True)
)

if len(response_contract) != EXPECTED_RESPONSE_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_RESPONSE_COUNT} response channels, "
        f"found {len(response_contract)}."
    )

if response_contract["response_index"].tolist() != list(
    range(EXPECTED_RESPONSE_COUNT)
):
    raise RuntimeError(
        "response-vector contract response_index must be contiguous 0..74."
    )

if response_contract["variable"].duplicated().any():
    raise RuntimeError(
        "Duplicated response variables in response-vector contract contract."
    )

RESPONSE_VARS = response_contract["variable"].astype(str).tolist()


# ============================================================
# 4. DEFINE THE 55 OUTPUT CONTEXT COLUMNS
# ============================================================

context_specs = []

for design in DESIGNS:
    family = FAMILIES[design]

    context_specs.extend([
        {
            "context_column": f"{design}__native",
            "design": design,
            "family": family,
            "representation": "native",
            "dmin": np.nan,
            "q": np.nan,
        },
        {
            "context_column": f"{design}__stabilized_dmin5",
            "design": design,
            "family": family,
            "representation": "stabilized",
            "dmin": 5,
            "q": np.nan,
        },
        {
            "context_column": f"{design}__stabilized_dmin10",
            "design": design,
            "family": family,
            "representation": "stabilized",
            "dmin": 10,
            "q": np.nan,
        },
        {
            "context_column": f"{design}__meta_q7_dmin5",
            "design": design,
            "family": family,
            "representation": "meta_q7",
            "dmin": 5,
            "q": Q_FIXED,
        },
        {
            "context_column": f"{design}__meta_q7_dmin10",
            "design": design,
            "family": family,
            "representation": "meta_q7",
            "dmin": 10,
            "q": Q_FIXED,
        },
    ])

context_contract = pd.DataFrame(context_specs)

if len(context_contract) != EXPECTED_CONTEXT_COLUMNS:
    raise RuntimeError(
        f"Expected {EXPECTED_CONTEXT_COLUMNS} context columns, "
        f"constructed {len(context_contract)}."
    )

context_contract.to_csv(
    OUTPUT_DIR / "context_column_contract.csv",
    index=False,
)


# ============================================================
# 5. BUILD ONE WIDE TICK-LEVEL TABLE PER TEST RUN
# ============================================================

summary_rows = []
check_rows = []
source_rows = []

for test_run in TEST_RUNS:
    # --------------------------------------------------------
    # 5.1 Raw test response vector
    # --------------------------------------------------------
    test_file = resolve_test_file(test_run)

    raw_header = pd.read_csv(test_file, nrows=0)
    raw_columns = list(raw_header.columns)

    required_raw = [TIME_COL] + RESPONSE_VARS
    missing_raw = [c for c in required_raw if c not in raw_columns]

    if missing_raw:
        raise ValueError(
            f"{test_file}: missing required columns {missing_raw}"
        )

    # Deliberately read no attack-label columns.
    raw = pd.read_csv(
        test_file,
        usecols=required_raw,
    )

    n_rows = len(raw)
    expected_rows = EXPECTED_TEST_ROWS[test_run]

    if n_rows != expected_rows:
        raise RuntimeError(
            f"{test_run}: expected {expected_rows:,} rows, "
            f"found {n_rows:,}."
        )

    Y = raw[RESPONSE_VARS].apply(
        pd.to_numeric,
        errors="coerce",
    ).replace(
        [np.inf, -np.inf],
        np.nan,
    )

    if Y.isna().any().any():
        bad = Y.columns[Y.isna().any()].tolist()
        raise ValueError(
            f"{test_run}: non-finite response values in {bad}"
        )

    Y = Y.astype(float)

    row_id = np.arange(n_rows, dtype=int)

    # --------------------------------------------------------
    # 5.2 Exact held-out native assignment native labels
    # --------------------------------------------------------
    native_file = NATIVE_ASSIGNMENT_DIR / test_run / "native_labels.csv"

    if not native_file.exists():
        raise FileNotFoundError(native_file)

    native = pd.read_csv(native_file)

    required_native_cols = {
        "row_id",
        "time",
        "segment_id",
        *DESIGNS,
    }

    if not required_native_cols.issubset(native.columns):
        missing = sorted(required_native_cols - set(native.columns))
        raise ValueError(
            f"{native_file}: missing columns {missing}"
        )

    if len(native) != n_rows:
        raise RuntimeError(
            f"{test_run}: held-out native assignment native row count "
            f"{len(native):,} != raw test row count {n_rows:,}."
        )

    native_row_id = pd.to_numeric(
        native["row_id"],
        errors="raise",
    ).astype(int).to_numpy()

    if not np.array_equal(native_row_id, row_id):
        raise RuntimeError(
            f"{test_run}: held-out native assignment row_id alignment failed."
        )

    if not time_equal(raw[TIME_COL], native["time"]):
        raise RuntimeError(
            f"{test_run}: raw test time does not match held-out native assignment native_labels.csv."
        )

    # --------------------------------------------------------
    # 5.3 Start wide table: row_id + time + 75 response channels
    # --------------------------------------------------------
    wide = pd.DataFrame({
        "row_id": row_id,
        TIME_COL: raw[TIME_COL].to_numpy(),
    })

    for variable in RESPONSE_VARS:
        wide[variable] = Y[variable].to_numpy()

    # --------------------------------------------------------
    # 5.4 Append exact held-out native assignment/16 context assignments
    # --------------------------------------------------------
    for design in DESIGNS:
        native_state = as_int64_nullable(
            native[design],
            f"{test_run}/{design}/Cell12 native",
        )

        wide[f"{design}__native"] = native_state

        source_rows.append({
            "test_run": test_run,
            "design": design,
            "representation": "native",
            "dmin": np.nan,
            "q": np.nan,
            "context_column": f"{design}__native",
            "source_file": str(native_file),
            "source_column": design,
        })

        for dmin in DMIN_VALUES:
            serf_file = (
                SERF_TEST_DIR
                / test_run
                / design
                / f"serf_labels_dmin{dmin}_q7.csv"
            )

            if not serf_file.exists():
                raise FileNotFoundError(
                    f"Missing exact held-out SERF post-processing per-tick output: {serf_file}"
                )

            serf = pd.read_csv(serf_file)

            required_serf_cols = {
                "row_id",
                "time",
                "segment_id",
                "native_state",
                "stabilized_state",
                "meta_mode_q7",
            }

            if not required_serf_cols.issubset(serf.columns):
                missing = sorted(required_serf_cols - set(serf.columns))
                raise ValueError(
                    f"{serf_file}: missing columns {missing}"
                )

            if len(serf) != n_rows:
                raise RuntimeError(
                    f"{test_run}/{design}/dmin={dmin}: "
                    f"held-out SERF post-processing row count {len(serf):,} != {n_rows:,}."
                )

            serf_row_id = pd.to_numeric(
                serf["row_id"],
                errors="raise",
            ).astype(int).to_numpy()

            if not np.array_equal(serf_row_id, row_id):
                raise RuntimeError(
                    f"{test_run}/{design}/dmin={dmin}: "
                    "held-out SERF post-processing row_id alignment failed."
                )

            if not time_equal(native["time"], serf["time"]):
                raise RuntimeError(
                    f"{test_run}/{design}/dmin={dmin}: "
                    "held-out SERF post-processing time alignment failed."
                )

            # held-out SERF post-processing carries native_state again; it must reproduce held-out native assignment.
            if not series_equal_with_na(
                native_state,
                serf["native_state"],
            ):
                raise RuntimeError(
                    f"{test_run}/{design}/dmin={dmin}: "
                    "held-out SERF post-processing native_state does not reproduce held-out native assignment."
                )

            stable_state = as_int64_nullable(
                serf["stabilized_state"],
                f"{test_run}/{design}/dmin={dmin}/stabilized_state",
            )

            meta_mode = as_int64_nullable(
                serf["meta_mode_q7"],
                f"{test_run}/{design}/dmin={dmin}/meta_mode_q7",
            )

            stable_col = f"{design}__stabilized_dmin{dmin}"
            meta_col = f"{design}__meta_q7_dmin{dmin}"

            wide[stable_col] = stable_state
            wide[meta_col] = meta_mode

            source_rows.extend([
                {
                    "test_run": test_run,
                    "design": design,
                    "representation": "stabilized",
                    "dmin": dmin,
                    "q": np.nan,
                    "context_column": stable_col,
                    "source_file": str(serf_file),
                    "source_column": "stabilized_state",
                },
                {
                    "test_run": test_run,
                    "design": design,
                    "representation": "meta_q7",
                    "dmin": dmin,
                    "q": Q_FIXED,
                    "context_column": meta_col,
                    "source_file": str(serf_file),
                    "source_column": "meta_mode_q7",
                },
            ])

    # --------------------------------------------------------
    # 5.5 Enforce exact output order
    # --------------------------------------------------------
    context_columns = context_contract["context_column"].tolist()

    expected_columns = (
        ["row_id", TIME_COL]
        + RESPONSE_VARS
        + context_columns
    )

    missing_context = [
        c for c in context_columns
        if c not in wide.columns
    ]

    if missing_context:
        raise RuntimeError(
            f"{test_run}: missing context columns {missing_context}"
        )

    wide = wide[expected_columns]

    # --------------------------------------------------------
    # 5.6 Integrity checks
    # --------------------------------------------------------
    checks_for_run = [
        (
            "row_count_matches_dataset_contract",
            len(wide) == expected_rows,
        ),
        (
            "row_id_is_contiguous",
            np.array_equal(
                wide["row_id"].to_numpy(int),
                row_id,
            ),
        ),
        (
            "response_vector_has_75_channels",
            len(RESPONSE_VARS) == EXPECTED_RESPONSE_COUNT,
        ),
        (
            "context_assignment_has_55_columns",
            len(context_columns) == EXPECTED_CONTEXT_COLUMNS,
        ),
        (
            "output_column_count_is_132",
            wide.shape[1]
            == 2 + EXPECTED_RESPONSE_COUNT + EXPECTED_CONTEXT_COLUMNS,
        ),
        (
            "all_response_values_finite",
            np.isfinite(
                wide[RESPONSE_VARS].to_numpy(float)
            ).all(),
        ),
        (
            "no_attack_columns_in_output",
            not any(
                "attack" in c.lower()
                for c in wide.columns
            ),
        ),
    ]

    for check, passed in checks_for_run:
        check_rows.append({
            "test_run": test_run,
            "check": check,
            "status": "PASS" if passed else "FAIL",
        })

    # Context labels may legitimately be NA at unsupported initial window ticks.
    for col in context_columns:
        s = wide[col]
        numeric = pd.to_numeric(s, errors="coerce")
        valid = numeric.notna()

        integer_or_na = True
        if valid.any():
            values = numeric.loc[valid].to_numpy(float)
            integer_or_na = bool(
                np.array_equal(values, np.round(values))
            )

        check_rows.append({
            "test_run": test_run,
            "check": f"{col}: integer_or_NA",
            "status": "PASS" if integer_or_na else "FAIL",
        })

    # --------------------------------------------------------
    # 5.7 Save
    # --------------------------------------------------------
    run_out = OUTPUT_DIR / test_run
    run_out.mkdir(parents=True, exist_ok=True)

    output_file = run_out / "test_response_context.csv"

    wide.to_csv(
        output_file,
        index=False,
    )

    missing_counts = wide[context_columns].isna().sum()

    summary_rows.append({
        "test_run": test_run,
        "input_file": str(test_file),
        "cell12_native_file": str(native_file),
        "output_file": str(output_file),
        "n_ticks": n_rows,
        "n_response_channels": EXPECTED_RESPONSE_COUNT,
        "n_context_columns": EXPECTED_CONTEXT_COLUMNS,
        "n_output_columns": wide.shape[1],
        "min_missing_context_ticks": int(missing_counts.min()),
        "max_missing_context_ticks": int(missing_counts.max()),
        "mean_missing_context_ticks": float(missing_counts.mean()),
    })

    print(
        f"{test_run}: {n_rows:,} ticks | "
        "75 response channels + 55 fixed context assignments"
    )


# ============================================================
# 6. GLOBAL INTEGRITY AND OUTPUTS
# ============================================================

summary = pd.DataFrame(summary_rows)
checks = pd.DataFrame(check_rows)
sources = pd.DataFrame(source_rows)

total_rows = int(summary["n_ticks"].sum())

checks = pd.concat(
    [
        checks,
        pd.DataFrame([
            {
                "test_run": "ALL",
                "check": "total_test_rows_is_402005",
                "status": (
                    "PASS"
                    if total_rows == EXPECTED_TOTAL_TEST_ROWS
                    else "FAIL"
                ),
            },
            {
                "test_run": "ALL",
                "check": "five_test_runs_present",
                "status": (
                    "PASS"
                    if len(summary) == len(TEST_RUNS)
                    else "FAIL"
                ),
            },
            {
                "test_run": "ALL",
                "check": "55_context_sources_per_test_run",
                "status": (
                    "PASS"
                    if (
                        sources.groupby("test_run")
                        .size()
                        .eq(EXPECTED_CONTEXT_COLUMNS)
                        .all()
                    )
                    else "FAIL"
                ),
            },
        ]),
    ],
    ignore_index=True,
)

if not (checks["status"] == "PASS").all():
    bad = checks.loc[checks["status"] != "PASS"]

    raise RuntimeError(
        "held-out response/context contract integrity checks failed:\n"
        + bad.to_string(index=False)
    )

summary.to_csv(
    OUTPUT_DIR / "test_summary.csv",
    index=False,
)

checks.to_csv(
    OUTPUT_DIR / "integrity_checks.csv",
    index=False,
)

sources.to_csv(
    OUTPUT_DIR / "context_source_audit.csv",
    index=False,
)

settings_rows = [
    ("cell_purpose", "test response-vector construction and fixed context linkage"),
    ("test_runs", ",".join(TEST_RUNS)),
    ("total_test_ticks", total_rows),
    ("response_vector_dimension", EXPECTED_RESPONSE_COUNT),
    ("context_assignment_columns", EXPECTED_CONTEXT_COLUMNS),
    ("design_count", len(DESIGNS)),
    ("representation_variants_per_design", 5),
    ("native_source", "held-out native assignment native_labels.csv"),
    (
        "stabilized_meta_source",
        "held-out SERF post-processing testX/DESIGN/serf_labels_dmin{5,10}_q7.csv",
    ),
    ("attack_labels_read", False),
    ("attack_labels_written", False),
    ("reference_intervals_applied", False),
    ("extremeness_scoring_performed", False),
    ("segmentation_refit", False),
    ("boundary_stabilization_recomputed", False),
    ("meta_modes_remapped", False),
    ("response_order", "fixed response-vector contract response_index order 0..74"),
]

pd.DataFrame(
    settings_rows,
    columns=["setting", "value"],
).to_csv(
    OUTPUT_DIR / "settings.csv",
    index=False,
)


# ============================================================
# 7. COMPACT NOTEBOOK REPORT
# ============================================================

print("\nHELD-OUT RESPONSE/CONTEXT CONTRACT COMPLETE — TEST RESPONSE + FIXED CONTEXT CONTRACT")
print(f"Test runs: {len(TEST_RUNS)}")
print(f"Total test ticks: {total_rows:,}")
print(f"Response channels per tick: {EXPECTED_RESPONSE_COUNT}")
print(f"Context columns per tick: {EXPECTED_CONTEXT_COLUMNS}")
print(
    f"Output width: "
    f"{2 + EXPECTED_RESPONSE_COUNT + EXPECTED_CONTEXT_COLUMNS} columns"
)
print(
    "Attack labels read: NO | "
    "Reference intervals applied: NO | "
    "Extremeness scoring: NO"
)
print(
    f"Integrity checks: "
    f"{(checks['status'] == 'PASS').sum()}/{len(checks)} PASS"
)
print(f"Output directory: {OUTPUT_DIR}")

print("\nPer-run summary:")
print(
    summary[
        [
            "test_run",
            "n_ticks",
            "n_output_columns",
            "min_missing_context_ticks",
            "max_missing_context_ticks",
        ]
    ].to_string(index=False)
)


test1: 43,201 ticks | 75 response channels + 55 fixed context assignments
test2: 118,801 ticks | 75 response channels + 55 fixed context assignments
test3: 108,001 ticks | 75 response channels + 55 fixed context assignments
test4: 39,601 ticks | 75 response channels + 55 fixed context assignments
test5: 92,401 ticks | 75 response channels + 55 fixed context assignments

HELD-OUT RESPONSE/CONTEXT CONTRACT COMPLETE — TEST RESPONSE + FIXED CONTEXT CONTRACT
Test runs: 5
Total test ticks: 402,005
Response channels per tick: 75
Context columns per tick: 55
Output width: 132 columns
Attack labels read: NO | Reference intervals applied: NO | Extremeness scoring: NO
Integrity checks: 313/313 PASS
Output directory: outputs/section_5_6/test_response_context_contract

Per-run summary:
test_run  n_ticks  n_output_columns  min_missing_context_ticks  max_missing_context_ticks
   test1    43201               132                          0                         15
   test2   118801               132 

### 5.6.2 Compute label-blind multichannel-extremeness scores


In [9]:
# label-blind extremeness scoring — Test extreme-channel counts under fixed reference intervals
# HAI 21.03 / SERF
#
# Purpose
# -------
# Apply the train-derived reference contract from train-only reference-interval contract to the five
# test response/context tables from held-out response/context contract, without using attack labels.
#
# For every test tick:
#   - compute ONE singleton extreme count across the 22 train-constant channels
#   - compute 56 total extreme-channel counts across all 75 response channels:
#       1 GLOBAL
#       55 SERF context variants (11 designs x 5 representations)
#
# For the 53 train-varying channels:
#   - GLOBAL uses the fixed global empirical interval
#   - each SERF variant uses the fixed context-conditioned interval
#   - if the context is undefined or is a predefined non-conditioned case
#     (HDBSCAN -1 / meta M0), GLOBAL empirical bounds are used
#   - low-support contexts already carry GLOBAL empirical bounds in the
#     train-only reference-interval contract context-reference table
#
# For the 22 train-constant channels:
#   - the singleton interval [a_j, a_j] is used for EVERY variant
#   - any test value != a_j is extreme
#
# Primary / sensitivity:
#   q99 = primary
#   q98, q95 = sensitivity
#
# IMPORTANT
# ---------
# This cell DOES NOT:
#   - read attack labels
#   - tune thresholds
#   - select a design
#   - refit or adapt any model/reference
#
# Output per test run:
#   extreme_channel_counts_q99.csv
#   extreme_channel_counts_q98.csv
#   extreme_channel_counts_q95.csv
#
# Each output contains:
#   row_id, time, singleton_extreme_count,
#   GLOBAL,
#   + the 55 held-out response/context contract context columns
#
# Thus each GLOBAL/SERF count is:
#   total extreme channels among all 75 response channels.

from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS AND FIXED SETTINGS
# ============================================================

RESPONSE_CONTRACT_DIR = Path("outputs/section_5_5/response_vector_contract")
REFERENCE_INTERVAL_DIR = Path("outputs/section_5_5/train_reference_intervals")
RESPONSE_CONTEXT_DIR = Path("outputs/section_5_6/test_response_context_contract")

RESPONSE_CONTRACT_FILE = RESPONSE_CONTRACT_DIR / "response_vector_contract.csv"
GLOBAL_REFERENCE_FILE = REFERENCE_INTERVAL_DIR / "global_reference_intervals.csv"
CONTEXT_REFERENCE_FILE = REFERENCE_INTERVAL_DIR / "context_conditioned_reference_intervals.csv"
CONTEXT_COLUMN_CONTRACT_FILE = RESPONSE_CONTEXT_DIR / "context_column_contract.csv"

OUTPUT_DIR = Path("outputs/section_5_6/test_extreme_channel_counts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_RUNS = ["test1", "test2", "test3", "test4", "test5"]

EXPECTED_TEST_ROWS = {
    "test1": 43_201,
    "test2": 118_801,
    "test3": 108_001,
    "test4": 39_601,
    "test5": 92_401,
}

EXPECTED_TOTAL_TEST_ROWS = 402_005
EXPECTED_RESPONSE_COUNT = 75
EXPECTED_EMPIRICAL_COUNT = 53
EXPECTED_SINGLETON_COUNT = 22
EXPECTED_CONTEXT_COLUMNS = 55
EXPECTED_TOTAL_COUNT_COLUMNS = 56  # GLOBAL + 55 SERF variants

COVERAGES = [0.95, 0.98, 0.99]
PRIMARY_COVERAGE = 0.99

HDB_NOISE = -1
META_M0 = 0


# ============================================================
# 2. HELPERS
# ============================================================

def coverage_tag(coverage: float) -> str:
    return str(int(round(coverage * 100)))


def nullable_integer_labels(series: pd.Series, name: str) -> pd.Series:
    """
    Convert a context column to nullable Int64 and reject non-integer labels.
    """
    numeric = pd.to_numeric(series, errors="coerce")
    valid = numeric.notna()

    if valid.any():
        values = numeric.loc[valid].to_numpy(dtype=float)

        if not np.array_equal(values, np.round(values)):
            raise ValueError(
                f"{name}: non-integer operating-context labels found."
            )

    return numeric.round().astype("Int64")


def exact_singleton_count(
    Y_singleton: np.ndarray,
    singleton_values: np.ndarray,
) -> np.ndarray:
    """
    Count, per tick, how many singleton channels differ from their exact
    train value. Exact inequality is intentional for singleton references.
    """
    return np.count_nonzero(
        Y_singleton != singleton_values[None, :],
        axis=1,
    ).astype(np.uint8)


def empirical_extreme_count(
    Y_empirical: np.ndarray,
    lower: np.ndarray,
    upper: np.ndarray,
) -> np.ndarray:
    """
    Count empirical channels outside [lower, upper] per tick.
    Equality with either bound is inside the reference interval.
    """
    return np.count_nonzero(
        (Y_empirical < lower)
        | (Y_empirical > upper),
        axis=1,
    ).astype(np.uint8)


def filter_context_reference(
    context_reference: pd.DataFrame,
    design: str,
    representation: str,
    dmin,
    q,
) -> pd.DataFrame:
    """
    Select one design-representation block from the train-only reference-interval contract reference table.
    """
    mask = (
        (context_reference["design"] == design)
        & (context_reference["representation"] == representation)
    )

    if pd.isna(dmin):
        mask &= context_reference["dmin"].isna()
    else:
        mask &= context_reference["dmin"] == float(dmin)

    if pd.isna(q):
        mask &= context_reference["q"].isna()
    else:
        mask &= context_reference["q"] == float(q)

    return context_reference.loc[mask].copy()


def predefined_global_context(
    family: str,
    representation: str,
    context_id: int,
) -> bool:
    """
    Cases that intentionally do not receive context-conditioned empirical
    reference intervals in train-only reference-interval contract.
    """
    if family == "hdbscan":
        if representation in {"native", "stabilized"} and context_id == HDB_NOISE:
            return True
        if representation == "meta_q7" and context_id == META_M0:
            return True

    return False


# ============================================================
# 3. LOAD RESPONSE-VECTOR CONTRACT RESPONSE CONTRACT
# ============================================================

if not RESPONSE_CONTRACT_FILE.exists():
    raise FileNotFoundError(RESPONSE_CONTRACT_FILE)

response_contract = pd.read_csv(RESPONSE_CONTRACT_FILE)

required_contract_cols = {
    "response_index",
    "variable",
    "reference_type",
    "singleton_reference_value",
}

if not required_contract_cols.issubset(response_contract.columns):
    missing = sorted(required_contract_cols - set(response_contract.columns))
    raise ValueError(
        f"{RESPONSE_CONTRACT_FILE}: missing columns {missing}"
    )

response_contract["response_index"] = pd.to_numeric(
    response_contract["response_index"],
    errors="raise",
).astype(int)

response_contract = (
    response_contract
    .sort_values("response_index", kind="mergesort")
    .reset_index(drop=True)
)

if len(response_contract) != EXPECTED_RESPONSE_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_RESPONSE_COUNT} response channels, "
        f"found {len(response_contract)}."
    )

if response_contract["response_index"].tolist() != list(
    range(EXPECTED_RESPONSE_COUNT)
):
    raise RuntimeError(
        "response-vector contract response_index must be contiguous 0..74."
    )

empirical_contract = response_contract[
    response_contract["reference_type"] == "empirical"
].copy()

singleton_contract = response_contract[
    response_contract["reference_type"] == "singleton"
].copy()

if len(empirical_contract) != EXPECTED_EMPIRICAL_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_EMPIRICAL_COUNT} empirical channels, "
        f"found {len(empirical_contract)}."
    )

if len(singleton_contract) != EXPECTED_SINGLETON_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_SINGLETON_COUNT} singleton channels, "
        f"found {len(singleton_contract)}."
    )

RESPONSE_VARS = response_contract["variable"].astype(str).tolist()
EMPIRICAL_VARS = empirical_contract["variable"].astype(str).tolist()
SINGLETON_VARS = singleton_contract["variable"].astype(str).tolist()

singleton_values = singleton_contract[
    "singleton_reference_value"
].to_numpy(dtype=float)

if not np.isfinite(singleton_values).all():
    raise RuntimeError(
        "response-vector contract singleton_reference_value contains non-finite values."
    )


# ============================================================
# 4. LOAD TRAIN-ONLY REFERENCE-INTERVAL CONTRACT GLOBAL + CONTEXT REFERENCES
# ============================================================

if not GLOBAL_REFERENCE_FILE.exists():
    raise FileNotFoundError(GLOBAL_REFERENCE_FILE)

if not CONTEXT_REFERENCE_FILE.exists():
    raise FileNotFoundError(CONTEXT_REFERENCE_FILE)

global_reference = pd.read_csv(GLOBAL_REFERENCE_FILE)
context_reference = pd.read_csv(CONTEXT_REFERENCE_FILE)

required_global_cols = {
    "response_index",
    "variable",
    "reference_type",
}

for coverage in COVERAGES:
    tag = coverage_tag(coverage)
    required_global_cols.update({
        f"lower_q{tag}",
        f"upper_q{tag}",
    })

if not required_global_cols.issubset(global_reference.columns):
    missing = sorted(required_global_cols - set(global_reference.columns))
    raise ValueError(
        f"{GLOBAL_REFERENCE_FILE}: missing columns {missing}"
    )

required_context_cols = {
    "response_index",
    "design",
    "family",
    "representation",
    "dmin",
    "q",
    "context_id",
    "variable",
    "reference_type",
}

for coverage in COVERAGES:
    tag = coverage_tag(coverage)
    required_context_cols.update({
        f"lower_q{tag}",
        f"upper_q{tag}",
        f"reference_source_q{tag}",
    })

if not required_context_cols.issubset(context_reference.columns):
    missing = sorted(required_context_cols - set(context_reference.columns))
    raise ValueError(
        f"{CONTEXT_REFERENCE_FILE}: missing columns {missing}"
    )

global_reference["response_index"] = pd.to_numeric(
    global_reference["response_index"],
    errors="raise",
).astype(int)

global_reference = (
    global_reference
    .sort_values("response_index", kind="mergesort")
    .reset_index(drop=True)
)

if len(global_reference) != EXPECTED_RESPONSE_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_RESPONSE_COUNT} global reference rows, "
        f"found {len(global_reference)}."
    )

if not np.array_equal(
    global_reference["variable"].astype(str).to_numpy(),
    np.array(RESPONSE_VARS, dtype=object),
):
    raise RuntimeError(
        "train-only reference-interval contract global reference order does not match response-vector contract response contract."
    )

global_empirical = (
    global_reference[
        global_reference["reference_type"] == "empirical"
    ]
    .set_index("variable")
    .loc[EMPIRICAL_VARS]
)

global_singleton = (
    global_reference[
        global_reference["reference_type"] == "singleton"
    ]
    .set_index("variable")
    .loc[SINGLETON_VARS]
)

# Verify singleton limits are identical for all coverages.
for coverage in COVERAGES:
    tag = coverage_tag(coverage)

    if not np.array_equal(
        global_singleton[f"lower_q{tag}"].to_numpy(dtype=float),
        singleton_values,
    ):
        raise RuntimeError(
            f"Global singleton lower bounds do not match response-vector contract at q{tag}."
        )

    if not np.array_equal(
        global_singleton[f"upper_q{tag}"].to_numpy(dtype=float),
        singleton_values,
    ):
        raise RuntimeError(
            f"Global singleton upper bounds do not match response-vector contract at q{tag}."
        )


# ============================================================
# 5. LOAD HELD-OUT RESPONSE/CONTEXT CONTRACT CONTEXT-COLUMN CONTRACT
# ============================================================

if not CONTEXT_COLUMN_CONTRACT_FILE.exists():
    raise FileNotFoundError(CONTEXT_COLUMN_CONTRACT_FILE)

context_contract = pd.read_csv(CONTEXT_COLUMN_CONTRACT_FILE)

required_context_contract_cols = {
    "context_column",
    "design",
    "family",
    "representation",
    "dmin",
    "q",
}

if not required_context_contract_cols.issubset(context_contract.columns):
    missing = sorted(
        required_context_contract_cols - set(context_contract.columns)
    )
    raise ValueError(
        f"{CONTEXT_COLUMN_CONTRACT_FILE}: missing columns {missing}"
    )

if len(context_contract) != EXPECTED_CONTEXT_COLUMNS:
    raise RuntimeError(
        f"Expected {EXPECTED_CONTEXT_COLUMNS} context columns, "
        f"found {len(context_contract)}."
    )

if context_contract["context_column"].duplicated().any():
    raise RuntimeError(
        "Duplicated context columns in held-out response/context contract contract."
    )

CONTEXT_COLUMNS = context_contract["context_column"].astype(str).tolist()


# ============================================================
# 6. PRECOMPUTE REFERENCE LOOKUPS FOR THE 55 CONTEXT VARIANTS
# ============================================================

# lookup[(context_column, coverage)][context_id]
#   -> (lower[53], upper[53], source)
reference_lookup = {}

lookup_audit_rows = []

for _, spec in context_contract.iterrows():
    context_column = str(spec["context_column"])
    design = str(spec["design"])
    family = str(spec["family"])
    representation = str(spec["representation"])
    dmin = spec["dmin"]
    q = spec["q"]

    block = filter_context_reference(
        context_reference=context_reference,
        design=design,
        representation=representation,
        dmin=dmin,
        q=q,
    )

    if block.empty:
        raise RuntimeError(
            f"No train-only reference-interval contract context-reference rows for {context_column}."
        )

    empirical_block = block[
        block["reference_type"] == "empirical"
    ].copy()

    if empirical_block.empty:
        raise RuntimeError(
            f"No empirical reference rows for {context_column}."
        )

    # Every stored context group must have all 53 empirical channels.
    per_context_sizes = empirical_block.groupby("context_id").size()

    if not (per_context_sizes == EXPECTED_EMPIRICAL_COUNT).all():
        bad = per_context_sizes[
            per_context_sizes != EXPECTED_EMPIRICAL_COUNT
        ]
        raise RuntimeError(
            f"{context_column}: expected 53 empirical rows per context; "
            f"found:\n{bad.to_string()}"
        )

    for coverage in COVERAGES:
        tag = coverage_tag(coverage)
        by_context = {}

        for context_id, ctx in empirical_block.groupby("context_id", sort=True):
            context_id = int(context_id)

            ctx = (
                ctx
                .set_index("variable")
                .loc[EMPIRICAL_VARS]
            )

            lower = ctx[f"lower_q{tag}"].to_numpy(dtype=float)
            upper = ctx[f"upper_q{tag}"].to_numpy(dtype=float)

            if not (
                np.isfinite(lower).all()
                and np.isfinite(upper).all()
            ):
                raise RuntimeError(
                    f"{context_column}, context={context_id}, q{tag}: "
                    "non-finite reference bounds."
                )

            if not (lower <= upper).all():
                raise RuntimeError(
                    f"{context_column}, context={context_id}, q{tag}: "
                    "lower > upper."
                )

            sources = ctx[f"reference_source_q{tag}"].astype(str).unique()

            if len(sources) != 1:
                raise RuntimeError(
                    f"{context_column}, context={context_id}, q{tag}: "
                    f"inconsistent reference sources {sources.tolist()}."
                )

            source = str(sources[0])

            if source not in {"context", "global_support"}:
                raise RuntimeError(
                    f"{context_column}, context={context_id}, q{tag}: "
                    f"unexpected empirical reference source '{source}'."
                )

            by_context[context_id] = {
                "lower": lower,
                "upper": upper,
                "source": source,
            }

            lookup_audit_rows.append({
                "context_column": context_column,
                "design": design,
                "family": family,
                "representation": representation,
                "dmin": dmin,
                "q": q,
                "coverage": coverage,
                "context_id": context_id,
                "reference_source": source,
            })

        reference_lookup[(context_column, coverage)] = by_context


# ============================================================
# 7. APPLY FIXED REFERENCES TO EACH TEST RUN
# ============================================================

summary_rows = []
fallback_rows = []
check_rows = []

for test_run in TEST_RUNS:
    input_file = (
        RESPONSE_CONTEXT_DIR
        / test_run
        / "test_response_context.csv"
    )

    if not input_file.exists():
        raise FileNotFoundError(input_file)

    usecols = (
        ["row_id", "time"]
        + RESPONSE_VARS
        + CONTEXT_COLUMNS
    )

    frame = pd.read_csv(
        input_file,
        usecols=usecols,
    )

    n_rows = len(frame)
    expected_rows = EXPECTED_TEST_ROWS[test_run]

    if n_rows != expected_rows:
        raise RuntimeError(
            f"{test_run}: expected {expected_rows:,} rows, "
            f"found {n_rows:,}."
        )

    row_id = pd.to_numeric(
        frame["row_id"],
        errors="raise",
    ).astype(int).to_numpy()

    if not np.array_equal(
        row_id,
        np.arange(n_rows, dtype=int),
    ):
        raise RuntimeError(
            f"{test_run}: row_id is not contiguous 0..n-1."
        )

    # --------------------------------------------------------
    # 7.1 Fixed 75D test response vector
    # --------------------------------------------------------
    Y = frame[RESPONSE_VARS].apply(
        pd.to_numeric,
        errors="coerce",
    ).replace(
        [np.inf, -np.inf],
        np.nan,
    )

    if Y.isna().any().any():
        bad = Y.columns[Y.isna().any()].tolist()
        raise ValueError(
            f"{test_run}: non-finite response values in {bad}"
        )

    Y_empirical = Y[EMPIRICAL_VARS].to_numpy(dtype=float)
    Y_singleton = Y[SINGLETON_VARS].to_numpy(dtype=float)

    # One singleton count per tick; identical for GLOBAL and all 55 variants.
    singleton_count = exact_singleton_count(
        Y_singleton=Y_singleton,
        singleton_values=singleton_values,
    )

    # --------------------------------------------------------
    # 7.2 Compute q95 / q98 / q99 outputs independently
    # --------------------------------------------------------
    for coverage in COVERAGES:
        tag = coverage_tag(coverage)

        global_lower = global_empirical[
            f"lower_q{tag}"
        ].to_numpy(dtype=float)

        global_upper = global_empirical[
            f"upper_q{tag}"
        ].to_numpy(dtype=float)

        global_empirical_count = empirical_extreme_count(
            Y_empirical=Y_empirical,
            lower=global_lower,
            upper=global_upper,
        )

        global_total_count = (
            global_empirical_count.astype(np.uint16)
            + singleton_count.astype(np.uint16)
        ).astype(np.uint8)

        out = pd.DataFrame({
            "row_id": row_id,
            "time": frame["time"].to_numpy(),
            "singleton_extreme_count": singleton_count,
            "GLOBAL": global_total_count,
        })

        # ----------------------------------------------------
        # 7.3 55 context-conditioned variants
        # ----------------------------------------------------
        for _, spec in context_contract.iterrows():
            context_column = str(spec["context_column"])
            design = str(spec["design"])
            family = str(spec["family"])
            representation = str(spec["representation"])

            labels = nullable_integer_labels(
                frame[context_column],
                f"{test_run}/{context_column}",
            )

            # Start from GLOBAL empirical count.
            # Rows with NA / predefined global contexts stay GLOBAL.
            empirical_count = global_empirical_count.copy()

            n_na = int(labels.isna().sum())
            n_predefined_global = 0
            n_context_reference = 0
            n_global_support = 0

            valid_labels = labels.dropna().astype(int)

            for context_id in sorted(valid_labels.unique()):
                context_id = int(context_id)

                mask = (
                    labels == context_id
                ).fillna(False).to_numpy()

                n_mask = int(mask.sum())

                if predefined_global_context(
                    family=family,
                    representation=representation,
                    context_id=context_id,
                ):
                    # Keep GLOBAL empirical count.
                    n_predefined_global += n_mask
                    continue

                lookup = reference_lookup[
                    (context_column, coverage)
                ]

                if context_id not in lookup:
                    raise RuntimeError(
                        f"{test_run}/{context_column}/q{tag}: "
                        f"context_id={context_id} has no train-only reference-interval contract reference "
                        "and is not an allowed predefined-global context."
                    )

                ref = lookup[context_id]

                context_empirical_count = empirical_extreme_count(
                    Y_empirical=Y_empirical[mask],
                    lower=ref["lower"],
                    upper=ref["upper"],
                )

                empirical_count[mask] = context_empirical_count

                if ref["source"] == "context":
                    n_context_reference += n_mask
                elif ref["source"] == "global_support":
                    n_global_support += n_mask
                else:
                    raise RuntimeError(
                        f"{test_run}/{context_column}/q{tag}: "
                        f"unexpected source {ref['source']}."
                    )

            total_count = (
                empirical_count.astype(np.uint16)
                + singleton_count.astype(np.uint16)
            ).astype(np.uint8)

            out[context_column] = total_count

            fallback_rows.append({
                "test_run": test_run,
                "coverage": coverage,
                "context_column": context_column,
                "design": design,
                "family": family,
                "representation": representation,
                "dmin": spec["dmin"],
                "q": spec["q"],
                "n_ticks": n_rows,
                "n_context_reference_ticks": n_context_reference,
                "n_global_support_ticks": n_global_support,
                "n_predefined_global_ticks": n_predefined_global,
                "n_undefined_context_ticks": n_na,
                "n_global_empirical_ticks_total": (
                    n_global_support
                    + n_predefined_global
                    + n_na
                ),
            })

        # ----------------------------------------------------
        # 7.4 Output checks
        # ----------------------------------------------------
        count_columns = ["GLOBAL"] + CONTEXT_COLUMNS

        if len(count_columns) != EXPECTED_TOTAL_COUNT_COLUMNS:
            raise RuntimeError(
                f"Expected {EXPECTED_TOTAL_COUNT_COLUMNS} total-count columns, "
                f"found {len(count_columns)}."
            )

        counts = out[count_columns].to_numpy(dtype=int)

        checks_for_file = [
            (
                "row_count_matches_test_contract",
                len(out) == expected_rows,
            ),
            (
                "singleton_count_in_0_22",
                (
                    (out["singleton_extreme_count"] >= 0)
                    & (out["singleton_extreme_count"] <= EXPECTED_SINGLETON_COUNT)
                ).all(),
            ),
            (
                "all_total_counts_in_0_75",
                bool(
                    (counts >= 0).all()
                    and (counts <= EXPECTED_RESPONSE_COUNT).all()
                ),
            ),
            (
                "all_total_counts_ge_singleton_count",
                bool(
                    (
                        counts
                        >= out["singleton_extreme_count"]
                        .to_numpy(dtype=int)[:, None]
                    ).all()
                ),
            ),
            (
                "output_has_56_total_count_columns",
                len(count_columns) == EXPECTED_TOTAL_COUNT_COLUMNS,
            ),
            (
                "attack_labels_absent",
                not any(
                    "attack" in c.lower()
                    for c in out.columns
                ),
            ),
        ]

        for check, passed in checks_for_file:
            check_rows.append({
                "test_run": test_run,
                "coverage": coverage,
                "check": check,
                "status": "PASS" if passed else "FAIL",
            })

        # ----------------------------------------------------
        # 7.5 Save one compact file per coverage
        # ----------------------------------------------------
        run_dir = OUTPUT_DIR / test_run
        run_dir.mkdir(parents=True, exist_ok=True)

        output_file = (
            run_dir
            / f"extreme_channel_counts_q{tag}.csv"
        )

        out.to_csv(
            output_file,
            index=False,
        )

        # Compact label-blind descriptive summary.
        summary_rows.append({
            "test_run": test_run,
            "coverage": coverage,
            "is_primary": coverage == PRIMARY_COVERAGE,
            "n_ticks": n_rows,
            "n_singleton_channels": EXPECTED_SINGLETON_COUNT,
            "n_empirical_channels": EXPECTED_EMPIRICAL_COUNT,
            "n_total_response_channels": EXPECTED_RESPONSE_COUNT,
            "n_total_count_variants": EXPECTED_TOTAL_COUNT_COLUMNS,
            "mean_singleton_extreme_count": float(
                out["singleton_extreme_count"].mean()
            ),
            "max_singleton_extreme_count": int(
                out["singleton_extreme_count"].max()
            ),
            "mean_global_extreme_count": float(
                out["GLOBAL"].mean()
            ),
            "max_global_extreme_count": int(
                out["GLOBAL"].max()
            ),
            "min_variant_mean_extreme_count": float(
                out[count_columns].mean().min()
            ),
            "max_variant_mean_extreme_count": float(
                out[count_columns].mean().max()
            ),
            "output_file": str(output_file),
        })

        print(
            f"{test_run} q{tag}: "
            f"singleton + GLOBAL + 55 SERF counts complete"
        )


# ============================================================
# 8. GLOBAL AUDITS
# ============================================================

summary = pd.DataFrame(summary_rows)
fallback = pd.DataFrame(fallback_rows)
checks = pd.DataFrame(check_rows)
lookup_audit = pd.DataFrame(lookup_audit_rows)

total_primary_rows = int(
    summary.loc[
        summary["coverage"] == PRIMARY_COVERAGE,
        "n_ticks",
    ].sum()
)

checks = pd.concat(
    [
        checks,
        pd.DataFrame([
            {
                "test_run": "ALL",
                "coverage": PRIMARY_COVERAGE,
                "check": "primary_q99_total_test_rows_is_402005",
                "status": (
                    "PASS"
                    if total_primary_rows == EXPECTED_TOTAL_TEST_ROWS
                    else "FAIL"
                ),
            },
            {
                "test_run": "ALL",
                "coverage": np.nan,
                "check": "three_coverage_levels_completed",
                "status": (
                    "PASS"
                    if set(summary["coverage"].unique()) == set(COVERAGES)
                    else "FAIL"
                ),
            },
            {
                "test_run": "ALL",
                "coverage": np.nan,
                "check": "five_test_runs_per_coverage",
                "status": (
                    "PASS"
                    if summary.groupby("coverage")["test_run"].nunique().eq(5).all()
                    else "FAIL"
                ),
            },
            {
                "test_run": "ALL",
                "coverage": np.nan,
                "check": "55_fallback_audit_rows_per_test_coverage",
                "status": (
                    "PASS"
                    if fallback.groupby(
                        ["test_run", "coverage"]
                    ).size().eq(EXPECTED_CONTEXT_COLUMNS).all()
                    else "FAIL"
                ),
            },
            {
                "test_run": "ALL",
                "coverage": np.nan,
                "check": "fallback_tick_accounting_matches_test_length",
                "status": (
                    "PASS"
                    if (
                        fallback[
                            [
                                "n_context_reference_ticks",
                                "n_global_support_ticks",
                                "n_predefined_global_ticks",
                                "n_undefined_context_ticks",
                            ]
                        ].sum(axis=1)
                        == fallback["n_ticks"]
                    ).all()
                    else "FAIL"
                ),
            },
        ]),
    ],
    ignore_index=True,
)

if not (checks["status"] == "PASS").all():
    bad = checks.loc[
        checks["status"] != "PASS"
    ]

    raise RuntimeError(
        "label-blind extremeness scoring integrity checks failed:\n"
        + bad.to_string(index=False)
    )


# ============================================================
# 9. SAVE AUDITS / SETTINGS
# ============================================================

summary.to_csv(
    OUTPUT_DIR / "test_extreme_count_summary.csv",
    index=False,
)

fallback.to_csv(
    OUTPUT_DIR / "reference_application_audit.csv",
    index=False,
)

lookup_audit.to_csv(
    OUTPUT_DIR / "reference_lookup_audit.csv",
    index=False,
)

checks.to_csv(
    OUTPUT_DIR / "integrity_checks.csv",
    index=False,
)

settings_rows = [
    ("cell_purpose", "label-blind test extreme-channel counting"),
    ("test_runs", ",".join(TEST_RUNS)),
    ("total_test_ticks", EXPECTED_TOTAL_TEST_ROWS),
    ("response_vector_dimension", EXPECTED_RESPONSE_COUNT),
    ("empirical_reference_channels", EXPECTED_EMPIRICAL_COUNT),
    ("singleton_reference_channels", EXPECTED_SINGLETON_COUNT),
    ("singleton_count_columns", 1),
    ("total_count_variants", EXPECTED_TOTAL_COUNT_COLUMNS),
    ("global_variants", 1),
    ("serf_context_variants", EXPECTED_CONTEXT_COLUMNS),
    ("primary_coverage", PRIMARY_COVERAGE),
    (
        "sensitivity_coverages",
        ",".join(str(c) for c in COVERAGES if c != PRIMARY_COVERAGE),
    ),
    (
        "empirical_extreme_rule",
        "y < lower OR y > upper; equality is inside",
    ),
    (
        "singleton_extreme_rule",
        "y != train singleton reference value",
    ),
    (
        "undefined_context_policy",
        "GLOBAL empirical interval of the same coverage",
    ),
    (
        "hdbscan_minus1_policy",
        "GLOBAL empirical interval of the same coverage",
    ),
    (
        "meta_m0_policy",
        "GLOBAL empirical interval of the same coverage",
    ),
    (
        "low_support_policy",
        "use train-only reference-interval contract bounds, which are already replaced by GLOBAL bounds",
    ),
    ("attack_labels_read", False),
    ("attack_labels_written", False),
    ("threshold_on_extreme_channel_count", "none"),
    ("design_selection", "none"),
]

pd.DataFrame(
    settings_rows,
    columns=["setting", "value"],
).to_csv(
    OUTPUT_DIR / "settings.csv",
    index=False,
)


# ============================================================
# 10. NOTEBOOK REPORT
# ============================================================

print("\nLABEL-BLIND EXTREMENESS SCORING COMPLETE")
print(f"Test ticks: {EXPECTED_TOTAL_TEST_ROWS:,}")
print(
    f"Response vector: {EXPECTED_RESPONSE_COUNT} channels "
    f"({EXPECTED_EMPIRICAL_COUNT} empirical + "
    f"{EXPECTED_SINGLETON_COUNT} singleton)"
)
print(
    f"Per coverage: 1 singleton count + "
    f"{EXPECTED_TOTAL_COUNT_COLUMNS} total counts "
    "(GLOBAL + 55 SERF variants)"
)
print("Primary: q99 | Sensitivity: q95, q98")
print("Attack labels used: NO")
print(
    f"Integrity checks: "
    f"{(checks['status'] == 'PASS').sum()}/{len(checks)} PASS"
)
print(f"Output directory: {OUTPUT_DIR}")

print("\nPrimary q99 label-blind summary:")
print(
    summary.loc[
        summary["coverage"] == PRIMARY_COVERAGE,
        [
            "test_run",
            "n_ticks",
            "mean_singleton_extreme_count",
            "max_singleton_extreme_count",
            "mean_global_extreme_count",
            "max_global_extreme_count",
            "min_variant_mean_extreme_count",
            "max_variant_mean_extreme_count",
        ],
    ].to_string(index=False)
)

test1 q95: singleton + GLOBAL + 55 SERF counts complete
test1 q98: singleton + GLOBAL + 55 SERF counts complete
test1 q99: singleton + GLOBAL + 55 SERF counts complete
test2 q95: singleton + GLOBAL + 55 SERF counts complete
test2 q98: singleton + GLOBAL + 55 SERF counts complete
test2 q99: singleton + GLOBAL + 55 SERF counts complete
test3 q95: singleton + GLOBAL + 55 SERF counts complete
test3 q98: singleton + GLOBAL + 55 SERF counts complete
test3 q99: singleton + GLOBAL + 55 SERF counts complete
test4 q95: singleton + GLOBAL + 55 SERF counts complete
test4 q98: singleton + GLOBAL + 55 SERF counts complete
test4 q99: singleton + GLOBAL + 55 SERF counts complete
test5 q95: singleton + GLOBAL + 55 SERF counts complete
test5 q98: singleton + GLOBAL + 55 SERF counts complete
test5 q99: singleton + GLOBAL + 55 SERF counts complete

LABEL-BLIND EXTREMENESS SCORING COMPLETE
Test ticks: 402,005
Response vector: 75 channels (53 empirical + 22 singleton)
Per coverage: 1 singleton count + 56 to

### 5.6.3 Compare label-blind extremeness within aligned meta-modes


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# HAI 21.03 — LABEL-BLIND ALIGNED-META-MODE ANALYSIS
# LABEL-BLIND EXTREMENESS BY ALIGNED META-MODE
#
# Purpose
# -------
# Combine the FIXED aligned meta-mode assignments from held-out SERF post-processing with
# the FIXED label-blind extreme-channel counts K_t from label-blind extremeness scoring.
#
# For each design x dmin aligned representation and shared meta-mode
# M1..M7, compare GLOBAL and meta-mode-conditioned K_t on exactly the
# same timestamps. The analysis remains fully label-blind.
#
# This cell DOES NOT:
#   - read attack labels or raw HAI test files;
#   - refit segmentation models;
#   - restabilize episodes;
#   - remap native states to meta-modes;
#   - recompute reference intervals;
#   - recompute channel-level extremeness.
#
# Inputs
# ------
# held-out SERF post-processing:
#   outputs/section_5_4/test_serf_postprocessing/
#       testX/DESIGN/serf_labels_dmin{5|10}_q7.csv
#
# label-blind extremeness scoring:
#   outputs/section_5_6/test_extreme_channel_counts/
#       testX/extreme_channel_counts_q{95|98|99}.csv
#
# Outputs
# -------
#   meta_mode_extremeness_by_run_all_coverages.csv
#   meta_mode_extremeness_pooled_all_coverages.csv
#   shared_meta_mode_ranges_all_coverages.csv
#   Supplementary_Table_S17_label_blind_extremeness_by_aligned_meta_mode_all_coverages.csv
#   Supplementary_Table_S17_compact_q99.csv
#   m0_fallback_audit_all_coverages.csv
#   undefined_meta_mode_audit.csv
#   integrity_checks.csv
#   settings.csv
# ============================================================


# ============================================================
# 1. FIXED SETTINGS
# ============================================================

TEST_IDS = [f"test{i}" for i in range(1, 6)]
EXPECTED_TEST_ROWS = {
    "test1": 43_201,
    "test2": 118_801,
    "test3": 108_001,
    "test4": 39_601,
    "test5": 92_401,
}
EXPECTED_TOTAL_TEST_ROWS = 402_005

DMIN_VALUES = [5, 10]
META_MODES = list(range(1, 8))
COVERAGES = [0.95, 0.98, 0.99]
PRIMARY_COVERAGE = 0.99
LAYERS = [1, 3, 5, 10, 15]

DESIGNS = {
    "KM4_8": "kmeans",
    "KM7_8": "kmeans",
    "KM13_8": "kmeans",
    "KM13_16": "kmeans",
    "HDB23_8": "hdbscan",
    "HDB18_8": "hdbscan",
    "HDB15_8": "hdbscan",
    "HDB18_16": "hdbscan",
    "HMM4": "hmm",
    "HMM5": "hmm",
    "HMM8": "hmm",
}

SERF_TEST_DIR_CANDIDATES = [
    Path("outputs/section_5_4/test_serf_postprocessing"),
]
EXTREMENESS_DIR_CANDIDATES = [
    Path("outputs/section_5_6/test_extreme_channel_counts"),
    Path("c20x"),
]

OUTPUT_DIR = Path("outputs/section_5_6/extremeness_by_meta_mode")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

META_COL = "meta_mode_q7"
GLOBAL_COL = "GLOBAL"


# ============================================================
# 2. HELPERS
# ============================================================

def coverage_tag(coverage: float) -> str:
    return str(int(round(coverage * 100)))


def first_existing_dir(candidates, required_relpath, label):
    for directory in candidates:
        if directory.exists() and (directory / required_relpath).exists():
            return directory
    searched = ", ".join(str(d / required_relpath) for d in candidates)
    raise FileNotFoundError(f"Could not locate {label}. Searched: {searched}")


def normalize_time(series: pd.Series, name: str) -> pd.Series:
    out = pd.to_datetime(series, errors="coerce")
    if out.isna().any():
        raise ValueError(f"{name}: invalid timestamps.")
    return out


def numeric_k(series: pd.Series, name: str) -> np.ndarray:
    x = pd.to_numeric(series, errors="coerce")
    if x.isna().any():
        raise ValueError(f"{name}: missing/non-numeric K_t values.")
    values = x.to_numpy(dtype=int)
    if ((values < 0) | (values > 75)).any():
        raise ValueError(f"{name}: K_t outside 0..75.")
    return values


def nullable_meta_mode(series: pd.Series, name: str) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    valid = x.notna()
    if valid.any():
        values = x.loc[valid].to_numpy(dtype=float)
        if not np.array_equal(values, np.round(values)):
            raise ValueError(f"{name}: non-integer meta-mode labels.")
    out = x.round().astype("Int64")
    used = set(out.dropna().astype(int).unique())
    if not used.issubset(set(range(0, 8))):
        raise ValueError(f"{name}: unexpected meta-mode ids {sorted(used)}.")
    return out


def quantile(values: np.ndarray, q: float) -> float:
    if len(values) == 0:
        return np.nan
    return float(np.quantile(values, q, method="linear"))


def pct(mask_count: int, denominator: int) -> float:
    return 100.0 * float(mask_count) / float(denominator) if denominator else np.nan


def summarize_k(values: np.ndarray, prefix: str) -> dict:
    if len(values) == 0:
        row = {
            f"{prefix}_mean_k": np.nan,
            f"{prefix}_median_k": np.nan,
            f"{prefix}_q90_k": np.nan,
            f"{prefix}_q95_k": np.nan,
            f"{prefix}_q99_k": np.nan,
            f"{prefix}_max_k": np.nan,
        }
        for k in LAYERS:
            row[f"{prefix}_share_k_ge_{k}_pct"] = np.nan
        return row

    row = {
        f"{prefix}_mean_k": float(np.mean(values)),
        f"{prefix}_median_k": float(np.median(values)),
        f"{prefix}_q90_k": quantile(values, 0.90),
        f"{prefix}_q95_k": quantile(values, 0.95),
        f"{prefix}_q99_k": quantile(values, 0.99),
        f"{prefix}_max_k": int(np.max(values)),
    }
    for k in LAYERS:
        row[f"{prefix}_share_k_ge_{k}_pct"] = pct(int(np.sum(values >= k)), len(values))
    return row


def add_deltas(row: dict) -> dict:
    for metric in ["mean_k", "median_k", "q90_k", "q95_k", "q99_k", "max_k"]:
        g = row[f"global_{metric}"]
        m = row[f"meta_{metric}"]
        row[f"delta_{metric}"] = m - g if pd.notna(g) and pd.notna(m) else np.nan

    for k in LAYERS:
        g = row[f"global_share_k_ge_{k}_pct"]
        m = row[f"meta_share_k_ge_{k}_pct"]
        row[f"delta_share_k_ge_{k}_pp"] = m - g if pd.notna(g) and pd.notna(m) else np.nan
    return row


def check_alignment(test_id: str, scores: pd.DataFrame, labels: pd.DataFrame):
    expected_n = EXPECTED_TEST_ROWS[test_id]
    if len(scores) != expected_n or len(labels) != expected_n:
        raise RuntimeError(
            f"{test_id}: row-count mismatch scores={len(scores)}, labels={len(labels)}, "
            f"expected={expected_n}."
        )

    expected = np.arange(expected_n, dtype=int)
    score_row = pd.to_numeric(scores["row_id"], errors="raise").to_numpy(dtype=int)
    label_row = pd.to_numeric(labels["row_id"], errors="raise").to_numpy(dtype=int)

    if not np.array_equal(score_row, expected):
        raise RuntimeError(f"{test_id}: label-blind extremeness scoring row_id is not 0..n-1.")
    if not np.array_equal(label_row, expected):
        raise RuntimeError(f"{test_id}: held-out SERF post-processing row_id is not 0..n-1.")

    score_time = normalize_time(scores["time"], f"{test_id}/Cell20/time")
    label_time = normalize_time(labels["time"], f"{test_id}/Cell16/time")
    if not np.array_equal(score_time.to_numpy(), label_time.to_numpy()):
        raise RuntimeError(f"{test_id}: held-out SERF post-processing vs label-blind extremeness scoring timestamp mismatch.")


def range_columns(part: pd.DataFrame, column: str) -> tuple:
    x = pd.to_numeric(part[column], errors="coerce").dropna()
    if x.empty:
        return np.nan, np.nan
    return float(x.min()), float(x.max())


def range_text(lo, hi, decimals=2, suffix=""):
    if pd.isna(lo) or pd.isna(hi):
        return ""
    return f"{float(lo):.{decimals}f}–{float(hi):.{decimals}f}{suffix}"


# ============================================================
# 3. LOCATE FROZEN INPUTS
# ============================================================

SERF_TEST_DIR = first_existing_dir(
    SERF_TEST_DIR_CANDIDATES,
    Path("test1") / "KM4_8" / "serf_labels_dmin5_q7.csv",
    "held-out SERF post-processing SERF post-processing output",
)
EXTREMENESS_DIR = first_existing_dir(
    EXTREMENESS_DIR_CANDIDATES,
    Path("test1") / "extreme_channel_counts_q99.csv",
    "label-blind extremeness scoring extreme-channel-count output",
)


# ============================================================
# 4. LABEL-BLIND META-MODE ANALYSIS BY RUN
# ============================================================

by_run_rows = []
m0_rows = []
undefined_rows = []
check_rows = []

# Accumulate exact tick-level values for pooled summaries across all five runs.
pooled_acc = {}

for coverage in COVERAGES:
    tag = coverage_tag(coverage)

    for test_id in TEST_IDS:
        score_file = EXTREMENESS_DIR / test_id / f"extreme_channel_counts_q{tag}.csv"
        if not score_file.exists():
            raise FileNotFoundError(score_file)

        scores = pd.read_csv(score_file)
        if not {"row_id", "time", GLOBAL_COL}.issubset(scores.columns):
            raise ValueError(f"{score_file}: missing row_id/time/GLOBAL.")
        if any("attack" in c.lower() for c in scores.columns):
            raise RuntimeError(f"{score_file}: attack-linked column found in label-blind input.")

        global_k = numeric_k(scores[GLOBAL_COL], f"{test_id}/q{tag}/GLOBAL")

        for design, family in DESIGNS.items():
            for dmin in DMIN_VALUES:
                label_file = (
                    SERF_TEST_DIR
                    / test_id
                    / design
                    / f"serf_labels_dmin{dmin}_q7.csv"
                )
                if not label_file.exists():
                    raise FileNotFoundError(label_file)

                labels = pd.read_csv(label_file)
                required_label_cols = {"row_id", "time", META_COL}
                if not required_label_cols.issubset(labels.columns):
                    missing = sorted(required_label_cols - set(labels.columns))
                    raise ValueError(f"{label_file}: missing columns {missing}.")
                if any("attack" in c.lower() for c in labels.columns):
                    raise RuntimeError(f"{label_file}: attack-linked column found in held-out SERF post-processing input.")

                check_alignment(test_id, scores, labels)

                meta_mode = nullable_meta_mode(
                    labels[META_COL],
                    f"{test_id}/{design}/dmin{dmin}/{META_COL}",
                )

                meta_col = f"{design}__meta_q7_dmin{dmin}"
                if meta_col not in scores.columns:
                    raise ValueError(f"{score_file}: missing '{meta_col}'.")
                meta_k = numeric_k(scores[meta_col], f"{test_id}/q{tag}/{meta_col}")

                # M0 is a predefined GLOBAL-fallback case, not a substantive regime.
                m0_mask = meta_mode.eq(0).fillna(False).to_numpy()
                n_m0 = int(m0_mask.sum())
                m0_mismatch = int(np.sum(global_k[m0_mask] != meta_k[m0_mask])) if n_m0 else 0
                m0_rows.append({
                    "coverage": coverage,
                    "test_id": test_id,
                    "design": design,
                    "family": family,
                    "dmin": dmin,
                    "n_ticks": len(labels),
                    "n_m0_ticks": n_m0,
                    "m0_share_of_run_pct": pct(n_m0, len(labels)),
                    "n_GLOBAL_vs_META_mismatches_on_m0": m0_mismatch,
                })

                undef_mask = meta_mode.isna().to_numpy()
                undefined_rows.append({
                    "test_id": test_id,
                    "design": design,
                    "family": family,
                    "dmin": dmin,
                    "n_ticks": len(labels),
                    "n_undefined_meta_mode_ticks": int(undef_mask.sum()),
                    "undefined_share_of_run_pct": pct(int(undef_mask.sum()), len(labels)),
                })

                # Integrity checks once per combination/coverage.
                used_modes = set(meta_mode.dropna().astype(int).unique())
                check_rows.extend([
                    {
                        "coverage": coverage,
                        "test_id": test_id,
                        "design": design,
                        "dmin": dmin,
                        "check": "meta_ids_subset_0_to_7",
                        "status": "PASS" if used_modes.issubset(set(range(0, 8))) else "FAIL",
                    },
                    {
                        "coverage": coverage,
                        "test_id": test_id,
                        "design": design,
                        "dmin": dmin,
                        "check": "m0_GLOBAL_equals_META",
                        "status": "PASS" if m0_mismatch == 0 else "FAIL",
                    },
                ])

                for mode in META_MODES:
                    mode_mask = meta_mode.eq(mode).fillna(False).to_numpy()
                    n_mode = int(mode_mask.sum())

                    g = global_k[mode_mask]
                    m = meta_k[mode_mask]

                    row = {
                        "coverage": coverage,
                        "is_primary": coverage == PRIMARY_COVERAGE,
                        "test_id": test_id,
                        "design": design,
                        "family": family,
                        "dmin": dmin,
                        "meta_context_column": meta_col,
                        "meta_mode": mode,
                        "n_ticks_in_meta_mode": n_mode,
                        "share_of_run_pct": pct(n_mode, len(labels)),
                    }
                    row.update(summarize_k(g, "global"))
                    row.update(summarize_k(m, "meta"))
                    row = add_deltas(row)
                    by_run_rows.append(row)

                    key = (coverage, design, family, dmin, meta_col, mode)
                    if key not in pooled_acc:
                        pooled_acc[key] = {"global": [], "meta": [], "n_by_run": []}
                    if n_mode:
                        pooled_acc[key]["global"].append(g.astype(np.uint8, copy=False))
                        pooled_acc[key]["meta"].append(m.astype(np.uint8, copy=False))
                    pooled_acc[key]["n_by_run"].append(n_mode)


by_run = pd.DataFrame(by_run_rows)
m0_audit = pd.DataFrame(m0_rows)
undefined_audit = pd.DataFrame(undefined_rows).drop_duplicates().reset_index(drop=True)
checks = pd.DataFrame(check_rows)

expected_by_run_rows = (
    len(COVERAGES) * len(TEST_IDS) * len(DESIGNS) * len(DMIN_VALUES) * len(META_MODES)
)
if len(by_run) != expected_by_run_rows:
    raise RuntimeError(f"Unexpected by-run row count: {len(by_run)} != {expected_by_run_rows}")

if not checks["status"].eq("PASS").all():
    failed = checks.loc[~checks["status"].eq("PASS")]
    raise RuntimeError("label-blind aligned-meta-mode analysis integrity failure:\n" + failed.to_string(index=False))


# ============================================================
# 5. EXACT POOLED SUMMARIES BY ALIGNED REPRESENTATION
# ============================================================

pooled_rows = []

for key, acc in pooled_acc.items():
    coverage, design, family, dmin, meta_col, mode = key
    g = np.concatenate(acc["global"]) if acc["global"] else np.array([], dtype=np.uint8)
    m = np.concatenate(acc["meta"]) if acc["meta"] else np.array([], dtype=np.uint8)

    if len(g) != len(m):
        raise RuntimeError(
            f"Pooled length mismatch for {design}, dmin={dmin}, M{mode}, coverage={coverage}."
        )

    row = {
        "coverage": coverage,
        "is_primary": coverage == PRIMARY_COVERAGE,
        "design": design,
        "family": family,
        "dmin": dmin,
        "meta_context_column": meta_col,
        "meta_mode": mode,
        "n_ticks_in_meta_mode": int(len(g)),
        "share_of_full_test_timeline_pct": pct(len(g), EXPECTED_TOTAL_TEST_ROWS),
        "runs_with_meta_mode_present": int(np.sum(np.asarray(acc["n_by_run"]) > 0)),
    }
    row.update(summarize_k(g, "global"))
    row.update(summarize_k(m, "meta"))
    row = add_deltas(row)
    pooled_rows.append(row)

pooled = pd.DataFrame(pooled_rows)
expected_pooled_rows = len(COVERAGES) * len(DESIGNS) * len(DMIN_VALUES) * len(META_MODES)
if len(pooled) != expected_pooled_rows:
    raise RuntimeError(f"Unexpected pooled row count: {len(pooled)} != {expected_pooled_rows}")


# ============================================================
# 6. SHARED M1..M7 SUMMARY ACROSS ALIGNED REPRESENTATIONS
# ============================================================

shared_rows = []

metric_columns = [
    "share_of_full_test_timeline_pct",
    "global_mean_k", "meta_mean_k", "delta_mean_k",
    "global_median_k", "meta_median_k", "delta_median_k",
    "global_q90_k", "meta_q90_k", "delta_q90_k",
    "global_q95_k", "meta_q95_k", "delta_q95_k",
    "global_q99_k", "meta_q99_k", "delta_q99_k",
    "global_max_k", "meta_max_k", "delta_max_k",
]
for k in LAYERS:
    metric_columns.extend([
        f"global_share_k_ge_{k}_pct",
        f"meta_share_k_ge_{k}_pct",
        f"delta_share_k_ge_{k}_pp",
    ])

for coverage in COVERAGES:
    for mode in META_MODES:
        part0 = pooled.loc[
            pooled["coverage"].eq(coverage)
            & pooled["meta_mode"].eq(mode)
        ].copy()

        # An aligned representation contributes to the regime summary only
        # when that meta-mode is actually present in its pooled test timeline.
        part = part0.loc[part0["n_ticks_in_meta_mode"] > 0].copy()

        row = {
            "coverage": coverage,
            "is_primary": coverage == PRIMARY_COVERAGE,
            "meta_mode": mode,
            "active_aligned_representations": int(len(part)),
            "total_aligned_representations": int(len(part0)),
            "n_ticks_in_meta_mode_min": int(part["n_ticks_in_meta_mode"].min()) if not part.empty else 0,
            "n_ticks_in_meta_mode_max": int(part["n_ticks_in_meta_mode"].max()) if not part.empty else 0,
            "runs_with_meta_mode_present_min": int(part["runs_with_meta_mode_present"].min()) if not part.empty else 0,
            "runs_with_meta_mode_present_max": int(part["runs_with_meta_mode_present"].max()) if not part.empty else 0,
        }

        for col in metric_columns:
            lo, hi = range_columns(part, col)
            row[f"{col}_min"] = lo
            row[f"{col}_max"] = hi

        shared_rows.append(row)

shared = pd.DataFrame(shared_rows)


# ============================================================
# 7. SUPPLEMENTARY TABLE S17 + COMPACT q99 TABLE
# ============================================================

# Full numeric S17: one row per coverage x shared meta-mode, with min/max
# across aligned design x dmin representations in which the mode is present.
s17 = shared.copy()

# Compact article-oriented q99 view. Keep numeric ranges readable while the
# full S17 retains all q95/q98/q99 metrics and all nested layers.
compact_rows = []
q99 = shared.loc[shared["coverage"].eq(PRIMARY_COVERAGE)].sort_values("meta_mode")

for _, r in q99.iterrows():
    compact_rows.append({
        "meta_mode": f"M{int(r['meta_mode'])}",
        "active_aligned_representations": int(r["active_aligned_representations"]),
        "support_pct_range": range_text(
            r["share_of_full_test_timeline_pct_min"],
            r["share_of_full_test_timeline_pct_max"],
            decimals=2,
            suffix="%",
        ),
        "GLOBAL_mean_Kt_range": range_text(
            r["global_mean_k_min"], r["global_mean_k_max"], decimals=2
        ),
        "META_mean_Kt_range": range_text(
            r["meta_mean_k_min"], r["meta_mean_k_max"], decimals=2
        ),
        "delta_mean_Kt_range": range_text(
            r["delta_mean_k_min"], r["delta_mean_k_max"], decimals=2
        ),
        "GLOBAL_Kt_ge_5_pct_range": range_text(
            r["global_share_k_ge_5_pct_min"],
            r["global_share_k_ge_5_pct_max"],
            decimals=2,
            suffix="%",
        ),
        "META_Kt_ge_5_pct_range": range_text(
            r["meta_share_k_ge_5_pct_min"],
            r["meta_share_k_ge_5_pct_max"],
            decimals=2,
            suffix="%",
        ),
        "delta_Kt_ge_5_pp_range": range_text(
            r["delta_share_k_ge_5_pp_min"],
            r["delta_share_k_ge_5_pp_max"],
            decimals=2,
            suffix=" pp",
        ),
        "GLOBAL_Kt_ge_10_pct_range": range_text(
            r["global_share_k_ge_10_pct_min"],
            r["global_share_k_ge_10_pct_max"],
            decimals=2,
            suffix="%",
        ),
        "META_Kt_ge_10_pct_range": range_text(
            r["meta_share_k_ge_10_pct_min"],
            r["meta_share_k_ge_10_pct_max"],
            decimals=2,
            suffix="%",
        ),
        "delta_Kt_ge_10_pp_range": range_text(
            r["delta_share_k_ge_10_pp_min"],
            r["delta_share_k_ge_10_pp_max"],
            decimals=2,
            suffix=" pp",
        ),
    })

s17_compact_q99 = pd.DataFrame(compact_rows)


# ============================================================
# 8. FINAL INTEGRITY CHECKS
# ============================================================

final_checks = [
    {
        "check": "by_run_expected_row_count",
        "value": len(by_run),
        "expected": expected_by_run_rows,
        "status": "PASS" if len(by_run) == expected_by_run_rows else "FAIL",
    },
    {
        "check": "pooled_expected_row_count",
        "value": len(pooled),
        "expected": expected_pooled_rows,
        "status": "PASS" if len(pooled) == expected_pooled_rows else "FAIL",
    },
    {
        "check": "shared_expected_row_count",
        "value": len(shared),
        "expected": len(COVERAGES) * len(META_MODES),
        "status": "PASS" if len(shared) == len(COVERAGES) * len(META_MODES) else "FAIL",
    },
    {
        "check": "all_m0_GLOBAL_equals_META",
        "value": int(m0_audit["n_GLOBAL_vs_META_mismatches_on_m0"].sum()),
        "expected": 0,
        "status": "PASS" if int(m0_audit["n_GLOBAL_vs_META_mismatches_on_m0"].sum()) == 0 else "FAIL",
    },
    {
        "check": "no_attack_linked_inputs_used",
        "value": 0,
        "expected": 0,
        "status": "PASS",
    },
]

final_checks_df = pd.DataFrame(final_checks)
if not final_checks_df["status"].eq("PASS").all():
    raise RuntimeError("Final label-blind aligned-meta-mode analysis integrity check failed.")

# Keep detailed alignment/M0 checks as well.
integrity = pd.concat(
    [
        checks.assign(value=np.nan, expected=np.nan),
        final_checks_df.assign(coverage=np.nan, test_id="", design="", dmin=np.nan),
    ],
    ignore_index=True,
    sort=False,
)


# ============================================================
# 9. SAVE OUTPUTS
# ============================================================

by_run.to_csv(
    OUTPUT_DIR / "meta_mode_extremeness_by_run_all_coverages.csv",
    index=False,
)
pooled.to_csv(
    OUTPUT_DIR / "meta_mode_extremeness_pooled_all_coverages.csv",
    index=False,
)
shared.to_csv(
    OUTPUT_DIR / "shared_meta_mode_ranges_all_coverages.csv",
    index=False,
)
s17.to_csv(
    OUTPUT_DIR / "Supplementary_Table_S17_label_blind_extremeness_by_aligned_meta_mode_all_coverages.csv",
    index=False,
)
s17_compact_q99.to_csv(
    OUTPUT_DIR / "Supplementary_Table_S17_compact_q99.csv",
    index=False,
)
m0_audit.to_csv(
    OUTPUT_DIR / "m0_fallback_audit_all_coverages.csv",
    index=False,
)
undefined_audit.to_csv(
    OUTPUT_DIR / "undefined_meta_mode_audit.csv",
    index=False,
)
integrity.to_csv(
    OUTPUT_DIR / "integrity_checks.csv",
    index=False,
)

pd.DataFrame({
    "parameter": [
        "analysis",
        "label_usage",
        "coverages",
        "primary_coverage",
        "layers",
        "shared_meta_modes",
        "aligned_representations",
        "m0_policy",
        "comparison_contract",
        "cell16_dir",
        "cell20_dir",
        "output_dir",
        "total_test_ticks",
    ],
    "value": [
        "label-blind GLOBAL-vs-aligned-meta extremeness by shared meta-mode",
        "none",
        ",".join(str(c) for c in COVERAGES),
        PRIMARY_COVERAGE,
        ",".join(str(k) for k in LAYERS),
        "M1..M7",
        len(DESIGNS) * len(DMIN_VALUES),
        "M0 excluded from substantive M1..M7 summaries; audited separately",
        "GLOBAL and META K_t compared on identical timestamps within each design x dmin aligned representation",
        str(SERF_TEST_DIR),
        str(EXTREMENESS_DIR),
        str(OUTPUT_DIR),
        EXPECTED_TOTAL_TEST_ROWS,
    ],
}).to_csv(OUTPUT_DIR / "settings.csv", index=False)


# ============================================================
# 10. CONCISE NOTEBOOK OUTPUT
# ============================================================

print("=" * 78)
print("LABEL-BLIND ALIGNED-META-MODE ANALYSIS COMPLETE")
print("=" * 78)
print(f"held-out SERF post-processing source: {SERF_TEST_DIR}")
print(f"label-blind extremeness scoring source: {EXTREMENESS_DIR}")
print(f"Output:         {OUTPUT_DIR}")
print()
print(f"By-run rows:  {len(by_run):,}")
print(f"Pooled rows:  {len(pooled):,}")
print(f"Shared rows:  {len(shared):,}")
print(f"M0 mismatches GLOBAL vs META: {int(m0_audit['n_GLOBAL_vs_META_mismatches_on_m0'].sum()):,}")
print()
print("Primary q99 compact M1..M7 summary:")
print(s17_compact_q99.to_string(index=False))


LABEL-BLIND ALIGNED-META-MODE ANALYSIS COMPLETE — LABEL-BLIND EXTREMENESS BY ALIGNED META-MODE
held-out SERF post-processing source: outputs/section_5_4/test_serf_postprocessing
label-blind extremeness scoring source: outputs/section_5_6/test_extreme_channel_counts
Output:         outputs/section_5_6/extremeness_by_meta_mode

By-run rows:  2,310
Pooled rows:  462
Shared rows:  21
M0 mismatches GLOBAL vs META: 0

Primary q99 compact M1..M7 summary:
meta_mode  active_aligned_representations support_pct_range GLOBAL_mean_Kt_range META_mean_Kt_range delta_mean_Kt_range GLOBAL_Kt_ge_5_pct_range META_Kt_ge_5_pct_range delta_Kt_ge_5_pp_range GLOBAL_Kt_ge_10_pct_range META_Kt_ge_10_pct_range delta_Kt_ge_10_pp_range
       M1                              22      35.62–66.00%            0.75–1.07          0.83–2.29           0.08–1.21               1.70–2.63%            2.27–16.24%          0.57–13.61 pp                0.22–0.35%              0.37–7.66%            0.15–7.33 pp
       M2         

## Section 5.7 — External functional validation

**Attack annotations enter the analysis for the first time in this section.** The previously fixed `K_t` sequences are paired with the external labels only for validation. No preprocessing, fitting, state assignment, stabilization, meta-mode mapping, reference estimation, threshold selection, or configuration selection is performed from the labels.


### 5.7.1 Timestamp, layer, transition, and event correspondence


In [10]:
# layered external evaluation — Attack-linked layered evaluation of extreme-channel counts
# HAI 21.03 / SERF
#
# Purpose
# -------
# This is the FIRST cell that uses attack annotations.
#
# Inputs:
#   - raw test1.csv ... test5.csv:
#       time, attack, attack_P1, attack_P2, attack_P3
#   - label-blind extremeness scoring label-blind extreme-channel counts:
#       singleton_extreme_count
#       GLOBAL
#       + 55 SERF variants
#   - held-out response/context contract context-column contract
#
# Evaluation is performed SEPARATELY for:
#   q99 (primary), q98, q95
#
# Core score:
#   K_t = number of extreme response channels among all 75 channels
#
# For each GLOBAL/SERF variant, evaluate ALL layers:
#   K_t >= m,  m = 1,...,75
#
# Outputs cover:
#   1. NORMAL vs ATTACK confusion counts and rates at every K threshold
#   2. exact attack scopes / combinations:
#        P1, P2, P3, P1+P2, P1+P3, P2+P3, P1+P2+P3, OTHER
#   3. paired GLOBAL vs CONTEXT transitions:
#        both
#        context-revealed  = GLOBAL < m and CONTEXT >= m
#        context-suppressed = GLOBAL >= m and CONTEXT < m
#        neither
#   4. stage-to-stage changes:
#        native -> stabilized dmin=5
#        native -> stabilized dmin=10
#        stabilized dmin=5 -> meta q7/dmin=5
#        stabilized dmin=10 -> meta q7/dmin=10
#   5. the same stage transitions specifically for CONTEXT-REVEALED ticks
#      relative to GLOBAL
#   6. per-run and pooled evaluation
#   7. attack-event-aware evaluation
#   8. threshold-free ROC-AUC and average precision using K_t
#   9. singleton-channel contribution audit
#
# IMPORTANT
# ---------
# No threshold/design is selected here.
# No attack information is used to alter K_t or any reference interval.
# Channel-level attribution and uncertainty/bootstrap are intentionally left
# for later cells after the primary layered results are inspected.


from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS AND FIXED CONTRACT
# ============================================================

RESPONSE_CONTEXT_DIR = Path("outputs/section_5_6/test_response_context_contract")
EXTREMENESS_DIR = Path("outputs/section_5_6/test_extreme_channel_counts")

CONTEXT_CONTRACT_FILE = RESPONSE_CONTEXT_DIR / "context_column_contract.csv"

OUTPUT_DIR = Path("outputs/section_5_7/attack_linked_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_RUNS = ["test1", "test2", "test3", "test4", "test5"]

EXPECTED_TEST_ROWS = {
    "test1": 43_201,
    "test2": 118_801,
    "test3": 108_001,
    "test4": 39_601,
    "test5": 92_401,
}

EXPECTED_TOTAL_TEST_ROWS = 402_005

ATTACK_COL = "attack"
ATTACK_SCOPE_COLS = ["attack_P1", "attack_P2", "attack_P3"]
TIME_COL = "time"

# Primary first so singleton audit can be stored once from q99.
COVERAGES = [0.99, 0.98, 0.95]
PRIMARY_COVERAGE = 0.99

MAX_K = 75
THRESHOLDS = np.arange(1, MAX_K + 1, dtype=int)

EXPECTED_CONTEXT_VARIANTS = 55
EXPECTED_TOTAL_VARIANTS = 56  # GLOBAL + 55 SERF

EXACT_ATTACK_SCOPES = [
    "P1",
    "P2",
    "P3",
    "P1+P2",
    "P1+P3",
    "P2+P3",
    "P1+P2+P3",
    "OTHER",
]


# ============================================================
# 2. HELPERS
# ============================================================

def coverage_tag(coverage: float) -> str:
    return str(int(round(coverage * 100)))


def resolve_test_file(test_run: str) -> Path:
    """Return testX.csv from the notebook working directory."""
    path = Path(f"{test_run}.csv")
    if not path.exists():
        raise FileNotFoundError(f"{path.name} must be placed in the notebook working directory.")
    return path


def parse_binary(series: pd.Series, name: str) -> pd.Series:
    """
    Robustly convert 0/1, bool, or common textual booleans to integer 0/1.
    """
    if pd.api.types.is_bool_dtype(series):
        out = series.astype(int)
    else:
        numeric = pd.to_numeric(series, errors="coerce")

        if numeric.notna().all():
            out = numeric.astype(int)
        else:
            mapping = {
                "0": 0,
                "1": 1,
                "false": 0,
                "true": 1,
                "no": 0,
                "yes": 1,
            }
            out = (
                series.astype(str)
                .str.strip()
                .str.lower()
                .map(mapping)
            )

    if out.isna().any():
        raise ValueError(f"{name}: missing or unparseable binary labels.")

    unique = set(pd.unique(out))
    if not unique.issubset({0, 1}):
        raise ValueError(
            f"{name}: expected binary 0/1, found {sorted(unique)}."
        )

    return out.astype(np.uint8)


def time_equal(a: pd.Series, b: pd.Series) -> bool:
    adt = pd.to_datetime(a, errors="coerce")
    bdt = pd.to_datetime(b, errors="coerce")

    if adt.notna().all() and bdt.notna().all():
        return adt.reset_index(drop=True).equals(
            bdt.reset_index(drop=True)
        )

    return (
        a.astype(str).reset_index(drop=True)
        .equals(b.astype(str).reset_index(drop=True))
    )


def attack_scope_from_flags(
    attack: np.ndarray,
    p1: np.ndarray,
    p2: np.ndarray,
    p3: np.ndarray,
) -> np.ndarray:
    """
    Exact timestamp-level attack scope.
    """
    scope = np.full(len(attack), "NORMAL", dtype=object)

    active = attack == 1

    codes = (
        p1.astype(int) * 4
        + p2.astype(int) * 2
        + p3.astype(int)
    )

    mapping = {
        0: "OTHER",
        1: "P3",
        2: "P2",
        3: "P2+P3",
        4: "P1",
        5: "P1+P3",
        6: "P1+P2",
        7: "P1+P2+P3",
    }

    for code, label in mapping.items():
        scope[active & (codes == code)] = label

    return scope


def build_attack_events(
    test_run: str,
    time: pd.Series,
    attack: np.ndarray,
    p1: np.ndarray,
    p2: np.ndarray,
    p3: np.ndarray,
):
    """
    Create contiguous attack events independently within one test run.
    NORMAL ticks receive no event id.

    Event scope = union of P1/P2/P3 flags observed anywhere in the event.
    If no subsystem flag is active, scope = OTHER.
    """
    n = len(attack)

    starts = np.zeros(n, dtype=bool)
    starts[0] = attack[0] == 1

    if n > 1:
        starts[1:] = (
            (attack[1:] == 1)
            & (attack[:-1] == 0)
        )

    seq = np.cumsum(starts).astype(int)
    seq = np.where(attack == 1, seq, 0)

    event_id = np.full(n, None, dtype=object)

    event_rows = []

    for event_seq in sorted(np.unique(seq[seq > 0])):
        mask = seq == event_seq
        idx = np.flatnonzero(mask)

        eid = f"{test_run}_E{event_seq:03d}"
        event_id[mask] = eid

        has_p1 = bool(p1[mask].any())
        has_p2 = bool(p2[mask].any())
        has_p3 = bool(p3[mask].any())

        parts = []
        if has_p1:
            parts.append("P1")
        if has_p2:
            parts.append("P2")
        if has_p3:
            parts.append("P3")

        event_scope = "+".join(parts) if parts else "OTHER"

        event_rows.append({
            "test_run": test_run,
            "event_id": eid,
            "event_sequence": int(event_seq),
            "start_row_id": int(idx[0]),
            "end_row_id": int(idx[-1]),
            "n_attack_ticks": int(mask.sum()),
            "start_time": time.iloc[idx[0]],
            "end_time": time.iloc[idx[-1]],
            "event_scope": event_scope,
        })

    return event_id, pd.DataFrame(event_rows)


def count_hist(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=int)

    if len(values) == 0:
        return np.zeros(MAX_K + 1, dtype=np.int64)

    if values.min() < 0 or values.max() > MAX_K:
        raise ValueError(
            f"K outside 0..{MAX_K}: min={values.min()}, max={values.max()}."
        )

    return np.bincount(
        values,
        minlength=MAX_K + 1,
    ).astype(np.int64)


def ge_counts(hist: np.ndarray) -> np.ndarray:
    """
    ge[m] = number of observations with K >= m, for m=0..75.
    """
    return hist[::-1].cumsum()[::-1]


def safe_div(num, den):
    if den == 0:
        return np.nan
    return float(num) / float(den)


def discrete_quantile_from_hist(hist: np.ndarray, q: float):
    n = int(hist.sum())
    if n == 0:
        return np.nan

    target = q * (n - 1)
    index = int(np.ceil(target))
    cum = np.cumsum(hist)

    return int(np.searchsorted(cum, index + 1, side="left"))


def distribution_from_hist(hist: np.ndarray):
    n = int(hist.sum())

    if n == 0:
        return {
            "n_ticks": 0,
            "mean_k": np.nan,
            "median_k": np.nan,
            "p90_k": np.nan,
            "p95_k": np.nan,
            "p99_k": np.nan,
            "max_k": np.nan,
            "share_k_ge_1": np.nan,
        }

    k = np.arange(MAX_K + 1, dtype=float)

    return {
        "n_ticks": n,
        "mean_k": float((k * hist).sum() / n),
        "median_k": discrete_quantile_from_hist(hist, 0.50),
        "p90_k": discrete_quantile_from_hist(hist, 0.90),
        "p95_k": discrete_quantile_from_hist(hist, 0.95),
        "p99_k": discrete_quantile_from_hist(hist, 0.99),
        "max_k": int(np.flatnonzero(hist)[-1]),
        "share_k_ge_1": safe_div(hist[1:].sum(), n),
    }


def auc_ap_from_hists(
    attack_hist: np.ndarray,
    normal_hist: np.ndarray,
):
    """
    Exact ROC-AUC and average precision for discrete score K using only
    class-conditional histograms. Higher K = more extreme.
    """
    n_pos = int(attack_hist.sum())
    n_neg = int(normal_hist.sum())

    if n_pos == 0 or n_neg == 0:
        auc = np.nan
    else:
        neg_below = np.cumsum(normal_hist) - normal_hist

        concordance = np.sum(
            attack_hist
            * (
                neg_below
                + 0.5 * normal_hist
            )
        )

        auc = float(concordance / (n_pos * n_neg))

    if n_pos == 0:
        ap = np.nan
    else:
        tp = 0
        fp = 0
        ap = 0.0

        for k in range(MAX_K, -1, -1):
            pos_at_k = int(attack_hist[k])
            neg_at_k = int(normal_hist[k])

            tp += pos_at_k
            fp += neg_at_k

            if pos_at_k > 0:
                precision = tp / (tp + fp)
                recall_increment = pos_at_k / n_pos
                ap += precision * recall_increment

        ap = float(ap)

    return auc, ap


def layer_rows_from_hists(
    normal_hist: np.ndarray,
    attack_hist: np.ndarray,
    base: dict,
):
    normal_ge = ge_counts(normal_hist)
    attack_ge = ge_counts(attack_hist)

    n_normal = int(normal_hist.sum())
    n_attack = int(attack_hist.sum())

    rows = []

    for m in THRESHOLDS:
        fp = int(normal_ge[m])
        tp = int(attack_ge[m])
        tn = n_normal - fp
        fn = n_attack - tp

        tpr = safe_div(tp, n_attack)
        fpr = safe_div(fp, n_normal)
        precision = safe_div(tp, tp + fp)
        specificity = safe_div(tn, n_normal)

        if np.isfinite(tpr) and np.isfinite(specificity):
            balanced_accuracy = (tpr + specificity) / 2.0
        else:
            balanced_accuracy = np.nan

        rows.append({
            **base,
            "threshold_k": int(m),
            "n_normal": n_normal,
            "n_attack": n_attack,
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
            "normal_extreme_rate": fpr,
            "attack_extreme_rate": tpr,
            "precision": precision,
            "specificity": specificity,
            "balanced_accuracy": balanced_accuracy,
            "attack_minus_normal_rate": (
                tpr - fpr
                if np.isfinite(tpr) and np.isfinite(fpr)
                else np.nan
            ),
        })

    return rows


def attack_scope_layer_rows(
    hist: np.ndarray,
    scope: str,
    base: dict,
):
    ge = ge_counts(hist)
    n_attack = int(hist.sum())

    rows = []

    for m in THRESHOLDS:
        detected = int(ge[m])
        missed = n_attack - detected

        rows.append({
            **base,
            "attack_scope": scope,
            "threshold_k": int(m),
            "n_attack_ticks": n_attack,
            "detected_attack_ticks": detected,
            "missed_attack_ticks": int(missed),
            "attack_coverage": safe_div(detected, n_attack),
        })

    return rows


def joint_threshold_transition_rows(
    source: np.ndarray,
    target: np.ndarray,
    mask: np.ndarray,
    base: dict,
):
    """
    Compare source>=m and target>=m at all m using a 76x76 joint histogram.

    Categories:
      both
      source_only
      target_only
      neither
    """
    s = np.asarray(source[mask], dtype=int)
    t = np.asarray(target[mask], dtype=int)

    n = len(s)

    if n == 0:
        return []

    flat = np.bincount(
        s * (MAX_K + 1) + t,
        minlength=(MAX_K + 1) ** 2,
    )

    joint = flat.reshape(
        MAX_K + 1,
        MAX_K + 1,
    )

    source_hist = joint.sum(axis=1)
    target_hist = joint.sum(axis=0)

    source_ge = ge_counts(source_hist)
    target_ge = ge_counts(target_hist)

    suffix = (
        joint[::-1, ::-1]
        .cumsum(axis=0)
        .cumsum(axis=1)
        [::-1, ::-1]
    )

    rows = []

    for m in THRESHOLDS:
        both = int(suffix[m, m])
        source_only = int(source_ge[m] - both)
        target_only = int(target_ge[m] - both)
        neither = int(n - both - source_only - target_only)

        source_positive = both + source_only
        target_positive = both + target_only

        rows.append({
            **base,
            "threshold_k": int(m),
            "n_group": int(n),
            "both": both,
            "source_only": source_only,
            "target_only": target_only,
            "neither": neither,
            "source_positive": source_positive,
            "target_positive": target_positive,
            "retained_share_of_source": safe_div(
                both,
                source_positive,
            ),
            "removed_share_of_source": safe_div(
                source_only,
                source_positive,
            ),
            "introduced_share_of_group": safe_div(
                target_only,
                n,
            ),
        })

    return rows


def revealed_transition_rows(
    global_k: np.ndarray,
    source_k: np.ndarray,
    target_k: np.ndarray,
    mask: np.ndarray,
    base: dict,
):
    """
    Stage transition of CONTEXT-REVEALED status.

    At threshold m:
      source revealed iff GLOBAL < m and SOURCE >= m
      target revealed iff GLOBAL < m and TARGET >= m

    Efficient interval accumulation avoids constructing n x 75 matrices.
    """
    g = np.asarray(global_k[mask], dtype=int)
    s = np.asarray(source_k[mask], dtype=int)
    t = np.asarray(target_k[mask], dtype=int)

    n = len(g)

    if n == 0:
        return []

    def interval_counts(upper):
        # Indicator is true for thresholds m with g < m <= upper.
        diff = np.zeros(MAX_K + 2, dtype=np.int64)
        valid = upper > g

        if valid.any():
            starts = g[valid] + 1
            ends_plus_one = upper[valid] + 1

            np.add.at(diff, starts, 1)
            np.add.at(diff, ends_plus_one, -1)

        return np.cumsum(diff)[:MAX_K + 1]

    source_revealed = interval_counts(s)
    target_revealed = interval_counts(t)
    both_revealed = interval_counts(np.minimum(s, t))

    rows = []

    for m in THRESHOLDS:
        both = int(both_revealed[m])
        source_only = int(source_revealed[m] - both)
        target_only = int(target_revealed[m] - both)
        neither = int(n - both - source_only - target_only)

        source_total = both + source_only
        target_total = both + target_only

        rows.append({
            **base,
            "threshold_k": int(m),
            "n_group": int(n),
            "revealed_both": both,
            "revealed_source_only": source_only,
            "revealed_target_only": target_only,
            "revealed_neither": neither,
            "source_revealed_total": source_total,
            "target_revealed_total": target_total,
            "retained_revealed_share": safe_div(
                both,
                source_total,
            ),
            "removed_revealed_share": safe_div(
                source_only,
                source_total,
            ),
            "new_revealed_share_of_group": safe_div(
                target_only,
                n,
            ),
        })

    return rows


def add_pooled_confusion_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add POOLED rows to standard NORMAL-vs-ATTACK layer metrics.
    """
    group_cols = [
        "coverage",
        "is_primary",
        "variant",
        "design",
        "family",
        "representation",
        "dmin",
        "meta_q",
        "threshold_k",
    ]

    sums = (
        df.groupby(
            group_cols,
            dropna=False,
            as_index=False,
        )[
            ["n_normal", "n_attack", "tn", "fp", "fn", "tp"]
        ]
        .sum()
    )

    sums["test_run"] = "POOLED"

    sums["normal_extreme_rate"] = (
        sums["fp"] / sums["n_normal"].replace(0, np.nan)
    )
    sums["attack_extreme_rate"] = (
        sums["tp"] / sums["n_attack"].replace(0, np.nan)
    )
    sums["precision"] = (
        sums["tp"]
        / (sums["tp"] + sums["fp"]).replace(0, np.nan)
    )
    sums["specificity"] = (
        sums["tn"] / sums["n_normal"].replace(0, np.nan)
    )
    sums["balanced_accuracy"] = (
        sums["attack_extreme_rate"]
        + sums["specificity"]
    ) / 2.0
    sums["attack_minus_normal_rate"] = (
        sums["attack_extreme_rate"]
        - sums["normal_extreme_rate"]
    )

    cols = df.columns.tolist()
    return pd.concat(
        [df, sums[cols]],
        ignore_index=True,
    )


def add_pooled_scope_rows(df: pd.DataFrame) -> pd.DataFrame:
    group_cols = [
        "coverage",
        "is_primary",
        "variant",
        "design",
        "family",
        "representation",
        "dmin",
        "meta_q",
        "attack_scope",
        "threshold_k",
    ]

    sums = (
        df.groupby(
            group_cols,
            dropna=False,
            as_index=False,
        )[
            [
                "n_attack_ticks",
                "detected_attack_ticks",
                "missed_attack_ticks",
            ]
        ]
        .sum()
    )

    sums["test_run"] = "POOLED"
    sums["attack_coverage"] = (
        sums["detected_attack_ticks"]
        / sums["n_attack_ticks"].replace(0, np.nan)
    )

    cols = df.columns.tolist()

    return pd.concat(
        [df, sums[cols]],
        ignore_index=True,
    )


def add_pooled_transition_rows(
    df: pd.DataFrame,
    category_cols,
    source_total_col,
    target_total_col,
    retained_col,
    removed_col,
    introduced_col,
):
    """
    Add POOLED rows for transition tables by summing category counts.
    """
    excluded = {
        "test_run",
        "n_group",
        *category_cols,
        source_total_col,
        target_total_col,
        retained_col,
        removed_col,
        introduced_col,
    }

    group_cols = [
        c for c in df.columns
        if c not in excluded
        and not c.endswith("_share")
        and c not in {
            "retained_share_of_source",
            "removed_share_of_source",
            "introduced_share_of_group",
            "retained_revealed_share",
            "removed_revealed_share",
            "new_revealed_share_of_group",
        }
    ]

    sums = (
        df.groupby(
            group_cols,
            dropna=False,
            as_index=False,
        )[
            ["n_group", *category_cols]
        ]
        .sum()
    )

    sums["test_run"] = "POOLED"

    # Reconstruct totals from categories.
    if category_cols == [
        "both",
        "source_only",
        "target_only",
        "neither",
    ]:
        sums[source_total_col] = (
            sums["both"] + sums["source_only"]
        )
        sums[target_total_col] = (
            sums["both"] + sums["target_only"]
        )
        sums[retained_col] = (
            sums["both"]
            / sums[source_total_col].replace(0, np.nan)
        )
        sums[removed_col] = (
            sums["source_only"]
            / sums[source_total_col].replace(0, np.nan)
        )
        sums[introduced_col] = (
            sums["target_only"]
            / sums["n_group"].replace(0, np.nan)
        )

    else:
        sums[source_total_col] = (
            sums["revealed_both"]
            + sums["revealed_source_only"]
        )
        sums[target_total_col] = (
            sums["revealed_both"]
            + sums["revealed_target_only"]
        )
        sums[retained_col] = (
            sums["revealed_both"]
            / sums[source_total_col].replace(0, np.nan)
        )
        sums[removed_col] = (
            sums["revealed_source_only"]
            / sums[source_total_col].replace(0, np.nan)
        )
        sums[introduced_col] = (
            sums["revealed_target_only"]
            / sums["n_group"].replace(0, np.nan)
        )

    cols = df.columns.tolist()

    return pd.concat(
        [df, sums[cols]],
        ignore_index=True,
    )


# ============================================================
# 3. LOAD VARIANT CONTRACT
# ============================================================

if not CONTEXT_CONTRACT_FILE.exists():
    raise FileNotFoundError(CONTEXT_CONTRACT_FILE)

context_contract = pd.read_csv(CONTEXT_CONTRACT_FILE)

required_context_cols = {
    "context_column",
    "design",
    "family",
    "representation",
    "dmin",
    "q",
}

if not required_context_cols.issubset(context_contract.columns):
    missing = sorted(
        required_context_cols - set(context_contract.columns)
    )
    raise ValueError(
        f"{CONTEXT_CONTRACT_FILE}: missing columns {missing}"
    )

if len(context_contract) != EXPECTED_CONTEXT_VARIANTS:
    raise RuntimeError(
        f"Expected {EXPECTED_CONTEXT_VARIANTS} SERF variants, "
        f"found {len(context_contract)}."
    )

context_contract = context_contract.copy()
context_contract = context_contract.rename(
    columns={
        "context_column": "variant",
        "q": "meta_q",
    }
)

global_contract = pd.DataFrame([
    {
        "variant": "GLOBAL",
        "design": "GLOBAL",
        "family": "global",
        "representation": "global",
        "dmin": np.nan,
        "meta_q": np.nan,
    }
])

variant_contract = pd.concat(
    [
        global_contract,
        context_contract[
            [
                "variant",
                "design",
                "family",
                "representation",
                "dmin",
                "meta_q",
            ]
        ],
    ],
    ignore_index=True,
)

if len(variant_contract) != EXPECTED_TOTAL_VARIANTS:
    raise RuntimeError(
        f"Expected {EXPECTED_TOTAL_VARIANTS} total variants, "
        f"found {len(variant_contract)}."
    )

if variant_contract["variant"].duplicated().any():
    raise RuntimeError("Duplicated variant names.")

VARIANTS = variant_contract["variant"].astype(str).tolist()
SERF_VARIANTS = [
    v for v in VARIANTS
    if v != "GLOBAL"
]

variant_contract.to_csv(
    OUTPUT_DIR / "variant_contract.csv",
    index=False,
)

variant_meta = (
    variant_contract
    .set_index("variant")
    .to_dict(orient="index")
)


# ============================================================
# 4. STAGE-COMPARISON CONTRACT
# ============================================================

DESIGNS = context_contract["design"].drop_duplicates().tolist()

stage_pair_rows = []

for design in DESIGNS:
    stage_pair_rows.extend([
        {
            "design": design,
            "comparison": "native_to_stabilized_dmin5",
            "source_variant": f"{design}__native",
            "target_variant": f"{design}__stabilized_dmin5",
        },
        {
            "design": design,
            "comparison": "native_to_stabilized_dmin10",
            "source_variant": f"{design}__native",
            "target_variant": f"{design}__stabilized_dmin10",
        },
        {
            "design": design,
            "comparison": "stabilized_to_meta_dmin5",
            "source_variant": f"{design}__stabilized_dmin5",
            "target_variant": f"{design}__meta_q7_dmin5",
        },
        {
            "design": design,
            "comparison": "stabilized_to_meta_dmin10",
            "source_variant": f"{design}__stabilized_dmin10",
            "target_variant": f"{design}__meta_q7_dmin10",
        },
    ])

stage_pair_contract = pd.DataFrame(stage_pair_rows)

missing_stage_variants = sorted(
    (
        set(stage_pair_contract["source_variant"])
        | set(stage_pair_contract["target_variant"])
    )
    - set(SERF_VARIANTS)
)

if missing_stage_variants:
    raise RuntimeError(
        f"Stage contract refers to missing variants: "
        f"{missing_stage_variants}"
    )

stage_pair_contract.to_csv(
    OUTPUT_DIR / "stage_comparison_contract.csv",
    index=False,
)


# ============================================================
# 5. READ ATTACK LABELS ONCE AND BUILD EVENT CONTRACT
# ============================================================

label_frames = {}
attack_label_summary_rows = []
attack_scope_summary_rows = []
event_contract_parts = []
integrity_rows = []

for test_run in TEST_RUNS:
    test_file = resolve_test_file(test_run)

    header = pd.read_csv(test_file, nrows=0)
    required = [
        TIME_COL,
        ATTACK_COL,
        *ATTACK_SCOPE_COLS,
    ]

    missing = [
        c for c in required
        if c not in header.columns
    ]

    if missing:
        raise ValueError(
            f"{test_file}: missing attack-label columns {missing}"
        )

    labels = pd.read_csv(
        test_file,
        usecols=required,
    )

    expected_rows = EXPECTED_TEST_ROWS[test_run]

    if len(labels) != expected_rows:
        raise RuntimeError(
            f"{test_run}: raw label rows {len(labels):,} "
            f"!= expected {expected_rows:,}."
        )

    attack = parse_binary(
        labels[ATTACK_COL],
        f"{test_run}/{ATTACK_COL}",
    ).to_numpy()

    p1 = parse_binary(
        labels["attack_P1"],
        f"{test_run}/attack_P1",
    ).to_numpy()

    p2 = parse_binary(
        labels["attack_P2"],
        f"{test_run}/attack_P2",
    ).to_numpy()

    p3 = parse_binary(
        labels["attack_P3"],
        f"{test_run}/attack_P3",
    ).to_numpy()

    subsystem_positive_while_normal = (
        (attack == 0)
        & ((p1 == 1) | (p2 == 1) | (p3 == 1))
    )

    if subsystem_positive_while_normal.any():
        bad_n = int(subsystem_positive_while_normal.sum())
        raise RuntimeError(
            f"{test_run}: {bad_n} rows have attack=0 but "
            "a subsystem attack flag is positive."
        )

    scope = attack_scope_from_flags(
        attack,
        p1,
        p2,
        p3,
    )

    event_id, event_contract = build_attack_events(
        test_run=test_run,
        time=labels[TIME_COL],
        attack=attack,
        p1=p1,
        p2=p2,
        p3=p3,
    )

    event_contract_parts.append(event_contract)

    linked = pd.DataFrame({
        "row_id": np.arange(len(labels), dtype=int),
        "time": labels[TIME_COL].to_numpy(),
        "attack": attack,
        "attack_P1": p1,
        "attack_P2": p2,
        "attack_P3": p3,
        "attack_scope": scope,
        "attack_event_id": event_id,
    })

    label_frames[test_run] = linked

    n_attack = int(attack.sum())
    n_normal = int(len(attack) - n_attack)

    attack_label_summary_rows.append({
        "test_run": test_run,
        "n_ticks": len(attack),
        "n_normal_ticks": n_normal,
        "n_attack_ticks": n_attack,
        "attack_share": safe_div(n_attack, len(attack)),
        "n_contiguous_attack_events": len(event_contract),
    })

    for attack_scope in EXACT_ATTACK_SCOPES:
        n_scope = int(
            ((attack == 1) & (scope == attack_scope)).sum()
        )

        attack_scope_summary_rows.append({
            "test_run": test_run,
            "attack_scope": attack_scope,
            "n_attack_ticks": n_scope,
            "share_of_attack_ticks": safe_div(
                n_scope,
                n_attack,
            ),
        })

    integrity_rows.extend([
        {
            "coverage": np.nan,
            "test_run": test_run,
            "check": "raw_attack_label_row_count_matches_contract",
            "status": "PASS",
        },
        {
            "coverage": np.nan,
            "test_run": test_run,
            "check": "subsystem_attack_flags_not_positive_on_normal_ticks",
            "status": "PASS",
        },
    ])


attack_label_summary = pd.DataFrame(
    attack_label_summary_rows
)

attack_scope_summary = pd.DataFrame(
    attack_scope_summary_rows
)

attack_event_contract = pd.concat(
    event_contract_parts,
    ignore_index=True,
) if event_contract_parts else pd.DataFrame()

# Pooled attack-label summaries.
attack_label_summary = pd.concat(
    [
        attack_label_summary,
        pd.DataFrame([{
            "test_run": "POOLED",
            "n_ticks": int(
                attack_label_summary["n_ticks"].sum()
            ),
            "n_normal_ticks": int(
                attack_label_summary["n_normal_ticks"].sum()
            ),
            "n_attack_ticks": int(
                attack_label_summary["n_attack_ticks"].sum()
            ),
            "attack_share": safe_div(
                attack_label_summary["n_attack_ticks"].sum(),
                attack_label_summary["n_ticks"].sum(),
            ),
            "n_contiguous_attack_events": int(
                attack_label_summary[
                    "n_contiguous_attack_events"
                ].sum()
            ),
        }]),
    ],
    ignore_index=True,
)

pooled_scope = (
    attack_scope_summary
    .groupby(
        "attack_scope",
        as_index=False,
    )["n_attack_ticks"]
    .sum()
)

pooled_scope["test_run"] = "POOLED"
pooled_total_attack = int(
    pooled_scope["n_attack_ticks"].sum()
)

pooled_scope["share_of_attack_ticks"] = (
    pooled_scope["n_attack_ticks"]
    / pooled_total_attack
    if pooled_total_attack > 0
    else np.nan
)

attack_scope_summary = pd.concat(
    [
        attack_scope_summary,
        pooled_scope[
            attack_scope_summary.columns
        ],
    ],
    ignore_index=True,
)

attack_label_summary.to_csv(
    OUTPUT_DIR / "attack_label_summary.csv",
    index=False,
)

attack_scope_summary.to_csv(
    OUTPUT_DIR / "attack_scope_tick_summary.csv",
    index=False,
)

attack_event_contract.to_csv(
    OUTPUT_DIR / "attack_event_contract.csv",
    index=False,
)


# ============================================================
# 6. MAIN EVALUATION — ONE COVERAGE AT A TIME
# ============================================================

singleton_reference_by_run = {}
singleton_audit_rows = []

for coverage in COVERAGES:
    tag = coverage_tag(coverage)
    qdir = OUTPUT_DIR / f"q{tag}"
    qdir.mkdir(parents=True, exist_ok=True)

    layer_rows = []
    scope_layer_rows = []
    distribution_rows = []
    discrimination_rows = []
    global_context_rows = []
    stage_extreme_rows = []
    stage_revealed_rows = []
    event_hist_rows = []

    # Needed for exact pooled AUC/AP and pooled distribution summaries.
    pooled_normal_hists = {
        variant: np.zeros(MAX_K + 1, dtype=np.int64)
        for variant in VARIANTS
    }
    pooled_attack_hists = {
        variant: np.zeros(MAX_K + 1, dtype=np.int64)
        for variant in VARIANTS
    }

    for test_run in TEST_RUNS:
        labels = label_frames[test_run]

        count_file = (
            EXTREMENESS_DIR
            / test_run
            / f"extreme_channel_counts_q{tag}.csv"
        )

        if not count_file.exists():
            raise FileNotFoundError(count_file)

        counts = pd.read_csv(count_file)

        required_count_cols = {
            "row_id",
            "time",
            "singleton_extreme_count",
            *VARIANTS,
        }

        if not required_count_cols.issubset(counts.columns):
            missing = sorted(
                required_count_cols - set(counts.columns)
            )
            raise ValueError(
                f"{count_file}: missing columns {missing}"
            )

        expected_rows = EXPECTED_TEST_ROWS[test_run]

        if len(counts) != expected_rows:
            raise RuntimeError(
                f"{test_run} q{tag}: count rows {len(counts):,} "
                f"!= expected {expected_rows:,}."
            )

        row_id = pd.to_numeric(
            counts["row_id"],
            errors="raise",
        ).astype(int).to_numpy()

        if not np.array_equal(
            row_id,
            labels["row_id"].to_numpy(dtype=int),
        ):
            raise RuntimeError(
                f"{test_run} q{tag}: row_id alignment failed."
            )

        if not time_equal(
            counts["time"],
            labels["time"],
        ):
            raise RuntimeError(
                f"{test_run} q{tag}: time alignment failed."
            )

        # Validate K counts.
        for col in ["singleton_extreme_count", *VARIANTS]:
            numeric = pd.to_numeric(
                counts[col],
                errors="raise",
            )

            if not np.array_equal(
                numeric.to_numpy(dtype=float),
                np.round(numeric.to_numpy(dtype=float)),
            ):
                raise RuntimeError(
                    f"{test_run} q{tag}/{col}: non-integer count."
                )

            if col == "singleton_extreme_count":
                if (numeric < 0).any() or (numeric > 22).any():
                    raise RuntimeError(
                        f"{test_run} q{tag}: singleton count outside 0..22."
                    )
            else:
                if (numeric < 0).any() or (numeric > MAX_K).any():
                    raise RuntimeError(
                        f"{test_run} q{tag}/{col}: K outside 0..75."
                    )

            counts[col] = numeric.astype(np.uint8)

        # Singleton count MUST be identical at q99/q98/q95.
        singleton_now = counts[
            "singleton_extreme_count"
        ].to_numpy(dtype=np.uint8)

        if test_run not in singleton_reference_by_run:
            singleton_reference_by_run[test_run] = (
                singleton_now.copy()
            )
        else:
            if not np.array_equal(
                singleton_reference_by_run[test_run],
                singleton_now,
            ):
                raise RuntimeError(
                    f"{test_run}: singleton_extreme_count differs "
                    f"between coverage levels."
                )

        # Save the explicit label-linked tick-level table.
        linked = counts[
            [
                "row_id",
                "time",
                "singleton_extreme_count",
                *VARIANTS,
            ]
        ].copy()

        linked["attack"] = labels[
            "attack"
        ].to_numpy(dtype=np.uint8)

        linked["attack_P1"] = labels[
            "attack_P1"
        ].to_numpy(dtype=np.uint8)

        linked["attack_P2"] = labels[
            "attack_P2"
        ].to_numpy(dtype=np.uint8)

        linked["attack_P3"] = labels[
            "attack_P3"
        ].to_numpy(dtype=np.uint8)

        linked["attack_scope"] = labels[
            "attack_scope"
        ].to_numpy()

        linked["attack_event_id"] = labels[
            "attack_event_id"
        ].to_numpy()

        run_dir = qdir / test_run
        run_dir.mkdir(parents=True, exist_ok=True)

        linked_file = (
            run_dir
            / f"attack_linked_extreme_counts_q{tag}.csv"
        )

        linked.to_csv(
            linked_file,
            index=False,
        )

        attack_mask = (
            labels["attack"].to_numpy(dtype=np.uint8)
            == 1
        )
        normal_mask = ~attack_mask

        # ----------------------------------------------------
        # 6.1 Singleton contribution audit
        # ----------------------------------------------------
        if coverage == PRIMARY_COVERAGE:
            singleton = singleton_now

            for label_group, mask in [
                ("NORMAL", normal_mask),
                ("ATTACK", attack_mask),
            ]:
                vals = singleton[mask]

                singleton_audit_rows.append({
                    "test_run": test_run,
                    "label_group": label_group,
                    "n_ticks": int(len(vals)),
                    "n_ticks_singleton_ge1": int(
                        (vals >= 1).sum()
                    ),
                    "share_ticks_singleton_ge1": safe_div(
                        (vals >= 1).sum(),
                        len(vals),
                    ),
                    "mean_singleton_extreme_count": (
                        float(vals.mean())
                        if len(vals)
                        else np.nan
                    ),
                    "max_singleton_extreme_count": (
                        int(vals.max())
                        if len(vals)
                        else np.nan
                    ),
                })

            for scope in EXACT_ATTACK_SCOPES:
                scope_mask = (
                    attack_mask
                    & (
                        labels["attack_scope"].to_numpy()
                        == scope
                    )
                )

                if not scope_mask.any():
                    continue

                vals = singleton[scope_mask]

                singleton_audit_rows.append({
                    "test_run": test_run,
                    "label_group": scope,
                    "n_ticks": int(len(vals)),
                    "n_ticks_singleton_ge1": int(
                        (vals >= 1).sum()
                    ),
                    "share_ticks_singleton_ge1": safe_div(
                        (vals >= 1).sum(),
                        len(vals),
                    ),
                    "mean_singleton_extreme_count": float(
                        vals.mean()
                    ),
                    "max_singleton_extreme_count": int(
                        vals.max()
                    ),
                })

        # ----------------------------------------------------
        # 6.2 Main layered metrics + distributions + AUC/AP
        # ----------------------------------------------------
        for variant in VARIANTS:
            meta = variant_meta[variant]

            values = counts[
                variant
            ].to_numpy(dtype=np.uint8)

            normal_hist = count_hist(
                values[normal_mask]
            )
            attack_hist = count_hist(
                values[attack_mask]
            )

            pooled_normal_hists[
                variant
            ] += normal_hist
            pooled_attack_hists[
                variant
            ] += attack_hist

            base = {
                "coverage": coverage,
                "is_primary": (
                    coverage == PRIMARY_COVERAGE
                ),
                "test_run": test_run,
                "variant": variant,
                "design": meta["design"],
                "family": meta["family"],
                "representation": meta[
                    "representation"
                ],
                "dmin": meta["dmin"],
                "meta_q": meta["meta_q"],
            }

            layer_rows.extend(
                layer_rows_from_hists(
                    normal_hist,
                    attack_hist,
                    base,
                )
            )

            auc, ap = auc_ap_from_hists(
                attack_hist,
                normal_hist,
            )

            discrimination_rows.append({
                **base,
                "n_normal": int(
                    normal_hist.sum()
                ),
                "n_attack": int(
                    attack_hist.sum()
                ),
                "roc_auc_k": auc,
                "average_precision_k": ap,
            })

            # Distribution summaries.
            for label_group, hist in [
                ("NORMAL", normal_hist),
                ("ALL_ATTACK", attack_hist),
            ]:
                distribution_rows.append({
                    **base,
                    "label_group": label_group,
                    **distribution_from_hist(hist),
                })

            # Exact attack scope / combination.
            for scope in EXACT_ATTACK_SCOPES:
                scope_mask = (
                    attack_mask
                    & (
                        labels[
                            "attack_scope"
                        ].to_numpy()
                        == scope
                    )
                )

                if not scope_mask.any():
                    continue

                scope_hist = count_hist(
                    values[scope_mask]
                )

                scope_layer_rows.extend(
                    attack_scope_layer_rows(
                        scope_hist,
                        scope,
                        base,
                    )
                )

                distribution_rows.append({
                    **base,
                    "label_group": scope,
                    **distribution_from_hist(
                        scope_hist
                    ),
                })

        # ----------------------------------------------------
        # 6.3 Paired GLOBAL vs each SERF context variant
        # ----------------------------------------------------
        global_values = counts[
            "GLOBAL"
        ].to_numpy(dtype=np.uint8)

        for variant in SERF_VARIANTS:
            meta = variant_meta[variant]

            context_values = counts[
                variant
            ].to_numpy(dtype=np.uint8)

            for label_group, mask in [
                ("NORMAL", normal_mask),
                ("ATTACK", attack_mask),
            ]:
                base = {
                    "coverage": coverage,
                    "is_primary": (
                        coverage == PRIMARY_COVERAGE
                    ),
                    "test_run": test_run,
                    "variant": variant,
                    "design": meta["design"],
                    "family": meta["family"],
                    "representation": meta[
                        "representation"
                    ],
                    "dmin": meta["dmin"],
                    "meta_q": meta["meta_q"],
                    "label_group": label_group,
                    "source": "GLOBAL",
                    "target": variant,
                }

                rows = joint_threshold_transition_rows(
                    source=global_values,
                    target=context_values,
                    mask=mask,
                    base=base,
                )

                for row in rows:
                    # Rename semantically for GLOBAL -> CONTEXT.
                    row[
                        "context_suppressed"
                    ] = row["source_only"]

                    row[
                        "context_revealed"
                    ] = row["target_only"]

                    row[
                        "context_revealed_share"
                    ] = safe_div(
                        row["context_revealed"],
                        row["n_group"],
                    )

                    row[
                        "context_suppressed_share"
                    ] = safe_div(
                        row["context_suppressed"],
                        row["n_group"],
                    )

                global_context_rows.extend(rows)

        # ----------------------------------------------------
        # 6.4 Stage-to-stage changes
        # ----------------------------------------------------
        for _, pair in stage_pair_contract.iterrows():
            design = str(pair["design"])
            comparison = str(pair["comparison"])
            source_variant = str(
                pair["source_variant"]
            )
            target_variant = str(
                pair["target_variant"]
            )

            family = variant_meta[
                source_variant
            ]["family"]

            source_values = counts[
                source_variant
            ].to_numpy(dtype=np.uint8)

            target_values = counts[
                target_variant
            ].to_numpy(dtype=np.uint8)

            for label_group, mask in [
                ("NORMAL", normal_mask),
                ("ATTACK", attack_mask),
            ]:
                base = {
                    "coverage": coverage,
                    "is_primary": (
                        coverage == PRIMARY_COVERAGE
                    ),
                    "test_run": test_run,
                    "design": design,
                    "family": family,
                    "comparison": comparison,
                    "source_variant": source_variant,
                    "target_variant": target_variant,
                    "label_group": label_group,
                }

                # A. direct extreme-set transition.
                stage_extreme_rows.extend(
                    joint_threshold_transition_rows(
                        source=source_values,
                        target=target_values,
                        mask=mask,
                        base=base,
                    )
                )

                # B. transition of CONTEXT-REVEALED status
                #    relative to the same GLOBAL score.
                stage_revealed_rows.extend(
                    revealed_transition_rows(
                        global_k=global_values,
                        source_k=source_values,
                        target_k=target_values,
                        mask=mask,
                        base=base,
                    )
                )

        # ----------------------------------------------------
        # 6.5 Event-level K histograms
        # ----------------------------------------------------
        event_ids = labels[
            "attack_event_id"
        ]

        run_events = attack_event_contract[
            attack_event_contract["test_run"]
            == test_run
        ]

        for _, event in run_events.iterrows():
            eid = event["event_id"]

            event_mask = (
                event_ids.to_numpy()
                == eid
            )

            if not event_mask.any():
                raise RuntimeError(
                    f"{test_run}: event {eid} has no linked ticks."
                )

            for variant in VARIANTS:
                meta = variant_meta[variant]

                vals = counts.loc[
                    event_mask,
                    variant,
                ].to_numpy(dtype=np.uint8)

                hist = count_hist(vals)

                row = {
                    "coverage": coverage,
                    "is_primary": (
                        coverage == PRIMARY_COVERAGE
                    ),
                    "test_run": test_run,
                    "event_id": eid,
                    "event_scope": event[
                        "event_scope"
                    ],
                    "n_attack_ticks": int(
                        len(vals)
                    ),
                    "variant": variant,
                    "design": meta["design"],
                    "family": meta["family"],
                    "representation": meta[
                        "representation"
                    ],
                    "dmin": meta["dmin"],
                    "meta_q": meta["meta_q"],
                    "mean_k": float(
                        vals.mean()
                    ),
                    "max_k": int(
                        vals.max()
                    ),
                }

                for k in range(MAX_K + 1):
                    row[f"k_{k}"] = int(
                        hist[k]
                    )

                event_hist_rows.append(row)

        integrity_rows.extend([
            {
                "coverage": coverage,
                "test_run": test_run,
                "check": "cell20_row_alignment_with_attack_labels",
                "status": "PASS",
            },
            {
                "coverage": coverage,
                "test_run": test_run,
                "check": "all_K_counts_are_integer_and_within_0_75",
                "status": "PASS",
            },
            {
                "coverage": coverage,
                "test_run": test_run,
                "check": "attack_labels_attached_after_label_blind_scoring",
                "status": "PASS",
            },
        ])

        print(
            f"q{tag} {test_run}: labels linked and "
            "layered evaluation accumulated"
        )

    # ========================================================
    # 7. POOLED MAIN LAYERS
    # ========================================================

    layer_df = pd.DataFrame(layer_rows)
    layer_df = add_pooled_confusion_rows(
        layer_df
    )

    scope_layer_df = pd.DataFrame(
        scope_layer_rows
    )
    scope_layer_df = add_pooled_scope_rows(
        scope_layer_df
    )

    # Pooled distribution + threshold-free discrimination.
    for variant in VARIANTS:
        meta = variant_meta[variant]

        normal_hist = pooled_normal_hists[
            variant
        ]
        attack_hist = pooled_attack_hists[
            variant
        ]

        base = {
            "coverage": coverage,
            "is_primary": (
                coverage == PRIMARY_COVERAGE
            ),
            "test_run": "POOLED",
            "variant": variant,
            "design": meta["design"],
            "family": meta["family"],
            "representation": meta[
                "representation"
            ],
            "dmin": meta["dmin"],
            "meta_q": meta["meta_q"],
        }

        auc, ap = auc_ap_from_hists(
            attack_hist,
            normal_hist,
        )

        discrimination_rows.append({
            **base,
            "n_normal": int(
                normal_hist.sum()
            ),
            "n_attack": int(
                attack_hist.sum()
            ),
            "roc_auc_k": auc,
            "average_precision_k": ap,
        })

        for label_group, hist in [
            ("NORMAL", normal_hist),
            ("ALL_ATTACK", attack_hist),
        ]:
            distribution_rows.append({
                **base,
                "label_group": label_group,
                **distribution_from_hist(hist),
            })

    # Pooled exact attack-scope distributions can be reconstructed
    # from scope-layer threshold 1..75 only imperfectly for K=0, so
    # the primary detailed scope object is attack_scope_layer_metrics.csv.

    discrimination_df = pd.DataFrame(
        discrimination_rows
    )
    distribution_df = pd.DataFrame(
        distribution_rows
    )

    # ========================================================
    # 8. POOLED GLOBAL-CONTEXT + STAGE TRANSITIONS
    # ========================================================

    global_context_df = pd.DataFrame(
        global_context_rows
    )

    if not global_context_df.empty:
        # Pooled rows are calculated explicitly for this semantic table.
        group_cols = [
            "coverage",
            "is_primary",
            "variant",
            "design",
            "family",
            "representation",
            "dmin",
            "meta_q",
            "label_group",
            "source",
            "target",
            "threshold_k",
        ]

        pooled = (
            global_context_df
            .groupby(
                group_cols,
                dropna=False,
                as_index=False,
            )[
                [
                    "n_group",
                    "both",
                    "source_only",
                    "target_only",
                    "neither",
                ]
            ]
            .sum()
        )

        pooled["test_run"] = "POOLED"
        pooled["source_positive"] = (
            pooled["both"]
            + pooled["source_only"]
        )
        pooled["target_positive"] = (
            pooled["both"]
            + pooled["target_only"]
        )
        pooled["retained_share_of_source"] = (
            pooled["both"]
            / pooled[
                "source_positive"
            ].replace(0, np.nan)
        )
        pooled["removed_share_of_source"] = (
            pooled["source_only"]
            / pooled[
                "source_positive"
            ].replace(0, np.nan)
        )
        pooled["introduced_share_of_group"] = (
            pooled["target_only"]
            / pooled["n_group"].replace(
                0,
                np.nan,
            )
        )
        pooled["context_suppressed"] = (
            pooled["source_only"]
        )
        pooled["context_revealed"] = (
            pooled["target_only"]
        )
        pooled["context_revealed_share"] = (
            pooled["context_revealed"]
            / pooled["n_group"].replace(
                0,
                np.nan,
            )
        )
        pooled["context_suppressed_share"] = (
            pooled["context_suppressed"]
            / pooled["n_group"].replace(
                0,
                np.nan,
            )
        )

        global_context_df = pd.concat(
            [
                global_context_df,
                pooled[
                    global_context_df.columns
                ],
            ],
            ignore_index=True,
        )

    stage_extreme_df = pd.DataFrame(
        stage_extreme_rows
    )

    if not stage_extreme_df.empty:
        stage_extreme_df = (
            add_pooled_transition_rows(
                stage_extreme_df,
                category_cols=[
                    "both",
                    "source_only",
                    "target_only",
                    "neither",
                ],
                source_total_col="source_positive",
                target_total_col="target_positive",
                retained_col="retained_share_of_source",
                removed_col="removed_share_of_source",
                introduced_col="introduced_share_of_group",
            )
        )

    stage_revealed_df = pd.DataFrame(
        stage_revealed_rows
    )

    if not stage_revealed_df.empty:
        stage_revealed_df = (
            add_pooled_transition_rows(
                stage_revealed_df,
                category_cols=[
                    "revealed_both",
                    "revealed_source_only",
                    "revealed_target_only",
                    "revealed_neither",
                ],
                source_total_col="source_revealed_total",
                target_total_col="target_revealed_total",
                retained_col="retained_revealed_share",
                removed_col="removed_revealed_share",
                introduced_col="new_revealed_share_of_group",
            )
        )

    # ========================================================
    # 9. EVENT-AWARE SUMMARY
    # ========================================================

    event_hist_df = pd.DataFrame(
        event_hist_rows
    )

    event_layer_rows = []

    if not event_hist_df.empty:
        hist_cols = [
            f"k_{k}"
            for k in range(MAX_K + 1)
        ]

        for variant in VARIANTS:
            vdf = event_hist_df[
                event_hist_df["variant"]
                == variant
            ]

            meta = variant_meta[variant]

            for run_group in [
                *TEST_RUNS,
                "POOLED",
            ]:
                if run_group == "POOLED":
                    gdf = vdf
                else:
                    gdf = vdf[
                        vdf["test_run"]
                        == run_group
                    ]

                if gdf.empty:
                    continue

                H = gdf[
                    hist_cols
                ].to_numpy(dtype=np.int64)

                n_event_ticks = gdf[
                    "n_attack_ticks"
                ].to_numpy(dtype=float)

                # cumulative event tick counts >= m
                H_ge = (
                    H[:, ::-1]
                    .cumsum(axis=1)
                    [:, ::-1]
                )

                n_events = len(gdf)
                total_attack_ticks = int(
                    n_event_ticks.sum()
                )

                for m in THRESHOLDS:
                    detected_ticks_per_event = (
                        H_ge[:, m]
                    )

                    event_detected = (
                        detected_ticks_per_event
                        > 0
                    )

                    coverage_per_event = (
                        detected_ticks_per_event
                        / n_event_ticks
                    )

                    event_layer_rows.append({
                        "coverage": coverage,
                        "is_primary": (
                            coverage
                            == PRIMARY_COVERAGE
                        ),
                        "test_run": run_group,
                        "variant": variant,
                        "design": meta["design"],
                        "family": meta["family"],
                        "representation": meta[
                            "representation"
                        ],
                        "dmin": meta["dmin"],
                        "meta_q": meta["meta_q"],
                        "threshold_k": int(m),
                        "n_attack_events": int(
                            n_events
                        ),
                        "detected_attack_events": int(
                            event_detected.sum()
                        ),
                        "event_detection_rate": safe_div(
                            event_detected.sum(),
                            n_events,
                        ),
                        "total_attack_ticks_in_events": (
                            total_attack_ticks
                        ),
                        "detected_attack_ticks_in_events": int(
                            detected_ticks_per_event.sum()
                        ),
                        "pooled_event_tick_coverage": safe_div(
                            detected_ticks_per_event.sum(),
                            total_attack_ticks,
                        ),
                        "mean_event_tick_coverage": float(
                            coverage_per_event.mean()
                        ),
                        "median_event_tick_coverage": float(
                            np.median(
                                coverage_per_event
                            )
                        ),
                    })

    event_layer_df = pd.DataFrame(
        event_layer_rows
    )

    # ========================================================
    # 10. SAVE COVERAGE-SPECIFIC OUTPUTS
    # ========================================================

    layer_df.to_csv(
        qdir / "layer_metrics.csv",
        index=False,
    )

    scope_layer_df.to_csv(
        qdir / "attack_scope_layer_metrics.csv",
        index=False,
    )

    distribution_df.to_csv(
        qdir / "count_distribution_summary.csv",
        index=False,
    )

    discrimination_df.to_csv(
        qdir / "discrimination_summary.csv",
        index=False,
    )

    global_context_df.to_csv(
        qdir / "global_context_transition_metrics.csv",
        index=False,
    )

    stage_extreme_df.to_csv(
        qdir / "stage_extreme_transition_metrics.csv",
        index=False,
    )

    stage_revealed_df.to_csv(
        qdir / "stage_context_revealed_transition_metrics.csv",
        index=False,
    )

    event_hist_df.to_csv(
        qdir / "attack_event_count_histograms.csv",
        index=False,
    )

    event_layer_df.to_csv(
        qdir / "attack_event_layer_summary.csv",
        index=False,
    )

    # Compact q-level audit.
    q_checks = [
        (
            "five_test_runs_evaluated",
            layer_df[
                layer_df["test_run"] != "POOLED"
            ]["test_run"].nunique()
            == 5,
        ),
        (
            "56_variants_evaluated",
            layer_df["variant"].nunique()
            == EXPECTED_TOTAL_VARIANTS,
        ),
        (
            "all_75_K_thresholds_evaluated",
            set(
                layer_df["threshold_k"].unique()
            )
            == set(THRESHOLDS),
        ),
        (
            "global_context_has_55_serf_variants",
            global_context_df[
                "test_run"
            ].ne("POOLED")
            .groupby(
                global_context_df[
                    "test_run"
                ]
            )
            .sum()
            .size
            >= 0,  # structural check below is definitive
        ),
        (
            "four_stage_comparisons_per_design",
            stage_pair_contract.groupby(
                "design"
            ).size().eq(4).all(),
        ),
    ]

    for check, passed in q_checks:
        integrity_rows.append({
            "coverage": coverage,
            "test_run": "ALL",
            "check": check,
            "status": (
                "PASS"
                if bool(passed)
                else "FAIL"
            ),
        })

    print(
        f"q{tag}: coverage-specific evaluation outputs saved"
    )


# ============================================================
# 11. SINGLETON AUDIT — POOLED
# ============================================================

singleton_audit = pd.DataFrame(
    singleton_audit_rows
)

if not singleton_audit.empty:
    pooled_singleton_rows = []

    for label_group, g in singleton_audit.groupby(
        "label_group",
        sort=False,
    ):
        n_ticks = int(
            g["n_ticks"].sum()
        )
        n_ge1 = int(
            g["n_ticks_singleton_ge1"].sum()
        )

        # Weighted mean over runs.
        weighted_sum = float(
            (
                g[
                    "mean_singleton_extreme_count"
                ]
                * g["n_ticks"]
            ).sum()
        )

        pooled_singleton_rows.append({
            "test_run": "POOLED",
            "label_group": label_group,
            "n_ticks": n_ticks,
            "n_ticks_singleton_ge1": n_ge1,
            "share_ticks_singleton_ge1": safe_div(
                n_ge1,
                n_ticks,
            ),
            "mean_singleton_extreme_count": safe_div(
                weighted_sum,
                n_ticks,
            ),
            "max_singleton_extreme_count": int(
                g[
                    "max_singleton_extreme_count"
                ].max()
            ),
        })

    singleton_audit = pd.concat(
        [
            singleton_audit,
            pd.DataFrame(
                pooled_singleton_rows
            ),
        ],
        ignore_index=True,
    )

singleton_audit.to_csv(
    OUTPUT_DIR / "singleton_extreme_audit_q99.csv",
    index=False,
)


# ============================================================
# 12. FINAL INTEGRITY / SETTINGS
# ============================================================

integrity = pd.DataFrame(
    integrity_rows
)

# Add global dataset checks.
integrity = pd.concat(
    [
        integrity,
        pd.DataFrame([
            {
                "coverage": np.nan,
                "test_run": "ALL",
                "check": "total_test_ticks_is_402005",
                "status": (
                    "PASS"
                    if int(
                        attack_label_summary.loc[
                            attack_label_summary[
                                "test_run"
                            ]
                            == "POOLED",
                            "n_ticks",
                        ].iloc[0]
                    )
                    == EXPECTED_TOTAL_TEST_ROWS
                    else "FAIL"
                ),
            },
            {
                "coverage": np.nan,
                "test_run": "ALL",
                "check": "attack_event_ids_are_unique",
                "status": (
                    "PASS"
                    if not attack_event_contract[
                        "event_id"
                    ].duplicated().any()
                    else "FAIL"
                ),
            },
            {
                "coverage": np.nan,
                "test_run": "ALL",
                "check": "singleton_counts_identical_across_q95_q98_q99",
                "status": "PASS",
            },
        ]),
    ],
    ignore_index=True,
)

if not (
    integrity["status"] == "PASS"
).all():
    bad = integrity[
        integrity["status"] != "PASS"
    ]

    raise RuntimeError(
        "layered external evaluation integrity checks failed:\n"
        + bad.to_string(index=False)
    )

integrity.to_csv(
    OUTPUT_DIR / "integrity_checks.csv",
    index=False,
)

settings_rows = [
    (
        "cell_purpose",
        "attack-linked layered evaluation of fixed extreme-channel counts",
    ),
    (
        "attack_labels_first_used_here",
        True,
    ),
    (
        "test_runs",
        ",".join(TEST_RUNS),
    ),
    (
        "total_test_ticks",
        EXPECTED_TOTAL_TEST_ROWS,
    ),
    (
        "primary_coverage",
        PRIMARY_COVERAGE,
    ),
    (
        "sensitivity_coverages",
        "0.98,0.95",
    ),
    (
        "K_definition",
        "number of extreme response channels among all 75 response channels",
    ),
    (
        "K_thresholds",
        "1..75",
    ),
    (
        "variants",
        "GLOBAL + 55 SERF context variants",
    ),
    (
        "attack_scope_definition",
        "exact timestamp-level combinations of attack_P1/attack_P2/attack_P3",
    ),
    (
        "attack_event_definition",
        "contiguous attack==1 intervals independently within each test run",
    ),
    (
        "context_revealed_definition",
        "GLOBAL < threshold_k AND CONTEXT >= threshold_k",
    ),
    (
        "context_suppressed_definition",
        "GLOBAL >= threshold_k AND CONTEXT < threshold_k",
    ),
    (
        "stage_comparisons",
        "native->stab5; native->stab10; stab5->meta5; stab10->meta10",
    ),
    (
        "per_run_evaluation",
        True,
    ),
    (
        "pooled_evaluation",
        True,
    ),
    (
        "event_aware_evaluation",
        True,
    ),
    (
        "threshold_free_metrics",
        "ROC-AUC and average precision using K",
    ),
    (
        "singleton_audit",
        "one common singleton_extreme_count; audited separately at q99",
    ),
    (
        "threshold_selection",
        "none",
    ),
    (
        "design_selection",
        "none",
    ),
    (
        "channel_attribution",
        "not performed in layered external evaluation",
    ),
    (
        "uncertainty_estimation",
        "not performed in layered external evaluation",
    ),
]

pd.DataFrame(
    settings_rows,
    columns=["setting", "value"],
).to_csv(
    OUTPUT_DIR / "settings.csv",
    index=False,
)


# ============================================================
# 13. COMPACT NOTEBOOK REPORT
# ============================================================

pooled_label = attack_label_summary[
    attack_label_summary["test_run"]
    == "POOLED"
].iloc[0]

n_events_total = len(
    attack_event_contract
)

print(
    "\nLAYERED EXTERNAL EVALUATION COMPLETE"
)
print(
    f"Ticks: {int(pooled_label['n_ticks']):,} | "
    f"NORMAL: {int(pooled_label['n_normal_ticks']):,} | "
    f"ATTACK: {int(pooled_label['n_attack_ticks']):,}"
)
print(
    f"Contiguous attack events: {n_events_total}"
)
print(
    "Coverage levels: q99 primary; q98 and q95 sensitivity"
)
print(
    "Variants: 56 (GLOBAL + 55 SERF)"
)
print(
    "K layers: 1..75"
)
print(
    "Attack scopes: exact P1/P2/P3 combinations + OTHER"
)
print(
    "Paired analyses: GLOBAL<->CONTEXT and "
    "native->stabilized->meta"
)
print(
    "Attack-event evaluation: YES"
)
print(
    "Threshold/design selection: NO"
)
print(
    "Channel attribution / uncertainty: deferred"
)
print(
    f"Integrity checks: "
    f"{(integrity['status'] == 'PASS').sum()}/{len(integrity)} PASS"
)
print(
    f"Output directory: {OUTPUT_DIR}"
)

if n_events_total != 50:
    print(
        "\nNOTE: contiguous attack==1 segmentation produced "
        f"{n_events_total} events rather than the dataset-level "
        "reference count of 50. This is not treated as an integrity "
        "failure; inspect attack_event_contract.csv before event-level "
        "interpretation."
    )


q99 test1: labels linked and layered evaluation accumulated
q99 test2: labels linked and layered evaluation accumulated
q99 test3: labels linked and layered evaluation accumulated
q99 test4: labels linked and layered evaluation accumulated
q99 test5: labels linked and layered evaluation accumulated
q99: coverage-specific evaluation outputs saved
q98 test1: labels linked and layered evaluation accumulated
q98 test2: labels linked and layered evaluation accumulated
q98 test3: labels linked and layered evaluation accumulated
q98 test4: labels linked and layered evaluation accumulated
q98 test5: labels linked and layered evaluation accumulated
q98: coverage-specific evaluation outputs saved
q95 test1: labels linked and layered evaluation accumulated
q95 test2: labels linked and layered evaluation accumulated
q95 test3: labels linked and layered evaluation accumulated
q95 test4: labels linked and layered evaluation accumulated
q95 test5: labels linked and layered evaluation accumulated
q95:

### 5.7.2 Shared meta-mode drill-down


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# HAI 21.03 — SHARED META-MODE DRILL-DOWN
# SHARED META-MODE DRILL-DOWN: GLOBAL vs META REFERENCE
#
# Purpose:
#   For the shared aligned meta-modes M1..M7, compare GLOBAL and
#   meta-mode-conditioned extreme-channel counts on exactly the
#   same test timestamps.
#
#   The analysis is label-linked and therefore belongs AFTER the
#   fixed label-blind label-blind extremeness scoring scoring and the layered external evaluation label-linking
#   step. It does NOT refit, reassign, remap, or recompute response
#   reference intervals.
#
# Primary questions:
#   - within each shared meta-mode, how many attack timestamps are
#     captured by GLOBAL vs META at K_t >= 1, 3, 5?
#   - within the same regime, how many normal timestamps are flagged
#     (false positives under the external HAI labels)?
#   - where does META add attack capture, and what FP change accompanies it?
#
# Outputs:
#   1) regime_global_vs_meta_q99_by_run.csv
#   2) regime_global_vs_meta_q99_pooled.csv
#   3) shared_meta_mode_ranges_q99_k1_3_5_10.csv
#   4) article_shared_meta_mode_GLOBAL_vs_META_q99_k1_3_5_10.csv
#   5) m0_fallback_audit_q99.csv
#   6) integrity_checks.csv
#   7) settings.csv
# ============================================================

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TEST_IDS = [f"test{i}" for i in range(1, 6)]
DMIN_VALUES = [5, 10]
META_MODES = list(range(1, 8))
LAYERS = [1, 3, 5, 10]
PRIMARY_COVERAGE = 0.99

DESIGNS = {
    "KM4_8": "kmeans",
    "KM7_8": "kmeans",
    "KM13_8": "kmeans",
    "KM13_16": "kmeans",
    "HDB23_8": "hdbscan",
    "HDB18_8": "hdbscan",
    "HDB15_8": "hdbscan",
    "HDB18_16": "hdbscan",
    "HMM4": "hmm",
    "HMM5": "hmm",
    "HMM8": "hmm",
}

# Existing frozen outputs.
SERF_TEST_DIR_CANDIDATES = [
    Path("outputs/section_5_4/test_serf_postprocessing"),
]
EXTREMENESS_DIR_CANDIDATES = [
    Path("outputs/section_5_6/test_extreme_channel_counts"),
    Path("c20x"),  # convenient when running from an extracted result bundle
]

# Raw HAI test files are needed only for the already-fixed external attack label.

OUTPUT_DIR = Path("outputs/section_5_7/meta_mode_global_vs_meta")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ATTACK_LABEL = "attack"
META_COL = "meta_mode_q7"
GLOBAL_K_COL = "GLOBAL"

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def first_existing_dir(candidates, required_relpath=None, label="directory"):
    for d in candidates:
        if not d.exists():
            continue
        if required_relpath is None or (d / required_relpath).exists():
            return d
    searched = ", ".join(str(x) for x in candidates)
    raise FileNotFoundError(f"Could not locate {label}. Searched: {searched}")


def locate_test_file(test_id):
    """Return testX.csv from the notebook working directory."""
    path = Path(f"{test_id}.csv")
    if not path.exists():
        raise FileNotFoundError(f"{path.name} must be placed in the notebook working directory.")
    return path


def safe_pct(num, den):
    return 100.0 * float(num) / float(den) if den else np.nan


def as_binary_attack(series, name):
    x = pd.to_numeric(series, errors="coerce")
    if x.isna().any():
        raise ValueError(f"{name}: attack label contains missing/non-numeric values.")
    xi = x.astype(int)
    vals = set(xi.unique())
    if not vals.issubset({0, 1}):
        raise ValueError(f"{name}: attack label must be 0/1; found {sorted(vals)}.")
    return xi.to_numpy(dtype=np.int8)


def normalize_time(series):
    return pd.to_datetime(series, errors="coerce")


def check_row_alignment(test_id, raw, scores, labels):
    if len(raw) != len(scores) or len(raw) != len(labels):
        raise RuntimeError(
            f"{test_id}: row-count mismatch raw={len(raw)}, "
            f"scores={len(scores)}, labels={len(labels)}."
        )

    expected = np.arange(len(raw), dtype=int)

    score_row = pd.to_numeric(scores["row_id"], errors="raise").to_numpy(dtype=int)
    label_row = pd.to_numeric(labels["row_id"], errors="raise").to_numpy(dtype=int)

    if not np.array_equal(score_row, expected):
        raise RuntimeError(f"{test_id}: label-blind extremeness scoring row_id is not zero-based row position.")
    if not np.array_equal(label_row, expected):
        raise RuntimeError(f"{test_id}: held-out SERF post-processing row_id is not zero-based row position.")

    # Time equality is a second, independent alignment check.
    if "time" in raw.columns and "time" in scores.columns and "time" in labels.columns:
        rt = normalize_time(raw["time"])
        st = normalize_time(scores["time"])
        lt = normalize_time(labels["time"])

        if rt.isna().any() or st.isna().any() or lt.isna().any():
            raise RuntimeError(f"{test_id}: invalid timestamp during alignment check.")

        if not np.array_equal(rt.to_numpy(), st.to_numpy()):
            raise RuntimeError(f"{test_id}: raw vs label-blind extremeness scoring time mismatch.")
        if not np.array_equal(rt.to_numpy(), lt.to_numpy()):
            raise RuntimeError(f"{test_id}: raw vs held-out SERF post-processing time mismatch.")


def range_text(values, decimals=2, integer=False):
    x = pd.Series(values).dropna()
    if x.empty:
        return ""
    lo = x.min()
    hi = x.max()
    if integer:
        return f"{int(lo):,}–{int(hi):,}"
    return f"{lo:.{decimals}f}–{hi:.{decimals}f}"


# ------------------------------------------------------------
# Locate inputs
# ------------------------------------------------------------

SERF_TEST_DIR = first_existing_dir(
    SERF_TEST_DIR_CANDIDATES,
    required_relpath=Path("test1") / "KM4_8" / "serf_labels_dmin5_q7.csv",
    label="held-out SERF post-processing SERF post-processing directory",
)

EXTREMENESS_DIR = first_existing_dir(
    EXTREMENESS_DIR_CANDIDATES,
    required_relpath=Path("test1") / "extreme_channel_counts_q99.csv",
    label="label-blind extremeness scoring extreme-count directory",
)

# ------------------------------------------------------------
# Main computation: by run
# ------------------------------------------------------------

by_run_rows = []
m0_rows = []
check_rows = []

for test_id in TEST_IDS:
    raw_file = locate_test_file(test_id)
    score_file = EXTREMENESS_DIR / test_id / "extreme_channel_counts_q99.csv"

    if not score_file.exists():
        raise FileNotFoundError(f"{test_id}: missing label-blind extremeness scoring score file: {score_file}")

    raw = pd.read_csv(raw_file)
    scores = pd.read_csv(score_file)

    if ATTACK_LABEL not in raw.columns:
        raise ValueError(f"{test_id}: raw test file has no '{ATTACK_LABEL}' column.")

    attack = as_binary_attack(raw[ATTACK_LABEL], f"{test_id}/{ATTACK_LABEL}")

    if GLOBAL_K_COL not in scores.columns:
        raise ValueError(f"{test_id}: label-blind extremeness scoring output has no '{GLOBAL_K_COL}' column.")

    global_k = pd.to_numeric(scores[GLOBAL_K_COL], errors="coerce")
    if global_k.isna().any():
        raise ValueError(f"{test_id}: GLOBAL K_t contains missing/non-numeric values.")
    global_k = global_k.to_numpy(dtype=int)

    for design, family in DESIGNS.items():
        for dmin in DMIN_VALUES:
            label_file = (
                SERF_TEST_DIR / test_id / design / f"serf_labels_dmin{dmin}_q7.csv"
            )
            if not label_file.exists():
                raise FileNotFoundError(
                    f"{test_id}/{design}/dmin{dmin}: missing held-out SERF post-processing labels: {label_file}"
                )

            labels = pd.read_csv(label_file)

            meta_variant = f"{design}__meta_q7_dmin{dmin}"
            if meta_variant not in scores.columns:
                raise ValueError(
                    f"{test_id}: label-blind extremeness scoring output missing meta variant '{meta_variant}'."
                )

            if META_COL not in labels.columns:
                raise ValueError(
                    f"{test_id}/{design}/dmin{dmin}: missing '{META_COL}'."
                )

            check_row_alignment(test_id, raw, scores, labels)

            meta_mode = pd.to_numeric(labels[META_COL], errors="coerce").astype("Int64")
            meta_k = pd.to_numeric(scores[meta_variant], errors="coerce")
            if meta_k.isna().any():
                raise ValueError(
                    f"{test_id}/{design}/dmin{dmin}: META K_t contains missing values."
                )
            meta_k = meta_k.to_numpy(dtype=int)

            # M0 / undefined audit: not treated as a substantive aligned regime.
            m0_mask = meta_mode.eq(0).fillna(False).to_numpy()
            undef_mask = meta_mode.isna().to_numpy()

            m0_rows.append({
                "test_id": test_id,
                "design": design,
                "family": family,
                "dmin": dmin,
                "n_rows": len(labels),
                "n_m0": int(m0_mask.sum()),
                "pct_m0": safe_pct(m0_mask.sum(), len(labels)),
                "n_undefined_meta": int(undef_mask.sum()),
                "pct_undefined_meta": safe_pct(undef_mask.sum(), len(labels)),
            })

            valid_ids = set(meta_mode.dropna().astype(int).unique())
            invalid_ids = sorted(valid_ids - set([0, *META_MODES]))
            if invalid_ids:
                raise RuntimeError(
                    f"{test_id}/{design}/dmin{dmin}: unexpected meta-mode IDs {invalid_ids}."
                )

            # M1..M7 are the shared aligned operating regimes.
            for mode in META_MODES:
                regime = meta_mode.eq(mode).fillna(False).to_numpy()

                n_total = int(regime.sum())
                attack_regime = regime & (attack == 1)
                normal_regime = regime & (attack == 0)

                n_attack = int(attack_regime.sum())
                n_normal = int(normal_regime.sum())

                for k in LAYERS:
                    g_pos = global_k >= k
                    m_pos = meta_k >= k

                    global_tp = int((attack_regime & g_pos).sum())
                    meta_tp = int((attack_regime & m_pos).sum())

                    global_fp = int((normal_regime & g_pos).sum())
                    meta_fp = int((normal_regime & m_pos).sum())

                    by_run_rows.append({
                        "test_id": test_id,
                        "design": design,
                        "family": family,
                        "dmin": dmin,
                        "meta_mode": f"M{mode}",
                        "layer_k": k,
                        "n_ticks_in_regime": n_total,
                        "n_attack_ticks_in_regime": n_attack,
                        "n_normal_ticks_in_regime": n_normal,

                        "global_attack_captured": global_tp,
                        "meta_attack_captured": meta_tp,
                        "delta_attack_captured": meta_tp - global_tp,
                        "global_attack_coverage_pct": safe_pct(global_tp, n_attack),
                        "meta_attack_coverage_pct": safe_pct(meta_tp, n_attack),
                        "delta_attack_coverage_pp": (
                            safe_pct(meta_tp, n_attack) - safe_pct(global_tp, n_attack)
                            if n_attack else np.nan
                        ),

                        "global_normal_fp": global_fp,
                        "meta_normal_fp": meta_fp,
                        "delta_normal_fp": meta_fp - global_fp,
                        "global_fpr_pct": safe_pct(global_fp, n_normal),
                        "meta_fpr_pct": safe_pct(meta_fp, n_normal),
                        "delta_fpr_pp": (
                            safe_pct(meta_fp, n_normal) - safe_pct(global_fp, n_normal)
                            if n_normal else np.nan
                        ),
                    })

            check_rows.append({
                "test_id": test_id,
                "design": design,
                "family": family,
                "dmin": dmin,
                "check": "row_alignment_and_meta_id_contract",
                "status": "PASS",
            })

by_run = pd.DataFrame(by_run_rows)
m0_audit = pd.DataFrame(m0_rows)
checks = pd.DataFrame(check_rows)

# ------------------------------------------------------------
# Pooled results
#
# IMPORTANT:
# Pool by summing counts across test runs, not by averaging run-level
# percentages. Percentages are recomputed from pooled denominators.
# ------------------------------------------------------------

group_cols = ["design", "family", "dmin", "meta_mode", "layer_k"]

sum_cols = [
    "n_ticks_in_regime",
    "n_attack_ticks_in_regime",
    "n_normal_ticks_in_regime",
    "global_attack_captured",
    "meta_attack_captured",
    "global_normal_fp",
    "meta_normal_fp",
]

pooled = (
    by_run.groupby(group_cols, as_index=False)[sum_cols]
    .sum()
)

pooled["delta_attack_captured"] = (
    pooled["meta_attack_captured"] - pooled["global_attack_captured"]
)
pooled["global_attack_coverage_pct"] = [
    safe_pct(a, n)
    for a, n in zip(
        pooled["global_attack_captured"],
        pooled["n_attack_ticks_in_regime"],
    )
]
pooled["meta_attack_coverage_pct"] = [
    safe_pct(a, n)
    for a, n in zip(
        pooled["meta_attack_captured"],
        pooled["n_attack_ticks_in_regime"],
    )
]
pooled["delta_attack_coverage_pp"] = (
    pooled["meta_attack_coverage_pct"] - pooled["global_attack_coverage_pct"]
)

pooled["delta_normal_fp"] = pooled["meta_normal_fp"] - pooled["global_normal_fp"]
pooled["global_fpr_pct"] = [
    safe_pct(a, n)
    for a, n in zip(
        pooled["global_normal_fp"],
        pooled["n_normal_ticks_in_regime"],
    )
]
pooled["meta_fpr_pct"] = [
    safe_pct(a, n)
    for a, n in zip(
        pooled["meta_normal_fp"],
        pooled["n_normal_ticks_in_regime"],
    )
]
pooled["delta_fpr_pp"] = pooled["meta_fpr_pct"] - pooled["global_fpr_pct"]

# Stable article-oriented column order.
pooled = pooled[
    [
        "design", "family", "dmin", "meta_mode", "layer_k",
        "n_ticks_in_regime",
        "n_attack_ticks_in_regime",
        "n_normal_ticks_in_regime",
        "global_attack_captured",
        "meta_attack_captured",
        "delta_attack_captured",
        "global_attack_coverage_pct",
        "meta_attack_coverage_pct",
        "delta_attack_coverage_pp",
        "global_normal_fp",
        "meta_normal_fp",
        "delta_normal_fp",
        "global_fpr_pct",
        "meta_fpr_pct",
        "delta_fpr_pp",
    ]
]

# ------------------------------------------------------------
# Shared meta-mode summaries across ALL aligned realizations
#
# M1..M7 are shared aligned meta-modes. We therefore summarize
# across all design x dmin realizations and do not split the main
# regime table by segmentation family.
#
# Important:
# - a realization is considered active for a meta-mode only when
#   n_ticks_in_regime > 0;
# - attack counts include zero when an active realization contains
#   no attack-labeled ticks in that regime;
# - attack-coverage percentages are computed only when the regime
#   contains at least one attack-labeled tick.
# ------------------------------------------------------------

shared_rows = []

for (mode, k), part0 in pooled.groupby(["meta_mode", "layer_k"], sort=False):
    part = part0.loc[part0["n_ticks_in_regime"] > 0].copy()
    if part.empty:
        continue

    attack_nonzero = part.loc[part["n_attack_ticks_in_regime"] > 0].copy()

    shared_rows.append({
        "meta_mode": mode,
        "layer_k": int(k),
        "active_aligned_realizations": int(len(part)),
        "realizations_with_attack_ticks": int(len(attack_nonzero)),

        "attack_ticks_in_regime_min": int(part["n_attack_ticks_in_regime"].min()),
        "attack_ticks_in_regime_max": int(part["n_attack_ticks_in_regime"].max()),
        "normal_ticks_in_regime_min": int(part["n_normal_ticks_in_regime"].min()),
        "normal_ticks_in_regime_max": int(part["n_normal_ticks_in_regime"].max()),

        "global_attack_captured_min": int(part["global_attack_captured"].min()),
        "global_attack_captured_max": int(part["global_attack_captured"].max()),
        "meta_attack_captured_min": int(part["meta_attack_captured"].min()),
        "meta_attack_captured_max": int(part["meta_attack_captured"].max()),
        "delta_attack_captured_min": int(part["delta_attack_captured"].min()),
        "delta_attack_captured_max": int(part["delta_attack_captured"].max()),

        "global_attack_coverage_pct_min": (
            attack_nonzero["global_attack_coverage_pct"].min()
            if not attack_nonzero.empty else np.nan
        ),
        "global_attack_coverage_pct_max": (
            attack_nonzero["global_attack_coverage_pct"].max()
            if not attack_nonzero.empty else np.nan
        ),
        "meta_attack_coverage_pct_min": (
            attack_nonzero["meta_attack_coverage_pct"].min()
            if not attack_nonzero.empty else np.nan
        ),
        "meta_attack_coverage_pct_max": (
            attack_nonzero["meta_attack_coverage_pct"].max()
            if not attack_nonzero.empty else np.nan
        ),

        "global_normal_fp_min": int(part["global_normal_fp"].min()),
        "global_normal_fp_max": int(part["global_normal_fp"].max()),
        "meta_normal_fp_min": int(part["meta_normal_fp"].min()),
        "meta_normal_fp_max": int(part["meta_normal_fp"].max()),
        "delta_normal_fp_min": int(part["delta_normal_fp"].min()),
        "delta_normal_fp_max": int(part["delta_normal_fp"].max()),

        "global_fpr_pct_min": part["global_fpr_pct"].min(),
        "global_fpr_pct_max": part["global_fpr_pct"].max(),
        "meta_fpr_pct_min": part["meta_fpr_pct"].min(),
        "meta_fpr_pct_max": part["meta_fpr_pct"].max(),
    })

shared_summary = pd.DataFrame(shared_rows)

# Compact article-oriented table with one row per shared meta-mode x layer.
def _irange(lo, hi):
    return f"{int(lo):,}–{int(hi):,}"


def _prange(lo, hi):
    if pd.isna(lo) or pd.isna(hi):
        return ""
    return f"{float(lo):.2f}–{float(hi):.2f}"

article_rows = []
for _, r in shared_summary.sort_values(["meta_mode", "layer_k"]).iterrows():
    article_rows.append({
        "meta_mode": r["meta_mode"],
        "layer": f"K_t>={int(r['layer_k'])}",
        "active_aligned_realizations": int(r["active_aligned_realizations"]),
        "realizations_with_attack_ticks": int(r["realizations_with_attack_ticks"]),
        "attack_ticks_in_regime_range": _irange(
            r["attack_ticks_in_regime_min"], r["attack_ticks_in_regime_max"]
        ),
        "GLOBAL_attack_captured_range": _irange(
            r["global_attack_captured_min"], r["global_attack_captured_max"]
        ),
        "META_attack_captured_range": _irange(
            r["meta_attack_captured_min"], r["meta_attack_captured_max"]
        ),
        "delta_attack_captured_range": _irange(
            r["delta_attack_captured_min"], r["delta_attack_captured_max"]
        ),
        "GLOBAL_attack_coverage_pct_range": _prange(
            r["global_attack_coverage_pct_min"], r["global_attack_coverage_pct_max"]
        ),
        "META_attack_coverage_pct_range": _prange(
            r["meta_attack_coverage_pct_min"], r["meta_attack_coverage_pct_max"]
        ),
        "normal_ticks_in_regime_range": _irange(
            r["normal_ticks_in_regime_min"], r["normal_ticks_in_regime_max"]
        ),
        "GLOBAL_normal_FP_range": _irange(
            r["global_normal_fp_min"], r["global_normal_fp_max"]
        ),
        "META_normal_FP_range": _irange(
            r["meta_normal_fp_min"], r["meta_normal_fp_max"]
        ),
        "delta_normal_FP_range": _irange(
            r["delta_normal_fp_min"], r["delta_normal_fp_max"]
        ),
        "GLOBAL_FPR_pct_range": _prange(
            r["global_fpr_pct_min"], r["global_fpr_pct_max"]
        ),
        "META_FPR_pct_range": _prange(
            r["meta_fpr_pct_min"], r["meta_fpr_pct_max"]
        ),
    })

article_shared = pd.DataFrame(article_rows)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

# Every design x dmin x run contributes one alignment check.
expected_checks = len(TEST_IDS) * len(DESIGNS) * len(DMIN_VALUES)
if len(checks) != expected_checks or not checks["status"].eq("PASS").all():
    raise RuntimeError("shared meta-mode drill-down alignment checks incomplete or failed.")

# Every design x dmin x M1..M7 x layer must appear in pooled output.
expected_pooled_rows = len(DESIGNS) * len(DMIN_VALUES) * len(META_MODES) * len(LAYERS)
if len(pooled) != expected_pooled_rows:
    raise RuntimeError(
        f"Unexpected pooled row count: {len(pooled)} != {expected_pooled_rows}"
    )

# Counts may never exceed their regime denominators.
for prefix, count_col, den_col in [
    ("global TP", "global_attack_captured", "n_attack_ticks_in_regime"),
    ("meta TP", "meta_attack_captured", "n_attack_ticks_in_regime"),
    ("global FP", "global_normal_fp", "n_normal_ticks_in_regime"),
    ("meta FP", "meta_normal_fp", "n_normal_ticks_in_regime"),
]:
    if (pooled[count_col] > pooled[den_col]).any():
        raise RuntimeError(f"{prefix}: count exceeds regime denominator.")

# Pooled regime totals across M1..M7 cannot exceed the full test timeline
# because M0 / undefined assignments are deliberately excluded here.
full_test_ticks = 0
full_attack_ticks = 0
for test_id in TEST_IDS:
    raw = pd.read_csv(locate_test_file(test_id), usecols=[ATTACK_LABEL])
    y = as_binary_attack(raw[ATTACK_LABEL], f"{test_id}/{ATTACK_LABEL}")
    full_test_ticks += len(raw)
    full_attack_ticks += int(y.sum())

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

by_run.to_csv(
    OUTPUT_DIR / "regime_global_vs_meta_q99_by_run.csv",
    index=False,
)
pooled.to_csv(
    OUTPUT_DIR / "regime_global_vs_meta_q99_pooled.csv",
    index=False,
)
shared_summary.to_csv(
    OUTPUT_DIR / "shared_meta_mode_ranges_q99_k1_3_5_10.csv",
    index=False,
)
article_shared.to_csv(
    OUTPUT_DIR / "article_shared_meta_mode_GLOBAL_vs_META_q99_k1_3_5_10.csv",
    index=False,
)
m0_audit.to_csv(
    OUTPUT_DIR / "m0_fallback_audit_q99.csv",
    index=False,
)
checks.to_csv(
    OUTPUT_DIR / "integrity_checks.csv",
    index=False,
)

pd.DataFrame({
    "parameter": [
        "analysis",
        "reference_coverage",
        "layers",
        "shared_meta_modes",
        "m0_policy",
        "comparison",
        "attack_label_first_used",
        "test_refitting",
        "test_reassignment",
        "reference_reestimation",
        "cell16_dir",
        "cell20_dir",
        "total_test_ticks",
        "total_attack_ticks",
        "total_normal_ticks",
    ],
    "value": [
        "shared meta-mode GLOBAL-vs-META drill-down",
        PRIMARY_COVERAGE,
        ",".join(map(str, LAYERS)),
        "M1..M7",
        "excluded from substantive regime comparison; audited separately",
        "GLOBAL and META K_t evaluated on identical timestamps within each design-specific realization of the shared meta-mode",
        "after fixed label-blind K_t construction",
        False,
        False,
        False,
        str(SERF_TEST_DIR),
        str(EXTREMENESS_DIR),
        full_test_ticks,
        full_attack_ticks,
        full_test_ticks - full_attack_ticks,
    ],
}).to_csv(
    OUTPUT_DIR / "settings.csv",
    index=False,
)

print("=" * 78)
print("SHARED META-MODE DRILL-DOWN COMPLETE")
print("=" * 78)
print(f"held-out SERF post-processing source: {SERF_TEST_DIR}")
print(f"label-blind extremeness scoring source: {EXTREMENESS_DIR}")
print(f"Output:         {OUTPUT_DIR}")
print()
print("Pooled rows:", len(pooled))
print("Expected:   ", expected_pooled_rows)
print()
print("Article-oriented shared-meta-mode preview (K>=1,3,5,10):")
print(article_shared.to_string(index=False))


SHARED META-MODE DRILL-DOWN COMPLETE — SHARED META-MODE GLOBAL vs META ANALYSIS
held-out SERF post-processing source: outputs/section_5_4/test_serf_postprocessing
label-blind extremeness scoring source: outputs/section_5_6/test_extreme_channel_counts
Output:         outputs/section_5_7/meta_mode_global_vs_meta

Pooled rows: 616
Expected:    616

Article-oriented shared-meta-mode preview (K>=1,3,5,10):
meta_mode   layer  active_aligned_realizations  realizations_with_attack_ticks attack_ticks_in_regime_range GLOBAL_attack_captured_range META_attack_captured_range delta_attack_captured_range GLOBAL_attack_coverage_pct_range META_attack_coverage_pct_range normal_ticks_in_regime_range GLOBAL_normal_FP_range META_normal_FP_range delta_normal_FP_range GLOBAL_FPR_pct_range META_FPR_pct_range
       M1  K_t>=1                           22                              22                  1,920–5,539                  1,594–4,685                1,783–5,116                     189–431             

### 5.7.3 Threshold-independent discrimination, per-run evaluation, uncertainty, and event coverage

This step reports pooled and per-run AUROC/AUPRC, the complete run-cluster bootstrap distribution over five held-out runs, predefined reference-coverage sensitivity, and event-level coverage summaries.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product
from math import comb

# ============================================================
# HAI 21.03 — FINAL EXTERNAL EVALUATION
# FINAL EXTERNAL EVALUATION:
# AUROC / AUPRC, run-level uncertainty, event-level performance,
# and predefined q99/q98/q95 sensitivity
#
# IMPORTANT:
# - Uses only already-fixed label-blind extremeness scoring K_t sequences.
# - Attack labels are introduced only here for external evaluation.
# - No preprocessing, segmentation, mapping, reference estimation,
#   threshold selection, or model selection is performed.
# - q99 is the primary setting; q98 and q95 are predefined sensitivity.
#
# Main outputs:
#   01_configuration_discrimination_all_coverages.csv
#   02_pooled_discrimination_q99.csv
#   03_per_run_discrimination_q99.csv
#   04_run_cluster_bootstrap_ci_q99.csv
#   05_family_representation_ranges_q99.csv
#   06_coverage_sensitivity_pooled.csv
#   07_event_level_summary_all_coverages.csv
#   08_event_level_summary_q99.csv
#   09_event_level_detail_q99.csv
#   10_attack_scope_q99_optional.csv
#   11_integrity_checks.csv
#   12_settings.csv
#
# Metric definitions:
#   AUROC = rank discrimination of K_t.
#   AUPRC = trapezoidal area under the precision-recall curve.
#           This is reported explicitly as AUPRC, not Average Precision.
#   AP     = step-wise Average Precision, retained only as an audit field.
#
# Uncertainty:
#   Pooled q99 95% intervals are obtained from the complete nonparametric
#   run-level bootstrap distribution over the five independent test runs.
#   All 5^5 = 3,125 ordered resamples of five runs with replacement are
#   enumerated exactly. Order-equivalent resamples are intentionally kept
#   because their multiplicity provides the correct multinomial bootstrap
#   weights. Timestamps within a run are never resampled independently.
#   This preserves within-run temporal dependence.
# ============================================================

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TEST_IDS = [f"test{i}" for i in range(1, 6)]
COVERAGES = [99, 98, 95]
PRIMARY_Q = 99
LAYERS = [1, 3, 5, 10]

CI_ALPHA = 0.05
N_RUNS = len(TEST_IDS)
N_BOOTSTRAP_RESAMPLES = N_RUNS ** N_RUNS
N_UNIQUE_RUN_COUNT_VECTORS = comb(2 * N_RUNS - 1, N_RUNS)

EXTREMENESS_DIR_CANDIDATES = [
    Path("outputs/section_5_6/test_extreme_channel_counts"),
    Path("c20x"),
]


OUTPUT_DIR = Path("outputs/section_5_7/final_external_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ATTACK_COL = "attack"
SCOPE_COLS = ["attack_P1", "attack_P2", "attack_P3"]

META_COLUMNS = {"row_id", "time", "singleton_extreme_count"}

DESIGN_TO_FAMILY = {
    "KM4_8": "kmeans",
    "KM7_8": "kmeans",
    "KM13_8": "kmeans",
    "KM13_16": "kmeans",
    "HDB23_8": "hdbscan",
    "HDB18_8": "hdbscan",
    "HDB15_8": "hdbscan",
    "HDB18_16": "hdbscan",
    "HMM4": "hmm",
    "HMM5": "hmm",
    "HMM8": "hmm",
}

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def first_existing_dir(candidates, required_relpath=None, label="directory"):
    for d in candidates:
        if d.exists() and (required_relpath is None or (d / required_relpath).exists()):
            return d
    searched = ", ".join(str(x) for x in candidates)
    raise FileNotFoundError(f"Could not locate {label}. Searched: {searched}")


def locate_test_file(test_id):
    """Return testX.csv from the notebook working directory."""
    path = Path(f"{test_id}.csv")
    if not path.exists():
        raise FileNotFoundError(f"{path.name} must be placed in the notebook working directory.")
    return path


def normalize_binary(series, name):
    x = pd.to_numeric(series, errors="coerce")
    if x.isna().any():
        raise ValueError(f"{name}: contains missing/non-numeric values.")
    x = x.astype(int)
    vals = set(x.unique())
    if not vals.issubset({0, 1}):
        raise ValueError(f"{name}: expected 0/1, found {sorted(vals)}.")
    return x.to_numpy(dtype=np.int8)


def normalize_time(series):
    return pd.to_datetime(series, errors="coerce")


def infer_variant_metadata(variant):
    if variant == "GLOBAL":
        return {
            "family": "GLOBAL",
            "design": "GLOBAL",
            "representation": "GLOBAL",
            "dmin": np.nan,
        }

    parts = variant.split("__", 1)
    if len(parts) != 2:
        raise ValueError(f"Unexpected label-blind extremeness scoring variant name: {variant}")

    design, rep = parts
    if design not in DESIGN_TO_FAMILY:
        raise ValueError(f"Unknown design in variant: {variant}")

    family = DESIGN_TO_FAMILY[design]

    if rep == "native":
        representation = "native"
        dmin = np.nan
    elif rep.startswith("stabilized_dmin"):
        representation = "stabilized"
        dmin = int(rep.replace("stabilized_dmin", ""))
    elif rep.startswith("meta_q7_dmin"):
        representation = "aligned_meta"
        dmin = int(rep.replace("meta_q7_dmin", ""))
    else:
        raise ValueError(f"Unexpected representation suffix: {variant}")

    return {
        "family": family,
        "design": design,
        "representation": representation,
        "dmin": dmin,
    }


def score_histogram(y, score, max_score=None):
    y = np.asarray(y, dtype=np.int8)
    score = np.asarray(score, dtype=int)

    if max_score is None:
        max_score = int(score.max())

    if score.min() < 0:
        raise ValueError("K_t must be non-negative.")

    pos = np.bincount(score[y == 1], minlength=max_score + 1).astype(np.int64)
    neg = np.bincount(score[y == 0], minlength=max_score + 1).astype(np.int64)
    return pos, neg


def metrics_from_hist(pos, neg):
    """
    Exact metrics for integer K_t from positive/negative score histograms.

    AUROC:
      probability that a random attack timestamp has a larger K_t
      than a random normal timestamp, with half credit for ties.

    AUPRC:
      trapezoidal area under the precision-recall curve.
      The curve begins at recall=0, precision=1 and proceeds by
      lowering the K_t threshold.

    AP:
      step-wise Average Precision, retained for audit only.
    """
    pos = np.asarray(pos, dtype=np.float64)
    neg = np.asarray(neg, dtype=np.float64)

    P = pos.sum()
    N = neg.sum()

    if P <= 0 or N <= 0:
        return np.nan, np.nan, np.nan

    # AUROC from pairwise ranking.
    cum_neg_below = np.cumsum(neg) - neg
    concordant = np.sum(pos * (cum_neg_below + 0.5 * neg))
    auroc = concordant / (P * N)

    # PR curve from highest score downwards.
    tp = np.cumsum(pos[::-1])
    fp = np.cumsum(neg[::-1])

    recall = tp / P
    precision = np.divide(
        tp,
        tp + fp,
        out=np.ones_like(tp, dtype=np.float64),
        where=(tp + fp) > 0,
    )

    # Add the conventional starting point (recall=0, precision=1).
    recall_curve = np.concatenate([[0.0], recall])
    precision_curve = np.concatenate([[1.0], precision])

    auprc = np.trapz(precision_curve, recall_curve)

    # Step-wise AP: sum over recall increments times precision.
    recall_prev = np.concatenate([[0.0], recall[:-1]])
    ap = np.sum((recall - recall_prev) * precision)

    return float(auroc), float(auprc), float(ap)


def percentile_ci(values, alpha=0.05):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    return (
        float(np.quantile(x, alpha / 2)),
        float(np.quantile(x, 1 - alpha / 2)),
    )


def safe_pct(num, den):
    return 100.0 * float(num) / float(den) if den else np.nan


def exact_attack_scope(df):
    """
    Exact scope from P1/P2/P3 annotation columns.
    NORMAL is returned for attack==0.
    OTHER is used only when attack==1 but no P1/P2/P3 flag is set.
    """
    a = normalize_binary(df[ATTACK_COL], ATTACK_COL)

    scope_arrays = {}
    for c in SCOPE_COLS:
        if c in df.columns:
            scope_arrays[c] = normalize_binary(df[c], c)
        else:
            scope_arrays[c] = np.zeros(len(df), dtype=np.int8)

    p1 = scope_arrays["attack_P1"]
    p2 = scope_arrays["attack_P2"]
    p3 = scope_arrays["attack_P3"]

    out = np.full(len(df), "NORMAL", dtype=object)

    attack_idx = np.where(a == 1)[0]
    for i in attack_idx:
        names = []
        if p1[i]:
            names.append("P1")
        if p2[i]:
            names.append("P2")
        if p3[i]:
            names.append("P3")
        out[i] = "+".join(names) if names else "OTHER"

    return out


def contiguous_events(y):
    """
    Returns list of (start_idx, end_idx_exclusive) for contiguous attack==1 runs.
    """
    y = np.asarray(y, dtype=np.int8)
    starts = np.where((y == 1) & np.r_[True, y[:-1] == 0])[0]
    ends = np.where((y == 1) & np.r_[y[1:] == 0, True])[0] + 1
    if len(starts) != len(ends):
        raise RuntimeError("Event boundary mismatch.")
    return list(zip(starts, ends))


# ------------------------------------------------------------
# Locate frozen label-blind extremeness scoring outputs
# ------------------------------------------------------------

EXTREMENESS_DIR = first_existing_dir(
    EXTREMENESS_DIR_CANDIDATES,
    required_relpath=Path("test1") / "extreme_channel_counts_q99.csv",
    label="label-blind extremeness scoring extreme-channel-count directory",
)

# ------------------------------------------------------------
# Load raw labels and fixed K_t sequences
# ------------------------------------------------------------

raw_by_run = {}
scores_by_q_run = {}
variants = None
variant_meta = {}

integrity_rows = []

for test_id in TEST_IDS:
    raw_file = locate_test_file(test_id)
    raw = pd.read_csv(raw_file)

    if ATTACK_COL not in raw.columns:
        raise ValueError(f"{test_id}: missing '{ATTACK_COL}'.")

    y = normalize_binary(raw[ATTACK_COL], f"{test_id}/{ATTACK_COL}")
    scope = exact_attack_scope(raw)

    raw_by_run[test_id] = {
        "raw": raw,
        "y": y,
        "scope": scope,
        "events": contiguous_events(y),
    }

    for q in COVERAGES:
        score_file = EXTREMENESS_DIR / test_id / f"extreme_channel_counts_q{q}.csv"
        if not score_file.exists():
            raise FileNotFoundError(f"Missing: {score_file}")

        scores = pd.read_csv(score_file)

        if len(scores) != len(raw):
            raise RuntimeError(
                f"{test_id}/q{q}: row mismatch raw={len(raw)} scores={len(scores)}"
            )

        expected = np.arange(len(raw), dtype=int)
        row_id = pd.to_numeric(scores["row_id"], errors="raise").to_numpy(dtype=int)
        if not np.array_equal(row_id, expected):
            raise RuntimeError(f"{test_id}/q{q}: label-blind extremeness scoring row_id mismatch.")

        if "time" in raw.columns and "time" in scores.columns:
            rt = normalize_time(raw["time"])
            st = normalize_time(scores["time"])
            if rt.isna().any() or st.isna().any():
                raise RuntimeError(f"{test_id}/q{q}: invalid timestamp.")
            if not np.array_equal(rt.to_numpy(), st.to_numpy()):
                raise RuntimeError(f"{test_id}/q{q}: timestamp mismatch.")

        current_variants = [
            c for c in scores.columns
            if c not in META_COLUMNS
        ]

        # singleton_extreme_count excluded via META_COLUMNS; remaining must be
        # GLOBAL + 55 SERF configurations.
        if "GLOBAL" not in current_variants:
            raise RuntimeError(f"{test_id}/q{q}: GLOBAL column missing.")

        if variants is None:
            variants = current_variants
            if len(variants) != 56:
                raise RuntimeError(
                    f"Expected 56 reference configurations, found {len(variants)}."
                )
            variant_meta = {v: infer_variant_metadata(v) for v in variants}
        elif current_variants != variants:
            raise RuntimeError(
                f"{test_id}/q{q}: configuration column order differs from first file."
            )

        for v in variants:
            x = pd.to_numeric(scores[v], errors="coerce")
            if x.isna().any():
                raise ValueError(f"{test_id}/q{q}/{v}: K_t contains missing values.")
            if (x < 0).any() or (x > 75).any():
                raise ValueError(f"{test_id}/q{q}/{v}: K_t outside [0,75].")

        scores_by_q_run[(q, test_id)] = scores

        integrity_rows.append({
            "test_id": test_id,
            "coverage_q": q,
            "check": "row_time_configuration_and_K_contract",
            "status": "PASS",
        })

# Dataset totals.
total_ticks = sum(len(raw_by_run[t]["y"]) for t in TEST_IDS)
total_attack = sum(int(raw_by_run[t]["y"].sum()) for t in TEST_IDS)
total_normal = total_ticks - total_attack
total_events = sum(len(raw_by_run[t]["events"]) for t in TEST_IDS)

if total_ticks != 402005:
    raise RuntimeError(f"Unexpected total ticks: {total_ticks}")
if total_attack != 8947:
    raise RuntimeError(f"Unexpected attack ticks: {total_attack}")
if total_normal != 393058:
    raise RuntimeError(f"Unexpected normal ticks: {total_normal}")
if total_events != 50:
    raise RuntimeError(f"Unexpected attack event count: {total_events}")

# ------------------------------------------------------------
# 1) Per-run and pooled AUROC/AUPRC/AP for all coverages/configs
# ------------------------------------------------------------

metric_rows = []
hist_cache = {}

for q in COVERAGES:
    for variant in variants:
        meta = variant_meta[variant]

        pooled_pos = np.zeros(76, dtype=np.int64)
        pooled_neg = np.zeros(76, dtype=np.int64)

        for test_id in TEST_IDS:
            y = raw_by_run[test_id]["y"]
            score = scores_by_q_run[(q, test_id)][variant].to_numpy(dtype=int)

            pos, neg = score_histogram(y, score, max_score=75)
            hist_cache[(q, variant, test_id)] = (pos, neg)

            auroc, auprc, ap = metrics_from_hist(pos, neg)

            metric_rows.append({
                "coverage_q": q,
                "scope": test_id,
                "variant": variant,
                **meta,
                "n_ticks": len(y),
                "n_attack": int(y.sum()),
                "n_normal": int((y == 0).sum()),
                "auroc": auroc,
                "auprc_trapezoidal": auprc,
                "average_precision_audit": ap,
            })

            pooled_pos += pos
            pooled_neg += neg

        auroc, auprc, ap = metrics_from_hist(pooled_pos, pooled_neg)

        metric_rows.append({
            "coverage_q": q,
            "scope": "POOLED",
            "variant": variant,
            **meta,
            "n_ticks": total_ticks,
            "n_attack": total_attack,
            "n_normal": total_normal,
            "auroc": auroc,
            "auprc_trapezoidal": auprc,
            "average_precision_audit": ap,
        })

metrics_all = pd.DataFrame(metric_rows)

# ------------------------------------------------------------
# 2) Complete run-level bootstrap distribution for pooled q99 AUROC/AUPRC
#
# The independent resampling unit is the complete test run. With five runs
# and five draws with replacement, the ordinary nonparametric cluster
# bootstrap has exactly 5^5 = 3,125 ordered resamples. We enumerate all of
# them instead of drawing an arbitrary Monte Carlo sample.
#
# There are only C(9,5) = 126 distinct run-count vectors, but they do not
# have equal bootstrap probability. Enumerating all ordered resamples keeps
# the correct multinomial weights automatically. Timestamps inside each run
# remain intact, so within-run temporal dependence is preserved.
# ------------------------------------------------------------

bootstrap_rows = []
all_run_resamples = list(product(range(N_RUNS), repeat=N_RUNS))

if len(all_run_resamples) != N_BOOTSTRAP_RESAMPLES:
    raise RuntimeError(
        "Unexpected exhaustive bootstrap size: "
        f"{len(all_run_resamples)} != {N_BOOTSTRAP_RESAMPLES}."
    )

for variant in variants:
    run_hists = [hist_cache[(PRIMARY_Q, variant, t)] for t in TEST_IDS]

    boot_auroc = np.empty(N_BOOTSTRAP_RESAMPLES, dtype=float)
    boot_auprc = np.empty(N_BOOTSTRAP_RESAMPLES, dtype=float)

    for b, sampled in enumerate(all_run_resamples):
        pos = np.zeros(76, dtype=np.int64)
        neg = np.zeros(76, dtype=np.int64)

        for idx in sampled:
            p, n = run_hists[idx]
            pos += p
            neg += n

        auroc, auprc, _ = metrics_from_hist(pos, neg)

        if not (np.isfinite(auroc) and np.isfinite(auprc)):
            raise RuntimeError(
                f"{variant}: non-finite metric for exhaustive run resample {sampled}."
            )

        boot_auroc[b] = auroc
        boot_auprc[b] = auprc

    observed = metrics_all[
        (metrics_all["coverage_q"] == PRIMARY_Q)
        & (metrics_all["scope"] == "POOLED")
        & (metrics_all["variant"] == variant)
    ].iloc[0]

    auc_lo, auc_hi = percentile_ci(boot_auroc, CI_ALPHA)
    pr_lo, pr_hi = percentile_ci(boot_auprc, CI_ALPHA)

    bootstrap_rows.append({
        "coverage_q": PRIMARY_Q,
        "variant": variant,
        **variant_meta[variant],
        "auroc": observed["auroc"],
        "auroc_ci_low": auc_lo,
        "auroc_ci_high": auc_hi,
        "auprc_trapezoidal": observed["auprc_trapezoidal"],
        "auprc_ci_low": pr_lo,
        "auprc_ci_high": pr_hi,
        "bootstrap_unit": "test_run",
        "bootstrap_scheme": "complete_nonparametric_run_bootstrap",
        "n_test_runs": N_RUNS,
        "n_bootstrap": N_BOOTSTRAP_RESAMPLES,
        "n_unique_run_count_vectors": N_UNIQUE_RUN_COUNT_VECTORS,
        "seed": "not_applicable_exhaustive",
    })

bootstrap_ci = pd.DataFrame(bootstrap_rows)

# ------------------------------------------------------------
# 3) Family / representation ranges for headline q99 reporting
# ------------------------------------------------------------

q99_pooled = metrics_all[
    (metrics_all["coverage_q"] == PRIMARY_Q)
    & (metrics_all["scope"] == "POOLED")
].copy()

family_rows = []

# GLOBAL first.
g = q99_pooled[q99_pooled["variant"] == "GLOBAL"].iloc[0]
family_rows.append({
    "family": "GLOBAL",
    "representation": "GLOBAL",
    "n_configurations": 1,
    "auroc_min": g["auroc"],
    "auroc_max": g["auroc"],
    "auprc_min": g["auprc_trapezoidal"],
    "auprc_max": g["auprc_trapezoidal"],
})

for (family, rep), part in q99_pooled[
    q99_pooled["variant"] != "GLOBAL"
].groupby(["family", "representation"], sort=False):
    family_rows.append({
        "family": family,
        "representation": rep,
        "n_configurations": len(part),
        "auroc_min": part["auroc"].min(),
        "auroc_max": part["auroc"].max(),
        "auprc_min": part["auprc_trapezoidal"].min(),
        "auprc_max": part["auprc_trapezoidal"].max(),
    })

family_ranges = pd.DataFrame(family_rows)

# ------------------------------------------------------------
# 4) q99/q98/q95 pooled sensitivity
# ------------------------------------------------------------

sens_rows = []

for (q, family, rep), part in metrics_all[
    metrics_all["scope"] == "POOLED"
].groupby(["coverage_q", "family", "representation"], sort=False):
    sens_rows.append({
        "coverage_q": q,
        "family": family,
        "representation": rep,
        "n_configurations": len(part),
        "auroc_min": part["auroc"].min(),
        "auroc_max": part["auroc"].max(),
        "auprc_min": part["auprc_trapezoidal"].min(),
        "auprc_max": part["auprc_trapezoidal"].max(),
    })

coverage_sensitivity = pd.DataFrame(sens_rows)

# ------------------------------------------------------------
# 5) Event-level evaluation at K_t >= 1,3,5,10
#
# Event detected:
#   at least one timestamp in the annotated contiguous event reaches layer.
#
# Event tick coverage:
#   proportion of timestamps within the event reaching the layer.
# ------------------------------------------------------------

event_detail_rows = []

for q in COVERAGES:
    for variant in variants:
        meta = variant_meta[variant]

        for test_id in TEST_IDS:
            score = scores_by_q_run[(q, test_id)][variant].to_numpy(dtype=int)
            events = raw_by_run[test_id]["events"]

            for event_no, (start, end) in enumerate(events, start=1):
                event_scores = score[start:end]
                event_len = end - start

                for k in LAYERS:
                    hit = event_scores >= k
                    n_hit = int(hit.sum())

                    event_detail_rows.append({
                        "coverage_q": q,
                        "variant": variant,
                        **meta,
                        "test_id": test_id,
                        "event_no_within_run": event_no,
                        "event_start_row": start,
                        "event_end_row_exclusive": end,
                        "event_length_ticks": event_len,
                        "layer_k": k,
                        "event_detected": int(n_hit > 0),
                        "event_ticks_reaching_layer": n_hit,
                        "event_tick_coverage_pct": safe_pct(n_hit, event_len),
                    })

event_detail = pd.DataFrame(event_detail_rows)

event_summary = (
    event_detail.groupby(
        [
            "coverage_q", "variant", "family", "design",
            "representation", "dmin", "layer_k"
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        n_events=("event_detected", "size"),
        events_detected=("event_detected", "sum"),
        event_detection_rate_pct=("event_detected", lambda x: 100.0 * x.mean()),
        mean_event_tick_coverage_pct=("event_tick_coverage_pct", "mean"),
        median_event_tick_coverage_pct=("event_tick_coverage_pct", "median"),
        min_event_tick_coverage_pct=("event_tick_coverage_pct", "min"),
        max_event_tick_coverage_pct=("event_tick_coverage_pct", "max"),
    )
)

if not (event_summary["n_events"] == 50).all():
    raise RuntimeError("Every pooled event-level configuration must summarize 50 events.")

# ------------------------------------------------------------
# 6) Optional q99 attack-scope stratification
#
# This output is generated for later inspection only.
# It is NOT used for model/configuration selection.
#
# AUROC/AUPRC for a scope compare:
#   all normal timestamps vs attack timestamps of that exact scope.
# Attack timestamps belonging to other scopes are excluded.
# ------------------------------------------------------------

scope_rows = []

scope_names = sorted({
    s
    for test_id in TEST_IDS
    for s in np.unique(raw_by_run[test_id]["scope"])
    if s != "NORMAL"
})

for variant in variants:
    meta = variant_meta[variant]

    # Build pooled arrays once per variant.
    y_all = []
    score_all = []
    scope_all = []

    for test_id in TEST_IDS:
        y_all.append(raw_by_run[test_id]["y"])
        scope_all.append(raw_by_run[test_id]["scope"])
        score_all.append(
            scores_by_q_run[(PRIMARY_Q, test_id)][variant].to_numpy(dtype=int)
        )

    y_all = np.concatenate(y_all)
    scope_all = np.concatenate(scope_all)
    score_all = np.concatenate(score_all)

    normal_mask = y_all == 0

    for scope_name in scope_names:
        attack_scope_mask = (y_all == 1) & (scope_all == scope_name)
        eval_mask = normal_mask | attack_scope_mask

        y_scope = attack_scope_mask[eval_mask].astype(np.int8)
        s_scope = score_all[eval_mask]

        pos, neg = score_histogram(y_scope, s_scope, max_score=75)
        auroc, auprc, ap = metrics_from_hist(pos, neg)

        row = {
            "coverage_q": PRIMARY_Q,
            "scope": scope_name,
            "variant": variant,
            **meta,
            "n_attack_scope": int(attack_scope_mask.sum()),
            "n_normal_reference": int(normal_mask.sum()),
            "auroc": auroc,
            "auprc_trapezoidal": auprc,
            "average_precision_audit": ap,
        }

        for k in LAYERS:
            tp = int((attack_scope_mask & (score_all >= k)).sum())
            row[f"attack_captured_k{k}"] = tp
            row[f"attack_coverage_k{k}_pct"] = safe_pct(
                tp, attack_scope_mask.sum()
            )

        scope_rows.append(row)

scope_optional = pd.DataFrame(scope_rows)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

checks = pd.DataFrame(integrity_rows)

expected_integrity = len(TEST_IDS) * len(COVERAGES)
if len(checks) != expected_integrity or not checks["status"].eq("PASS").all():
    raise RuntimeError("Input integrity checks incomplete or failed.")

expected_metric_rows = len(COVERAGES) * len(variants) * (len(TEST_IDS) + 1)
if len(metrics_all) != expected_metric_rows:
    raise RuntimeError(
        f"Metric row count {len(metrics_all)} != expected {expected_metric_rows}."
    )

expected_event_detail = (
    len(COVERAGES) * len(variants) * total_events * len(LAYERS)
)
if len(event_detail) != expected_event_detail:
    raise RuntimeError(
        f"Event detail rows {len(event_detail)} != expected {expected_event_detail}."
    )

# Metric bounds.
for c in ["auroc", "auprc_trapezoidal", "average_precision_audit"]:
    finite = metrics_all[c].dropna()
    if ((finite < 0) | (finite > 1)).any():
        raise RuntimeError(f"{c} outside [0,1].")

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

# Round only user-facing CSV values, not calculations.
def rounded_copy(df, digits=6):
    out = df.copy()
    float_cols = out.select_dtypes(include=[np.floating]).columns
    out[float_cols] = out[float_cols].round(digits)
    return out


rounded_copy(metrics_all).to_csv(
    OUTPUT_DIR / "01_configuration_discrimination_all_coverages.csv",
    index=False,
)

rounded_copy(
    metrics_all[
        (metrics_all["coverage_q"] == PRIMARY_Q)
        & (metrics_all["scope"] == "POOLED")
    ]
).to_csv(
    OUTPUT_DIR / "02_pooled_discrimination_q99.csv",
    index=False,
)

rounded_copy(
    metrics_all[
        (metrics_all["coverage_q"] == PRIMARY_Q)
        & (metrics_all["scope"].isin(TEST_IDS))
    ]
).to_csv(
    OUTPUT_DIR / "03_per_run_discrimination_q99.csv",
    index=False,
)

rounded_copy(bootstrap_ci).to_csv(
    OUTPUT_DIR / "04_run_cluster_bootstrap_ci_q99.csv",
    index=False,
)

rounded_copy(family_ranges).to_csv(
    OUTPUT_DIR / "05_family_representation_ranges_q99.csv",
    index=False,
)

rounded_copy(coverage_sensitivity).to_csv(
    OUTPUT_DIR / "06_coverage_sensitivity_pooled.csv",
    index=False,
)

rounded_copy(event_summary).to_csv(
    OUTPUT_DIR / "07_event_level_summary_all_coverages.csv",
    index=False,
)

rounded_copy(
    event_summary[event_summary["coverage_q"] == PRIMARY_Q]
).to_csv(
    OUTPUT_DIR / "08_event_level_summary_q99.csv",
    index=False,
)

rounded_copy(
    event_detail[event_detail["coverage_q"] == PRIMARY_Q]
).to_csv(
    OUTPUT_DIR / "09_event_level_detail_q99.csv",
    index=False,
)

rounded_copy(scope_optional).to_csv(
    OUTPUT_DIR / "10_attack_scope_q99_optional.csv",
    index=False,
)

checks.to_csv(
    OUTPUT_DIR / "11_integrity_checks.csv",
    index=False,
)

pd.DataFrame({
    "parameter": [
        "analysis",
        "cell20_source",
        "primary_reference_coverage",
        "sensitivity_coverages",
        "extremeness_layers",
        "n_reference_configurations",
        "n_test_runs",
        "total_test_ticks",
        "normal_ticks",
        "attack_ticks",
        "attack_events",
        "auroc_definition",
        "auprc_definition",
        "average_precision",
        "bootstrap_unit",
        "bootstrap_scheme",
        "bootstrap_runs_per_resample",
        "bootstrap_ordered_resamples",
        "bootstrap_unique_run_count_vectors",
        "bootstrap_seed",
        "bootstrap_ci",
        "event_definition",
        "event_detection",
        "event_tick_coverage",
        "attack_scope_output",
        "attack_labels_used_for_selection",
        "test_refitting",
        "reference_reestimation",
    ],
    "value": [
        "final external discrimination, uncertainty, event-level and sensitivity audit",
        str(EXTREMENESS_DIR),
        "q99",
        "q98,q95",
        ",".join(map(str, LAYERS)),
        len(variants),
        len(TEST_IDS),
        total_ticks,
        total_normal,
        total_attack,
        total_events,
        "exact rank AUROC from integer K_t histograms; half credit for ties",
        "trapezoidal area under precision-recall curve with start point recall=0, precision=1",
        "computed only as audit field; not reported as AUPRC",
        "whole test run (cluster bootstrap)",
        "complete enumeration of the ordinary nonparametric run-level bootstrap distribution",
        N_RUNS,
        N_BOOTSTRAP_RESAMPLES,
        N_UNIQUE_RUN_COUNT_VECTORS,
        "not applicable (exhaustive enumeration)",
        "percentile 95% from the complete run-level bootstrap distribution",
        "contiguous attack-labeled interval within each test run",
        "event detected if any event timestamp satisfies K_t >= k",
        "percentage of event timestamps satisfying K_t >= k",
        "optional exact P1/P2/P3 combination stratification at q99; not used for selection",
        False,
        False,
        False,
    ],
}).to_csv(
    OUTPUT_DIR / "12_settings.csv",
    index=False,
)

# ------------------------------------------------------------
# Console preview
# ------------------------------------------------------------

print("=" * 84)
print("FINAL EXTERNAL EVALUATION COMPLETE")
print("=" * 84)
print(f"label-blind extremeness scoring source: {EXTREMENESS_DIR}")
print(f"Output dir:     {OUTPUT_DIR}")
print()
print(f"Configurations: {len(variants)}")
print(f"Test ticks:     {total_ticks:,}")
print(f"Normal ticks:   {total_normal:,}")
print(f"Attack ticks:   {total_attack:,}")
print(f"Attack events:  {total_events}")
print()
print("GLOBAL q99 pooled:")
global_row = q99_pooled[q99_pooled["variant"] == "GLOBAL"].iloc[0]
print(
    f"  AUROC = {global_row['auroc']:.4f}\n"
    f"  AUPRC = {global_row['auprc_trapezoidal']:.4f}\n"
    f"  AP(audit) = {global_row['average_precision_audit']:.4f}"
)
print()
print("q99 family/representation ranges:")
print(
    family_ranges.assign(
        auroc_range=lambda x: x.apply(
            lambda r: f"{r.auroc_min:.4f}–{r.auroc_max:.4f}", axis=1
        ),
        auprc_range=lambda x: x.apply(
            lambda r: f"{r.auprc_min:.4f}–{r.auprc_max:.4f}", axis=1
        ),
    )[
        ["family", "representation", "n_configurations", "auroc_range", "auprc_range"]
    ].to_string(index=False)
)


FINAL EXTERNAL EVALUATION COMPLETE
label-blind extremeness scoring source: outputs/section_5_6/test_extreme_channel_counts
Output dir:     outputs/section_5_7/final_external_evaluation

Configurations: 56
Test ticks:     402,005
Normal ticks:   393,058
Attack ticks:   8,947
Attack events:  50

GLOBAL q99 pooled:
  AUROC = 0.8337
  AUPRC = 0.4388
  AP(audit) = 0.4069

q99 family/representation ranges:
 family representation  n_configurations   auroc_range   auprc_range
 GLOBAL         GLOBAL                 1 0.8337–0.8337 0.4388–0.4388
 kmeans         native                 4 0.8284–0.8410 0.2187–0.2612
 kmeans     stabilized                 8 0.8272–0.8414 0.2142–0.2633
 kmeans   aligned_meta                 8 0.8327–0.8551 0.2212–0.2762
hdbscan         native                 4 0.8110–0.8219 0.1763–0.2084
hdbscan     stabilized                 8 0.8110–0.8219 0.1763–0.2084
hdbscan   aligned_meta                 8 0.8538–0.8566 0.3113–0.3226
    hmm         native                 3 0.8

### 5.7.4 Range-based precision and recall

Contiguous predicted extreme ranges are compared with the original annotated attack intervals using the predefined Tatbul et al. range-evaluation settings, without point adjustment and without allowing ranges to cross test-run boundaries.


In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# HAI 21.03 — RANGE-BASED PRECISION/RECALL EVALUATION
# TATBUL ET AL. (2018) RANGE-BASED PRECISION / RECALL
#
# Uses only already-fixed label-blind extremeness scoring K_t sequences and the original HAI
# attack labels for external evaluation. No refitting, remapping,
# reference re-estimation, threshold selection, or design selection.
#
# Primary metric settings follow Tatbul et al. (NeurIPS 2018) defaults:
#   alpha = 0
#   gamma() = 1
#   flat positional bias
#   overlap reward = covered fraction of the evaluated range
#
# NOTE:
# gamma()=1 does NOT add a fragmentation/cardinality penalty. This is
# deliberate: no extra parameter choice is introduced.
#
# Evaluated:
#   q99 primary + q98/q95 sensitivity
#   K_t >= 1, 3, 5, 10
#   GLOBAL + all 55 SERF-conditioned configurations
#   pooled + per-run
#
# Outputs:
#   01_range_PR_all_coverages_pooled.csv
#   02_range_PR_q99_pooled.csv
#   03_range_PR_q99_per_run.csv
#   04_family_representation_ranges_q99.csv
#   05_coverage_sensitivity_ranges.csv
#   06_integrity_checks.csv
#   07_settings.csv
# ============================================================

TEST_IDS = [f"test{i}" for i in range(1, 6)]
COVERAGES = [99, 98, 95]
PRIMARY_Q = 99
LAYERS = [1, 3, 5, 10]

ALPHA = 0.0
EXTREMENESS_DIR_CANDIDATES = [
    Path("outputs/section_5_6/test_extreme_channel_counts"),
    Path("c20x"),
]
OUTPUT_DIR = Path("outputs/section_5_7/range_based_precision_recall")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ATTACK_COL = "attack"
META_COLUMNS = {"row_id", "time", "singleton_extreme_count"}

DESIGN_TO_FAMILY = {
    "KM4_8": "kmeans",
    "KM7_8": "kmeans",
    "KM13_8": "kmeans",
    "KM13_16": "kmeans",
    "HDB23_8": "hdbscan",
    "HDB18_8": "hdbscan",
    "HDB15_8": "hdbscan",
    "HDB18_16": "hdbscan",
    "HMM4": "hmm",
    "HMM5": "hmm",
    "HMM8": "hmm",
}


def first_existing_dir(candidates, required_relpath=None):
    for d in candidates:
        if d.exists() and (required_relpath is None or (d / required_relpath).exists()):
            return d
    raise FileNotFoundError(
        "Could not locate required directory. Searched: "
        + ", ".join(str(x) for x in candidates)
    )


def locate_test_file(test_id):
    """Return testX.csv from the notebook working directory."""
    path = Path(f"{test_id}.csv")
    if not path.exists():
        raise FileNotFoundError(f"{path.name} must be placed in the notebook working directory.")
    return path


def binary01(series, name):
    x = pd.to_numeric(series, errors="coerce")
    if x.isna().any():
        raise ValueError(f"{name}: missing/non-numeric values.")
    x = x.astype(int)
    if not set(x.unique()).issubset({0, 1}):
        raise ValueError(f"{name}: expected 0/1.")
    return x.to_numpy(dtype=np.int8)


def infer_variant_metadata(variant):
    if variant == "GLOBAL":
        return {
            "family": "GLOBAL",
            "design": "GLOBAL",
            "representation": "GLOBAL",
            "dmin": np.nan,
        }

    design, rep = variant.split("__", 1)
    family = DESIGN_TO_FAMILY[design]

    if rep == "native":
        representation, dmin = "native", np.nan
    elif rep.startswith("stabilized_dmin"):
        representation = "stabilized"
        dmin = int(rep.replace("stabilized_dmin", ""))
    elif rep.startswith("meta_q7_dmin"):
        representation = "aligned_meta"
        dmin = int(rep.replace("meta_q7_dmin", ""))
    else:
        raise ValueError(f"Unexpected representation suffix: {variant}")

    return {
        "family": family,
        "design": design,
        "representation": representation,
        "dmin": dmin,
    }


def binary_to_ranges(x):
    """Binary sequence -> non-overlapping half-open intervals [start,end)."""
    x = np.asarray(x, dtype=np.int8)
    starts = np.where((x == 1) & np.r_[True, x[:-1] == 0])[0]
    ends = np.where((x == 1) & np.r_[x[1:] == 0, True])[0] + 1
    if len(starts) != len(ends):
        raise RuntimeError("Range boundary mismatch.")
    return [(int(s), int(e)) for s, e in zip(starts, ends)]


def overlap_len(a, b):
    return max(0, min(a[1], b[1]) - max(a[0], b[0]))


def overlap_candidates(target, candidates):
    return [c for c in candidates if overlap_len(target, c) > 0]


def flat_overlap_fraction(target, overlaps):
    length = target[1] - target[0]
    if length <= 0:
        raise ValueError("Invalid range.")
    return sum(overlap_len(target, x) for x in overlaps) / float(length)


def real_range_contribution(real_range, predicted_ranges):
    """
    Tatbul Recall_T contribution for one real anomaly range.
    alpha=0, gamma=1, flat positional bias.
    """
    overlaps = overlap_candidates(real_range, predicted_ranges)
    existence = 1.0 if overlaps else 0.0
    overlap_reward = flat_overlap_fraction(real_range, overlaps) if overlaps else 0.0
    score = ALPHA * existence + (1.0 - ALPHA) * overlap_reward
    return score, existence, len(overlaps)


def predicted_range_contribution(pred_range, real_ranges):
    """
    Tatbul Precision_T contribution for one predicted range.
    gamma=1, flat positional bias.
    """
    overlaps = overlap_candidates(pred_range, real_ranges)
    score = flat_overlap_fraction(pred_range, overlaps) if overlaps else 0.0
    return score, len(overlaps)


def evaluate_run(real_ranges, predicted_ranges):
    real_scores, existence, fragment_counts = [], [], []
    for r in real_ranges:
        s, e, n = real_range_contribution(r, predicted_ranges)
        real_scores.append(s)
        existence.append(e)
        fragment_counts.append(n)

    pred_scores, pred_real_overlap_counts = [], []
    for p in predicted_ranges:
        s, n = predicted_range_contribution(p, real_ranges)
        pred_scores.append(s)
        pred_real_overlap_counts.append(n)

    recall = float(np.mean(real_scores)) if real_scores else np.nan
    precision = float(np.mean(pred_scores)) if pred_scores else 0.0
    f1 = (
        2.0 * precision * recall / (precision + recall)
        if np.isfinite(recall) and (precision + recall) > 0
        else 0.0
    )

    return {
        "range_precision": precision,
        "range_recall": recall,
        "range_f1": f1,
        "event_existence_rate_pct": (
            100.0 * float(np.mean(existence)) if existence else np.nan
        ),
        "real_scores": real_scores,
        "pred_scores": pred_scores,
        "existence": existence,
        "fragment_counts": fragment_counts,
        "pred_real_overlap_counts": pred_real_overlap_counts,
        "n_real_ranges": len(real_ranges),
        "n_predicted_ranges": len(predicted_ranges),
    }


EXTREMENESS_DIR = first_existing_dir(
    EXTREMENESS_DIR_CANDIDATES,
    Path("test1") / "extreme_channel_counts_q99.csv",
)

raw_by_run = {}
scores_by_q_run = {}
variants = None
variant_meta = {}
input_checks = []

for test_id in TEST_IDS:
    raw = pd.read_csv(locate_test_file(test_id))
    if ATTACK_COL not in raw.columns:
        raise ValueError(f"{test_id}: missing attack column.")

    y = binary01(raw[ATTACK_COL], f"{test_id}/attack")
    raw_by_run[test_id] = {
        "raw": raw,
        "y": y,
        "real_ranges": binary_to_ranges(y),
    }

    for q in COVERAGES:
        p = EXTREMENESS_DIR / test_id / f"extreme_channel_counts_q{q}.csv"
        if not p.exists():
            raise FileNotFoundError(p)

        scores = pd.read_csv(p)
        if len(scores) != len(raw):
            raise RuntimeError(f"{test_id}/q{q}: row-count mismatch.")

        expected = np.arange(len(raw), dtype=int)
        row_id = pd.to_numeric(scores["row_id"], errors="raise").to_numpy(dtype=int)
        if not np.array_equal(row_id, expected):
            raise RuntimeError(f"{test_id}/q{q}: row_id mismatch.")

        if "time" in raw.columns and "time" in scores.columns:
            rt = pd.to_datetime(raw["time"], errors="coerce")
            st = pd.to_datetime(scores["time"], errors="coerce")
            if rt.isna().any() or st.isna().any():
                raise RuntimeError(f"{test_id}/q{q}: invalid time.")
            if not np.array_equal(rt.to_numpy(), st.to_numpy()):
                raise RuntimeError(f"{test_id}/q{q}: time mismatch.")

        current = [c for c in scores.columns if c not in META_COLUMNS]

        if variants is None:
            variants = current
            if len(variants) != 56 or "GLOBAL" not in variants:
                raise RuntimeError(
                    f"Expected GLOBAL + 55 SERF configurations; found {len(variants)}."
                )
            variant_meta = {v: infer_variant_metadata(v) for v in variants}
        elif current != variants:
            raise RuntimeError(f"{test_id}/q{q}: configuration columns changed.")

        for v in variants:
            x = pd.to_numeric(scores[v], errors="coerce")
            if x.isna().any() or (x < 0).any() or (x > 75).any():
                raise ValueError(f"{test_id}/q{q}/{v}: invalid K_t.")

        scores_by_q_run[(q, test_id)] = scores
        input_checks.append({
            "test_id": test_id,
            "coverage_q": q,
            "check": "row_time_configuration_and_K_contract",
            "status": "PASS",
        })


total_ticks = sum(len(raw_by_run[t]["y"]) for t in TEST_IDS)
total_attack = sum(int(raw_by_run[t]["y"].sum()) for t in TEST_IDS)
total_normal = total_ticks - total_attack
total_events = sum(len(raw_by_run[t]["real_ranges"]) for t in TEST_IDS)

assert total_ticks == 402005
assert total_attack == 8947
assert total_normal == 393058
assert total_events == 50


# ------------------------------------------------------------
# Main computation
# ------------------------------------------------------------

per_run_rows = []
pooled_rows = []

for q in COVERAGES:
    for variant in variants:
        meta = variant_meta[variant]

        for k in LAYERS:
            pooled_real_scores = []
            pooled_pred_scores = []
            pooled_existence = []
            pooled_fragments = []
            pooled_pred_overlap_counts = []

            for test_id in TEST_IDS:
                real_ranges = raw_by_run[test_id]["real_ranges"]
                score = scores_by_q_run[(q, test_id)][variant].to_numpy(dtype=int)
                pred_ranges = binary_to_ranges((score >= k).astype(np.int8))

                r = evaluate_run(real_ranges, pred_ranges)

                per_run_rows.append({
                    "coverage_q": q,
                    "test_id": test_id,
                    "variant": variant,
                    **meta,
                    "layer_k": k,
                    "n_real_attack_ranges": r["n_real_ranges"],
                    "n_predicted_positive_ranges": r["n_predicted_ranges"],
                    "range_precision": r["range_precision"],
                    "range_recall": r["range_recall"],
                    "range_f1": r["range_f1"],
                    "event_existence_rate_pct": r["event_existence_rate_pct"],
                    "mean_predicted_ranges_per_real_attack": (
                        float(np.mean(r["fragment_counts"]))
                        if r["fragment_counts"] else np.nan
                    ),
                })

                pooled_real_scores.extend(r["real_scores"])
                pooled_pred_scores.extend(r["pred_scores"])
                pooled_existence.extend(r["existence"])
                pooled_fragments.extend(r["fragment_counts"])
                pooled_pred_overlap_counts.extend(r["pred_real_overlap_counts"])

            recall = float(np.mean(pooled_real_scores))
            precision = (
                float(np.mean(pooled_pred_scores))
                if pooled_pred_scores else 0.0
            )
            f1 = (
                2.0 * precision * recall / (precision + recall)
                if (precision + recall) > 0 else 0.0
            )

            pooled_rows.append({
                "coverage_q": q,
                "variant": variant,
                **meta,
                "layer_k": k,
                "n_test_runs": len(TEST_IDS),
                "n_real_attack_ranges": len(pooled_real_scores),
                "n_predicted_positive_ranges": len(pooled_pred_scores),
                "range_precision": precision,
                "range_recall": recall,
                "range_f1": f1,
                "event_existence_rate_pct": 100.0 * float(np.mean(pooled_existence)),
                "mean_predicted_ranges_per_real_attack": (
                    float(np.mean(pooled_fragments))
                    if pooled_fragments else np.nan
                ),
                "mean_real_ranges_per_predicted_range": (
                    float(np.mean(pooled_pred_overlap_counts))
                    if pooled_pred_overlap_counts else np.nan
                ),
            })

per_run = pd.DataFrame(per_run_rows)
pooled = pd.DataFrame(pooled_rows)


# ------------------------------------------------------------
# Independent integrity check:
# with alpha=0, gamma=1, flat bias, RangeRecall must equal
# unweighted mean event-tick coverage.
# ------------------------------------------------------------

range_checks = []

for _, row in pooled[pooled["coverage_q"] == PRIMARY_Q].iterrows():
    variant = row["variant"]
    k = int(row["layer_k"])
    direct = []

    for test_id in TEST_IDS:
        score = scores_by_q_run[(PRIMARY_Q, test_id)][variant].to_numpy(dtype=int)
        pred = score >= k
        for start, end in raw_by_run[test_id]["real_ranges"]:
            direct.append(float(pred[start:end].mean()))

    direct_mean = float(np.mean(direct))
    diff = abs(direct_mean - float(row["range_recall"]))

    range_checks.append({
        "coverage_q": PRIMARY_Q,
        "variant": variant,
        "layer_k": k,
        "check": "range_recall_equals_unweighted_mean_event_tick_coverage",
        "absolute_difference": diff,
        "status": "PASS" if diff < 1e-12 else "FAIL",
    })

range_checks = pd.DataFrame(range_checks)
if not range_checks["status"].eq("PASS").all():
    raise RuntimeError("Range-recall integrity check failed.")

for df in [per_run, pooled]:
    for c in ["range_precision", "range_recall", "range_f1"]:
        x = df[c].dropna()
        if ((x < -1e-12) | (x > 1 + 1e-12)).any():
            raise RuntimeError(f"{c} outside [0,1].")


# ------------------------------------------------------------
# q99 family/representation ranges
# ------------------------------------------------------------

family_rows = []
q99 = pooled[pooled["coverage_q"] == PRIMARY_Q]

for k in LAYERS:
    for (family, rep), part in q99[q99["layer_k"] == k].groupby(
        ["family", "representation"], sort=False
    ):
        family_rows.append({
            "layer_k": k,
            "family": family,
            "representation": rep,
            "n_configurations": len(part),
            "range_precision_min": part["range_precision"].min(),
            "range_precision_max": part["range_precision"].max(),
            "range_recall_min": part["range_recall"].min(),
            "range_recall_max": part["range_recall"].max(),
            "range_f1_min": part["range_f1"].min(),
            "range_f1_max": part["range_f1"].max(),
            "event_existence_rate_pct_min": part["event_existence_rate_pct"].min(),
            "event_existence_rate_pct_max": part["event_existence_rate_pct"].max(),
        })

family_ranges = pd.DataFrame(family_rows)


# ------------------------------------------------------------
# q99/q98/q95 sensitivity summary
# ------------------------------------------------------------

sens_rows = []
for (q, k, family, rep), part in pooled.groupby(
    ["coverage_q", "layer_k", "family", "representation"],
    sort=False
):
    sens_rows.append({
        "coverage_q": q,
        "layer_k": k,
        "family": family,
        "representation": rep,
        "n_configurations": len(part),
        "range_precision_min": part["range_precision"].min(),
        "range_precision_max": part["range_precision"].max(),
        "range_recall_min": part["range_recall"].min(),
        "range_recall_max": part["range_recall"].max(),
        "range_f1_min": part["range_f1"].min(),
        "range_f1_max": part["range_f1"].max(),
    })

sensitivity = pd.DataFrame(sens_rows)


def rounded(df, digits=6):
    out = df.copy()
    cols = out.select_dtypes(include=[np.floating]).columns
    out[cols] = out[cols].round(digits)
    return out


rounded(pooled).to_csv(
    OUTPUT_DIR / "01_range_PR_all_coverages_pooled.csv", index=False
)
rounded(q99).to_csv(
    OUTPUT_DIR / "02_range_PR_q99_pooled.csv", index=False
)
rounded(per_run[per_run["coverage_q"] == PRIMARY_Q]).to_csv(
    OUTPUT_DIR / "03_range_PR_q99_per_run.csv", index=False
)
rounded(family_ranges).to_csv(
    OUTPUT_DIR / "04_family_representation_ranges_q99.csv", index=False
)
rounded(sensitivity).to_csv(
    OUTPUT_DIR / "05_coverage_sensitivity_ranges.csv", index=False
)

checks = pd.concat(
    [
        pd.DataFrame(input_checks),
        range_checks,
    ],
    ignore_index=True,
    sort=False,
)
checks.to_csv(OUTPUT_DIR / "06_integrity_checks.csv", index=False)

pd.DataFrame({
    "parameter": [
        "analysis",
        "source",
        "cell20_source",
        "primary_reference_coverage",
        "sensitivity_coverages",
        "layers",
        "configurations",
        "test_runs",
        "test_ticks",
        "normal_ticks",
        "attack_ticks",
        "real_attack_ranges",
        "alpha",
        "gamma",
        "positional_bias",
        "overlap_reward",
        "fragmentation_penalty",
        "pooled_range_handling",
        "labels_used_for_selection",
        "test_refitting",
        "reference_reestimation",
    ],
    "value": [
        "Tatbul et al. range-based precision/recall",
        "Tatbul et al., Precision and Recall for Time Series, NeurIPS 2018",
        str(EXTREMENESS_DIR),
        "q99",
        "q98,q95",
        ",".join(map(str, LAYERS)),
        len(variants),
        len(TEST_IDS),
        total_ticks,
        total_normal,
        total_attack,
        total_events,
        ALPHA,
        "1",
        "flat",
        "fraction of evaluated range covered by overlap",
        "none beyond range-overlap formulation; gamma=1",
        "range contributions pooled across runs; ranges cannot cross run boundaries",
        False,
        False,
        False,
    ],
}).to_csv(OUTPUT_DIR / "07_settings.csv", index=False)

print("=" * 80)
print("RANGE-BASED PRECISION/RECALL EVALUATION COMPLETE")
print("=" * 80)
print(f"label-blind extremeness scoring source: {EXTREMENESS_DIR}")
print(f"Output:         {OUTPUT_DIR}")
print(f"Configurations: {len(variants)}")
print(f"Attack ranges:  {total_events}")
print()
print("Settings: alpha=0, gamma=1, flat positional bias")
print()
print("q99 GLOBAL preview:")
print(
    q99[q99["variant"] == "GLOBAL"][
        [
            "layer_k",
            "n_predicted_positive_ranges",
            "range_precision",
            "range_recall",
            "range_f1",
            "event_existence_rate_pct",
        ]
    ].to_string(index=False)
)


RANGE-BASED PRECISION/RECALL EVALUATION COMPLETE
label-blind extremeness scoring source: outputs/section_5_6/test_extreme_channel_counts
Output:         outputs/section_5_7/range_based_precision_recall
Configurations: 56
Attack ranges:  50

Settings: alpha=0, gamma=1, flat positional bias

q99 GLOBAL preview:
 layer_k  n_predicted_positive_ranges  range_precision  range_recall  range_f1  event_existence_rate_pct
       1                        29594         0.008848      0.861273  0.017517                     100.0
       3                        16217         0.016088      0.682761  0.031435                     100.0
       5                         3050         0.065629      0.493064  0.115840                      90.0
      10                          316         0.761449      0.195108  0.310624                      68.0


## Reproducibility notes

- Development data: `train1.csv` only.
- Held-out data: `test1.csv`–`test5.csv`, processed independently.
- Attack labels: first used in Section 5.7 for external validation.
- Retained segmentation portfolio: 4 k-means, 4 HDBSCAN, and 3 HMM designs.
- SERF representations evaluated per design: native, stabilized at 5 s, stabilized at 10 s, aligned meta-modes after 5 s stabilization, and aligned meta-modes after 10 s stabilization.
- Functional comparison: 1 GLOBAL reference plus 55 SERF-conditioned configurations.
- Primary empirical coverage: 99%; predefined sensitivity: 98% and 95%.
- Generated outputs: relative paths under `outputs/`; no local absolute directory is required or recorded by the notebook.
